# Predicting a penalty kick before the foot meets the ball

**The task.** A short broadcast clip stops 0.02 s before the kicker's foot touches the ball. From that clip alone (plus the kicker's preferred foot and the competition's gender), predict where the kick goes (`kick_direction`: left / middle / right, from the kicker's point of view), which way the goalkeeper moves (`goalkeeper_direction`: left / stands in the middle / right) and whether it is a goal (`goal_outcome`). The score is `0.60 * acc(kick) + 0.25 * acc(keeper) + 0.15 * acc(goal)`. 1,300 labelled clips are available for training, 900 unlabelled clips are predicted; the clips come from 215 source videos and almost every source sits entirely on one side of that split, so the model is judged on stadiums, broadcasters and goalkeepers it has never seen.

This notebook runs the whole pipeline from the raw clips to the submission CSV: every stage is a script in `src/`, called in order below with a short explanation of what it does, which pretrained model it uses, roughly how long it takes and what it writes. The last cells fit the level-2 stack and write the CSV.

**The final model in one paragraph.** Nothing after contact is visible, so the pipeline reads body posture in the last moments of the run-up. A person detector (YOLO11x) finds everyone in time-sampled frames; a click-supervised picker identifies the kicker and the goalkeeper and defines a per-clip *kicker frame* (the axis kicker -> keeper and the kicker-left/right axis), so that "left" and "right" mean the same thing in every camera view. Pose estimators (RTMPose-m and RTMPose-x with the Halpe-26 skeleton, which adds toe and heel keypoints, plus YOLO11m-pose) run on up-scaled crops of the two players over the last 0.4 s, with mirrored-crop test-time augmentation, and the keypoint tables are merged by confidence. Hand-crafted kinematic features (foot orientation, shank/thigh direction, hip and trunk motion, keeper lateral displacement, optical flow) feed gradient-boosted trees, extra-trees, XGBoost and a small temporal keypoint network, all trained with mirror augmentation (features mirrored, left/right labels swapped) and flip test-time augmentation on source-grouped 5-fold cross-validation. The two decisions that mattered most for the kick: the kick heads see **kicker features only, without the goalkeeper's motion** (a head that sees the keeper learns to copy a keeper who guesses right only 56% of the time), and **only the features from the last 0.04 s** (earlier time steps are noise for the kick). A per-target logistic-regression stack combines the members; the goalkeeper target stacks goalkeeper heads only. `goal_outcome` is the constant `goal` because no learned head beat the 0.722 base rate.

| Stage | Script | Hardware | Approx. time (2,200 clips) | Writes (under the data root) |
|---|---|---|---|---|
| Folds + chance baselines | `src/cv.py`, `src/baseline_meta.py` | CPU | seconds | `cache/folds_5_42.csv` |
| 1. Clip inventory | `src/inventory.py` | CPU, 8 threads | 3-5 min | `cache/clip_inventory.csv` |
| 2. Frame cache A | `src/framecache.py` | CPU | ~10 min | `cache/framesA/*.npz` (1.4 GB) |
| 3. Person + ball detection | `src/detect.py` | GPU (YOLO11x) | ~1 h on an RTX 4060 | `cache/detections.parquet` |
| 4. Kicker/keeper roles + geometry | `src/roles.py` | CPU | ~2 min | `cache/roles_clip.csv`, `roles_frames.parquet`, `tracks.parquet` |
| 5a. YOLO pose on crops | `src/pose_crops.py` | GPU (YOLO11m-pose) | ~0.5-1 h | `cache/pose_crops.parquet` |
| 5b. RTMPose Halpe-26 (m and x), mirrored-crop TTA, merges | `src/pose_rtm.py`, `src/pose_merge.py` | CPU (onnxruntime) | ~20 min (m) + 2-4 h (x), TTA passes multiply this | `cache/pose_rtm_{m,x,mf4,xf,a,mm,xx,a2b,a3}.parquet` |
| 6. Canonical crop caches (video models only) | `src/crops.py` | CPU | ~10 min | `cache/crops_{kicker,keeper}/` |
| 7. Feature tables + LightGBM heads | `src/features_*.py` | CPU | ~1.5 h total | `cache/features_v*.parquet`, `runs/gbm_*` |
| 8. Subset heads and the pure late-window kicker heads | `src/late_table.py`, `src/gbm_subset.py`, `src/model_subset.py`, `src/avg_members.py` | CPU | ~1 h | `runs/*_late04pure`, `runs/gbm_kick_ens_pure10`, `runs/et_*`, `runs/xgb_*` |
| 9. Raw-keypoint heads (last two time steps) + temporal keypoint nets | `src/gbm_kpraw.py`, `src/train_kpnet.py` | CPU | ~40 min | `runs/gbm_kpraw_*_late04`, `runs/gbm_kpraw_late_ens`, `runs/kpnet_*` |
| 9b. Small extra kick members: camera-corrected run-up, 3D pose | `src/camera.py`, `src/features_cam.py`, `src/pose3d.py`, `src/features_3d.py` | CPU | ~1 h + ~9 h (3D, optional) | `runs/gbm_camka`, `runs/gbm_3d_only` |
| 10. VideoMAE members (optional, GPU) | `src/train_videomae.py`, `src/assemble_folds.py` | GPU (Kaggle / Colab) | ~2.5 h per fold and run | `runs/vmae_keeper_ens6`, `runs/vmae_kicker_nf_ens5` |
| 11. Stack + submission | `src/stack.py` | CPU | < 1 min | `runs/stack_tab88/submission.csv` -> `submissions/` |

**Caches make re-runs fast.** Every heavy stage is resumable and skips clips that are already in its output (detections, poses, crops), and the frame/keypoint tensors are cached on disk, so re-running the whole notebook on a machine that already has the cache folder takes about two hours (feature tables + heads + stack) instead of days. Delete the corresponding cache file to force a stage to recompute. Members whose run folder is missing (for example the GPU-trained VideoMAE ensembles) are skipped by the stack with a message; the result then differs slightly from the reported one.

## 0. Setup

**Environment used:** Windows 11 laptop, Python 3.12, 16 GB RAM, RTX 4060 8 GB (only detection and YOLO pose use the GPU); the VideoMAE members were trained on Kaggle T4 / Colab A100 GPUs. Package versions: `torch 2.14.0+cu126`, `ultralytics 8.4.190`, `timm 1.0.30`, `lightgbm 4.7.0`, `xgboost 3.4.1`, `scikit-learn 1.8.0`, `rtmlib 0.0.16`, `onnxruntime 1.26.0`, `pyarrow 24.0.0`, `opencv-python 5.0.0`, `transformers 5.20.0` (VideoMAE only).

**Data root.** All data, caches, pretrained weights and run outputs live under one root (default `D:\cv3`), set in `src/config.py` and overridable with the environment variable `CV3_ROOT`. Expected layout:

```
<root>\data\train\0001_public.mp4 ... 1300_public.mp4   (the 1,300 labelled clips)
<root>\data\val\001_private.mp4 ... 900_private.mp4      (the 900 unlabelled clips to predict)
<root>\data\train_ground_truth.csv                        (labels + metadata of the labelled clips)
<root>\data\private_metadata.csv
<root>\data\public_position_clicks.csv                    (hand-clicked kicker / keeper positions, 879 clips)
<root>\data\submission_template.csv
<root>\models\yolo11x.pt, yolo11m-pose.pt                 (downloaded below if missing)
<root>\cache, <root>\runs                                 (created automatically)
```

On Windows use `python -m pip`, not a bare `pip`. The `!python` cells below must run with the same interpreter as the notebook kernel (checked in the next cell). The notebook lives next to `src/`.

In [ ]:
# Install the dependencies (idempotent: pinned to the versions used for the submissions).
# torch is installed separately with the CUDA index, e.g.
#   python -m pip install torch==2.14.0 torchvision --index-url https://download.pytorch.org/whl/cu126
!python -m pip install -q ultralytics==8.4.190 timm==1.0.30 lightgbm==4.7.0 xgboost==3.4.1 scikit-learn==1.8.0 rtmlib==0.0.16 onnxruntime==1.26.0 pyarrow==24.0.0 opencv-python==5.0.0 transformers==5.20.0 pandas numpy nbformat

## Source code

The pipeline is a set of Python modules. They are embedded here so that this notebook is a single file: running the cells below writes them to `src/` next to the notebook (`config.py` first; it holds every path). 46 modules, collapsed by default; expand a cell to read the code. Modules that were only used for negative experiments are included for completeness and are not called by the pipeline.

Modules: `config.py`, `assemble_folds.py`, `avg_members.py`, `ball_detect.py`, `baseline_meta.py`, `camera.py`, `crops.py`, `cv.py`, `detect.py`, `dupes.py`, `feat_frozen.py`, `features_3d.py`, `features_ball.py`, `features_cam.py`, `features_flow.py`, `features_flow2.py`, `features_rtm.py`, `features_rtm2.py`, `features_rtm2g.py`, `features_rtm3.py`, `features_union.py`, `features_union2.py`, `features_v1.py`, `features_v2.py`, `features_v3.py`, `framecache.py`, `gbm_kpraw.py`, `gbm_subset.py`, `hier_head.py`, `inventory.py`, `late_table.py`, `model_subset.py`, `pose3d.py`, `pose_crops.py`, `pose_merge.py`, `pose_rtm.py`, `probe_frozen.py`, `roles.py`, `selftrain.py`, `stack.py`, `stack2.py`, `train_kpnet.py`, `train_stack7.py`, `train_videomae.py`, `video.py`, `viz_contact.py`

In [ ]:
import os
os.makedirs('src', exist_ok=True)
print('writing', 46, 'modules to', os.path.abspath('src'))

In [ ]:
%%writefile src/config.py
"""Central paths and constants for the penalty-kick competition.

Data lives on the external disk (outside OneDrive); code lives in this folder.
Override the data root with the environment variable CV3_ROOT if the disk
letter changes.
"""
import os
from pathlib import Path

ROOT = Path(os.environ.get("CV3_ROOT", r"D:\cv3"))
DATA_DIR = ROOT / "data"
CACHE_DIR = ROOT / "cache"
MODELS_DIR = ROOT / "models"
RUNS_DIR = ROOT / "runs"

# Keep every pretrained-weight cache on the big disk (C: is nearly full).
# Must be set before transformers / torch hub / ultralytics are imported.
os.environ.setdefault("HF_HOME", str(MODELS_DIR / "hf"))   # dinov2-base, videomae-ssv2, siglip2, vjepa2 already here
os.environ.setdefault("TORCH_HOME", str(ROOT / "torch"))
os.environ.setdefault("YOLO_CONFIG_DIR", str(ROOT / "yolo_cfg"))

# The Kaggle download names the partitions train/ (= 1,300 labeled "public"
# clips, 0001_public.mp4 ...) and val/ (= 900 unlabeled "private" clips,
# 001_private.mp4 ...). Labels are in train_ground_truth.csv.
PUBLIC_DIR = DATA_DIR / "train"
PRIVATE_DIR = DATA_DIR / "val"
GT_CSV = DATA_DIR / "train_ground_truth.csv"
PRIVATE_CSV = DATA_DIR / "private_metadata.csv"
CLICKS_CSV = DATA_DIR / "public_position_clicks.csv"
TEMPLATE_CSV = DATA_DIR / "submission_template.csv"

TARGETS = ["kick_direction", "goalkeeper_direction", "goal_outcome"]
CLASSES = {
    "kick_direction": ["left", "middle", "right"],
    "goalkeeper_direction": ["left", "stands in the middle", "right"],
    "goal_outcome": ["goal", "no_goal"],
}
WEIGHTS = {"kick_direction": 0.60, "goalkeeper_direction": 0.25, "goal_outcome": 0.15}

# Horizontal-flip label swap (mirror image => kicker's left becomes right).
FLIP_MAP = {
    "kick_direction": {"left": "right", "right": "left", "middle": "middle"},
    "goalkeeper_direction": {"left": "right", "right": "left",
                             "stands in the middle": "stands in the middle"},
    "goal_outcome": {"goal": "goal", "no_goal": "no_goal"},
    "preferred_foot": {"left": "right", "right": "left"},
}

SEED = 42
N_FOLDS = 5

for _d in (CACHE_DIR, MODELS_DIR, RUNS_DIR):
    _d.mkdir(parents=True, exist_ok=True)


In [ ]:
%%writefile src/assemble_folds.py
"""Assemble a stack member (oof_*.npy / test_*.npy + metrics.json) from per-fold prediction files
(fold{k}_preds.npz with val_idx, val_<target>, test_<target>) of train_videomae.py / train_stack7.py runs,
tolerating missing folds (their OOF rows stay NaN; stack.py fills them with the class prior).

Run:  python src/assemble_folds.py --src D:\cv3\kaggle_out\cv3-video-models-2\cv3\runs\vmae_ssv2_keeper_crop --run vmae_keeper
"""
import argparse
import json
import os
import sys
from pathlib import Path

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import pyarrow  # noqa: F401
import numpy as np
import pandas as pd

from config import CLASSES, RUNS_DIR, TARGETS
from cv import get_folds, load_private, load_public, score_report


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--src", required=True, help="directory with fold{k}_preds.npz")
    ap.add_argument("--run", required=True)
    a = ap.parse_args()
    pub, priv = load_public(), load_private()
    folds = get_folds(pub)
    oof = {t: np.full((len(pub), len(CLASSES[t])), np.nan, np.float32) for t in TARGETS}
    test = {t: [] for t in TARGETS}
    done = []
    for k in range(5):
        p = Path(a.src) / f"fold{k}_preds.npz"
        if not p.exists():
            continue
        z = np.load(p)
        vi = z["val_idx"]
        assert set(vi.tolist()) == set(np.where(folds.values == k)[0].tolist()), f"fold {k}: val_idx does not match the folds"
        for t in TARGETS:
            oof[t][vi] = z[f"val_{t}"]
            test[t].append(z[f"test_{t}"])
        done.append(k)
    out = RUNS_DIR / a.run
    out.mkdir(parents=True, exist_ok=True)
    metrics = {"folds": done}
    for t in TARGETS:
        key = t.split("_")[0]
        np.save(out / f"oof_{key}.npy", oof[t])
        np.save(out / f"test_{key}.npy", np.mean(test[t], 0).astype(np.float32))
        m = ~np.isnan(oof[t][:, 0])
        metrics[t] = float((np.asarray(CLASSES[t])[oof[t][m].argmax(1)] == pub[t].values[m]).mean())
    cover = ~np.isnan(oof[TARGETS[0]][:, 0])
    pred = pd.DataFrame({t: np.asarray(CLASSES[t])[np.nan_to_num(oof[t], nan=0.0).argmax(1)] for t in TARGETS})
    print(f"{a.run}: folds {done}, OOF rows covered {cover.sum()}/{len(pub)}; accuracy on covered rows: "
          + ", ".join(f"{t}={metrics[t]:.3f}" for t in TARGETS))
    metrics["score_covered"] = score_report(pub[cover].reset_index(drop=True), pred[cover].reset_index(drop=True),
                                            folds[cover].reset_index(drop=True), a.run)
    json.dump(metrics, open(out / "metrics.json", "w"), indent=1)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/avg_members.py
"""Average several stack members (same model family, different seeds / inits) into one member: NaN-aware mean of
the OOF probabilities and mean of the test probabilities. Fewer, less noisy inputs for the logistic-regression stack.

Run:  python src/avg_members.py --run vmae_keeper_ens --members vmae_keeper_s1 vmae_keeper_s2 vmae_keeper_s3 vmae_keeper
"""
import argparse
import json
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import pyarrow  # noqa: F401
import numpy as np
import pandas as pd

from config import CLASSES, RUNS_DIR, TARGETS
from cv import get_folds, load_public, score_report

if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--run", required=True)
    ap.add_argument("--members", nargs="+", required=True)
    a = ap.parse_args()
    pub = load_public()
    out = RUNS_DIR / a.run
    out.mkdir(parents=True, exist_ok=True)
    metrics = {"members": a.members}
    oof_all = {}
    for t in TARGETS:
        key = t.split("_")[0]
        O = np.stack([np.load(RUNS_DIR / m / f"oof_{key}.npy") for m in a.members])
        T = np.stack([np.load(RUNS_DIR / m / f"test_{key}.npy") for m in a.members])
        with np.errstate(all="ignore"):
            oof = np.nanmean(O, 0)          # rows missing in some members use the others
        test = T.mean(0)
        np.save(out / f"oof_{key}.npy", oof.astype(np.float32))
        np.save(out / f"test_{key}.npy", test.astype(np.float32))
        oof_all[t] = oof
        m = ~np.isnan(oof[:, 0])
        metrics[t] = float((np.asarray(CLASSES[t])[oof[m].argmax(1)] == pub[t].values[m]).mean())
    pred = pd.DataFrame({t: np.asarray(CLASSES[t])[np.nan_to_num(oof_all[t], nan=0.0).argmax(1)] for t in TARGETS})
    metrics["score"] = score_report(pub, pred, get_folds(pub), a.run)
    json.dump(metrics, open(out / "metrics.json", "w"), indent=1)


In [ ]:
%%writefile src/ball_detect.py
"""Dense ball detection in the final 0.45 s of every clip, on a crop around the kicker's feet.

The stage-1 detector (detect.py) sees the ball in only ~74% of the clips, from 1-2 sparse frames at imgsz 1280, with a
median confidence of 0.34: the ball is 10-20 px wide. Here every frame of the last 0.45 s is cropped to a square of
3 kicker heights around the kicker's feet (from roles_clip.csv), upsampled to 640 px and run through YOLO11x with
classes=[sports ball] at a low threshold. Boxes are written back in full-frame pixels, one row per detection
(det_id -1 marks a processed frame without a ball), to CACHE_DIR/ball_dets.parquet (resumable).

Run:  python src/ball_detect.py            python src/ball_detect.py --limit 10 (smoke)
"""
import argparse
import os
import sys
import time
from concurrent.futures import ThreadPoolExecutor

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import cv2
import numpy as np
import pandas as pd

from config import CACHE_DIR, MODELS_DIR, PRIVATE_DIR, PUBLIC_DIR
from detect import frame_timestamps, read_selected

cv2.setNumThreads(1)

OUT = CACHE_DIR / "ball_dets.parquet"
BALL = 32
WINDOW_S = 0.45
CROP_HEIGHTS = 3.0      # crop side in kicker heights
MIN_CROP = 256          # px


def crop_box(rc):
    """Square crop (x0, y0, side) around the kicker's feet, clamped to the frame."""
    W, H = int(rc.W), int(rc.H)
    kx, ky, kh = rc.kcx * W, rc.kcy * H, rc.kh * H
    cx, cy = kx, ky + 0.4 * kh
    side = int(max(MIN_CROP, CROP_HEIGHTS * kh))
    side = min(side, W, H)
    x0 = int(np.clip(cx - side / 2, 0, W - side))
    y0 = int(np.clip(cy - side / 2, 0, H - side))
    return x0, y0, side


def load_clip(job):
    clip, path = job
    for _ in range(4):
        try:
            ts = frame_timestamps(path)
            if len(ts) == 0:
                return clip, None, None, None
            t_end = ts[-1]
            wanted = [i for i in range(len(ts)) if (t_end - ts[i]) / 1000.0 <= WINDOW_S]
            frames = read_selected(path, wanted)
            if all(i in frames for i in wanted):
                return clip, ts, wanted, frames
        except Exception as e:  # noqa: BLE001
            print(f"  decode error on {clip}: {str(e)[:80]}", flush=True)
        time.sleep(10)
    return clip, None, None, None


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--limit", type=int, default=0)
    ap.add_argument("--imgsz", type=int, default=640)
    ap.add_argument("--conf", type=float, default=0.05)
    ap.add_argument("--model", default=str(MODELS_DIR / "yolo11x.pt"))
    a = ap.parse_args()
    from ultralytics import YOLO
    import torch
    model = YOLO(a.model)

    RC = pd.read_csv(CACHE_DIR / "roles_clip.csv").set_index("clip")
    RC = RC[RC.kicker_found == 1]
    jobs = [(c, str((PUBLIC_DIR if RC.loc[c, "partition"] == "public" else PRIVATE_DIR) / c)) for c in RC.index]
    if a.limit:
        jobs = jobs[: a.limit]
    existing = pd.read_parquet(OUT) if OUT.exists() else None
    done = set(existing["clip"]) if existing is not None else set()
    jobs = [j for j in jobs if j[0] not in done]
    print(f"{len(done)} clips cached, {len(jobs)} to do", flush=True)

    rows, n_done, t0 = [], 0, time.time()

    def flush():
        nonlocal existing, rows
        if not rows:
            return
        new = pd.DataFrame(rows)
        existing = new if existing is None else pd.concat([existing, new], ignore_index=True)
        tmp = OUT.with_suffix(".tmp.parquet")
        existing.to_parquet(tmp, index=False)
        os.replace(tmp, OUT)
        rows = []

    with ThreadPoolExecutor(max_workers=2) as ex:
        futs = [ex.submit(load_clip, j) for j in jobs[:4]]
        nxt = 4
        while futs:
            clip, ts, wanted, frames = futs.pop(0).result()
            if nxt < len(jobs):
                futs.append(ex.submit(load_clip, jobs[nxt]))
                nxt += 1
            if ts is None:
                print("DECODE FAIL", clip, flush=True)
                rows.append({"clip": clip, "frame_idx": -1, "t_from_end_s": np.nan, "W": 0, "H": 0, "det_id": -1,
                             "conf": np.nan, "x1": np.nan, "y1": np.nan, "x2": np.nan, "y2": np.nan})
                continue
            rc = RC.loc[clip]
            x0, y0, side = crop_box(rc)
            H, W = frames[wanted[0]].shape[:2]
            crops = [frames[i][y0:y0 + side, x0:x0 + side] for i in wanted]
            res, i, chunk, fails = [], 0, 8, 0
            while i < len(crops):
                try:
                    res.extend(model.predict(crops[i:i + chunk], imgsz=a.imgsz, conf=a.conf, classes=[BALL],
                                             device=0, half=True, verbose=False))
                    i += chunk
                except Exception as e:  # noqa: BLE001
                    fails += 1
                    if fails > 10:
                        raise
                    torch.cuda.empty_cache()
                    chunk = 1
                    print(f"  {type(e).__name__} -> batch 1", flush=True)
            t_end = ts[-1]
            for fi, r in zip(wanted, res):
                base = {"clip": clip, "frame_idx": fi, "t_from_end_s": float((t_end - ts[fi]) / 1000.0), "W": W, "H": H}
                if len(r.boxes) == 0:
                    rows.append({**base, "det_id": -1, "conf": np.nan, "x1": np.nan, "y1": np.nan, "x2": np.nan, "y2": np.nan})
                    continue
                xyxy = r.boxes.xyxy.cpu().numpy()
                conf = r.boxes.conf.cpu().numpy()
                for k in range(len(conf)):
                    rows.append({**base, "det_id": k, "conf": float(conf[k]),
                                 "x1": float(xyxy[k, 0] + x0), "y1": float(xyxy[k, 1] + y0),
                                 "x2": float(xyxy[k, 2] + x0), "y2": float(xyxy[k, 3] + y0)})
            n_done += 1
            if n_done % 25 == 0:
                flush()
                el = time.time() - t0
                print(f"  {n_done}/{len(jobs)} clips  {el:.0f}s  ({el / n_done:.2f}s/clip)", flush=True)
    flush()
    print(f"done {n_done} clips in {time.time() - t0:.0f}s -> {OUT}")
    if existing is not None:
        d = existing[existing.det_id >= 0]
        per_clip = d.groupby("clip").size()
        print(f"clips processed {existing['clip'].nunique()} | with >=1 ball det {d['clip'].nunique()} | "
              f"median dets/clip {per_clip.median() if len(per_clip) else 0} | conf median {d.conf.median():.3f}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/baseline_meta.py
"""Metadata-only baselines (no video). Establishes the local 'educated guess' bar.

For each target and each fold, predict the majority class within groups of
metadata columns learned on the training folds (falling back to the global
majority when a group is unseen). Reports OOF competition score per grouping.

Run:  python src/baseline_meta.py
"""
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import pandas as pd

from config import TARGETS
from cv import get_folds, load_public, score_report


def group_majority_predict(train: pd.DataFrame, test: pd.DataFrame, keys: list[str]) -> pd.DataFrame:
    out = pd.DataFrame(index=test.index)
    for t in TARGETS:
        glob = train[t].mode()[0]
        if keys:
            tab = train.groupby(keys)[t].agg(lambda s: s.mode()[0])
            idx = pd.MultiIndex.from_frame(test[keys]) if len(keys) > 1 else test[keys[0]]
            out[t] = tab.reindex(idx).fillna(glob).values
        else:
            out[t] = glob
    return out


def main():
    pub = load_public()
    folds = get_folds(pub)
    pub["res_w"] = pub["resolution"].str.split("x").str[0].astype(int)
    pub["fps_r"] = pub["fps"].round(0).astype(int)
    groupings = {
        "global majority": [],
        "preferred_foot": ["preferred_foot"],
        "gender": ["gender"],
        "foot x gender": ["preferred_foot", "gender"],
        "foot x gender x fps": ["preferred_foot", "gender", "fps_r"],
        "foot x gender x res": ["preferred_foot", "gender", "res_w"],
    }
    print("label x preferred_foot (public):")
    for t in TARGETS:
        print(pd.crosstab(pub["preferred_foot"], pub[t], normalize="index").round(3))
    print()
    for name, keys in groupings.items():
        oof = pd.DataFrame(index=pub.index, columns=TARGETS, dtype=object)
        for k in sorted(folds.unique()):
            tr, te = pub[folds != k], pub[folds == k]
            oof.loc[te.index] = group_majority_predict(tr, te, keys).values
        score_report(pub, oof, folds, name)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/camera.py
"""Camera motion per clip (reference notebook notebook, section 6): broadcast cameras pan / zoom to follow the kicker, so raw
pixel positions mix player motion with camera motion. For every clip we estimate the frame-to-frame camera transform
from BACKGROUND points (corners outside every tracked person box, followed with Lucas-Kanade optical flow, fitted with
RANSAC) and chain the transforms so that every frame can be mapped into the coordinates of the LAST frame.

The fit is a 4-DoF similarity (translation + rotation + scale = pan + zoom), which is more stable than a full
homography on low-texture pitch frames; it is stored as a 3x3 matrix in ORIGINAL pixel coordinates.

Output: CACHE_DIR/camera/<clip>.npz with
    to_ref  [T, 3, 3]  frame t -> last frame (pixels), identity where the fit failed
    inliers [T-1]      RANSAC inliers per step (0 = fallback to identity)
    ts_ms   [T]        frame timestamps
Run:  python src/camera.py [--workers 8] [--limit N] [--viz]     (resumable: cached clips are skipped)
"""
import argparse
import os
import sys
import time
from multiprocessing import Pool

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import pyarrow  # noqa: F401
import cv2
import numpy as np
import pandas as pd

from config import CACHE_DIR, PRIVATE_DIR, PUBLIC_DIR

OUT = CACHE_DIR / "camera"
SHORT = 360            # analysis resolution (short side); transforms are rescaled to original pixels
MAX_CORNERS, QUALITY, MIN_DIST = 800, 0.01, 6
RANSAC_T = 2.0         # px at analysis resolution
MIN_INL = 12
PAD = 0.4              # person boxes are enlarged by this fraction of their height before masking
NEAR_FRAMES = 8        # use the tracked boxes of the nearest sampled frame if it is this close, else all boxes


def person_mask(shape, boxes, pad=PAD):
    m = np.full(shape, 255, np.uint8)
    for x1, y1, x2, y2 in boxes:
        p = pad * (y2 - y1)
        m[int(max(y1 - p, 0)):int(min(y2 + p, shape[0])), int(max(x1 - p, 0)):int(min(x2 + p, shape[1]))] = 0
    return m


def step_transform(g0, g1, mask):
    """Similarity transform (3x3) mapping frame-t pixels to frame-(t+1) pixels at analysis resolution, and inliers."""
    p0 = cv2.goodFeaturesToTrack(g0, MAX_CORNERS, QUALITY, MIN_DIST, mask=mask)
    if p0 is None or len(p0) < 8:
        return None, 0
    p1, ok, _ = cv2.calcOpticalFlowPyrLK(g0, g1, p0, None, winSize=(21, 21), maxLevel=3,
                                         criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01))
    ok = ok.ravel() == 1
    if ok.sum() < 8:
        return None, 0
    A, inl = cv2.estimateAffinePartial2D(p0[ok], p1[ok], method=cv2.RANSAC, ransacReprojThreshold=RANSAC_T,
                                         maxIters=2000, confidence=0.995)
    if A is None or inl is None or int(inl.sum()) < MIN_INL:
        return None, 0
    M = np.eye(3)
    M[:2] = A
    return M, int(inl.sum())


def stream_gray(path, short_side):
    """Yield (gray frame at analysis resolution, timestamp ms) one frame at a time (same decode loop as video.decode_clip,
    so frame indices agree with the cached detections, but without holding the whole clip in memory)."""
    cap = cv2.VideoCapture(str(path))
    try:
        while True:
            ok, f = cap.read()
            if not ok:
                break
            t = cap.get(cv2.CAP_PROP_POS_MSEC)
            h, w = f.shape[:2]
            sc = short_side / min(h, w)
            if sc < 1:
                f = cv2.resize(f, (round(w * sc), round(h * sc)), interpolation=cv2.INTER_AREA)
            yield cv2.cvtColor(f, cv2.COLOR_BGR2GRAY), t
    finally:
        cap.release()


def clip_camera(args):
    try:
        return _clip_camera(args)
    except Exception as e:  # noqa: BLE001  (one bad clip must not kill the pool)
        return args[0], f"FAILED: {type(e).__name__}: {e}"


def _clip_camera(args):
    clip, path, W, boxes_by_frame, all_boxes = args
    out = OUT / f"{clip}.npz"
    if out.exists():
        return clip, "cached"
    keys = np.array(sorted(boxes_by_frame)) if boxes_by_frame else None
    steps, inl, ts = [], [], []
    prev, s = None, None
    for t, (g, tms) in enumerate(stream_gray(path, SHORT)):
        ts.append(tms)
        if prev is not None:
            if keys is not None:
                j = keys[np.argmin(np.abs(keys - (t - 1)))]
                bx = boxes_by_frame[j] if abs(j - (t - 1)) <= NEAR_FRAMES else all_boxes
            else:
                bx = all_boxes
            mask = person_mask(prev.shape, np.asarray(bx, float) * s if len(bx) else [])
            M, n = step_transform(prev, g, mask)
            steps.append(M if M is not None else np.eye(3))
            inl.append(n)
        else:
            s = g.shape[1] / W                       # analysis px per original px
        prev = g
    T = len(ts)
    if T == 0:
        return clip, "DECODE FAILED: no frames"
    ts = np.asarray(ts, np.float64)
    if len(ts) > 1 and np.all(np.diff(ts) <= 0):     # containers with zero timestamps: nominal fps (as decode_clip)
        fps = cv2.VideoCapture(str(path)).get(cv2.CAP_PROP_FPS) or 25.0
        ts = np.arange(T) * 1000.0 / fps
    steps = np.array(steps) if steps else np.zeros((0, 3, 3))
    inl = np.array(inl, np.int32)
    to_ref = np.tile(np.eye(3), (T, 1, 1))
    for t in range(T - 2, -1, -1):
        to_ref[t] = to_ref[t + 1] @ steps[t]
    S, Si = np.diag([s, s, 1.0]), np.diag([1 / s, 1 / s, 1.0])
    to_ref = Si @ to_ref @ S                           # original px (frame t) -> original px (last frame)
    OUT.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(out, to_ref=to_ref.astype(np.float64), inliers=inl, ts_ms=ts, scale=s)
    c = np.array([W / 2, W / 2 * prev.shape[0] / prev.shape[1], 1.0])
    pan = (to_ref[0] @ c)[:2] - c[:2]
    return clip, f"{T} fr, inliers median {np.median(inl) if len(inl) else 0:.0f}, fallback {int((inl == 0).sum())}, pan over clip {pan[0]:+.0f},{pan[1]:+.0f} px"


def build_args(limit=None):
    RC = pd.read_csv(CACHE_DIR / "roles_clip.csv").set_index("clip")
    TR = pd.read_parquet(CACHE_DIR / "tracks.parquet", columns=["clip", "frame_idx", "x1", "y1", "x2", "y2"])
    by_clip = {c: g for c, g in TR.groupby("clip")}
    args = []
    for clip, rc in RC.iterrows():
        if (OUT / f"{clip}.npz").exists():
            continue
        path = (PUBLIC_DIR if clip.endswith("_public.mp4") else PRIVATE_DIR) / clip
        g = by_clip.get(clip)
        if g is None or len(g) == 0:
            bbf, allb = {}, []
        else:
            bbf = {int(k): v[["x1", "y1", "x2", "y2"]].values.tolist() for k, v in g.groupby("frame_idx")}
            allb = g[["x1", "y1", "x2", "y2"]].values.tolist()
        args.append((clip, path, float(rc.W), bbf, allb))
        if limit and len(args) >= limit:
            break
    return args


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--workers", type=int, default=8)
    ap.add_argument("--limit", type=int, default=None)
    a = ap.parse_args()
    OUT.mkdir(parents=True, exist_ok=True)
    args = build_args(a.limit)
    print(f"{len(args)} clips to do ({len(list(OUT.glob('*.npz')))} cached)", flush=True)
    t0 = time.time()
    done = 0
    if a.workers <= 1:
        it = map(clip_camera, args)
    else:
        pool = Pool(a.workers)
        it = pool.imap_unordered(clip_camera, args, chunksize=4)
    for clip, msg in it:
        done += 1
        if a.limit or done % 100 == 0 or "FAILED" in msg:
            print(f"[{done}/{len(args)} {time.time() - t0:.0f}s] {clip}: {msg}", flush=True)
    print(f"done {done} clips in {time.time() - t0:.0f}s", flush=True)


In [ ]:
%%writefile src/crops.py
"""Stage 2a: canonical kicker / keeper crop caches for the crop CNNs.

For every clip and view in {kicker, keeper}: 16 frames at tau = 0.60, 0.56, ..., 0.04,
0.00 s before the last frame (chronological; offsets before the clip start clamp to the
first frame with valid=False), decoded natively (framecache.frame_timestamps +
read_selected) and cropped with ONE square box fixed in image coordinates across the 16
frames, so the players' motion is visible inside the crop:

  kicker view: side = clip(3.0 * h_k, 160 px, 0.6 H), centred on the mean of the kicker
               box centres at tau 0.24 and 0.00 (role boxes interpolated over
               t_from_end_s with np.interp, clamped);
  keeper view: side = clip(4.0 * h_g, 160 px, H), centred on the keeper centre at tau 0.30.
  h_k, h_g = the picked box heights at the reference frame (roles_clip.csv kh, gh).
  Grey (114) padding outside the frame.

Canonical orientation: goal_side == 'R' -> the crop is mirrored horizontally so the goal is
always on the image-left side (mirrored=1). goal_side NaN -> not mirrored, side_unknown=1.
Fallbacks when a role was not picked (fallback=1): kicker = square 0.5 H at (0.55 W, 0.55 H);
keeper = square 0.6 H at (0.25 W | 0.75 W by goal side, else 0.5 W; 0.4 H).

Output (JPEG q90 224x224, stored like framecache.py: concatenated bytes + offsets):
  CACHE_DIR/crops_{view}/{clip_stem}.npz   + CACHE_DIR/crops_{view}_index.csv
  CACHE_DIR/viz_crops/{view}_tau0.jpg      (4x6 contact sheets of the tau=0 crops)
  CACHE_DIR/viz_crops/{view}_strip.jpg     (16-frame strips of 4 clips: motion check)

Clips whose detections/roles are not available yet (no W/H in roles_clip.csv) are skipped
unless --include-missing is given (then they get fallback crops). The build is resumable
(existing npz are skipped; --force rebuilds, e.g. after roles.py was rerun).

Run:   python src/crops.py [--view kicker|keeper|both] [--limit N] [--force] [--include-missing] [--viz-only]
Load:  from crops import load_crops, load_crop_meta, STACK7_TAUS, stack_indices
"""
import argparse
import os
import sys
import time
from concurrent.futures import ThreadPoolExecutor

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import cv2
import numpy as np
import pandas as pd

from config import CACHE_DIR, PRIVATE_DIR, PUBLIC_DIR, SEED
from framecache import frame_timestamps, read_selected
from video import indices_at_offsets

cv2.setNumThreads(1)

TAUS16 = [round(0.04 * k, 2) for k in range(15, -1, -1)]   # 0.60 ... 0.00, chronological
N_FRAMES = len(TAUS16)
STACK7_TAUS = [0.40, 0.32, 0.24, 0.16, 0.08, 0.04, 0.00]
CROP_PX = 224
JPEG_Q = 90
PAD_VALUE = 114
VIEWS = ("kicker", "keeper")
MIN_SIDE = 160.0
ROLES_CLIP = CACHE_DIR / "roles_clip.csv"
ROLES_FRAMES = CACHE_DIR / "roles_frames.parquet"
VIZ_DIR = CACHE_DIR / "viz_crops"


def crops_dir(view):
    return CACHE_DIR / f"crops_{view}"


def index_csv(view):
    return CACHE_DIR / f"crops_{view}_index.csv"


def stack_indices(taus=STACK7_TAUS, shift=0):
    """Cache indices (into the 16-frame grid) of the given taus; shift=1 -> one step earlier."""
    idx = []
    for tau in taus:
        k = int(round(tau / 0.04))
        assert abs(k * 0.04 - tau) < 1e-6 and 0 <= k < N_FRAMES, tau
        j = N_FRAMES - 1 - k - shift
        assert 0 <= j < N_FRAMES, (tau, shift)
        assert abs(TAUS16[j] - (tau + 0.04 * shift)) < 1e-6
        idx.append(j)
    return idx


# ----------------------------------------------------------------------------- boxes
def interp_role(tr: pd.DataFrame, taus):
    """Role boxes (pixels) at the sampled t_from_end_s -> (cx, cy, h) at the given taus (np.interp, clamped)."""
    tr = tr.sort_values("t_from_end_s")
    t = tr.t_from_end_s.values.astype(np.float64)
    cx = ((tr.x1 + tr.x2) / 2).values
    cy = ((tr.y1 + tr.y2) / 2).values
    h = (tr.y2 - tr.y1).values
    taus = np.asarray(taus, np.float64)
    return np.interp(taus, t, cx), np.interp(taus, t, cy), np.interp(taus, t, h)


def crop_box(view, row, roles_tr, W, H):
    """Square crop (x0, y0, side) in ORIGINAL image coordinates + fallback flag."""
    side_known = isinstance(row.get("goal_side"), str) and row["goal_side"] in ("L", "R")
    tr = roles_tr[roles_tr.role == view] if roles_tr is not None else None
    found = int(row.get(f"{'kicker' if view == 'kicker' else 'keeper'}_found", 0)) == 1 and tr is not None and len(tr) > 0
    if view == "kicker":
        if found:
            cx, cy, h = interp_role(tr, [0.24, 0.0])
            cxc, cyc = float(cx.mean()), float(cy.mean())
            h_k = float(row["kh"]) * H if pd.notna(row.get("kh", np.nan)) else float(h.mean())
            side = float(np.clip(3.0 * h_k, MIN_SIDE, 0.6 * H))
            fallback = 0
        else:
            cxc, cyc, side, fallback = 0.55 * W, 0.55 * H, 0.5 * H, 1
    else:
        if found:
            cx, cy, h = interp_role(tr, [0.30])
            cxc, cyc = float(cx[0]), float(cy[0])
            h_g = float(row["gh"]) * H if pd.notna(row.get("gh", np.nan)) else float(h[0])
            side = float(np.clip(4.0 * h_g, MIN_SIDE, H))
            fallback = 0
        else:
            cxc = {"L": 0.25 * W, "R": 0.75 * W}.get(row.get("goal_side"), 0.5 * W) if side_known else 0.5 * W
            cyc, side, fallback = 0.4 * H, 0.6 * H, 1
    side = int(round(side))
    x0, y0 = int(round(cxc - side / 2)), int(round(cyc - side / 2))
    return x0, y0, side, fallback


def crop_square(frame, x0, y0, side, out_px=CROP_PX, mirror=False):
    """Crop [y0:y0+side, x0:x0+side] with grey padding, resize to out_px, optional horizontal mirror."""
    H, W = frame.shape[:2]
    out = np.full((side, side, 3), PAD_VALUE, np.uint8)
    sx0, sy0 = max(0, x0), max(0, y0)
    sx1, sy1 = min(W, x0 + side), min(H, y0 + side)
    if sx1 > sx0 and sy1 > sy0:
        out[sy0 - y0:sy1 - y0, sx0 - x0:sx1 - x0] = frame[sy0:sy1, sx0:sx1]
    out = cv2.resize(out, (out_px, out_px), interpolation=cv2.INTER_AREA if side > out_px else cv2.INTER_LINEAR)
    if mirror:
        out = np.ascontiguousarray(out[:, ::-1])
    return out


# ----------------------------------------------------------------------------- build
def clip_path(clip):
    return str((PUBLIC_DIR if clip.endswith("_public.mp4") else PRIVATE_DIR) / clip)


def decode_job(job, retries=4):
    """Thread worker: timestamps + the 16 wanted native frames of one clip (nothing else is kept).
    A transient OpenCV allocation failure (the machine is out of RAM) is retried after a pause."""
    clip, path = job[0], job[1]
    for attempt in range(retries + 1):
        try:
            ts = frame_timestamps(path)
            if len(ts) == 0:
                return clip, None
            idx = indices_at_offsets(ts, TAUS16)
            frames = read_selected(path, set(idx))
            if not frames:
                return clip, None
            return clip, (ts, idx, frames)
        except cv2.error as e:  # noqa: PERF203
            if attempt == retries:
                print(f"DECODE ERROR {clip}: {str(e)[:120]}", flush=True)
                return clip, None
            time.sleep(15 * (attempt + 1))
    return clip, None


def build_clip(clip, decoded, row, roles_tr, views):
    """Crop + encode + save one clip for the requested views. Returns per-view index rows."""
    ts, idx, frames = decoded
    avail = sorted(frames)
    # a frame that failed to decode (decoder stopped early) -> nearest decoded frame before it
    fixed = []
    for i in idx:
        if i in frames:
            fixed.append(i)
        else:
            earlier = [a for a in avail if a <= i]
            fixed.append(earlier[-1] if earlier else avail[0])
    t_end = ts[-1]
    valid = np.array([(t_end - tau * 1000.0) >= ts[0] - 1.0 for tau in TAUS16], dtype=bool) & np.array(
        [i in frames for i in idx], dtype=bool)
    idx = fixed
    H, W = frames[idx[0]].shape[:2]
    if pd.notna(row.get("W", np.nan)) and (int(row["W"]) != W or int(row["H"]) != H):
        print(f"WARNING {clip}: roles W,H=({row['W']},{row['H']}) != decoded ({W},{H})", flush=True)
    gs = row.get("goal_side")
    side_known = isinstance(gs, str) and gs in ("L", "R")
    mirrored = int(side_known and gs == "R")
    partition = "public" if clip.endswith("_public.mp4") else "private"
    stem = clip.replace(".mp4", "")
    out_rows = []
    for view in views:
        x0, y0, side, fallback = crop_box(view, row, roles_tr, W, H)
        bufs, offs = [], [0]
        for i in idx:
            tile = crop_square(frames[i], x0, y0, side, mirror=bool(mirrored))
            ok, enc = cv2.imencode(".jpg", tile, [cv2.IMWRITE_JPEG_QUALITY, JPEG_Q])
            b = enc.tobytes()
            bufs.append(b)
            offs.append(offs[-1] + len(b))
        out = crops_dir(view) / f"{stem}.npz"
        tmp = out.with_suffix(".tmp.npz")
        np.savez(tmp, jpeg=np.frombuffer(b"".join(bufs), dtype=np.uint8), offsets=np.asarray(offs, np.int64),
                 frame_idx=np.asarray(idx, np.int32), ts_ms=ts[idx].astype(np.float32),
                 t_from_end=np.asarray(TAUS16, np.float32), valid=valid,
                 mirrored=np.int8(mirrored), side_unknown=np.int8(not side_known), fallback=np.int8(fallback),
                 box=np.asarray([x0, y0, side], np.float32), W=W, H=H, n_frames=len(ts),
                 goal_side=(gs if side_known else "U"), partition=partition, view=view)
        os.replace(tmp, out)
        out_rows.append({"clip": clip, "partition": partition, "view": view, "W": W, "H": H, "n_frames": len(ts),
                         "n_valid": int(valid.sum()), "goal_side": gs if side_known else "U", "mirrored": mirrored,
                         "side_unknown": int(not side_known), "fallback": fallback, "x0": x0, "y0": y0, "side": side})
    return out_rows


def load_crops(clip, view, idx=None, rgb=True):
    """Decode cached crops. clip may be '0001_public.mp4' or '0001_public'.
    idx: list of cache indices (0..15, chronological, 0 = tau 0.60, 15 = tau 0.00); default all 16.
    Returns uint8 [T, 224, 224, 3] RGB (BGR if rgb=False)."""
    stem = clip.replace(".mp4", "")
    z = np.load(crops_dir(view) / f"{stem}.npz")
    buf, offs = z["jpeg"], z["offsets"]
    idx = range(N_FRAMES) if idx is None else idx
    out = np.empty((len(idx), CROP_PX, CROP_PX, 3), np.uint8)
    for j, k in enumerate(idx):
        img = cv2.imdecode(buf[offs[k]:offs[k + 1]], cv2.IMREAD_COLOR)
        out[j] = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if rgb else img
    return out


def load_crop_meta(clip, view):
    stem = clip.replace(".mp4", "")
    z = np.load(crops_dir(view) / f"{stem}.npz")
    m = {k: z[k] for k in ("frame_idx", "ts_ms", "t_from_end", "valid", "box", "W", "H", "n_frames")}
    for k in ("mirrored", "side_unknown", "fallback"):
        m[k] = int(z[k])
    m["goal_side"] = str(z["goal_side"])
    m["partition"] = str(z["partition"])
    return m


def write_index(view):
    """Rebuild crops_{view}_index.csv from every npz present. Returns the DataFrame."""
    rows = []
    for f in sorted(crops_dir(view).glob("*.npz")):
        try:
            m = load_crop_meta(f.stem, view)
        except Exception as e:  # noqa: BLE001
            print(f"index: cannot read {f.name}: {e}", flush=True)
            continue
        rows.append({"clip": f.stem + ".mp4", "partition": m["partition"], "view": view, "W": int(m["W"]), "H": int(m["H"]),
                     "n_frames": int(m["n_frames"]), "n_valid": int(m["valid"].sum()), "goal_side": m["goal_side"],
                     "mirrored": m["mirrored"], "side_unknown": m["side_unknown"], "fallback": m["fallback"],
                     "x0": int(m["box"][0]), "y0": int(m["box"][1]), "side": int(m["box"][2])})
    df = pd.DataFrame(rows)
    df.to_csv(index_csv(view), index=False)
    return df


def clip_rows(clip_list):
    """clip -> roles_clip row (Series) and roles_frames sub-table (or None)."""
    RC = pd.read_csv(ROLES_CLIP).set_index("clip")
    RF = pd.read_parquet(ROLES_FRAMES)
    groups = {c: g for c, g in RF.groupby("clip", sort=False)}
    return RC, groups


# ----------------------------------------------------------------------------- viz
def put_label(img, text):
    cv2.rectangle(img, (0, 0), (img.shape[1], 14), (0, 0, 0), -1)
    cv2.putText(img, text, (2, 11), cv2.FONT_HERSHEY_SIMPLEX, 0.38, (255, 255, 255), 1, cv2.LINE_AA)
    return img


def viz(views, n=24, seed=SEED):
    """4x6 contact sheets of the tau=0 crops (goal should be on the LEFT, player centred) + 16-frame strips."""
    VIZ_DIR.mkdir(exist_ok=True)
    rng = np.random.RandomState(seed)
    for view in views:
        df = pd.read_csv(index_csv(view)) if index_csv(view).exists() else write_index(view)
        if len(df) == 0:
            continue
        pick = df.sample(min(n, len(df)), random_state=rng).sort_values("clip")
        tiles = []
        for _, r in pick.iterrows():
            img = load_crops(r["clip"], view, [N_FRAMES - 1], rgb=False)[0].copy()
            lab = f"{r['clip'][:4]}{'p' if r['partition'] == 'private' else ''} {r['goal_side']}{' m' if r['mirrored'] else ''}" \
                  f"{' FB' if r['fallback'] else ''} s{int(r['side'])}"
            tiles.append(put_label(img, lab))
        while len(tiles) % 4:
            tiles.append(np.zeros((CROP_PX, CROP_PX, 3), np.uint8))
        rows = [np.hstack(tiles[i:i + 4]) for i in range(0, len(tiles), 4)]
        sheet = np.vstack(rows[:6])
        p = VIZ_DIR / f"{view}_tau0.jpg"
        cv2.imwrite(str(p), sheet, [cv2.IMWRITE_JPEG_QUALITY, 90])
        print("wrote", p)
        # motion strips: 4 clips x 16 frames (downscaled tiles)
        strip_clips = pick["clip"].values[:4]
        strips = []
        for c in strip_clips:
            fr = load_crops(c, view, None, rgb=False)
            small = [cv2.resize(f, (112, 112), interpolation=cv2.INTER_AREA) for f in fr]
            small[0] = put_label(small[0].copy(), c[:4])
            strips.append(np.hstack(small))
        p = VIZ_DIR / f"{view}_strip.jpg"
        cv2.imwrite(str(p), np.vstack(strips), [cv2.IMWRITE_JPEG_QUALITY, 90])
        print("wrote", p)


# ----------------------------------------------------------------------------- main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--view", default="both", choices=["kicker", "keeper", "both"])
    ap.add_argument("--limit", type=int, default=0)
    ap.add_argument("--force", action="store_true", help="rebuild existing npz (after roles.py changed)")
    ap.add_argument("--include-missing", action="store_true",
                    help="also build (fallback) crops for clips without detections/roles yet")
    ap.add_argument("--viz-only", action="store_true")
    ap.add_argument("--threads", type=int, default=2)
    a = ap.parse_args()
    views = list(VIEWS) if a.view == "both" else [a.view]
    for v in views:
        crops_dir(v).mkdir(parents=True, exist_ok=True)
    if a.viz_only:
        viz(views)
        return

    RC = pd.read_csv(ROLES_CLIP)
    RF = pd.read_parquet(ROLES_FRAMES)
    groups = {c: g for c, g in RF.groupby("clip", sort=False)}
    have_roles = RC.W.notna()
    todo = RC if a.include_missing else RC[have_roles]
    print(f"roles_clip: {len(RC)} clips, {int(have_roles.sum())} with detections "
          f"(kicker {int(RC.kicker_found.sum())}, keeper {int(RC.keeper_found.sum())}, "
          f"side known {int(RC.goal_side.isin(['L', 'R']).sum())}); candidates {len(todo)}", flush=True)
    jobs = []
    for _, r in todo.iterrows():
        c = r["clip"]
        if not a.force and all((crops_dir(v) / c.replace(".mp4", ".npz")).exists() for v in views):
            continue
        jobs.append((c, clip_path(c), r))
    if a.limit:
        jobs = jobs[: a.limit]
    print(f"{len(todo) - len(jobs) if not a.limit else '?'} cached, {len(jobs)} clips to build for views {views}", flush=True)

    row_of = {j[0]: j[2] for j in jobs}
    t0, n_done, n_fail, rows = time.time(), 0, 0, []
    with ThreadPoolExecutor(max_workers=a.threads) as ex:
        ahead = a.threads + 1          # at most threads+1 clips' native frames alive at once
        futs = [ex.submit(decode_job, j) for j in jobs[:ahead]]
        nxt = ahead
        while futs:
            clip, decoded = futs.pop(0).result()
            if nxt < len(jobs):
                futs.append(ex.submit(decode_job, jobs[nxt]))
                nxt += 1
            row = row_of[clip]
            if decoded is None:
                print("DECODE FAIL", clip, flush=True)
                n_fail += 1
                continue
            try:
                rows += build_clip(clip, decoded, row, groups.get(clip), views)
            except Exception as e:  # noqa: BLE001
                print(f"BUILD FAIL {clip}: {e!r}", flush=True)
                n_fail += 1
                continue
            finally:
                del decoded
            n_done += 1
            if n_done % 100 == 0:
                el = time.time() - t0
                print(f"  {n_done}/{len(jobs)}  {el:.0f}s ({el / n_done:.2f} s/clip)", flush=True)
    el = time.time() - t0
    print(f"built {n_done} clips ({n_fail} failed) in {el:.0f}s" + (f" = {el / max(1, n_done):.2f} s/clip" if n_done else ""),
          flush=True)
    for v in views:
        df = write_index(v)
        size = sum(f.stat().st_size for f in crops_dir(v).glob("*.npz")) / 2**20
        if len(df):
            print(f"{v}: {len(df)} clips indexed ({(df.partition == 'public').sum()} public, "
                  f"{(df.partition == 'private').sum()} private) | mirrored {df.mirrored.mean():.3f} "
                  f"side_unknown {df.side_unknown.mean():.3f} fallback {df.fallback.mean():.3f} | "
                  f"side px median {df['side'].median():.0f} (min {df['side'].min()}, max {df['side'].max()}) | "
                  f"{size:.0f} MB ({size / len(df) * 1024:.0f} KB/clip)", flush=True)
    viz(views)
    if n_done:
        x = load_crops(jobs[0][0], views[0], stack_indices())
        print("sample stack-7 load:", x.shape, x.dtype, "indices", stack_indices())


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/cv.py
"""Validation harness shared by every model.

* source-grouped folds (StratifiedGroupKFold on kick_direction, groups = source_id),
  cached to disk so every experiment uses the identical split
* the competition metric
* flip-label helper
* a submission writer that validates the file against submission_template.csv
"""
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

from config import (CACHE_DIR, CLASSES, FLIP_MAP, GT_CSV, N_FOLDS, PRIVATE_CSV,
                    SEED, TARGETS, TEMPLATE_CSV, WEIGHTS)


def load_public() -> pd.DataFrame:
    return pd.read_csv(GT_CSV)


def load_private() -> pd.DataFrame:
    return pd.read_csv(PRIVATE_CSV)


def make_folds(df: pd.DataFrame, n_splits: int = N_FOLDS, seed: int = SEED,
               strat_col: str = "kick_direction") -> np.ndarray:
    """Fold id per row. Grouped by source_id so no source leaks across folds."""
    folds = np.full(len(df), -1, dtype=int)
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for k, (_, va) in enumerate(sgkf.split(df, df[strat_col], groups=df["source_id"])):
        folds[va] = k
    assert (folds >= 0).all()
    return folds


def get_folds(df: pd.DataFrame | None = None, n_splits: int = N_FOLDS, seed: int = SEED) -> pd.Series:
    """Cached fold assignment indexed by clip (CACHE_DIR/folds_{n}_{seed}.csv)."""
    path = CACHE_DIR / f"folds_{n_splits}_{seed}.csv"
    if path.exists():
        f = pd.read_csv(path).set_index("clip")["fold"]
    else:
        if df is None:
            df = load_public()
        f = pd.Series(make_folds(df, n_splits, seed), index=df["clip"].values, name="fold")
        f.rename_axis("clip").reset_index().to_csv(path, index=False)
    if df is not None:
        return f.reindex(df["clip"].values).reset_index(drop=True)
    return f


def accuracy_by_target(y_true: pd.DataFrame, y_pred: pd.DataFrame) -> dict:
    return {t: float((y_true[t].values == y_pred[t].values).mean()) for t in TARGETS}


def competition_score(y_true: pd.DataFrame, y_pred: pd.DataFrame) -> float:
    acc = accuracy_by_target(y_true, y_pred)
    return float(sum(WEIGHTS[t] * acc[t] for t in TARGETS))


def score_report(y_true: pd.DataFrame, y_pred: pd.DataFrame, folds: pd.Series | None = None,
                 name: str = "") -> float:
    """Print overall + per-fold score; returns the overall score."""
    acc = accuracy_by_target(y_true, y_pred)
    total = competition_score(y_true, y_pred)
    line = " | ".join(f"{t.split('_')[0]} {acc[t]:.3f}" for t in TARGETS)
    print(f"[{name}] score {total:.4f}  ({line})")
    if folds is not None:
        per = []
        for k in sorted(folds.unique()):
            m = (folds == k).values
            per.append(competition_score(y_true[m], y_pred[m]))
        print(f"   folds: {' '.join(f'{s:.3f}' for s in per)}   mean {np.mean(per):.4f} +- {np.std(per):.4f}")
    return total


def flip_labels(df: pd.DataFrame) -> pd.DataFrame:
    """Return a copy with left/right swapped in every label column present (mirror image)."""
    out = df.copy()
    for col, mp in FLIP_MAP.items():
        if col in out.columns:
            out[col] = out[col].map(mp)
    return out


def probs_to_labels(probs: dict[str, np.ndarray]) -> pd.DataFrame:
    """probs[target] is (n, n_classes) in CLASSES[target] order."""
    return pd.DataFrame({t: np.asarray(CLASSES[t])[np.argmax(probs[t], axis=1)] for t in TARGETS})


def write_submission(pred: pd.DataFrame, path) -> pd.DataFrame:
    """Validate against the template (exact clip set, exact label strings) and write."""
    tmpl = pd.read_csv(TEMPLATE_CSV)
    cols = ["clip"] + TARGETS
    missing = [c for c in cols if c not in pred.columns]
    if missing:
        raise ValueError(f"missing columns {missing}")
    sub = pred[cols].copy()
    if sub["clip"].duplicated().any():
        raise ValueError("duplicate clips")
    if set(sub["clip"]) != set(tmpl["clip"]):
        raise ValueError(f"clip set differs from template: {len(set(sub.clip) ^ set(tmpl.clip))} mismatches")
    for t in TARGETS:
        bad = ~sub[t].isin(CLASSES[t])
        if bad.any():
            raise ValueError(f"{t}: invalid values {sub.loc[bad, t].unique()[:5]}")
    sub = sub.set_index("clip").loc[tmpl["clip"]].reset_index()
    sub.to_csv(path, index=False)
    print(f"wrote {path}  rows={len(sub)}")
    for t in TARGETS:
        print(f"   {t}: {sub[t].value_counts().to_dict()}")
    return sub


if __name__ == "__main__":
    pub = load_public()
    f = get_folds(pub)
    print("fold sizes:", f.value_counts().sort_index().to_dict())
    print("sources per fold:", pub.groupby(f)["source_id"].nunique().to_dict())
    for t in TARGETS:
        print(t, pd.crosstab(f, pub[t]).to_dict("index"))
    # sanity: metric on the majority-class prediction
    maj = pd.DataFrame({t: [pub[t].mode()[0]] * len(pub) for t in TARGETS})
    score_report(pub, maj, f, "majority")


In [ ]:
%%writefile src/detect.py
"""Stage 1: person (and ball) detection on time-sampled frames of every clip.

For each clip we pick frames at fixed offsets before the last frame (dense over
the final second, sparse up to 3 s) and run YOLO11x detect at imgsz=1280 (the
players are only 30-80 px tall). One row per detected box is written to
CACHE_DIR/detections.parquet, incrementally, so the job can be resumed.

Run:  python src/detect.py            (both partitions)
      python src/detect.py --limit 20 (smoke test)
"""
import argparse
import os
import sys
import time
from concurrent.futures import ThreadPoolExecutor

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import cv2
import numpy as np
import pandas as pd

from config import CACHE_DIR, GT_CSV, MODELS_DIR, PRIVATE_CSV, PRIVATE_DIR, PUBLIC_DIR
from video import indices_at_offsets

cv2.setNumThreads(1)

# seconds before the last frame
OFFSETS = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.25, 1.5, 2.0, 2.5, 3.0]
OUT = CACHE_DIR / "detections.parquet"
PERSON, BALL = 0, 32


def frame_timestamps(path):
    """First pass: timestamps of every frame via grab() (no colour conversion)."""
    cap = cv2.VideoCapture(str(path))
    ts = []
    try:
        while cap.grab():
            ts.append(cap.get(cv2.CAP_PROP_POS_MSEC))
    finally:
        cap.release()
    ts = np.asarray(ts, dtype=np.float64)
    if len(ts) > 1 and np.all(np.diff(ts) <= 0):  # broken timestamps -> nominal fps
        fps = cv2.VideoCapture(str(path)).get(cv2.CAP_PROP_FPS) or 25.0
        ts = np.arange(len(ts)) * 1000.0 / fps
    return ts


def read_selected(path, wanted):
    """Second pass: decode sequentially, keep only the wanted frame indices."""
    wanted = set(wanted)
    out = {}
    cap = cv2.VideoCapture(str(path))
    i = 0
    try:
        while i <= max(wanted):
            ok, f = cap.read()
            if not ok:
                break
            if i in wanted:
                out[i] = f
            i += 1
    finally:
        cap.release()
    return out


def predict_chunked(model, imgs, imgsz, conf, chunk=4):
    """Run the detector on a list of frames in small batches; on CUDA OOM
    (the GPU is shared with other jobs) free the cache, shrink the batch and retry."""
    import torch
    out, i, fails = [], 0, 0
    while i < len(imgs):
        try:
            out.extend(model.predict(imgs[i:i + chunk], imgsz=imgsz, conf=conf, classes=[PERSON, BALL],
                                     device=0, half=True, verbose=False))
            i += chunk
            fails = 0
        except Exception as e:  # noqa: BLE001
            msg = str(e).lower()
            transient = ("out of memory" in msg or "insufficient memory" in msg
                         or isinstance(e, (SystemError, MemoryError)))
            fails += 1
            if not transient or fails > 10:
                raise
            torch.cuda.empty_cache()
            if chunk > 1:
                chunk = 1
                print(f"  {type(e).__name__} -> batch 1", flush=True)
            else:
                print(f"  {type(e).__name__} at batch 1 -> waiting 30 s (RAM/GPU pressure)", flush=True)
                time.sleep(30)
    return out


def load_clip(job):
    partition, clip, path = job
    for attempt in range(6):   # decoding can fail transiently under memory pressure
        try:
            ts = frame_timestamps(path)
            if len(ts) == 0:
                return partition, clip, None, None, None
            idx = indices_at_offsets(ts, OFFSETS)
            uniq = sorted(set(idx))
            frames = read_selected(path, uniq)
            if all(i in frames for i in uniq):
                return partition, clip, ts, uniq, frames
        except Exception as e:  # noqa: BLE001
            print(f"  decode error on {clip}: {str(e)[:80]}", flush=True)
        time.sleep(15)
    return partition, clip, None, None, None


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--limit", type=int, default=0)
    ap.add_argument("--imgsz", type=int, default=1280)
    ap.add_argument("--conf", type=float, default=0.10)
    ap.add_argument("--model", default=str(MODELS_DIR / "yolo11x.pt"))
    a = ap.parse_args()

    from ultralytics import YOLO
    model = YOLO(a.model)

    gt = pd.read_csv(GT_CSV)
    pm = pd.read_csv(PRIVATE_CSV)
    jobs = [("public", c, str(PUBLIC_DIR / c)) for c in gt["clip"]]
    jobs += [("private", c, str(PRIVATE_DIR / c)) for c in pm["clip"]]
    if a.limit:
        jobs = jobs[: a.limit // 2] + jobs[len(gt): len(gt) + a.limit // 2]

    existing = pd.read_parquet(OUT) if OUT.exists() else None
    done = set(existing["clip"]) if existing is not None else set()
    jobs = [j for j in jobs if j[1] not in done]
    print(f"{len(done)} clips already cached, {len(jobs)} to do", flush=True)

    rows, n_done, t0 = [], 0, time.time()

    def flush():
        nonlocal existing, rows
        if not rows:
            return
        new = pd.DataFrame(rows)
        existing = new if existing is None else pd.concat([existing, new], ignore_index=True)
        tmp = OUT.with_suffix(".tmp.parquet")   # atomic replace so concurrent readers never see a partial file
        existing.to_parquet(tmp, index=False)
        os.replace(tmp, OUT)
        rows = []

    with ThreadPoolExecutor(max_workers=2) as ex:
        futs = [ex.submit(load_clip, j) for j in jobs[:4]]
        nxt = 4
        while futs:
            partition, clip, ts, uniq, frames = futs.pop(0).result()
            if nxt < len(jobs):
                futs.append(ex.submit(load_clip, jobs[nxt]))
                nxt += 1
            if ts is None:
                print("DECODE FAIL", clip, flush=True)
                rows.append({"clip": clip, "partition": partition, "n_frames": 0, "frame_idx": -1,
                             "ts_ms": np.nan, "t_from_end_s": np.nan, "W": 0, "H": 0, "det_id": -1,
                             "cls": -1, "conf": np.nan, "x1": np.nan, "y1": np.nan, "x2": np.nan, "y2": np.nan})
                continue
            imgs = [frames[i] for i in uniq]
            res = predict_chunked(model, imgs, a.imgsz, a.conf)
            H, W = imgs[0].shape[:2]
            t_end = ts[-1]
            for i, r in zip(uniq, res):
                base = {"clip": clip, "partition": partition, "n_frames": len(ts), "frame_idx": i,
                        "ts_ms": float(ts[i]), "t_from_end_s": float((t_end - ts[i]) / 1000.0), "W": W, "H": H}
                if len(r.boxes) == 0:
                    rows.append({**base, "det_id": -1, "cls": -1, "conf": np.nan,
                                 "x1": np.nan, "y1": np.nan, "x2": np.nan, "y2": np.nan})
                    continue
                xyxy = r.boxes.xyxy.cpu().numpy()
                conf = r.boxes.conf.cpu().numpy()
                cls = r.boxes.cls.cpu().numpy().astype(int)
                for k in range(len(conf)):
                    rows.append({**base, "det_id": k, "cls": int(cls[k]), "conf": float(conf[k]),
                                 "x1": float(xyxy[k, 0]), "y1": float(xyxy[k, 1]),
                                 "x2": float(xyxy[k, 2]), "y2": float(xyxy[k, 3])})
            n_done += 1
            if n_done % 25 == 0:
                flush()
                el = time.time() - t0
                print(f"  {n_done}/{len(jobs)} clips  {el:.0f}s  ({el / n_done:.2f}s/clip)", flush=True)
    flush()
    print(f"done {n_done} clips in {time.time() - t0:.0f}s -> {OUT}")
    if existing is not None:
        d = existing[existing.cls == PERSON]
        print("rows:", len(existing), "| persons per frame:", round(d.groupby(["clip", "frame_idx"]).size().mean(), 2),
              "| frames with 0 persons:", int((existing.groupby(["clip", "frame_idx"]).cls.max() < 0).sum()),
              "| clips with a ball:", existing[existing.cls == BALL]["clip"].nunique())


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/dupes.py
"""Near-duplicate clip detection (same kick re-used across source videos, e.g. compilations).

Builds a small appearance descriptor per clip from frame cache A (last ~1 s, 4 frames,
grayscale 44x26, per-frame standardised), compares all 2,200 clips by cosine similarity
(also against horizontally flipped descriptors) and reports
  * source_id overlap between the public and private partitions,
  * public<->public near-duplicate pairs and how often their labels agree (sanity check),
  * private clips that have a near-duplicate public clip (label transfer candidates).
Writes CACHE_DIR/dupes_pairs.csv (all pairs above a loose threshold) and dupes_nn.csv (nearest
neighbour per clip).

Run: python src/dupes.py
"""
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import cv2
import numpy as np
import pandas as pd

from config import CACHE_DIR, TARGETS
from cv import load_private, load_public
from framecache import load_frames

IDX = [0, 6, 12, 23]           # t_end, -0.25 s, -0.5 s, -0.96 s
DW, DH = 44, 26


def descriptor(clip):
    fr = load_frames(clip, IDX, rgb=False)          # [4, 208, 352, 3] BGR
    parts, parts_flip = [], []
    for f in fr:
        g = cv2.cvtColor(f, cv2.COLOR_BGR2GRAY).astype(np.float32)
        g = cv2.resize(g, (DW, DH), interpolation=cv2.INTER_AREA)
        g = (g - g.mean()) / (g.std() + 1e-6)
        parts.append(g.ravel())
        parts_flip.append(g[:, ::-1].ravel())
    d = np.concatenate(parts)
    df = np.concatenate(parts_flip)
    return d / (np.linalg.norm(d) + 1e-6), df / (np.linalg.norm(df) + 1e-6)


def main(loose=0.80, tight=0.92):
    pub, prv = load_public(), load_private()
    pub["partition"], prv["partition"] = "public", "private"
    meta = pd.concat([pub, prv], ignore_index=True)
    clips = meta["clip"].tolist()

    s_pub, s_prv = set(pub["source_id"]), set(prv["source_id"])
    both = s_pub & s_prv
    print(f"sources: public {len(s_pub)}, private {len(s_prv)}, shared {len(both)}; "
          f"private clips whose source also appears in public: {prv['source_id'].isin(s_pub).sum()}/{len(prv)}")

    D, DF = [], []
    for i, c in enumerate(clips):
        d, df = descriptor(c)
        D.append(d)
        DF.append(df)
        if (i + 1) % 500 == 0:
            print(f"  descriptors {i + 1}/{len(clips)}", flush=True)
    D, DF = np.stack(D), np.stack(DF)
    S = D @ D.T
    SF = D @ DF.T                                    # similarity to the mirrored version of the other clip
    np.fill_diagonal(S, -1)
    np.fill_diagonal(SF, -1)
    S_best = np.maximum(S, SF)
    flipped = SF > S

    # nearest neighbour per clip
    nn = S_best.argmax(1)
    nn_df = pd.DataFrame({"clip": clips, "partition": meta["partition"], "source_id": meta["source_id"],
                          "nn_clip": [clips[j] for j in nn], "nn_partition": meta["partition"].values[nn],
                          "nn_source": meta["source_id"].values[nn],
                          "sim": S_best[np.arange(len(clips)), nn], "mirrored": flipped[np.arange(len(clips)), nn]})
    nn_df.to_csv(CACHE_DIR / "dupes_nn.csv", index=False)

    # all pairs above the loose threshold
    ii, jj = np.where(np.triu(S_best, 1) > loose)
    pairs = pd.DataFrame({"a": [clips[i] for i in ii], "b": [clips[j] for j in jj], "sim": S_best[ii, jj],
                          "mirrored": flipped[ii, jj],
                          "part_a": meta["partition"].values[ii], "part_b": meta["partition"].values[jj],
                          "src_a": meta["source_id"].values[ii], "src_b": meta["source_id"].values[jj]})
    lab = pub.set_index("clip")
    for t in TARGETS:
        pairs[f"{t}_a"] = pairs["a"].map(lab[t])
        pairs[f"{t}_b"] = pairs["b"].map(lab[t])
    pairs = pairs.sort_values("sim", ascending=False)
    pairs.to_csv(CACHE_DIR / "dupes_pairs.csv", index=False)

    print(f"\npairs with sim > {loose}: {len(pairs)}  (same source: {(pairs['src_a'] == pairs['src_b']).sum()}, "
          f"mirrored: {pairs['mirrored'].sum()})")
    pp = pairs[(pairs["part_a"] == "public") & (pairs["part_b"] == "public")]
    for thr in (loose, 0.85, 0.90, tight, 0.95, 0.98):
        q = pp[pp["sim"] > thr]
        if len(q) == 0:
            print(f"  public-public sim>{thr:.2f}: 0 pairs")
            continue
        agree = {t: (q[f"{t}_a"] == q[f"{t}_b"]).mean() for t in TARGETS}
        # under a mirror, left/right swap: count agreement after swapping for mirrored pairs
        kd = np.where(q["mirrored"], q["kick_direction_b"].map({"left": "right", "right": "left", "middle": "middle"}),
                      q["kick_direction_b"])
        print(f"  public-public sim>{thr:.2f}: {len(q):4d} pairs, label agreement "
              + ", ".join(f"{t}={v:.2f}" for t, v in agree.items())
              + f", kick(mirror-aware)={(q['kick_direction_a'].values == kd).mean():.2f}"
              + f", same source {(q['src_a'] == q['src_b']).mean():.2f}")
    x = pairs[(pairs["part_a"] != pairs["part_b"])]
    for thr in (loose, 0.85, 0.90, tight, 0.95):
        q = x[x["sim"] > thr]
        n_prv = pd.unique(np.where(q["part_a"] == "private", q["a"], q["b"])).size
        print(f"  private clips with a public match sim>{thr:.2f}: {n_prv}/{len(prv)}  ({len(q)} pairs)")
    print("\nnearest-neighbour similarity quantiles (all clips):",
          np.round(np.quantile(nn_df["sim"], [0.5, 0.75, 0.9, 0.95, 0.99]), 3))
    print("top cross-partition pairs:")
    print(x.head(15)[["a", "b", "sim", "mirrored", "src_a", "src_b"]].to_string(index=False))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/feat_frozen.py
"""Frozen-encoder feature tables (fp16) computed from frame cache A.

Row order of every table: cv.load_public() clips (1300) then cv.load_private() clips (900).
Outputs in CACHE_DIR/feats/:
  index.csv               row, clip, partition
  dinov2b_full.npy        [2200, 2, 30, 1536]  facebook/dinov2-base on every cached frame resized to
                                                224x392 (16:9, multiples of 14); [CLS | mean patch];
                                                axis 1: 0 = original, 1 = horizontally flipped
  videomae_ssv2_full.npy  [2200, 2, 942]       MCG-NJU/videomae-base-finetuned-ssv2 on the DENSE16 frames
                                                (chronological) squashed to 224x224;
                                                [fc_norm(mean token) (768) | 174 SSv2 logits]

Usage:
  python src/feat_frozen.py --smoke 30               # first 30 public clips -> CACHE_DIR/feats_smoke/
  python src/feat_frozen.py                          # both tables (waits for framesA/_DONE, checks RAM)
  python src/feat_frozen.py --models dinov2          # one table only
A table whose .npy already exists is skipped; an interrupted table resumes from its .partial.npy.
"""
import argparse
import gc
import os
import sys
import threading
import time
from pathlib import Path
from queue import Queue

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import config  # noqa: E402  (sets HF_HOME / TORCH_HOME on D: before transformers is imported)

# The encoders were downloaded by the benchmark into D:\cv3\models\hf\hub; reuse that cache.
_HUB_ALT = config.ROOT / "models" / "hf" / "hub"
if _HUB_ALT.exists():
    os.environ.setdefault("HF_HUB_CACHE", str(_HUB_ALT))

import cv2  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import psutil  # noqa: E402
import torch  # noqa: E402

from cv import load_private, load_public  # noqa: E402
from framecache import CACHE_A, DENSE16, N_FRAMES_A, load_frames  # noqa: E402

cv2.setNumThreads(2)

FEATS_DIR = config.CACHE_DIR / "feats"
DINO_NAME = "facebook/dinov2-base"
VMAE_NAME = "MCG-NJU/videomae-base-finetuned-ssv2"
DINO_HW = (224, 392)           # H, W: 16:9-ish, multiples of 14 -> 16 x 28 patches
VMAE_HW = (224, 224)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

TABLES = {
    "dinov2": ("dinov2b_full.npy", (2, N_FRAMES_A, 1536)),
    "videomae": ("videomae_ssv2_full.npy", (2, 942)),
}


# ----------------------------------------------------------------------------- helpers
def clip_index() -> pd.DataFrame:
    pub, priv = load_public(), load_private()
    rows = [(c, "public") for c in pub["clip"]] + [(c, "private") for c in priv["clip"]]
    return pd.DataFrame({"row": range(len(rows)), "clip": [r[0] for r in rows], "partition": [r[1] for r in rows]})


def free_ram_gb() -> float:
    return psutil.virtual_memory().available / 2**30


def wait_for_ram(min_gb: float = 2.5, poll_s: int = 30, max_wait_s: int = 1200):
    t0 = time.time()
    while free_ram_gb() < min_gb and time.time() - t0 < max_wait_s:
        print(f"free RAM {free_ram_gb():.2f} GB < {min_gb} GB; waiting {poll_s}s", flush=True)
        time.sleep(poll_s)
    print(f"free RAM {free_ram_gb():.2f} GB", flush=True)


def wait_for_cache(poll_s: int = 30, max_wait_s: int = 1200) -> bool:
    marker = CACHE_A / "_DONE"
    t0 = time.time()
    while not marker.exists() and time.time() - t0 < max_wait_s:
        n = len(list(CACHE_A.glob("*.npz")))
        print(f"waiting for {marker} ({n} npz present) ...", flush=True)
        time.sleep(poll_s)
    return marker.exists()


class TableWriter:
    """Row-wise fp16 table on disk; resumable through <name>.partial.npy + <name>.progress."""

    def __init__(self, path: Path, row_shape, dtype=np.float16):
        self.final = path
        self.partial = path.with_name(path.stem + ".partial.npy")
        self.progress = path.with_name(path.stem + ".progress")
        shape = (None, *row_shape)
        if self.partial.exists() and self.progress.exists():
            self.arr = np.lib.format.open_memmap(self.partial, mode="r+")
            self.done = int(self.progress.read_text().strip() or 0)
            print(f"resuming {self.partial.name} at row {self.done}")
        else:
            self.arr = None
            self.done = 0
        self.row_shape, self.dtype = tuple(row_shape), dtype

    def open(self, n_rows: int):
        if self.arr is None:
            self.arr = np.lib.format.open_memmap(self.partial, mode="w+", dtype=self.dtype,
                                                 shape=(n_rows, *self.row_shape))
        assert self.arr.shape == (n_rows, *self.row_shape), (self.arr.shape, n_rows, self.row_shape)

    def write(self, row: int, value: np.ndarray):
        self.arr[row] = value

    def checkpoint(self, n_done: int):
        self.arr.flush()
        self.progress.write_text(str(n_done))

    def finish(self):
        self.arr.flush()
        self.arr = None
        gc.collect()
        os.replace(self.partial, self.final)
        self.progress.unlink(missing_ok=True)


def prefetch(items, fn, depth: int = 4):
    """Yield fn(item) for each item, computed in a background thread (decode/resize overlap the GPU)."""
    q: Queue = Queue(maxsize=depth)
    stop = object()

    def worker():
        try:
            for it in items:
                for attempt in range(8):
                    try:
                        q.put(fn(it))
                        break
                    except MemoryError:
                        # Windows commit-limit hiccups (pagefile on the full C: drive) are transient.
                        print(f"MemoryError in prefetch (free RAM {free_ram_gb():.2f} GB), retry {attempt + 1}/8 in 15s",
                              flush=True)
                        gc.collect()
                        time.sleep(15)
                else:
                    raise MemoryError("prefetch failed 8 times")
        except Exception as e:  # noqa: BLE001
            q.put(e)
        q.put(stop)

    threading.Thread(target=worker, daemon=True).start()
    while True:
        x = q.get()
        if x is stop:
            return
        if isinstance(x, Exception):
            raise x
        yield x


def normalize_on_gpu(x_uint8: torch.Tensor, mean, std) -> torch.Tensor:
    """x_uint8 [..., H, W, 3] uint8 (cpu) -> [..., 3, H, W] fp16 normalized on cuda."""
    x = x_uint8.cuda(non_blocking=True).half().div_(255.0)
    m = torch.tensor(mean, dtype=torch.half, device="cuda")
    s = torch.tensor(std, dtype=torch.half, device="cuda")
    x = (x - m) / s
    return x.movedim(-1, -3).contiguous()


# ----------------------------------------------------------------------------- DINOv2
def dino_prep(clip: str) -> np.ndarray:
    """-> uint8 [2, 30, 224, 392, 3]: original and horizontally flipped resized frames."""
    fr = load_frames(clip)                                     # [30, 208, 352, 3] RGB
    h, w = DINO_HW
    out = np.empty((2, fr.shape[0], h, w, 3), np.uint8)
    for k in range(fr.shape[0]):
        out[0, k] = cv2.resize(fr[k], (w, h), interpolation=cv2.INTER_LINEAR)
    out[1] = out[0][:, :, ::-1]
    return out


def load_dino():
    from transformers import Dinov2Model
    m = Dinov2Model.from_pretrained(DINO_NAME, dtype=torch.float16).cuda().eval()
    return m


@torch.inference_mode()
def dino_encode(model, x_uint8: np.ndarray, batch: int = 64) -> np.ndarray:
    """x_uint8 [N, H, W, 3] -> [N, 1536] fp16 ([CLS | mean patch])."""
    xs = torch.from_numpy(np.ascontiguousarray(x_uint8))
    outs = []
    for i in range(0, len(xs), batch):
        px = normalize_on_gpu(xs[i:i + batch], IMAGENET_MEAN, IMAGENET_STD)
        hs = model(pixel_values=px).last_hidden_state          # [b, 1+N, 768], layer-normed
        outs.append(torch.cat([hs[:, 0], hs[:, 1:].mean(1)], dim=1).half().cpu())
    return torch.cat(outs).numpy()


def run_dino(index: pd.DataFrame, out_dir: Path, log_every: int = 100):
    name, row_shape = TABLES["dinov2"]
    path = out_dir / name
    if path.exists():
        print(f"{path} exists, skipping")
        return
    model = load_dino()
    wr = TableWriter(path, row_shape)
    wr.open(len(index))
    todo = index.iloc[wr.done:]
    torch.cuda.reset_peak_memory_stats()
    t0, n_img = time.time(), 0
    for n, (row, x) in enumerate(prefetch(todo.itertuples(index=False), lambda r: (r.row, dino_prep(r.clip))), 1):
        feats = dino_encode(model, x.reshape(-1, *x.shape[2:]))           # [60, 1536]
        wr.write(row, feats.reshape(2, N_FRAMES_A, 1536))
        n_img += x.shape[0] * x.shape[1]
        if n % log_every == 0 or n == len(todo):
            wr.checkpoint(row + 1)
            el = time.time() - t0
            print(f"  dinov2 {wr.done + n}/{len(index)}  {n_img / el:.0f} img/s  "
                  f"peak {torch.cuda.max_memory_allocated() / 2**30:.2f} GB  {el:.0f}s", flush=True)
    wr.finish()
    el = time.time() - t0
    print(f"dinov2 done: {n_img} images in {el:.0f}s = {n_img / max(el, 1e-6):.0f} img/s; "
          f"peak VRAM {torch.cuda.max_memory_allocated() / 2**30:.2f} GB -> {path}")
    del model
    torch.cuda.empty_cache()


# ----------------------------------------------------------------------------- VideoMAE
def vmae_prep(clip: str) -> np.ndarray:
    """-> uint8 [2, 16, 224, 224, 3]: DENSE16 frames (chronological), original and flipped."""
    fr = load_frames(clip, DENSE16)                            # [16, 208, 352, 3] RGB, chronological
    h, w = VMAE_HW
    out = np.empty((2, fr.shape[0], h, w, 3), np.uint8)
    for k in range(fr.shape[0]):
        out[0, k] = cv2.resize(fr[k], (w, h), interpolation=cv2.INTER_AREA)
    out[1] = out[0][:, :, ::-1]
    return out


def load_vmae():
    from transformers import VideoMAEForVideoClassification
    m = VideoMAEForVideoClassification.from_pretrained(VMAE_NAME, dtype=torch.float16).cuda().eval()
    assert m.fc_norm is not None and m.config.num_labels == 174, (m.fc_norm, m.config.num_labels)
    return m


@torch.inference_mode()
def vmae_encode(model, x_uint8: np.ndarray) -> np.ndarray:
    """x_uint8 [B, 16, 224, 224, 3] -> [B, 942] fp16 ([fc_norm(mean token) | logits])."""
    px = normalize_on_gpu(torch.from_numpy(np.ascontiguousarray(x_uint8)), IMAGENET_MEAN, IMAGENET_STD)
    hs = model.videomae(pixel_values=px).last_hidden_state     # [B, 1568, 768]
    pooled = model.fc_norm(hs.mean(1))                         # [B, 768]  (what the classifier sees)
    logits = model.classifier(pooled)                          # [B, 174]
    return torch.cat([pooled, logits], dim=1).half().cpu().numpy()


def run_vmae(index: pd.DataFrame, out_dir: Path, clips_per_batch: int = 4, log_every: int = 25):
    name, row_shape = TABLES["videomae"]
    path = out_dir / name
    if path.exists():
        print(f"{path} exists, skipping")
        return
    model = load_vmae()
    wr = TableWriter(path, row_shape)
    wr.open(len(index))
    todo = index.iloc[wr.done:]
    batches = [todo.iloc[i:i + clips_per_batch] for i in range(0, len(todo), clips_per_batch)]

    def prep(b):
        return b["row"].to_numpy(), np.stack([vmae_prep(c) for c in b["clip"]])   # [B, 2, 16, 224, 224, 3]

    torch.cuda.reset_peak_memory_stats()
    t0, n_vid = time.time(), 0
    for n, (rows, x) in enumerate(prefetch(batches, prep), 1):
        feats = vmae_encode(model, x.reshape(-1, *x.shape[2:]))           # [2B, 942]
        feats = feats.reshape(len(rows), 2, 942)
        for r, f in zip(rows, feats):
            wr.write(r, f)
        n_vid += 2 * len(rows)
        if n % log_every == 0 or n == len(batches):
            wr.checkpoint(int(rows[-1]) + 1)
            el = time.time() - t0
            print(f"  videomae {int(rows[-1]) + 1}/{len(index)}  {n_vid / el:.1f} videos/s  "
                  f"peak {torch.cuda.max_memory_allocated() / 2**30:.2f} GB  {el:.0f}s", flush=True)
    wr.finish()
    el = time.time() - t0
    print(f"videomae done: {n_vid} videos in {el:.0f}s = {n_vid / max(el, 1e-6):.1f} videos/s "
          f"({n_vid / 2 / max(el, 1e-6):.1f} clips/s incl. flip); "
          f"peak VRAM {torch.cuda.max_memory_allocated() / 2**30:.2f} GB -> {path}")
    del model
    torch.cuda.empty_cache()


# ----------------------------------------------------------------------------- smoke checks
@torch.inference_mode()
def smoke_checks(index: pd.DataFrame, out_dir: Path):
    d = np.load(out_dir / TABLES["dinov2"][0]) if (out_dir / TABLES["dinov2"][0]).exists() else None
    v = np.load(out_dir / TABLES["videomae"][0]) if (out_dir / TABLES["videomae"][0]).exists() else None
    if d is not None:
        print("dinov2 table", d.shape, d.dtype, "finite:", np.isfinite(d.astype(np.float32)).all())
        cls = d[:, :, :, :768].astype(np.float32)
        cos = (cls[:, 0, 0] * cls[:, 1, 0]).sum(1) / (np.linalg.norm(cls[:, 0, 0], axis=1) * np.linalg.norm(cls[:, 1, 0], axis=1))
        cos_t = (cls[:, 0, 0] * cls[:, 0, 7]).sum(1) / (np.linalg.norm(cls[:, 0, 0], axis=1) * np.linalg.norm(cls[:, 0, 7], axis=1))
        print(f"  CLS cos(orig,flip) k=0: mean {cos.mean():.3f}  | cos(k=0,k=7): {cos_t.mean():.3f}  "
              f"| CLS std {cls.std():.3f}  patch-mean std {d[:, :, :, 768:].astype(np.float32).std():.3f}")
    if v is not None:
        print("videomae table", v.shape, v.dtype, "finite:", np.isfinite(v.astype(np.float32)).all())
        model = load_vmae()
        id2label = model.config.id2label
        x = vmae_prep(index["clip"].iloc[0])
        full = model(pixel_values=normalize_on_gpu(torch.from_numpy(np.ascontiguousarray(x)), IMAGENET_MEAN, IMAGENET_STD))
        ref = full.logits.float().cpu().numpy()
        mine = v[0, :, 768:].astype(np.float32)
        print(f"  logits via base+fc_norm+classifier vs model().logits: max|diff| {np.abs(ref - mine).max():.4f}")
        for i in range(min(3, len(v))):
            lg = v[i, 0, 768:].astype(np.float32)
            top = np.argsort(-lg)[:3]
            print(f"  {index['clip'].iloc[i]}: top SSv2 classes {[id2label[int(t)] for t in top]}")
        del model
        torch.cuda.empty_cache()


# ----------------------------------------------------------------------------- main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--smoke", type=int, default=0, help="run on the first N public clips present -> feats_smoke/")
    ap.add_argument("--models", nargs="+", default=["dinov2", "videomae"], choices=list(TABLES))
    ap.add_argument("--no-wait", action="store_true", help="do not wait for framesA/_DONE")
    args = ap.parse_args()

    index = clip_index()
    if args.smoke:
        present = [c for c in index["clip"] if (CACHE_A / (c.replace(".mp4", "") + ".npz")).exists()]
        index = index[index["clip"].isin(present[:args.smoke])].reset_index(drop=True)
        index["row"] = range(len(index))
        out_dir = config.CACHE_DIR / "feats_smoke"
        print(f"SMOKE: {len(index)} clips -> {out_dir}")
    else:
        if not args.no_wait and not wait_for_cache():
            print("frame cache not finished (no _DONE after 20 min); aborting")
            sys.exit(1)
        missing = [c for c in index["clip"] if not (CACHE_A / (c.replace(".mp4", "") + ".npz")).exists()]
        if missing:
            print(f"{len(missing)} clips missing from frame cache, e.g. {missing[:5]}; aborting")
            sys.exit(1)
        out_dir = FEATS_DIR
    out_dir.mkdir(parents=True, exist_ok=True)
    index.to_csv(out_dir / "index.csv", index=False)
    print(f"index: {len(index)} rows ({(index.partition == 'public').sum()} public, "
          f"{(index.partition == 'private').sum()} private)")

    t_all = time.time()
    for m in args.models:
        if not args.smoke:
            wait_for_ram(2.5)
        t0 = time.time()
        (run_dino if m == "dinov2" else run_vmae)(index, out_dir)
        print(f"[{m}] wall {time.time() - t0:.0f}s", flush=True)
    if args.smoke:
        smoke_checks(index, out_dir)
    print(f"all done in {time.time() - t_all:.0f}s; tables: "
          f"{[p.name for p in out_dir.glob('*.npy')]}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/features_3d.py
"""Kicker features from the RTMW3D 3D pose (pose3d.py): the depth axis z is the kick's lateral axis in these side
views, so the depth components of the feet, legs, hips, shoulders and trunk are measured directly.

Kicker frame: u = kicker->keeper image direction (roles_clip), l = (u_y, -u_x) image-plane perpendicular (~ up).
For a 3D vector v = (dx, dy, dz) (dx, dy in model-crop pixels, dz in model depth units):
  fwd = (dx, dy) . u / torso      up = (dx, dy) . l / torso      lat = -dz * sign(u_x)   (kicker's left positive)
Mirror tags: side-agnostic ('disp') vectors -> lat _N, fwd _I, up _I; left-minus-right ('lr') vectors -> lat _I,
fwd _N, up _I (the joint swap and the axis flip cancel). Angles atan2(lat, fwd) -> _N (disp) / _I (lr).

features_3d.parquet (k_3d prefix = kicker subset) ; join: features_v9<sfx> + 3d -> gbm_v9k<sfx>_3d, plus a
3D-only head gbm_3d_only to measure the standalone signal.
Run:  python src/features_3d.py --base a2b [--no-train]
"""
import argparse
import glob
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import pyarrow  # noqa: F401
import numpy as np
import pandas as pd

from config import CACHE_DIR
from features_rtm import table_sfx
from features_v1 import train
from gbm_subset import subset_columns

NOSE, LSHO, RSHO, LHIP, RHIP, LKNEE, RKNEE, LANK, RANK = 0, 5, 6, 11, 12, 13, 14, 15, 16
LBTOE, LSTOE, LHEEL, RBTOE, RSTOE, RHEEL = 17, 18, 19, 20, 21, 22
NK = 23
TAUS = [0.0, 0.04, 0.08, 0.12, 0.16, 0.24, 0.40]
S_MIN = 0.3


def load_pose3d():
    files = sorted(glob.glob(str(CACHE_DIR / "pose3d_k*.parquet")))
    files = [f for f in files if not f.endswith(".tmp.parquet")]
    P = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
    return P.drop_duplicates(["clip", "frame_idx"])


def sample(g):
    """g: rows of one clip -> dict tau -> [NK, 3] (cx, cy, z; NaN below S_MIN), torso length (crop px), ankle speeds,
    mean score."""
    g = g.sort_values("t_from_end_s")
    t = g.t_from_end_s.values
    dt = np.median(np.diff(t)) if len(t) > 1 else 0.04
    tol = max(0.03, 0.6 * dt)
    X = np.stack([g[[f"cx{k}" for k in range(NK)]].values, g[[f"cy{k}" for k in range(NK)]].values,
                  g[[f"z{k}" for k in range(NK)]].values], 2)                        # [T, NK, 3]
    S = g[[f"s{k}" for k in range(NK)]].values
    X[S < S_MIN] = np.nan
    # temporal median over 3 frames
    if len(t) >= 3:
        Xs = X.copy()
        for k in range(NK):
            for d in range(3):
                Xs[:, k, d] = pd.Series(X[:, k, d]).rolling(3, center=True, min_periods=1).median().values
        X = Xs
    out = {}
    for tau in TAUS:
        j = int(np.argmin(np.abs(t - tau)))
        out[tau] = X[j] if abs(t[j] - tau) <= tol else np.full((NK, 3), np.nan)
    sh = (X[:, LSHO, :2] + X[:, RSHO, :2]) / 2
    hp = (X[:, LHIP, :2] + X[:, RHIP, :2]) / 2
    torso = np.linalg.norm(sh - hp, axis=1)
    s = float(np.nanmedian(torso)) if np.isfinite(torso).any() else np.nan
    if not np.isfinite(s) or s < 1e-3:
        s = 0.35 * 384 / 1.25                       # fallback: crop is 1.25-padded box -> 384 px
    m = t <= 0.16 + 1e-6
    speeds = {}
    for name, k in (("L", LANK), ("R", RANK)):
        p = X[m, k, :2]
        d = np.linalg.norm(np.diff(p, axis=0), axis=1)
        speeds[name] = np.nanmean(d) / s if np.isfinite(d).any() else np.nan
    return out, s, speeds, float(S[:, :17].mean())


def comps(v, u, l, s, lat_sign):
    """(fwd, up, lat) of a 3D vector in the kicker frame; None if missing."""
    if v is None or not np.all(np.isfinite(v)):
        return None
    return float(v[:2] @ u) / s, float(v[:2] @ l) / s, float(-v[2] * lat_sign)


def put(f, name, kind, c):
    """kind 'disp': lat _N, fwd _I, up _I, ang _N ; kind 'lr': lat _I, fwd _N, up _I, ang _I."""
    tl, tf, ta = ("N", "I", "N") if kind == "disp" else ("I", "N", "I")
    if c is None:
        f[f"{name}_lat_{tl}"] = f[f"{name}_fwd_{tf}"] = f[f"{name}_up_I"] = f[f"{name}_ang_{ta}"] = np.nan
        return
    fwd, up, lat = c
    f[f"{name}_lat_{tl}"], f[f"{name}_fwd_{tf}"], f[f"{name}_up_I"] = lat, fwd, up
    f[f"{name}_ang_{ta}"] = float(np.degrees(np.arctan2(lat, fwd)))


def feats(A, s, speeds, conf, u, l):
    lat_sign = 1.0 if u[0] >= 0 else -1.0
    f = {"k_3d_conf_I": conf, "k_3d_scale_I": s}
    sL, sR = speeds["L"], speeds["R"]
    has = np.isfinite(sL) and np.isfinite(sR)
    plant_right = (sR < sL) if has else None
    if plant_right is None:
        pt = ph = pa = pk = kt = kh = ka = kk = None
    else:
        pt, ph, pa, pk = (RBTOE, RHEEL, RANK, RKNEE) if plant_right else (LBTOE, LHEEL, LANK, LKNEE)
        kt, kh, ka, kk = (LBTOE, LHEEL, LANK, LKNEE) if plant_right else (RBTOE, RHEEL, RANK, RKNEE)
    C = lambda v: comps(v, u, l, s, lat_sign)  # noqa: E731
    V = lambda P, a, b: (P[a] - P[b]) if (a is not None and b is not None) else None  # noqa: E731
    for tau in TAUS:
        P = A[tau]
        t = f"{tau}"
        # feet orientation (toe - heel): the depth component is what 2D cannot see
        put(f, f"k_3d_kickfoot{t}", "disp", C(V(P, kt, kh)))
        put(f, f"k_3d_plantfoot{t}", "disp", C(V(P, pt, ph)))
        # legs
        put(f, f"k_3d_kickshank{t}", "disp", C(V(P, ka, kk)))
        put(f, f"k_3d_kickthigh{t}", "disp", C(V(P, kk, (LHIP if kk == LKNEE else RHIP) if kk is not None else None)))
        put(f, f"k_3d_plantshank{t}", "disp", C(V(P, pa, pk)))
        put(f, f"k_3d_feet{t}", "disp", C(V(P, kt, pt)))                 # kick toe relative to plant toe
        put(f, f"k_3d_ankles{t}", "disp", C(V(P, ka, pa)))
        # body axes: left minus right (lr kind)
        put(f, f"k_3d_hips{t}", "lr", C(P[LHIP] - P[RHIP]))
        put(f, f"k_3d_shoulders{t}", "lr", C(P[LSHO] - P[RSHO]))
        # trunk and gaze (disp)
        put(f, f"k_3d_trunk{t}", "disp", C((P[LSHO] + P[RSHO]) / 2 - (P[LHIP] + P[RHIP]) / 2))
        put(f, f"k_3d_gaze{t}", "disp", C(P[NOSE] - (P[LSHO] + P[RSHO]) / 2))
        # plant toe relative to the hip centre (how far 'across' the body the plant foot stands)
        put(f, f"k_3d_planthip{t}", "disp", C((P[pt] - (P[LHIP] + P[RHIP]) / 2) if pt is not None else None))
    # velocities of the kick toe and kick ankle (0.08 -> 0, 0.16 -> 0) and hip centre (0.16 -> 0)
    for a_, b_, name in ((0.08, 0.0, "08"), (0.16, 0.0, "16")):
        put(f, f"k_3d_kicktoe_v{name}", "disp", C((A[b_][kt] - A[a_][kt]) if kt is not None else None))
        put(f, f"k_3d_kickank_v{name}", "disp", C((A[b_][ka] - A[a_][ka]) if ka is not None else None))
    hip0, hip16 = (A[0.0][LHIP] + A[0.0][RHIP]) / 2, (A[0.16][LHIP] + A[0.16][RHIP]) / 2
    put(f, "k_3d_hip_v16", "disp", C(hip0 - hip16))
    # hip / shoulder rotation rate about the vertical: change of the lr-axis angle between 0.16 and 0
    for name, (ja, jb) in (("hips", (LHIP, RHIP)), ("shoulders", (LSHO, RSHO))):
        c0, c16 = C(A[0.0][ja] - A[0.0][jb]), C(A[0.16][ja] - A[0.16][jb])
        f[f"k_3d_{name}_turn_N"] = float(np.degrees(np.arctan2(c0[2], c0[0]) - np.arctan2(c16[2], c16[0]))) if (c0 and c16) else np.nan
    return f


def build():
    P = load_pose3d()
    RC = pd.read_csv(CACHE_DIR / "roles_clip.csv").set_index("clip")
    rows = {}
    for clip, g in P.groupby("clip"):
        rc = RC.loc[clip]
        if pd.isna(rc.u_x):
            continue
        u = np.array([rc.u_x, rc.u_y])
        l = np.array([u[1], -u[0]])
        A, s, speeds, conf = sample(g)
        rows[clip] = feats(A, s, speeds, conf, u, l)
    R = pd.DataFrame.from_dict(rows, orient="index")
    R.index.name = "clip"
    bad = [c for c in R.columns if not (c.endswith("_N") or c.endswith("_I"))]
    assert not bad, bad
    R.to_parquet(CACHE_DIR / "features_3d.parquet")
    print(f"3d features: {R.shape}, NaN share {R.isna().mean().mean():.3f}")
    return R


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--base", default="a2b")
    ap.add_argument("--no-train", action="store_true")
    ap.add_argument("--suffix", default="")
    a = ap.parse_args()
    R = build()
    if a.no_train:
        sys.exit(0)
    sfx = table_sfx(a.base)
    V9 = pd.read_parquet(CACHE_DIR / f"features_v9{sfx}.parquet")
    F = V9.join(R, how="left")
    F.to_parquet(CACHE_DIR / f"features_v15{sfx}.parquet")
    meta = [c for c in ("foot_N", "gender_I", "u_x_N", "u_y_I") if c in F.columns]
    print(f"\n=== gbm_3d_only{a.suffix} ({R.shape[1] + len(meta)} cols) ===")
    train(F[list(R.columns) + meta], f"gbm_3d_only{a.suffix}")
    cols = subset_columns(F.columns, "kicker")
    print(f"\n=== gbm_v9k{sfx}_3d{a.suffix} (kicker-only + 3d, {len(cols)} cols) ===")
    train(F[cols], f"gbm_v9k{sfx}_3d{a.suffix}")


In [ ]:
%%writefile src/features_ball.py
"""Ball-anchored kicker geometry, round 2: dense ball detections (ball_detect.py, every frame of the last 0.45 s on a
crop around the kicker's feet) + RTMPose Halpe-26 toes / heels / ankles / knees / hips.

Anchor = robust confidence-weighted position of the ball while it is at rest (detections 0.08-0.45 s before the end;
outliers > 0.6 kicker heights from the weighted median are dropped). Features, in the kicker frame (lat -> _N,
fwd -> _I, body scales):
  plant toe / plant heel / kick toe / kick heel / kick ankle / kick knee / hip-mid relative to the ball at
  tau = 0, 0.04, 0.08, 0.12, 0.16 (where the plant foot stands relative to the ball decides how open the hips are;
  where the kick foot meets the ball decides the contact point);
  kick-foot approach line (toe 0.08 -> 0) and the ball's perpendicular offset from it (miss distance = contact point);
  plant toe -> ball signed angle; run-up line (hip 0.4 -> 0) angle and the ball's offset from it;
  ball at the last frame relative to the anchor (lat / fwd: is the ball already moving) and ball size / kicker height.

features_ball_<mode>.parquet (b_ prefix -> kicker subset).  Join: features_v9<sfx> + ball -> gbm_v9k<sfx>_ball.
Run:  python src/features_ball.py --mode a2b [--no-train]
"""
import argparse
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import pyarrow  # noqa: F401
import numpy as np
import pandas as pd

from config import CACHE_DIR
from features_rtm import pose_file, table_sfx
from features_rtm2 import sample_smooth
from features_v1 import train
from features_v2 import LANK, LHIP, LKNEE, RANK, RHIP, RKNEE, mid, proj
from gbm_subset import subset_columns

L_TOE, R_TOE, L_HEEL, R_HEEL = 20, 21, 24, 25
TAUS_B = [0.0, 0.04, 0.08, 0.12, 0.16]
REST = (0.08, 0.45)       # the ball is at rest in this window before the end


def ball_anchors():
    """clip -> dict(ax, ay [W units], conf, n, w_px, last: (x, y) at the last frame or None)."""
    D = pd.read_parquet(CACHE_DIR / "ball_dets.parquet")
    D = D[D.det_id >= 0]
    RC = pd.read_csv(CACHE_DIR / "roles_clip.csv").set_index("clip")
    out = {}
    for clip, g in D.groupby("clip"):
        rc = RC.loc[clip]
        W, H = rc.W, rc.H
        kh = rc.kh * H
        cx, cy = ((g.x1 + g.x2) / 2).values, ((g.y1 + g.y2) / 2).values
        w = g.conf.values
        t = g.t_from_end_s.values
        rest = (t >= REST[0]) & (t <= REST[1])
        if rest.sum() == 0:
            rest = t >= 0.04
        if rest.sum() == 0:
            continue
        # weighted median, then drop outliers, then weighted mean
        def wmed(v, w_):
            o = np.argsort(v)
            c = np.cumsum(w_[o])
            return v[o][np.searchsorted(c, c[-1] / 2)]
        mx, my = wmed(cx[rest], w[rest]), wmed(cy[rest], w[rest])
        ok = rest & (np.hypot(cx - mx, cy - my) < 0.6 * kh)
        if ok.sum() == 0:
            continue
        ax, ay = np.average(cx[ok], weights=w[ok]), np.average(cy[ok], weights=w[ok])
        n = int(ok.sum())
        conf = float(w[ok].max())
        if n < 2 and conf < 0.15:
            continue
        wpx = float(np.average((g.x2 - g.x1).values[ok], weights=w[ok]))
        # ball in the last 0.03 s (nearest to the end), only if near the anchor (< 1.5 kh)
        last = None
        m_last = t <= 0.03
        if m_last.any():
            j = np.argmax(np.where(m_last, w, -1))
            if np.hypot(cx[j] - ax, cy[j] - ay) < 1.5 * kh:
                last = (cx[j] / W, cy[j] / W)
        out[clip] = dict(ax=ax / W, ay=ay / W, conf=conf, n=n, w_kh=wpx / kh, last=last)
    return out


def unit(v):
    if v is None or not np.all(np.isfinite(v)):
        return None
    n = np.linalg.norm(v)
    return v / n if n > 1e-6 else None


def sang(v, l, u):
    """Signed angle (deg) of v relative to the forward axis u, positive toward l."""
    v = unit(v)
    return float(np.degrees(np.arctan2(v @ l, v @ u))) if v is not None else np.nan


def ball_features(A, s, speeds, l, u, b):
    f = {"b_found_I": 1.0, "b_conf_I": b["conf"], "b_n_I": float(b["n"]), "b_size_I": b["w_kh"]}
    ball = np.array([b["ax"], b["ay"]])
    sL, sR = speeds["L"], speeds["R"]
    has_side = np.isfinite(sL) and np.isfinite(sR)
    plant_right = (sR < sL) if has_side else None
    if plant_right is None:
        ptoe = pheel = ktoe = kheel = kank = kknee = None
    else:
        ptoe, pheel, ktoe, kheel = (R_TOE, R_HEEL, L_TOE, L_HEEL) if plant_right else (L_TOE, L_HEEL, R_TOE, R_HEEL)
        kank, kknee = (LANK, LKNEE) if plant_right else (RANK, RKNEE)
    for tau in TAUS_B:
        P = A[tau]
        proj(mid(P, LHIP, RHIP) - ball, l, u, f"b_hip{tau}", "disp", f, s)
        for name, k in (("ptoe", ptoe), ("pheel", pheel), ("ktoe", ktoe), ("kheel", kheel), ("kank", kank), ("kknee", kknee)):
            proj(P[k] - ball if k is not None else None, l, u, f"b_{name}{tau}", "disp", f, s)
    # distances (invariant)
    for tau in (0.0, 0.08):
        for name, k in (("ptoe", ptoe), ("ktoe", ktoe)):
            v = A[tau][k] - ball if k is not None else None
            f[f"b_{name}{tau}_dist_I"] = float(np.linalg.norm(v)) / s if v is not None and np.all(np.isfinite(v)) else np.nan
    # plant toe -> ball direction, signed angle vs the kicker->keeper axis
    f["b_ptoe_ball_ang_N"] = sang(ball - A[0.0][ptoe], l, u) if ptoe is not None else np.nan
    f["b_pheel_ball_ang_N"] = sang(ball - A[0.0][pheel], l, u) if pheel is not None else np.nan
    # kick-foot approach line (toe 0.08 -> 0.0): direction and the ball's perpendicular offset (contact point)
    if ktoe is not None:
        p8, p0 = A[0.08][ktoe], A[0.0][ktoe]
        d = unit(p0 - p8)
        if d is not None and np.all(np.isfinite(p0)):
            perp = np.array([d[1], -d[0]])
            sign = 1.0 if (perp @ l) >= 0 else -1.0       # orient 'perp' toward the kicker's left so the tag _N is right
            f["b_kickline_ang_N"] = sang(d, l, u)
            f["b_ball_off_kickline_N"] = float((ball - p0) @ perp) / s * sign
            f["b_ball_along_kickline_I"] = float((ball - p0) @ d) / s
        else:
            f["b_kickline_ang_N"] = f["b_ball_off_kickline_N"] = f["b_ball_along_kickline_I"] = np.nan
        # same for the kick ankle over 0.12 -> 0
        p12, p0a = A[0.12][kank], A[0.0][kank]
        d = unit(p0a - p12)
        if d is not None and np.all(np.isfinite(p0a)):
            perp = np.array([d[1], -d[0]])
            sign = 1.0 if (perp @ l) >= 0 else -1.0
            f["b_ball_off_ankline_N"] = float((ball - p0a) @ perp) / s * sign
        else:
            f["b_ball_off_ankline_N"] = np.nan
    else:
        for n_ in ("b_kickline_ang_N", "b_ball_off_kickline_N", "b_ball_along_kickline_I", "b_ball_off_ankline_N"):
            f[n_] = np.nan
    # run-up line (hip 0.40 -> 0.0): angle to the forward axis, ball offset from the line
    h40, h0 = mid(A[0.40], LHIP, RHIP), mid(A[0.0], LHIP, RHIP)
    d = unit(h0 - h40)
    if d is not None and np.all(np.isfinite(h0)):
        perp = np.array([d[1], -d[0]])
        sign = 1.0 if (perp @ l) >= 0 else -1.0
        f["b_runup_ang_N"] = sang(d, l, u)
        f["b_ball_off_runup_N"] = float((ball - h0) @ perp) / s * sign
        f["b_ball_along_runup_I"] = float((ball - h0) @ d) / s
    else:
        f["b_runup_ang_N"] = f["b_ball_off_runup_N"] = f["b_ball_along_runup_I"] = np.nan
    # ball at the last frame relative to the rest anchor (already moving?)
    if b["last"] is not None:
        proj(np.array(b["last"]) - ball, l, u, "b_lastmove", "disp", f, s)
        f["b_lastmove_dist_I"] = float(np.hypot(b["last"][0] - ball[0], b["last"][1] - ball[1])) / s
    else:
        proj(None, l, u, "b_lastmove", "disp", f)
        f["b_lastmove_dist_I"] = np.nan
    return f


def build(mode):
    P = pd.read_parquet(CACHE_DIR / f"pose_rtm_{pose_file(mode)}.parquet")
    RC = pd.read_csv(CACHE_DIR / "roles_clip.csv").set_index("clip")
    balls = ball_anchors()
    print(f"ball anchors: {len(balls)} clips")
    rows = {}
    for (clip, role), g in P.groupby(["clip", "role"]):
        if role != "kicker":
            continue
        rc = RC.loc[clip]
        f = rows.setdefault(clip, {})
        if pd.isna(rc.u_x) or clip not in balls:
            f["b_found_I"] = 0.0
            continue
        W = int(rc.W)
        u = np.array([rc.u_x, rc.u_y])
        l = np.array([u[1], -u[0]])
        A, s, speeds = sample_smooth(g, W)
        b = balls[clip]
        # plausibility: at the last frame the ball must be near the kicker (toes / ankles / hips within 3.5 torsos)
        ball = np.array([b["ax"], b["ay"]])
        P0 = A[0.0]
        cands = [P0[k] for k in (L_TOE, R_TOE, LANK, RANK)] + [mid(P0, LHIP, RHIP)]
        dists = [np.linalg.norm(p - ball) / s for p in cands if np.all(np.isfinite(p))]
        if dists and min(dists) > 3.5:
            f["b_found_I"] = 0.0
            continue
        f.update(ball_features(A, s, speeds, l, u, b))
        f["b_ball_kicker_dist_I"] = float(min(dists)) if dists else np.nan
    R = pd.DataFrame.from_dict(rows, orient="index")
    R.index.name = "clip"
    bad = [c for c in R.columns if not (c.endswith("_N") or c.endswith("_I"))]
    assert not bad, bad
    R.to_parquet(CACHE_DIR / f"features_ball_{mode}.parquet")
    print(f"ball features ({mode}): {R.shape}, anchored {int(R.b_found_I.sum())}, NaN share {R.isna().mean().mean():.3f}")
    return R


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--mode", default="a2b")
    ap.add_argument("--no-train", action="store_true")
    ap.add_argument("--suffix", default="")
    a = ap.parse_args()
    R = build(a.mode)
    if a.no_train:
        sys.exit(0)
    sfx = table_sfx(a.mode)
    V9 = pd.read_parquet(CACHE_DIR / f"features_v9{sfx}.parquet")
    F = V9.join(R, how="left")
    F.to_parquet(CACHE_DIR / f"features_v14{sfx}.parquet")
    cols = subset_columns(F.columns, "kicker")
    print(f"\n=== gbm_v9k{sfx}_ball{a.suffix} (kicker-only + ball, {len(cols)} cols) ===")
    train(F[cols], f"gbm_v9k{sfx}_ball{a.suffix}")


In [ ]:
%%writefile src/features_cam.py
"""Camera-corrected PATH features (after the reference tracking notebook, sections 6-9), built on camera.py (per-frame transforms to the
last frame). Raw pixel positions mix player motion with camera pan / zoom; here the kicker's run-up and the keeper's
movement are measured in the coordinates of the last frame, in the kicker frame (u = kicker -> keeper, l = kicker's
left) and in body heights:

  kicker  kc_run{t}_*     ground-point displacement from t s before the end to the end (lat / fwd / angle / speed)
          kc_curv_*       heading change of the run-up (late vs mid, mid vs early segment)
          kc_decel_I      speed change over the approach, kc_straight_I path straightness over the last second
          kc_hip*/kc_kick*/kc_plant*  camera-corrected hip and foot displacements from the RTMPose tracks (last 0.4 s)
          kcam_pan{t}_*   the camera's own pan over the last t s (in kicker heights) and zoom change
  keeper  gc_run{t}_*     camera-corrected keeper displacement (lat / fwd), velocity and commit time

features_v13{sfx} = features_v9{sfx} + cam  ->  gbm_v13k{sfx} (kicker-only), gbm_camk{sfx} (cam-only kicker signal)
Run:  python src/features_cam.py [--mode avg|balanced|performance] [--no-train]
"""
import argparse
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import pyarrow  # noqa: F401
import numpy as np
import pandas as pd

from config import CACHE_DIR
from features_rtm import POSE_FILE, SFX, signed_angle
from features_rtm2 import unit
from features_v1 import train
from features_v2 import KP_CONF, LANK, LHIP, RANK, RHIP, proj
from gbm_subset import subset_columns

CAM = CACHE_DIR / "camera"
RUN_T = [0.2, 0.3, 0.5, 0.7, 1.0, 1.5, 2.0]
GK_T = [0.1, 0.2, 0.3, 0.5, 1.0]
PAN_T = [0.1, 0.2, 0.4, 1.0]
POSE_T = [0.0, 0.04, 0.08, 0.16, 0.24, 0.40]


def warp(M, p):
    """Apply a 3x3 transform to points p [..., 2] (pixels)."""
    q = np.c_[p.reshape(-1, 2), np.ones(p.reshape(-1, 2).shape[0])] @ M.T
    return (q[:, :2] / q[:, 2:3]).reshape(p.shape)


def dang(a, b):
    """Angle difference a - b in degrees wrapped to (-180, 180]."""
    if not (np.isfinite(a) and np.isfinite(b)):
        return np.nan
    d = (a - b + 180.0) % 360.0 - 180.0
    return float(d)


def tol_at(t):
    return max(0.06, 0.15 * t)


def at(points, t, tol=None):
    """Nearest entry of {t_from_end: value} within tol of t, else None."""
    if not points:
        return None
    ts = np.fromiter(points.keys(), float)
    j = int(np.argmin(np.abs(ts - t)))
    return points[ts[j]] if abs(ts[j] - t) <= (tol_at(t) if tol is None else tol) else None


def role_points(rf_role, to_ref, correct=True):
    """{t_from_end: (ground xy, centre xy, box h)} in last-frame pixels (or raw pixels when correct=False)."""
    out = {}
    T = len(to_ref)
    for r in rf_role.itertuples():
        fi = int(r.frame_idx)
        if fi < 0 or fi >= T:
            continue
        pts = np.array([[(r.x1 + r.x2) / 2, r.y2], [(r.x1 + r.x2) / 2, (r.y1 + r.y2) / 2]])
        if correct:
            pts = warp(to_ref[fi], pts)
        out[float(r.t_from_end_s)] = (pts[0], pts[1], r.y2 - r.y1)
    return out


def kicker_cam(pts, l, u, W, kh, f):
    p0 = at(pts, 0.0)
    if p0 is None:
        return
    g0 = p0[0] / W
    seg = {}
    for t in RUN_T:
        pt = at(pts, t)
        if pt is None:
            proj(None, l, u, f"kc_run{t}", "disp", f)
            f[f"kc_run{t}_ang_N"] = f[f"kc_speed{t}_I"] = np.nan
            continue
        d = g0 - pt[0] / W
        seg[t] = d
        proj(d, l, u, f"kc_run{t}", "disp", f, kh)
        f[f"kc_run{t}_ang_N"] = signed_angle(unit(d), l, u)
        f[f"kc_speed{t}_I"] = float(np.linalg.norm(d)) / kh / t

    def disp(a, b):  # displacement from t=a (earlier) to t=b (later, a > b)
        pa, pb = at(pts, a), (p0 if b == 0.0 else at(pts, b))
        return None if pa is None or pb is None else (pb[0] - pa[0]) / W

    late, mid, early = disp(0.3, 0.0), disp(0.7, 0.3), disp(1.5, 0.7)
    f["kc_curv_late_N"] = dang(signed_angle(unit(late), l, u), signed_angle(unit(mid), l, u))
    f["kc_curv_early_N"] = dang(signed_angle(unit(mid), l, u), signed_angle(unit(early), l, u))
    sp = lambda d, dt: float(np.linalg.norm(d)) / kh / dt if d is not None else np.nan  # noqa: E731
    f["kc_speed_late_I"], f["kc_speed_mid_I"] = sp(late, 0.3), sp(mid, 0.4)
    f["kc_decel_I"] = f["kc_speed_late_I"] - f["kc_speed_mid_I"]
    ts = sorted(t for t in pts if t <= 1.0 + 1e-6)
    P = np.array([pts[t][0] for t in ts]) / W
    if len(P) >= 3:
        f["kc_straight_I"] = float(np.linalg.norm(P[-1] - P[0]) / (np.linalg.norm(np.diff(P, axis=0), axis=1).sum() + 1e-9))
    else:
        f["kc_straight_I"] = np.nan
    # lateral position of the run-up START relative to the final kicker->keeper axis, at the earliest tracked point
    t_first = max(pts)
    if t_first >= 0.5:
        d = g0 - pts[t_first][0] / W
        proj(d, l, u, "kc_start", "disp", f, kh)
        f["kc_start_t_I"] = t_first
    else:
        proj(None, l, u, "kc_start", "disp", f)
        f["kc_start_t_I"] = np.nan


def keeper_cam(pts, raw, l, u, W, gh, f):
    p0 = at(pts, 0.0)
    if p0 is None:
        return
    c0 = p0[1] / W
    for t in GK_T:
        pt = at(pts, t)
        if pt is None:
            proj(None, l, u, f"gc_run{t}", "disp", f)
            continue
        proj(c0 - pt[1] / W, l, u, f"gc_run{t}", "disp", f, gh)
    p1, p2 = at(pts, 0.1), at(pts, 0.2)
    f["gc_vel_N"] = ((c0 - p1[1] / W) @ l) / gh / 0.1 if p1 is not None else np.nan
    f["gc_acc_N"] = ((((c0 - p1[1] / W) - (p1[1] - p2[1]) / W)) @ l) / gh / 0.01 if (p1 is not None and p2 is not None) else np.nan
    moved = [(t, abs(f.get(f"gc_run{t}_lat_N", np.nan))) for t in GK_T]
    f["gc_commit_I"] = next((t for t, m in reversed(moved) if not np.isnan(m) and m < 0.15), 0.0)
    # diagnostic (dropped later): keeper displacement 1.0 -> 0.3 s, raw vs corrected (the keeper mostly stands still)
    a, b = at(pts, 1.0), at(pts, 0.3)
    ra, rb = at(raw, 1.0), at(raw, 0.3)
    if a is not None and b is not None and ra is not None and rb is not None:
        f["_diag_cor"] = float(np.linalg.norm(b[1] - a[1]) / W / gh)
        f["_diag_raw"] = float(np.linalg.norm(rb[1] - ra[1]) / W / gh)


def camera_feats(to_ref, ts_ms, l, u, W, H, kh, f):
    c = np.array([[W / 2, H / 2]])
    end = ts_ms[-1]
    for t in PAN_T:
        if end - ts_ms[0] < t * 1000 - 40:
            proj(None, l, u, f"kcam_pan{t}", "disp", f)
            f[f"kcam_zoom{t}_I"] = np.nan
            continue
        i = int(np.argmin(np.abs(ts_ms - (end - t * 1000))))
        M = to_ref[i]
        proj((warp(M, c)[0] - c[0]) / W, l, u, f"kcam_pan{t}", "disp", f, kh)
        f[f"kcam_zoom{t}_I"] = float(np.sqrt(abs(np.linalg.det(M[:2, :2]))) - 1.0)


def pose_cam(g, to_ref, l, u, W, kh, f):
    """Camera-corrected hip / foot displacements from the RTMPose kicker track (last 0.4 s)."""
    g = g.sort_values("t_from_end_s")
    T = len(to_ref)
    pts = {}
    for r in g.itertuples():
        fi = int(r.frame_idx)
        if fi < 0 or fi >= T:
            continue
        P = np.full((4, 2), np.nan)
        for k, j in enumerate((LHIP, RHIP, LANK, RANK)):
            if getattr(r, f"kc{j}") >= KP_CONF:
                P[k] = (getattr(r, f"kx{j}"), getattr(r, f"ky{j}"))
        ok = np.isfinite(P[:, 0])
        if ok.any():
            P[ok] = warp(to_ref[fi], P[ok])
        pts[float(r.t_from_end_s)] = P / W
    if not pts:
        return

    def joint(t, k):
        P = at(pts, t, 0.03)
        return None if P is None or not np.all(np.isfinite(P[k])) else P[k]

    def hip(t):
        a, b = joint(t, 0), joint(t, 1)
        return None if a is None or b is None else (a + b) / 2

    h = {t: hip(t) for t in POSE_T}
    for a, b in ((0.08, 0.0), (0.16, 0.08), (0.40, 0.16), (0.40, 0.0)):
        d = None if h[a] is None or h[b] is None else h[b] - h[a]
        proj(d, l, u, f"kc_hipv{a}_{b}", "disp", f, kh * (a - b))
    d1 = None if h[0.08] is None or h[0.0] is None else h[0.0] - h[0.08]
    d2 = None if h[0.24] is None or h[0.08] is None else h[0.08] - h[0.24]
    f["kc_hip_turn_N"] = dang(signed_angle(unit(d1), l, u), signed_angle(unit(d2), l, u))
    # plant / kick foot from corrected ankle speeds over the last 0.16 s
    sp = {}
    for name, k in (("L", 2), ("R", 3)):
        a, b = joint(0.16, k), joint(0.0, k)
        sp[name] = np.nan if a is None or b is None else float(np.linalg.norm(b - a)) / kh / 0.16
    if np.isfinite(sp["L"]) and np.isfinite(sp["R"]):
        plant, kick = (2, 3) if sp["L"] < sp["R"] else (3, 2)
        for a in (0.08, 0.16):
            pa, pb = joint(a, kick), joint(0.0, kick)
            proj(None if pa is None or pb is None else pb - pa, l, u, f"kc_kick{a}", "disp", f, kh)
        pa, pb = joint(0.16, plant), joint(0.0, plant)
        d = None if pa is None or pb is None else pb - pa
        proj(d, l, u, "kc_plant0.16", "disp", f, kh)
        f["kc_plant_slide_I"] = float(np.linalg.norm(d)) / kh if d is not None else np.nan
        f["kc_kickspeed_I"], f["kc_plantspeed_I"] = max(sp.values()), min(sp.values())
    else:
        for n in ("kc_kick0.08", "kc_kick0.16", "kc_plant0.16"):
            proj(None, l, u, n, "disp", f)
        f["kc_plant_slide_I"] = f["kc_kickspeed_I"] = f["kc_plantspeed_I"] = np.nan


def build(mode):
    RC = pd.read_csv(CACHE_DIR / "roles_clip.csv").set_index("clip")
    RF = pd.read_parquet(CACHE_DIR / "roles_frames.parquet")
    rf_g = {c: g for c, g in RF.groupby("clip")}
    cols = ["clip", "role", "frame_idx", "t_from_end_s"] + [f"{a}{j}" for j in (LHIP, RHIP, LANK, RANK) for a in ("kx", "ky", "kc")]
    PO = pd.read_parquet(CACHE_DIR / f"pose_rtm_{POSE_FILE[mode]}.parquet", columns=cols)
    PO = PO[PO.role == "kicker"]
    po_g = {c: g for c, g in PO.groupby("clip")}
    rows, missing = {}, 0
    for clip, rc in RC.iterrows():
        f = {}
        rows[clip] = f
        npz = CAM / f"{clip}.npz"
        if pd.isna(rc.u_x) or not npz.exists():
            missing += int(not npz.exists())
            continue
        z = np.load(npz)
        to_ref, ts_ms = z["to_ref"], z["ts_ms"]
        f["kcam_fallback_I"] = float((z["inliers"] == 0).mean()) if len(z["inliers"]) else np.nan
        W, H = float(rc.W), float(rc.H)
        u = np.array([rc.u_x, rc.u_y])
        l = np.array([u[1], -u[0]])
        kh, gh = rc.kh * H / W, rc.gh * H / W            # heights in units of W
        rf = rf_g.get(clip)
        if rf is not None:
            kicker_cam(role_points(rf[rf.role == "kicker"], to_ref), l, u, W, kh, f)
            gk = rf[rf.role == "keeper"]
            keeper_cam(role_points(gk, to_ref), role_points(gk, to_ref, correct=False), l, u, W, gh, f)
        camera_feats(to_ref, ts_ms, l, u, W, H, kh, f)
        if clip in po_g:
            pose_cam(po_g[clip], to_ref, l, u, W, kh, f)
    R = pd.DataFrame.from_dict(rows, orient="index")
    R.index.name = "clip"
    d = R[["_diag_raw", "_diag_cor"]].dropna()
    print(f"camera files missing: {missing}; keeper displacement 1.0->0.3 s (keeper heights): raw median {d._diag_raw.median():.3f}, "
          f"corrected median {d._diag_cor.median():.3f} on {len(d)} clips (should drop if the correction works)")
    R = R.drop(columns=["_diag_raw", "_diag_cor"])
    bad = [c for c in R.columns if not (c.endswith("_N") or c.endswith("_I"))]
    assert not bad, bad
    R.to_parquet(CACHE_DIR / f"features_cam_{mode}.parquet")
    print(f"cam features ({mode}): {R.shape}, NaN share {R.isna().mean().mean():.3f}")
    return R


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--mode", default="avg", choices=list(POSE_FILE))
    ap.add_argument("--no-train", action="store_true")
    a = ap.parse_args()
    R = build(a.mode)
    if a.no_train:
        sys.exit(0)
    sfx = SFX[a.mode]
    V9 = pd.read_parquet(CACHE_DIR / f"features_v9{sfx}.parquet")
    F13 = V9.join(R, how="left")
    F13.to_parquet(CACHE_DIR / f"features_v13{sfx}.parquet")
    cam_k = [c for c in R.columns if c.startswith(("kc_", "kcam_"))]
    print(f"\n=== gbm_camk{sfx} (camera-corrected kicker path features only, {len(cam_k)} cols) ===")
    train(R[cam_k], f"gbm_camk{sfx}")
    cols = subset_columns(F13.columns, "kicker")
    print(f"\n=== gbm_v13k{sfx} (kicker-only, {len(cols)} cols) ===")
    train(F13[cols], f"gbm_v13k{sfx}")
    gcols = subset_columns(F13.columns, "keeper")
    print(f"\n=== gbm_v13g{sfx} (keeper-only, {len(gcols)} cols) ===")
    train(F13[gcols], f"gbm_v13g{sfx}")


In [ ]:
%%writefile src/features_flow.py
"""Optical-flow features (Farneback, as in Lecture 7) inside the kicker and keeper
boxes over the final frames, expressed in the kicker's frame.

For each role we crop a window around the role box at the last frame (fixed in
image coordinates), compute dense flow between (last-1 -> last) and
(~0.08 s before -> last), subtract the median flow of the crop border (camera
pan), and summarise the body-region flow: mean lateral (`_N`) / forward (`_I`)
components in body heights per second, magnitude, and for the kicker the
lower-half (legs) vs upper-half flow separately.

Output: CACHE_DIR/features_flow.parquet; trains gbm_v4 = v2 + flow.
Run:  python src/features_flow.py
"""
import os
import sys
import time
from concurrent.futures import ThreadPoolExecutor

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import pyarrow  # noqa: F401
import cv2
import numpy as np
import pandas as pd

from config import CACHE_DIR, PRIVATE_DIR, PUBLIC_DIR
from cv import load_private, load_public
from features_v1 import train
from framecache import frame_timestamps, read_selected

cv2.setNumThreads(1)
CROP_K = {"kicker": 1.8, "keeper": 2.2}
CROP_PX = 256
FB = dict(pyr_scale=0.5, levels=3, winsize=15, iterations=3, poly_n=5, poly_sigma=1.2, flags=0)


def role_box_at(tr: pd.DataFrame, t: float):
    tr = tr.sort_values("t_from_end_s")
    tt = tr.t_from_end_s.values
    vals = [np.interp(t, tt, v) for v in (((tr.x1 + tr.x2) / 2).values, ((tr.y1 + tr.y2) / 2).values,
                                          (tr.x2 - tr.x1).values, (tr.y2 - tr.y1).values)]
    return vals  # cx, cy, w, h (pixels)


def crop_gray(frame, cx, cy, side):
    H, W = frame.shape[:2]
    side = int(round(side))
    x0, y0 = int(round(cx - side / 2)), int(round(cy - side / 2))
    out = np.full((side, side), 114, np.uint8)
    g = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    sx0, sy0, sx1, sy1 = max(0, x0), max(0, y0), min(W, x0 + side), min(H, y0 + side)
    if sx1 > sx0 and sy1 > sy0:
        out[sy0 - y0:sy1 - y0, sx0 - x0:sx1 - x0] = g[sy0:sy1, sx0:sx1]
    return cv2.resize(out, (CROP_PX, CROP_PX), interpolation=cv2.INTER_AREA if side > CROP_PX else cv2.INTER_LINEAR), CROP_PX / side


def flow_feats(prev, cur, scale, h_px, dt, l, u, prefix, f, split_halves):
    flow = cv2.calcOpticalFlowFarneback(prev, cur, None, **FB)  # [H, W, 2] in crop px
    flow = flow / scale / max(h_px, 1.0) / max(dt, 1e-3)       # body heights per second, original-pixel geometry
    m = np.zeros((CROP_PX, CROP_PX), bool)
    b = CROP_PX // 8
    m[b:-b, b:-b] = True
    border = flow[~m]
    pan = np.median(border.reshape(-1, 2), axis=0)
    body = flow[m].reshape(-1, 2) - pan
    # central body region (the role box occupies the middle ~1/CROP_K of the crop)
    c0, c1 = int(CROP_PX * 0.3), int(CROP_PX * 0.7)
    core = (flow[c0:c1, c0:c1].reshape(-1, 2) - pan)
    def put(name, v):
        f[f"{prefix}{name}_lat_N"] = float(v @ l)
        f[f"{prefix}{name}_fwd_I"] = float(v @ u)
    put("core", core.mean(0))
    put("body", body.mean(0))
    f[f"{prefix}core_mag_I"] = float(np.linalg.norm(core, axis=1).mean())
    f[f"{prefix}core_magstd_I"] = float(np.linalg.norm(core, axis=1).std())
    f[f"{prefix}pan_mag_I"] = float(np.linalg.norm(pan))
    if split_halves:
        top = flow[c0:CROP_PX // 2, c0:c1].reshape(-1, 2) - pan
        bot = flow[CROP_PX // 2:c1, c0:c1].reshape(-1, 2) - pan
        put("legs", bot.mean(0))
        put("torso", top.mean(0))
        f[f"{prefix}legs_mag_I"] = float(np.linalg.norm(bot, axis=1).mean())
        # dominant leg motion: the 20% fastest pixels in the lower half
        mag = np.linalg.norm(bot, axis=1)
        k = max(1, int(0.2 * len(mag)))
        fast = bot[np.argsort(mag)[-k:]].mean(0)
        put("legsfast", fast)


def one_clip(job):
    clip, path, rc, rf = job
    f = {"clip": clip}
    if pd.isna(rc.u_x):
        return f
    W, H = int(rc.W), int(rc.H)
    u = np.array([rc.u_x, rc.u_y])
    l = np.array([u[1], -u[0]])
    ts = frame_timestamps(path)
    if len(ts) < 3:
        return f
    t_end = ts[-1]
    tfe = (t_end - ts) / 1000.0
    i0 = len(ts) - 1
    i1 = i0 - 1
    i8 = int(np.argmin(np.abs(tfe - 0.08)))
    i8 = min(i8, i1)
    frames = read_selected(path, {i0, i1, i8})
    for role in ("kicker", "keeper"):
        tr = rf[rf.role == role]
        if len(tr) == 0:
            continue
        cx, cy, w, h = role_box_at(tr, 0.0)
        side = max(CROP_K[role] * h, 1.4 * w, 48)
        g0, scale = crop_gray(frames[i0], cx, cy, side)
        g1, _ = crop_gray(frames[i1], cx, cy, side)
        g8, _ = crop_gray(frames[i8], cx, cy, side)
        p = "fk_" if role == "kicker" else "fg_"
        flow_feats(g1, g0, scale, h, (ts[i0] - ts[i1]) / 1000.0, l, u, p + "1f_", f, role == "kicker")
        flow_feats(g8, g0, scale, h, (ts[i0] - ts[i8]) / 1000.0, l, u, p + "80_", f, role == "kicker")
        f[p + "box_h_I"] = h / W
    return f


def build():
    RC = pd.read_csv(CACHE_DIR / "roles_clip.csv").set_index("clip")
    RF = pd.read_parquet(CACHE_DIR / "roles_frames.parquet")
    rf_g = {c: g for c, g in RF.groupby("clip")}
    pub, priv = load_public(), load_private()
    jobs = [(c, str(PUBLIC_DIR / c), RC.loc[c], rf_g.get(c, RF.iloc[0:0])) for c in pub["clip"]]
    jobs += [(c, str(PRIVATE_DIR / c), RC.loc[c], rf_g.get(c, RF.iloc[0:0])) for c in priv["clip"]]
    t0, rows = time.time(), []
    with ThreadPoolExecutor(max_workers=6) as ex:
        for n, f in enumerate(ex.map(one_clip, jobs), 1):
            rows.append(f)
            if n % 400 == 0:
                print(f"  {n}/{len(jobs)} {time.time() - t0:.0f}s", flush=True)
    F = pd.DataFrame(rows).set_index("clip")
    bad = [c for c in F.columns if not (c.endswith("_N") or c.endswith("_I"))]
    assert not bad, bad
    F.to_parquet(CACHE_DIR / "features_flow.parquet")
    print(f"flow features: {F.shape}, NaN share {F.isna().mean().mean():.3f}, {time.time() - t0:.0f}s")
    return F


if __name__ == "__main__":
    Fl = build()
    V2 = pd.read_parquet(CACHE_DIR / "features_v2.parquet")
    F = V2.join(Fl, how="left")
    F.to_parquet(CACHE_DIR / "features_v4.parquet")
    print("\n=== gbm_v4 (v2 + optical flow) ===")
    train(F, "gbm_v4")
    print("\n=== gbm_flow_only (flow + metadata) ===")
    cols = [c for c in F.columns if c.startswith(("fk_", "fg_")) or c in ("foot_N", "gender_I", "kicker_found_I", "keeper_found_I", "u_x_N", "u_y_I")]
    train(F[cols], "gbm_flow_only", seeds=(0,))


In [ ]:
%%writefile src/features_flow2.py
"""Extended optical-flow features: more frame pairs over the last 0.3 s
(each earlier frame -> last frame, and consecutive pairs 0.16->0.08, 0.24->0.16)
plus a 2x2 quadrant split of the core region. Same conventions as features_flow.

Output: CACHE_DIR/features_flow2.parquet; trains gbm_v5 = v2 + flow2.
Run:  python src/features_flow2.py
"""
import os
import sys
import time
from concurrent.futures import ThreadPoolExecutor

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import pyarrow  # noqa: F401
import cv2
import numpy as np
import pandas as pd

from config import CACHE_DIR, PRIVATE_DIR, PUBLIC_DIR
from cv import load_private, load_public
from features_flow import CROP_K, CROP_PX, FB, crop_gray, role_box_at
from features_v1 import train
from framecache import frame_timestamps, read_selected

cv2.setNumThreads(1)
PAIRS = [("1f", "prev", 0), ("08", 0.08, 0), ("16", 0.16, 0), ("24", 0.24, 0), ("1608", 0.16, 0.08), ("2416", 0.24, 0.16)]


def flow_summary(prev, cur, scale, h_px, dt, l, u, prefix, f, kicker):
    flow = cv2.calcOpticalFlowFarneback(prev, cur, None, **FB) / scale / max(h_px, 1.0) / max(dt, 1e-3)
    b = CROP_PX // 8
    m = np.zeros((CROP_PX, CROP_PX), bool)
    m[b:-b, b:-b] = True
    pan = np.median(flow[~m].reshape(-1, 2), axis=0)
    c0, c1 = int(CROP_PX * 0.3), int(CROP_PX * 0.7)
    core = flow[c0:c1, c0:c1] - pan

    def put(name, v):
        f[f"{prefix}{name}_lat_N"] = float(v @ l)
        f[f"{prefix}{name}_fwd_I"] = float(v @ u)
    put("core", core.reshape(-1, 2).mean(0))
    f[f"{prefix}core_mag_I"] = float(np.linalg.norm(core.reshape(-1, 2), axis=1).mean())
    mid = (c1 - c0) // 2
    put("q_tl", core[:mid, :mid].reshape(-1, 2).mean(0))
    put("q_tr", core[:mid, mid:].reshape(-1, 2).mean(0))
    put("q_bl", core[mid:, :mid].reshape(-1, 2).mean(0))
    put("q_br", core[mid:, mid:].reshape(-1, 2).mean(0))
    if kicker:
        bot = core[mid:].reshape(-1, 2)
        mag = np.linalg.norm(bot, axis=1)
        k = max(1, int(0.2 * len(mag)))
        put("legsfast", bot[np.argsort(mag)[-k:]].mean(0))
        f[f"{prefix}legs_mag_I"] = float(mag.mean())


def one_clip(job):
    clip, path, rc, rf = job
    f = {"clip": clip}
    if pd.isna(rc.u_x):
        return f
    W = int(rc.W)
    u = np.array([rc.u_x, rc.u_y])
    l = np.array([u[1], -u[0]])
    ts = frame_timestamps(path)
    if len(ts) < 3:
        return f
    tfe = (ts[-1] - ts) / 1000.0
    i_last = len(ts) - 1
    def idx_at(t):
        return i_last - 1 if t == "prev" else min(int(np.argmin(np.abs(tfe - t))), i_last - 1) if t > 0 else i_last
    need = {i_last}
    pairs = []
    for name, a, b_ in PAIRS:
        ia, ib = idx_at(a), idx_at(b_)
        if ia == ib:
            continue
        need.update((ia, ib))
        pairs.append((name, ia, ib))
    frames = read_selected(path, need)
    for role in ("kicker", "keeper"):
        tr = rf[rf.role == role]
        if len(tr) == 0:
            continue
        cx, cy, w, h = role_box_at(tr, 0.0)
        side = max(CROP_K[role] * h, 1.4 * w, 48)
        gray = {}
        for i in need:
            gray[i], scale = crop_gray(frames[i], cx, cy, side)
        p = "f2k_" if role == "kicker" else "f2g_"
        for name, ia, ib in pairs:
            flow_summary(gray[ia], gray[ib], scale, h, (ts[ib] - ts[ia]) / 1000.0, l, u, f"{p}{name}_", f, role == "kicker")
    return f


def build():
    RC = pd.read_csv(CACHE_DIR / "roles_clip.csv").set_index("clip")
    RF = pd.read_parquet(CACHE_DIR / "roles_frames.parquet")
    rf_g = {c: g for c, g in RF.groupby("clip")}
    pub, priv = load_public(), load_private()
    jobs = [(c, str(PUBLIC_DIR / c), RC.loc[c], rf_g.get(c, RF.iloc[0:0])) for c in pub["clip"]]
    jobs += [(c, str(PRIVATE_DIR / c), RC.loc[c], rf_g.get(c, RF.iloc[0:0])) for c in priv["clip"]]
    t0, rows = time.time(), []
    with ThreadPoolExecutor(max_workers=6) as ex:
        for n, f in enumerate(ex.map(one_clip, jobs), 1):
            rows.append(f)
            if n % 400 == 0:
                print(f"  {n}/{len(jobs)} {time.time() - t0:.0f}s", flush=True)
    F = pd.DataFrame(rows).set_index("clip")
    bad = [c for c in F.columns if not (c.endswith("_N") or c.endswith("_I"))]
    assert not bad, bad
    F.to_parquet(CACHE_DIR / "features_flow2.parquet")
    print(f"flow2 features: {F.shape}, NaN share {F.isna().mean().mean():.3f}, {time.time() - t0:.0f}s")
    return F


if __name__ == "__main__":
    F2 = build()
    V2 = pd.read_parquet(CACHE_DIR / "features_v2.parquet")
    F = V2.join(F2, how="left")
    F.to_parquet(CACHE_DIR / "features_v5.parquet")
    print("\n=== gbm_v5 (v2 + flow2) ===")
    train(F, "gbm_v5")
    Fl = pd.read_parquet(CACHE_DIR / "features_flow.parquet")
    F6 = F.join(Fl, how="left")
    F6.to_parquet(CACHE_DIR / "features_v6.parquet")
    print("\n=== gbm_v6 (v2 + flow + flow2) ===")
    train(F6, "gbm_v6")


In [ ]:
%%writefile src/features_rtm.py
"""Tabular features v7: RTMPose (Halpe-26) kinematics of the kicker and the keeper from
pose_rtm_{m,x}.parquet (same body features as features_v2, computed on the RTMPose COCO-17
keypoints, prefixed "r") plus FOOT ORIENTATION features from the big-toe / heel keypoints:

  kicker: plant-foot and kick-foot direction (toe - heel, unit vector) at several taus,
          projected on the kicker-left / forward axes ("disp" mirror tags), the signed foot angle
          relative to the forward axis and relative to the run-up direction;
  keeper: mean foot direction ("disp") and left-minus-right foot direction ("lr") at tau 0 / 0.16.

Joined with the v1 box features and the optical-flow features:
  features_v7.parquet = v1 + rtm pose + flow            -> gbm_v7
  features_v8.parquet = v2 (v1 + yolo pose) + rtm + flow -> gbm_v8
Run:  python src/features_rtm.py [--mode balanced|performance] [--no-train]
"""
import argparse
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import pyarrow  # noqa: F401
import numpy as np
import pandas as pd

from config import CACHE_DIR
from cv import load_private, load_public
from features_v1 import train
from features_v2 import KP_CONF, LANK, LHIP, RANK, RHIP, TAUS, keeper_features, kicker_features, mid, proj, sample_track

L_TOE, R_TOE, L_STOE, R_STOE, L_HEEL, R_HEEL = 20, 21, 22, 23, 24, 25
KP26 = 26


def sample_all(g: pd.DataFrame, W: int):
    """All 26 keypoints at TAUS (NaN below KP_CONF or when no frame is close enough)."""
    g = g.sort_values("t_from_end_s")
    t = g.t_from_end_s.values
    dt = np.median(np.diff(t)) if len(t) > 1 else 0.04
    tol = max(0.03, 0.6 * dt)
    kx = g[[f"kx{k}" for k in range(KP26)]].values / W
    ky = g[[f"ky{k}" for k in range(KP26)]].values / W
    kc = g[[f"kc{k}" for k in range(KP26)]].values
    kx[kc < KP_CONF] = np.nan
    ky[kc < KP_CONF] = np.nan
    out = {}
    for tau in TAUS:
        j = int(np.argmin(np.abs(t - tau)))
        out[tau] = np.stack([kx[j], ky[j]], 1) if abs(t[j] - tau) <= tol else np.full((KP26, 2), np.nan)
    return out


def foot_vec(P, side):
    """Unit toe-heel vector of one foot (None if missing). side: 'L' or 'R'."""
    toe, heel = (L_TOE, L_HEEL) if side == "L" else (R_TOE, R_HEEL)
    v = P[toe] - P[heel]
    if not np.all(np.isfinite(v)):
        return None, np.nan
    n = float(np.linalg.norm(v))
    if n < 1e-6:
        return None, np.nan
    return v / n, n


def signed_angle(v, l, u):
    return float(np.degrees(np.arctan2(v @ l, v @ u))) if v is not None and np.all(np.isfinite(v)) else np.nan


def kicker_foot_features(A, s, speeds, l, u):
    f = {}
    sL, sR = speeds["L"], speeds["R"]
    if not (np.isfinite(sL) and np.isfinite(sR)):
        for n in ("plantfoot0.0", "plantfoot0.08", "plantfoot0.16", "kickfoot0.0", "kickfoot0.04"):
            proj(None, l, u, f"rk_{n}", "disp", f)
            f[f"rk_{n}_len_I"] = f[f"rk_{n}_ang_N"] = np.nan
        f["rk_plantfoot_vs_runup_N"] = f["rk_feet_angle_diff_N"] = np.nan
        return f
    plant, kick = ("R", "L") if sR < sL else ("L", "R")
    hip0, hip16 = mid(A[0.0], LHIP, RHIP), mid(A[0.16], LHIP, RHIP)
    runup = hip0 - hip16 if np.all(np.isfinite(hip0)) and np.all(np.isfinite(hip16)) else None
    for name, side, tau in (("plantfoot0.0", plant, 0.0), ("plantfoot0.08", plant, 0.08), ("plantfoot0.16", plant, 0.16),
                            ("kickfoot0.0", kick, 0.0), ("kickfoot0.04", kick, 0.04)):
        v, n = foot_vec(A[tau], side)
        proj(v, l, u, f"rk_{name}", "disp", f)
        f[f"rk_{name}_len_I"] = n / s if np.isfinite(n) else np.nan
        f[f"rk_{name}_ang_N"] = signed_angle(v, l, u)
    vp, _ = foot_vec(A[0.0], plant)
    if vp is not None and runup is not None and np.linalg.norm(runup) > 1e-6:
        r = runup / np.linalg.norm(runup)
        f["rk_plantfoot_vs_runup_N"] = float(np.degrees(np.arctan2(vp @ np.array([r[1], -r[0]]), vp @ r)))
    else:
        f["rk_plantfoot_vs_runup_N"] = np.nan
    vk, _ = foot_vec(A[0.0], kick)
    f["rk_feet_angle_diff_N"] = signed_angle(vk, l, u) - signed_angle(vp, l, u) if (vk is not None and vp is not None) else np.nan
    # plant-foot position relative to the kick-foot and the hips using the toe (more stable than the ankle)
    toe_p, toe_k = (L_TOE if plant == "L" else R_TOE), (L_TOE if kick == "L" else R_TOE)
    proj(A[0.0][toe_k] - A[0.0][toe_p], l, u, "rk_toes0.0", "disp", f, s)
    proj(A[0.0][toe_p] - hip0, l, u, "rk_planttoe_hip", "disp", f, s)
    return f


def keeper_foot_features(A, s, l, u):
    f = {}
    for tau in (0.0, 0.16):
        vl, nl = foot_vec(A[tau], "L")
        vr, nr = foot_vec(A[tau], "R")
        both = vl is not None and vr is not None
        proj((vl + vr) / 2 if both else None, l, u, f"rg_feetdir{tau}", "disp", f)
        proj((vl - vr) if both else None, l, u, f"rg_feetdiff{tau}", "lr", f)
        f[f"rg_feet_angle_sum{tau}_N"] = (signed_angle(vl, l, u) + signed_angle(vr, l, u)) if both else np.nan
        f[f"rg_feet_angle_diff{tau}_I"] = (signed_angle(vl, l, u) - signed_angle(vr, l, u)) if both else np.nan
        # toes relative to ankles: weight shift forward/lateral
        proj(mid(A[tau], L_TOE, R_TOE) - mid(A[tau], LANK, RANK), l, u, f"rg_toes_ankles{tau}", "disp", f, s)
    return f


POSE_FILE = {"balanced": "m", "performance": "x", "avg": "a", "avg2": "a2", "avg3": "a3"}   # avg = pose_merge.py (m+x), avg2 = m+x+mf4, avg3 = m+x+mf4+xf
SFX = {"balanced": "", "performance": "x", "avg": "a", "avg2": "a2", "avg3": "a3"}


def force_kick_foot(speeds, pref_right):
    """FOOT_PREF=1: take the metadata's preferred foot as the kicking foot (the project PDF says it is guaranteed to
    be the foot used), keeping the two ankle-speed magnitudes so the speed-ratio feature survives: the kicking foot
    gets the larger of the two speeds, the plant foot the smaller (plant = slower ankle downstream)."""
    sL, sR = speeds["L"], speeds["R"]
    if not (np.isfinite(sL) and np.isfinite(sR)):
        sL, sR = 0.0, 1.0
    lo, hi = min(sL, sR), max(sL, sR)
    return {"L": lo, "R": hi} if pref_right else {"L": hi, "R": lo}


def pose_file(mode):
    """Named modes map to their table; any other mode is taken as the pose_rtm_<mode>.parquet suffix itself."""
    return POSE_FILE.get(mode, mode)


def table_sfx(mode):
    return SFX.get(mode, mode)


def build(mode="balanced"):
    P = pd.read_parquet(CACHE_DIR / f"pose_rtm_{pose_file(mode)}.parquet")
    RC = pd.read_csv(CACHE_DIR / "roles_clip.csv").set_index("clip")
    meta = pd.concat([load_public(), load_private()], ignore_index=True).set_index("clip")
    rows = {}
    for (clip, role), g in P.groupby(["clip", "role"]):
        rc = RC.loc[clip]
        if pd.isna(rc.u_x):
            continue
        W = int(rc.W)
        u = np.array([rc.u_x, rc.u_y])
        l = np.array([u[1], -u[0]])
        K, s, qual, speeds = sample_track(g, W)
        A = sample_all(g, W)
        f = rows.setdefault(clip, {})
        if role == "kicker" and os.environ.get("FOOT_PREF"):
            speeds = force_kick_foot(speeds, meta.loc[clip, "preferred_foot"] == "right")
        if role == "kicker":
            base = kicker_features(K, s, qual, speeds, l, u, meta.loc[clip, "preferred_foot"] == "right")
            f.update({"r" + k: v for k, v in base.items()})
            f.update(kicker_foot_features(A, s, speeds, l, u))
        else:
            base = keeper_features(K, s, qual, l, u)
            f.update({"r" + k: v for k, v in base.items()})
            f.update(keeper_foot_features(A, s, l, u))
    R = pd.DataFrame.from_dict(rows, orient="index")
    R.index.name = "clip"
    bad = [c for c in R.columns if not (c.endswith("_N") or c.endswith("_I"))]
    assert not bad, bad
    R.to_parquet(CACHE_DIR / f"features_rtm_{mode}.parquet")
    print(f"rtm features ({mode}): {R.shape}, clips {len(R)}, NaN share {R.isna().mean().mean():.3f}")
    return R


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--mode", default="balanced", help="balanced|performance|avg|avg2|avg3 or any pose_rtm_<mode> suffix")
    ap.add_argument("--no-train", action="store_true", help="build the feature tables only")
    ap.add_argument("--suffix", default="", help="run-name suffix")
    a = ap.parse_args()
    R = build(a.mode)
    sfx = table_sfx(a.mode)                              # table suffix: '' (m), 'x', 'a', ...
    V1 = pd.read_parquet(CACHE_DIR / "features_v1.parquet")
    V2 = pd.read_parquet(CACHE_DIR / "features_v2.parquet")
    Fl = pd.read_parquet(CACHE_DIR / "features_flow.parquet")
    F7 = V1.join(R, how="left").join(Fl, how="left")
    F7.to_parquet(CACHE_DIR / f"features_v7{sfx}.parquet")
    F8 = V2.join(R, how="left").join(Fl, how="left")
    F8.to_parquet(CACHE_DIR / f"features_v8{sfx}.parquet")
    print(f"wrote features_v7{sfx} {F7.shape}, features_v8{sfx} {F8.shape}")
    if a.no_train:
        sys.exit(0)
    print(f"\n=== gbm_v7{a.suffix} (v1 box + rtm pose + flow) {F7.shape} ===")
    train(F7, f"gbm_v7{a.suffix}")
    print(f"\n=== gbm_v8{a.suffix} (v2 box+yolo pose + rtm pose + flow) {F8.shape} ===")
    train(F8, f"gbm_v8{a.suffix}")


In [ ]:
%%writefile src/features_rtm2.py
"""Kicker features round 2 from the RTMPose Halpe-26 tracks (pose_rtm_{m,x}.parquet):
temporally smoothed keypoints (centred median over 3 frames), limb orientations (plant/kick foot, shank, thigh)
at more time points, angular velocities, kick-foot swing, knee angles, trunk lean / gaze / hip velocity over
time, arm positions. Side assignment (plant vs kick leg) is kinematic (slower ankle = plant), so these are
side-agnostic "disp"-type vectors (lat -> _N, fwd -> _I); signed angles -> _N; magnitudes / knee angles -> _I;
left-minus-right vectors -> "lr" (lat _I, fwd _N).

features_v9.parquet = features_v8 + rtm2  ->  gbm_v9k (kicker-only subset) / gbm_v9 (all)
Run:  python src/features_rtm2.py [--mode balanced|performance] [--no-train]
"""
import argparse
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import pyarrow  # noqa: F401
import numpy as np
import pandas as pd

from config import CACHE_DIR
from cv import load_private, load_public
from features_rtm import KP26, L_HEEL, L_TOE, R_HEEL, R_TOE, foot_vec, signed_angle
from features_v1 import train
from features_v2 import KP_CONF, LANK, LEAR, LEL, LHIP, LKNEE, LSH, LWR, NOSE, RANK, REAR, REL, RHIP, RKNEE, RSH, RWR, mid, proj
from gbm_subset import subset_columns

TAUS2 = [0.0, 0.04, 0.08, 0.12, 0.16, 0.20, 0.24, 0.30, 0.40]
HEAD, NECK = 17, 18


def sample_smooth(g: pd.DataFrame, W: int, win: int = 3):
    """All 26 keypoints at TAUS2 after a centred rolling median (win frames) per coordinate."""
    g = g.sort_values("t_from_end_s")
    t = g.t_from_end_s.values
    dt = np.median(np.diff(t)) if len(t) > 1 else 0.04
    tol = max(0.03, 0.6 * dt)
    kx = g[[f"kx{k}" for k in range(KP26)]].values / W
    ky = g[[f"ky{k}" for k in range(KP26)]].values / W
    kc = g[[f"kc{k}" for k in range(KP26)]].values
    kx[kc < KP_CONF] = np.nan
    ky[kc < KP_CONF] = np.nan
    if win > 1 and len(t) >= win:
        kx = pd.DataFrame(kx).rolling(win, center=True, min_periods=1).median().values
        ky = pd.DataFrame(ky).rolling(win, center=True, min_periods=1).median().values
    out = {}
    for tau in TAUS2:
        j = int(np.argmin(np.abs(t - tau)))
        out[tau] = np.stack([kx[j], ky[j]], 1) if abs(t[j] - tau) <= tol else np.full((KP26, 2), np.nan)
    # body scale (median torso length) and ankle speeds over the last 0.16 s (plant = slower ankle)
    sh = (np.stack([kx[:, LSH], ky[:, LSH]], 1) + np.stack([kx[:, RSH], ky[:, RSH]], 1)) / 2
    hp = (np.stack([kx[:, LHIP], ky[:, LHIP]], 1) + np.stack([kx[:, RHIP], ky[:, RHIP]], 1)) / 2
    torso = np.linalg.norm(sh - hp, axis=1)
    s = np.nanmedian(torso) if np.isfinite(torso).any() else np.nan
    if not np.isfinite(s) or s < 1e-3:
        s = 0.35 * np.nanmedian(g.box_h.values) / W
    m = t <= 0.16 + 1e-6
    speeds = {}
    for name, k in (("L", LANK), ("R", RANK)):
        p = np.stack([kx[m, k], ky[m, k]], 1)
        d = np.linalg.norm(np.diff(p, axis=0), axis=1)
        speeds[name] = np.nanmean(d) / s if np.isfinite(d).any() else np.nan
    return out, s, speeds


def unit(v):
    if v is None or not np.all(np.isfinite(v)):
        return None
    n = np.linalg.norm(v)
    return v / n if n > 1e-6 else None


def ang(P, a, b, c):
    """Angle at b (degrees) between b->a and b->c."""
    u_, v_ = P[a] - P[b], P[c] - P[b]
    if not (np.all(np.isfinite(u_)) and np.all(np.isfinite(v_))):
        return np.nan
    cs = u_ @ v_ / (np.linalg.norm(u_) * np.linalg.norm(v_) + 1e-9)
    return float(np.degrees(np.arccos(np.clip(cs, -1, 1))))


def kicker_features2(A, s, speeds, l, u):
    f = {}
    sL, sR = speeds["L"], speeds["R"]
    known = np.isfinite(sL) and np.isfinite(sR)
    plant = ("R" if sR < sL else "L") if known else None
    kick = ("L" if plant == "R" else "R") if known else None
    J = {"L": dict(toe=L_TOE, heel=L_HEEL, ank=LANK, knee=LKNEE, hip=LHIP), "R": dict(toe=R_TOE, heel=R_HEEL, ank=RANK, knee=RKNEE, hip=RHIP)}

    def limb(P, side, a, b):
        if side is None:
            return None
        return unit(P[J[side][a]] - P[J[side][b]])

    # limb orientations over time
    for tau in (0.0, 0.04, 0.08, 0.12, 0.16, 0.24):
        P = A[tau]
        for leg, side in (("plant", plant), ("kick", kick)):
            vf = foot_vec(P, side)[0] if side else None
            proj(vf, l, u, f"k2_{leg}foot{tau}", "disp", f)
            f[f"k2_{leg}foot{tau}_ang_N"] = signed_angle(vf, l, u)
            vs = limb(P, side, "ank", "knee")
            proj(vs, l, u, f"k2_{leg}shank{tau}", "disp", f)
            f[f"k2_{leg}shank{tau}_ang_N"] = signed_angle(vs, l, u)
            vt = limb(P, side, "knee", "hip")
            proj(vt, l, u, f"k2_{leg}thigh{tau}", "disp", f)
            f[f"k2_{leg}knee{tau}_I"] = ang(P, J[side]["hip"], J[side]["knee"], J[side]["ank"]) if side else np.nan
        # trunk, gaze, head over time
        proj(mid(P, LSH, RSH) - mid(P, LHIP, RHIP), l, u, f"k2_trunk{tau}", "disp", f, s)
        proj(P[NOSE] - mid(P, LEAR, REAR), l, u, f"k2_gaze{tau}", "disp", f, s)
        proj(P[HEAD] - mid(P, LHIP, RHIP), l, u, f"k2_head_hip{tau}", "disp", f, s)
        proj(P[LHIP] - P[RHIP], l, u, f"k2_hipline{tau}", "lr", f, s)
        proj(P[LSH] - P[RSH], l, u, f"k2_shline{tau}", "lr", f, s)
        f[f"k2_twist{tau}_N"] = f.get(f"k2_shline{tau}_fwd_N", np.nan) - f.get(f"k2_hipline{tau}_fwd_N", np.nan)
        # arms: wrists relative to shoulders (sum = disp, difference = lr)
        wl, wr = P[LWR] - P[LSH], P[RWR] - P[RSH]
        both = np.all(np.isfinite(wl)) and np.all(np.isfinite(wr))
        proj((wl + wr) / 2 if both else None, l, u, f"k2_arms_sum{tau}", "disp", f, s)
        proj((wl - wr) if both else None, l, u, f"k2_arms_diff{tau}", "lr", f, s)
        f[f"k2_elbow_dy{tau}_N"] = float(P[LEL][1] - P[REL][1]) / s if np.all(np.isfinite([P[LEL][1], P[REL][1]])) else np.nan
    # angular velocities and swing
    for leg in ("plant", "kick"):
        for a_, b_ in ((0.0, 0.08), (0.0, 0.16), (0.08, 0.16)):
            f[f"k2_{leg}foot_dang{a_}_{b_}_N"] = f.get(f"k2_{leg}foot{a_}_ang_N", np.nan) - f.get(f"k2_{leg}foot{b_}_ang_N", np.nan)
            f[f"k2_{leg}shank_dang{a_}_{b_}_N"] = f.get(f"k2_{leg}shank{a_}_ang_N", np.nan) - f.get(f"k2_{leg}shank{b_}_ang_N", np.nan)
    if known:
        for a_, b_ in ((0.0, 0.04), (0.0, 0.08), (0.04, 0.12)):
            sw = A[a_][J[kick]["toe"]] - A[b_][J[kick]["toe"]]
            proj(sw, l, u, f"k2_swing{a_}_{b_}", "disp", f, s)
            f[f"k2_swing{a_}_{b_}_ang_N"] = signed_angle(unit(sw), l, u)
            swa = A[a_][J[kick]["ank"]] - A[b_][J[kick]["ank"]]
            proj(swa, l, u, f"k2_swingank{a_}_{b_}", "disp", f, s)
        vp = foot_vec(A[0.0], plant)[0]
        sw = unit(A[0.0][J[kick]["toe"]] - A[0.08][J[kick]["toe"]])
        f["k2_swing_vs_plantfoot_N"] = (signed_angle(sw, l, u) - signed_angle(vp, l, u)) if (sw is not None and vp is not None) else np.nan
        # kick toe relative to plant toe / plant heel at contact and 0.04 before
        for tau in (0.0, 0.04):
            proj(A[tau][J[kick]["toe"]] - A[tau][J[plant]["toe"]], l, u, f"k2_toes{tau}", "disp", f, s)
            proj(A[tau][J[kick]["ank"]] - A[tau][J[plant]["heel"]], l, u, f"k2_kickank_plantheel{tau}", "disp", f, s)
        f["k2_knee_diff0.0_I"] = f.get("k2_plantknee0.0_I", np.nan) - f.get("k2_kickknee0.0_I", np.nan)
        f["k2_plant_right_N"] = 1.0 if plant == "R" else -1.0
    else:
        for n in ("k2_swing0.0_0.04", "k2_swing0.0_0.08", "k2_swing0.04_0.12", "k2_swingank0.0_0.04", "k2_swingank0.0_0.08",
                  "k2_swingank0.04_0.12", "k2_toes0.0", "k2_toes0.04", "k2_kickank_plantheel0.0", "k2_kickank_plantheel0.04"):
            proj(None, l, u, n, "disp", f)
        for n in ("k2_swing0.0_0.04_ang_N", "k2_swing0.0_0.08_ang_N", "k2_swing0.04_0.12_ang_N", "k2_swing_vs_plantfoot_N",
                  "k2_knee_diff0.0_I", "k2_plant_right_N"):
            f[n] = np.nan
    # hip centre velocity / trajectory and height change
    h = {tau: mid(A[tau], LHIP, RHIP) for tau in TAUS2}
    for a_, b_ in ((0.0, 0.04), (0.0, 0.08), (0.08, 0.16), (0.16, 0.30), (0.0, 0.40)):
        proj(h[a_] - h[b_], l, u, f"k2_hipvel{a_}_{b_}", "disp", f, s)
    f["k2_hip_dy0.0_0.16_I"] = float(h[0.0][1] - h[0.16][1]) / s if np.all(np.isfinite([h[0.0][1], h[0.16][1]])) else np.nan
    f["k2_trunk_dlat0.0_0.08_N"] = f.get("k2_trunk0.0_lat_N", np.nan) - f.get("k2_trunk0.08_lat_N", np.nan)
    f["k2_gaze_dlat0.0_0.16_N"] = f.get("k2_gaze0.0_lat_N", np.nan) - f.get("k2_gaze0.16_lat_N", np.nan)
    return f


def build(mode="balanced"):
    from features_rtm import pose_file
    P = pd.read_parquet(CACHE_DIR / f"pose_rtm_{pose_file(mode)}.parquet")
    P = P[P.role == "kicker"]
    RC = pd.read_csv(CACHE_DIR / "roles_clip.csv").set_index("clip")
    meta = None
    if os.environ.get("FOOT_PREF"):
        from cv import load_private, load_public
        from features_rtm import force_kick_foot
        meta = pd.concat([load_public(), load_private()], ignore_index=True).set_index("clip")
    rows = {}
    for clip, g in P.groupby("clip"):
        rc = RC.loc[clip]
        if pd.isna(rc.u_x):
            continue
        u = np.array([rc.u_x, rc.u_y])
        l = np.array([u[1], -u[0]])
        A, s, speeds = sample_smooth(g, int(rc.W))
        if meta is not None:
            speeds = force_kick_foot(speeds, meta.loc[clip, "preferred_foot"] == "right")
        rows[clip] = kicker_features2(A, s, speeds, l, u)
    R = pd.DataFrame.from_dict(rows, orient="index")
    R.index.name = "clip"
    bad = [c for c in R.columns if not (c.endswith("_N") or c.endswith("_I"))]
    assert not bad, bad
    R.to_parquet(CACHE_DIR / f"features_rtm2_{mode}.parquet")
    print(f"rtm2 kicker features ({mode}): {R.shape}, NaN share {R.isna().mean().mean():.3f}")
    return R


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    from features_rtm import table_sfx
    ap.add_argument("--mode", default="balanced", help="balanced|performance|avg|avg2|avg3 or any pose_rtm_<mode> suffix")
    ap.add_argument("--no-train", action="store_true")
    ap.add_argument("--kick-only", action="store_true", help="train only the kicker-only head gbm_v9k<suffix>")
    ap.add_argument("--suffix", default="")
    a = ap.parse_args()
    R = build(a.mode)
    if a.no_train:
        sys.exit(0)
    sfx = table_sfx(a.mode)                              # table suffix: '' (m), 'x', 'a', ...
    V8 = pd.read_parquet(CACHE_DIR / f"features_v8{sfx}.parquet")
    F9 = V8.join(R, how="left")
    F9.to_parquet(CACHE_DIR / f"features_v9{sfx}.parquet")
    cols = subset_columns(F9.columns, "kicker")
    print(f"\n=== gbm_v9k{a.suffix} (kicker-only: v8 kicker cols + rtm2) {len(cols)} cols ===")
    train(F9[cols], f"gbm_v9k{a.suffix}")
    if a.kick_only:
        sys.exit(0)
    print(f"\n=== gbm_v9{a.suffix} (all) {F9.shape[1]} cols ===")
    train(F9, f"gbm_v9{a.suffix}")


In [ ]:
%%writefile src/features_rtm2g.py
"""Keeper features round 2 from the RTMPose Halpe-26 tracks (smoothed), analogous to features_rtm2 for the kicker:
displacements of hip / shoulder / feet centres over 9 taus (origin = position 0.40 s before the end), trunk and
head lean, foot directions (mean = disp, L-R = lr), knee angles, arms, hip velocities, centre of mass over the
feet, and the onset time / sign of the lateral hip motion.

features_v11 = features_v10 + rtm2 keeper features (prefix g2_ / xg2_ for the x estimator)
Run:  python src/features_rtm2g.py        -> gbm_v11g, et_v11g (keeper-only subsets)
"""
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import pyarrow  # noqa: F401
import numpy as np
import pandas as pd

from config import CACHE_DIR
from features_rtm import foot_vec, signed_angle
from features_rtm2 import HEAD, TAUS2, ang, sample_smooth, unit
from features_v1 import train
from features_v2 import LANK, LEAR, LHIP, LKNEE, LSH, LWR, NOSE, RANK, REAR, RHIP, RKNEE, RSH, RWR, mid, proj
from gbm_subset import subset_columns


def keeper_features2(A, s, l, u, pre="g2_"):
    f = {}
    o = mid(A[0.40], LHIP, RHIP)
    if not np.all(np.isfinite(o)):
        o = mid(A[0.30], LHIP, RHIP)
    for tau in TAUS2:
        P = A[tau]
        hip, sh, feet = mid(P, LHIP, RHIP), mid(P, LSH, RSH), mid(P, LANK, RANK)
        proj(hip - o, l, u, f"{pre}hip{tau}", "disp", f, s)
        proj(sh - o, l, u, f"{pre}sh{tau}", "disp", f, s)
        proj(feet - o, l, u, f"{pre}feet{tau}", "disp", f, s)
        proj(sh - hip, l, u, f"{pre}trunk{tau}", "disp", f, s)
        proj(P[HEAD] - hip, l, u, f"{pre}head{tau}", "disp", f, s)
        proj(hip - feet, l, u, f"{pre}com{tau}", "disp", f, s)
        proj(P[NOSE] - mid(P, LEAR, REAR), l, u, f"{pre}gaze{tau}", "disp", f, s)
        proj(P[LANK] - P[RANK], l, u, f"{pre}ankles{tau}", "lr", f, s)
        proj(P[LWR] - P[RWR], l, u, f"{pre}wrists{tau}", "lr", f, s)
        proj(P[LHIP] - P[RHIP], l, u, f"{pre}hipline{tau}", "lr", f, s)
        proj(P[LSH] - P[RSH], l, u, f"{pre}shline{tau}", "lr", f, s)
        wl, wr = P[LWR] - P[LSH], P[RWR] - P[RSH]
        both = np.all(np.isfinite(wl)) and np.all(np.isfinite(wr))
        proj((wl + wr) / 2 if both else None, l, u, f"{pre}arms{tau}", "disp", f, s)
        vl, vr = foot_vec(P, "L")[0], foot_vec(P, "R")[0]
        fb = vl is not None and vr is not None
        proj((vl + vr) / 2 if fb else None, l, u, f"{pre}footdir{tau}", "disp", f)
        proj((vl - vr) if fb else None, l, u, f"{pre}footdiff{tau}", "lr", f)
        kl, kr = ang(P, LHIP, LKNEE, LANK), ang(P, RHIP, RKNEE, RANK)
        f[f"{pre}knee_sum{tau}_I"] = kl + kr
        f[f"{pre}knee_diff{tau}_N"] = kl - kr
        f[f"{pre}feet_spread{tau}_I"] = float(np.linalg.norm(P[LANK] - P[RANK])) / s if np.all(np.isfinite(P[LANK] - P[RANK])) else np.nan
        f[f"{pre}wrist_dy{tau}_N"] = float(P[LWR][1] - P[RWR][1]) / s if np.all(np.isfinite([P[LWR][1], P[RWR][1]])) else np.nan
    # hip velocities between consecutive taus and the lateral-motion onset
    hips = {tau: mid(A[tau], LHIP, RHIP) for tau in TAUS2}
    lat_v = {}
    for a_, b_ in zip(TAUS2[:-1], TAUS2[1:]):
        v = (hips[a_] - hips[b_]) / (b_ - a_)
        proj(v, l, u, f"{pre}hipvel{a_}_{b_}", "disp", f, s)
        lat_v[a_] = float(v @ l) / s if np.all(np.isfinite(v)) else np.nan
    onset, sign = np.nan, np.nan
    for tau in sorted(lat_v, reverse=True):            # from early (0.30) to late (0.0)
        if np.isfinite(lat_v[tau]) and abs(lat_v[tau]) > 1.0:   # > 1 body-scale per second
            onset, sign = tau, float(np.sign(lat_v[tau]))
            break
    f[f"{pre}onset_tau_I"], f[f"{pre}onset_sign_N"] = onset, sign
    f[f"{pre}max_abs_latvel_I"] = float(np.nanmax(np.abs(list(lat_v.values())))) if np.isfinite(list(lat_v.values())).any() else np.nan
    return f


def build(mode):
    P = pd.read_parquet(CACHE_DIR / f"pose_rtm_{'m' if mode == 'balanced' else 'x'}.parquet")
    P = P[P.role == "keeper"]
    RC = pd.read_csv(CACHE_DIR / "roles_clip.csv").set_index("clip")
    rows = {}
    pre = "g2_" if mode == "balanced" else "xg2_"
    for clip, g in P.groupby("clip"):
        rc = RC.loc[clip]
        if pd.isna(rc.u_x):
            continue
        u = np.array([rc.u_x, rc.u_y])
        l = np.array([u[1], -u[0]])
        A, s, _ = sample_smooth(g, int(rc.W))
        rows[clip] = keeper_features2(A, s, l, u, pre)
    R = pd.DataFrame.from_dict(rows, orient="index")
    R.index.name = "clip"
    bad = [c for c in R.columns if not (c.endswith("_N") or c.endswith("_I"))]
    assert not bad, bad
    print(f"rtm2 keeper features ({mode}): {R.shape}, NaN share {R.isna().mean().mean():.3f}")
    return R


if __name__ == "__main__":
    Rm, Rx = build("balanced"), build("performance")
    V10 = pd.read_parquet(CACHE_DIR / "features_v10.parquet")
    F11 = V10.join(Rm, how="left").join(Rx, how="left")
    F11.to_parquet(CACHE_DIR / "features_v11.parquet")
    print(f"features v11: {F11.shape}")
    cols = subset_columns(F11.columns, "keeper")
    print(f"keeper subset {len(cols)} cols")
    print("\n=== gbm_v11g ===")
    train(F11[cols], "gbm_v11g")


In [ ]:
%%writefile src/features_rtm3.py
"""Kicker features round 3: EVENT-ALIGNED features. The clips end at a human-marked contact frame (one frame = 17-67
ms), so time-indexed features blur. Here the plant-foot touchdown is detected per clip (last frame before contact at
which the plant ankle's speed drops below a threshold and stays low) and the kicker's configuration is described
at touchdown and relative to it: time to contact, plant-foot / shank / trunk orientation at touchdown, kick foot and
hips relative to the plant toe at touchdown, kick-foot velocity at touchdown, run-up direction over the window and
its angle to the kick-foot swing, plus the hip-speed profile (max, at touchdown, at contact).

features_v12 = features_v9a (averaged RTMPose, round 1+2) + rtm3  ->  gbm_v12k / et_v12k (kicker-only)
Run:  python src/features_rtm3.py [--mode avg|balanced|performance] [--no-train]
"""
import argparse
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import pyarrow  # noqa: F401
import numpy as np
import pandas as pd

from config import CACHE_DIR
from features_rtm import KP26, L_HEEL, L_TOE, POSE_FILE, R_HEEL, R_TOE, SFX, signed_angle
from features_rtm2 import ang, unit
from features_v1 import train
from features_v2 import KP_CONF, LANK, LHIP, LKNEE, LSH, RANK, RHIP, RKNEE, RSH, mid, proj
from gbm_subset import subset_columns

J = {"L": dict(toe=L_TOE, heel=L_HEEL, ank=LANK, knee=LKNEE, hip=LHIP), "R": dict(toe=R_TOE, heel=R_HEEL, ank=RANK, knee=RKNEE, hip=RHIP)}
SPEED_TD = 1.5          # body scales per second: below this the plant ankle is "down"


def smooth_tracks(g: pd.DataFrame, W: int, win: int = 3):
    """Full per-frame smoothed keypoints [n, 26, 2] (units of W), times from end (ascending = later), body scale."""
    g = g.sort_values("t_from_end_s", ascending=False)        # chronological
    t = g.t_from_end_s.values
    kx = g[[f"kx{k}" for k in range(KP26)]].values / W
    ky = g[[f"ky{k}" for k in range(KP26)]].values / W
    kc = g[[f"kc{k}" for k in range(KP26)]].values
    kx[kc < KP_CONF] = np.nan
    ky[kc < KP_CONF] = np.nan
    if win > 1 and len(t) >= win:
        kx = pd.DataFrame(kx).rolling(win, center=True, min_periods=1).median().values
        ky = pd.DataFrame(ky).rolling(win, center=True, min_periods=1).median().values
    P = np.stack([kx, ky], 2)
    sh = (P[:, LSH] + P[:, RSH]) / 2
    hp = (P[:, LHIP] + P[:, RHIP]) / 2
    torso = np.linalg.norm(sh - hp, axis=1)
    s = np.nanmedian(torso) if np.isfinite(torso).any() else np.nan
    if not np.isfinite(s) or s < 1e-3:
        s = 0.35 * np.nanmedian(g.box_h.values) / W
    return P, t, s


def speeds(P, t, k, s):
    """Speed (body scales / s) of joint k between consecutive frames, aligned to the later frame."""
    d = np.linalg.norm(np.diff(P[:, k], axis=0), axis=1)
    dt = -np.diff(t)
    v = np.full(len(t), np.nan)
    v[1:] = d / np.maximum(dt, 1e-3) / s
    return v


def kicker_features3(P, t, s, l, u):
    f = {}
    n = len(t)
    if n < 4:
        return f
    vL, vR = speeds(P, t, LANK, s), speeds(P, t, RANK, s)
    last = t <= 0.16 + 1e-6
    sL, sR = np.nanmean(vL[last]), np.nanmean(vR[last])
    if not (np.isfinite(sL) and np.isfinite(sR)):
        return f
    plant, kick = ("R", "L") if sR < sL else ("L", "R")
    vp = vL if plant == "L" else vR
    vk = vL if kick == "L" else vR
    # touchdown: last index at which the plant ankle speed is below the threshold, searching back from the end
    td = None
    for i in range(n - 1, 0, -1):
        if np.isfinite(vp[i]) and vp[i] < SPEED_TD:
            td = i
            # extend back while still slow
            while td - 1 > 0 and np.isfinite(vp[td - 1]) and vp[td - 1] < SPEED_TD:
                td -= 1
            break
    f["k3_td_found_I"] = 1.0 if td is not None else 0.0
    if td is None:
        return f
    f["k3_t_td_I"] = float(t[td])                       # seconds before contact at which the plant foot settled
    f["k3_plant_speed_min_I"] = float(np.nanmin(vp[td:]))
    f["k3_kick_speed_td_I"] = float(vk[td]) if np.isfinite(vk[td]) else np.nan
    f["k3_kick_speed_max_I"] = float(np.nanmax(vk[td:])) if np.isfinite(vk[td:]).any() else np.nan
    Ptd, P0 = P[td], P[-1]
    jp, jk = J[plant], J[kick]
    # orientations at touchdown
    vf = unit(Ptd[jp["toe"]] - Ptd[jp["heel"]])
    proj(vf, l, u, "k3_plantfoot_td", "disp", f)
    f["k3_plantfoot_td_ang_N"] = signed_angle(vf, l, u)
    vs = unit(Ptd[jp["ank"]] - Ptd[jp["knee"]])
    proj(vs, l, u, "k3_plantshank_td", "disp", f)
    f["k3_plantshank_td_ang_N"] = signed_angle(vs, l, u)
    vkf = unit(Ptd[jk["toe"]] - Ptd[jk["heel"]])
    proj(vkf, l, u, "k3_kickfoot_td", "disp", f)
    f["k3_kickfoot_td_ang_N"] = signed_angle(vkf, l, u)
    proj(mid(Ptd, LSH, RSH) - mid(Ptd, LHIP, RHIP), l, u, "k3_trunk_td", "disp", f, s)
    proj(Ptd[LHIP] - Ptd[RHIP], l, u, "k3_hipline_td", "lr", f, s)
    proj(Ptd[LSH] - Ptd[RSH], l, u, "k3_shline_td", "lr", f, s)
    f["k3_twist_td_N"] = f.get("k3_shline_td_fwd_N", np.nan) - f.get("k3_hipline_td_fwd_N", np.nan)
    f["k3_plantknee_td_I"] = ang(Ptd, jp["hip"], jp["knee"], jp["ank"])
    f["k3_kickknee_td_I"] = ang(Ptd, jk["hip"], jk["knee"], jk["ank"])
    # positions relative to the plant toe at touchdown (where the plant foot is relative to the body / ball)
    proj(Ptd[jk["ank"]] - Ptd[jp["toe"]], l, u, "k3_kickank_planttoe_td", "disp", f, s)
    proj(mid(Ptd, LHIP, RHIP) - Ptd[jp["toe"]], l, u, "k3_hip_planttoe_td", "disp", f, s)
    proj(mid(P0, LHIP, RHIP) - Ptd[jp["toe"]], l, u, "k3_hip0_planttoe_td", "disp", f, s)
    proj(P0[jk["toe"]] - Ptd[jp["toe"]], l, u, "k3_kicktoe0_planttoe_td", "disp", f, s)
    # kick-foot swing from touchdown to contact and its angle vs the plant-foot direction / run-up
    sw = P0[jk["toe"]] - Ptd[jk["toe"]]
    proj(sw, l, u, "k3_swing_td", "disp", f, s)
    f["k3_swing_td_ang_N"] = signed_angle(unit(sw), l, u)
    f["k3_swing_vs_plantfoot_N"] = (signed_angle(unit(sw), l, u) - signed_angle(vf, l, u)) if (unit(sw) is not None and vf is not None) else np.nan
    hips = (P[:, LHIP] + P[:, RHIP]) / 2
    run = hips[td] - hips[0]
    proj(run, l, u, "k3_runup", "disp", f, s)
    f["k3_runup_ang_N"] = signed_angle(unit(run), l, u)
    f["k3_swing_vs_runup_N"] = (signed_angle(unit(sw), l, u) - signed_angle(unit(run), l, u)) if (unit(sw) is not None and unit(run) is not None) else np.nan
    # hip speed profile
    vh = np.full(n, np.nan)
    d = np.linalg.norm(np.diff(hips, axis=0), axis=1)
    vh[1:] = d / np.maximum(-np.diff(t), 1e-3) / s
    f["k3_hipspeed_max_I"] = float(np.nanmax(vh)) if np.isfinite(vh).any() else np.nan
    f["k3_hipspeed_td_I"] = float(vh[td]) if np.isfinite(vh[td]) else np.nan
    f["k3_hipspeed_0_I"] = float(vh[-1]) if np.isfinite(vh[-1]) else np.nan
    f["k3_hipspeed_decel_I"] = f["k3_hipspeed_max_I"] - f["k3_hipspeed_0_I"] if np.isfinite(f["k3_hipspeed_max_I"]) and np.isfinite(f["k3_hipspeed_0_I"]) else np.nan
    # lateral hip drift between touchdown and contact (body lean into the kick)
    proj(hips[-1] - hips[td], l, u, "k3_hipdrift_td0", "disp", f, s)
    return f


def build(mode):
    P = pd.read_parquet(CACHE_DIR / f"pose_rtm_{POSE_FILE[mode]}.parquet")
    P = P[P.role == "kicker"]
    RC = pd.read_csv(CACHE_DIR / "roles_clip.csv").set_index("clip")
    rows = {}
    for clip, g in P.groupby("clip"):
        rc = RC.loc[clip]
        if pd.isna(rc.u_x):
            continue
        u = np.array([rc.u_x, rc.u_y])
        l = np.array([u[1], -u[0]])
        Pk, t, s = smooth_tracks(g, int(rc.W))
        rows[clip] = kicker_features3(Pk, t, s, l, u)
    R = pd.DataFrame.from_dict(rows, orient="index")
    R.index.name = "clip"
    bad = [c for c in R.columns if not (c.endswith("_N") or c.endswith("_I"))]
    assert not bad, bad
    R.to_parquet(CACHE_DIR / f"features_rtm3_{mode}.parquet")
    print(f"rtm3 event-aligned kicker features ({mode}): {R.shape}, touchdown found {R['k3_td_found_I'].mean():.3f}, "
          f"t_td median {R['k3_t_td_I'].median():.3f} s, NaN share {R.isna().mean().mean():.3f}")
    return R


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--mode", default="avg", choices=list(POSE_FILE))
    ap.add_argument("--no-train", action="store_true")
    a = ap.parse_args()
    R = build(a.mode)
    if a.no_train:
        sys.exit(0)
    sfx = SFX[a.mode]
    V9 = pd.read_parquet(CACHE_DIR / f"features_v9{sfx}.parquet")
    F12 = V9.join(R, how="left")
    F12.to_parquet(CACHE_DIR / f"features_v12{sfx}.parquet")
    cols = subset_columns(F12.columns, "kicker")
    print(f"\n=== gbm_v12k{sfx} (kicker-only, {len(cols)} cols) ===")
    train(F12[cols], f"gbm_v12k{sfx}")


In [ ]:
%%writefile src/features_union.py
"""features_v10 = features_v9 (RTMPose-m based) + the RTMPose-x based pose features (rtm + rtm2, columns prefixed
'x') so one model sees both pose estimators (agreement between them is informative). Trains kicker-only heads.

Run:  python src/features_union.py
"""
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import pyarrow  # noqa: F401
import pandas as pd

from config import CACHE_DIR
from features_v1 import train
from gbm_subset import subset_columns

if __name__ == "__main__":
    V9 = pd.read_parquet(CACHE_DIR / "features_v9.parquet")
    RX = pd.read_parquet(CACHE_DIR / "features_rtm_performance.parquet")
    R2X = pd.read_parquet(CACHE_DIR / "features_rtm2_performance.parquet")
    X = RX.join(R2X, how="outer")
    X.columns = ["x" + c for c in X.columns]          # xrk_*, xrg_*, xk2_*  (tags _N/_I preserved at the end)
    F10 = V9.join(X, how="left")
    F10.to_parquet(CACHE_DIR / "features_v10.parquet")
    print(f"features v10: {F10.shape} (v9 {V9.shape[1]} + x-pose {X.shape[1]})")
    cols = subset_columns(F10.columns, "kicker")
    cols = [c for c in cols if not c.startswith(("xrg_", "xg_"))]
    print(f"kicker subset {len(cols)} cols")
    print("\n=== gbm_v10k ===")
    train(F10[cols], "gbm_v10k")


In [ ]:
%%writefile src/features_union2.py
"""Union of pose-feature tables from several keypoint estimators: features_v9<base> + the rtm/rtm2 feature columns
computed from other pose tables (prefixed with the table's suffix, e.g. 'xa3'), so one model sees several
estimates of the same quantities (their agreement is informative; features_v10 = v9 + x copies gave +2 points).
Trains the kicker-only head gbm_v10k<base>_<extras>.

Run:  python src/features_union2.py --base a2b --extra a3 performance balanced
      (extras are features_rtm_<extra> / features_rtm2_<extra> table names)
"""
import argparse
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import pyarrow  # noqa: F401
import pandas as pd

from config import CACHE_DIR
from features_rtm import table_sfx
from features_v1 import train
from gbm_subset import subset_columns

if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--base", default="a2b")
    ap.add_argument("--extra", nargs="+", default=["a3"])
    ap.add_argument("--run", default=None)
    a = ap.parse_args()
    bsfx = table_sfx(a.base)
    F = pd.read_parquet(CACHE_DIR / f"features_v9{bsfx}.parquet")
    n0 = F.shape[1]
    for e in a.extra:
        R = pd.read_parquet(CACHE_DIR / f"features_rtm_{e}.parquet")
        R2 = pd.read_parquet(CACHE_DIR / f"features_rtm2_{e}.parquet")
        X = R.join(R2, how="outer")
        X = X[[c for c in X.columns if not c.startswith(("rg_", "g_"))]]       # kicker columns only
        X.columns = ["x" + table_sfx(e) + c for c in X.columns]                 # keeps the _N/_I tag at the end
        F = F.join(X, how="left")
    name = a.run or f"gbm_v10k{bsfx}_" + "_".join(table_sfx(e) for e in a.extra)
    F.to_parquet(CACHE_DIR / f"features_union_{name}.parquet")
    cols = subset_columns(F.columns, "kicker")
    print(f"union: base v9{bsfx} {n0} cols + extras -> {F.shape[1]} cols; kicker subset {len(cols)}")
    print(f"\n=== {name} ===")
    train(F[cols], name)


In [ ]:
%%writefile src/features_v1.py
"""Tabular features v1 (boxes only, no keypoints) + LightGBM with feature-level
mirror augmentation, flip-TTA and the three sanity tests.

Geometry: all coordinates are made pixel-isotropic in units of the frame width
(X = x/W, Y = y/W). u = unit vector kicker -> keeper, kicker-right r = (-u_y, u_x),
kicker-left l = -r. "Lateral" = projection on l (positive = kicker's left).
Every feature name ends in `_N` (negated under mirroring) or `_I` (invariant).

Inputs:  roles_clip.csv, roles_frames.parquet, tracks.parquet (from roles.py)
Outputs: CACHE_DIR/features_v1.parquet, RUNS_DIR/gbm_v1/{oof,test}_*.npy, submission.csv
Run:     python src/features_v1.py [--no-train]
"""
import argparse
import json
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import numpy as np
import pandas as pd

from config import CACHE_DIR, CLASSES, FLIP_MAP, RUNS_DIR, SEED, TARGETS
from cv import get_folds, load_private, load_public, score_report, write_submission

OFFS = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.8, 1.0]
LAT_OFFS = [0.1, 0.2, 0.3, 0.5]


def centre_at(tr: pd.DataFrame, t: float, W: int, tol: float = 0.06):
    """(X, Y, w, h) of the role box at sampled offset t (units of W), or None."""
    if len(tr) == 0:
        return None
    j = np.argmin(np.abs(tr.t_from_end_s.values - t))
    if abs(tr.t_from_end_s.values[j] - t) > tol:
        return None
    b = tr.iloc[j]
    return np.array([(b.x1 + b.x2) / 2 / W, (b.y1 + b.y2) / 2 / W, (b.x2 - b.x1) / W, (b.y2 - b.y1) / W])


def pan_estimate(tracks: pd.DataFrame, exclude: set, t: float, W: int):
    """Median displacement (t -> 0) of all other tracked people = camera pan proxy (units of W)."""
    d = []
    for tid, g in tracks[~tracks.track.isin(exclude)].groupby("track"):
        a, b = centre_at(g, t, W), centre_at(g, 0.0, W)
        if a is not None and b is not None:
            d.append(b[:2] - a[:2])
    if len(d) < 2:
        return None
    return np.median(np.array(d), axis=0)


def clip_features(rc, rf, tracks, W, H):
    f = {}
    found_k, found_g = bool(rc.kicker_found), bool(rc.keeper_found)
    f["kicker_found_I"], f["keeper_found_I"] = int(found_k), int(found_g)
    f["n_cand_I"] = rc.n_cand if pd.notna(rc.n_cand) else np.nan
    f["p_kicker_I"] = rc.p_kicker if pd.notna(rc.p_kicker) else np.nan
    f["p_keeper_I"] = rc.p_keeper if pd.notna(rc.p_keeper) else np.nan
    if not (found_k and found_g) or pd.isna(rc.u_x):
        return f
    u = np.array([rc.u_x, rc.u_y])
    l = np.array([u[1], -u[0]])          # kicker-left
    f["u_x_N"], f["u_y_I"] = u[0], u[1]   # u_x flips sign under mirroring, u_y does not
    f["dist_kk_I"] = rc.dist_kk
    kt = rf[rf.role == "kicker"]
    gt = rf[rf.role == "keeper"]
    # role track ids (to exclude from the pan estimate): nearest track to the role boxes at t=0
    exclude = set()
    for role_tr in (kt, gt):
        b0 = centre_at(role_tr, 0.0, W)
        if b0 is not None:
            tt = tracks[np.isclose(tracks.t_from_end_s, role_tr.t_from_end_s.min(), atol=1e-6)]
            if len(tt):
                c = np.c_[(tt.x1 + tt.x2) / 2 / W, (tt.y1 + tt.y2) / 2 / W]
                exclude.add(int(tt.track.values[np.argmin(np.hypot(c[:, 0] - b0[0], c[:, 1] - b0[1]))]))
    g0, k0 = centre_at(gt, 0.0, W), centre_at(kt, 0.0, W)
    gh = g0[3] if g0 is not None else rc.gh * H / W
    kh = k0[3] if k0 is not None else rc.kh * H / W
    f["keeper_h_I"], f["kicker_h_I"] = gh, kh
    # ---- keeper kinematics
    if g0 is not None:
        f["gk_aspect0_I"] = g0[2] / max(g0[3], 1e-6)
        for t in LAT_OFFS:
            gt_ = centre_at(gt, t, W)
            if gt_ is None:
                f[f"gk_lat_{t}_N"] = f[f"gk_fwd_{t}_I"] = f[f"gk_latc_{t}_N"] = np.nan
                f[f"gk_wchg_{t}_I"] = f[f"gk_hchg_{t}_I"] = np.nan
                continue
            d = g0[:2] - gt_[:2]
            f[f"gk_lat_{t}_N"] = d @ l / gh
            f[f"gk_fwd_{t}_I"] = d @ u / gh
            pan = pan_estimate(tracks, exclude, t, W)
            f[f"gk_latc_{t}_N"] = ((d - pan) @ l / gh) if pan is not None else f[f"gk_lat_{t}_N"]
            f[f"gk_wchg_{t}_I"] = g0[2] / max(gt_[2], 1e-6) - 1
            f[f"gk_hchg_{t}_I"] = g0[3] / max(gt_[3], 1e-6) - 1
        # velocity / acceleration over the last 0.2 s (per second, in keeper heights)
        g1, g2 = centre_at(gt, 0.1, W), centre_at(gt, 0.2, W)
        if g1 is not None:
            f["gk_vel_N"] = ((g0[:2] - g1[:2]) @ l) / gh / 0.1
            if g2 is not None:
                f["gk_acc_N"] = (((g0[:2] - g1[:2]) - (g1[:2] - g2[:2])) @ l) / gh / 0.01
        # early commit: largest offset at which the keeper had not yet moved laterally > 0.15 gh
        moved = [(t, abs(f.get(f"gk_lat_{t}_N", np.nan))) for t in LAT_OFFS]
        f["gk_commit_I"] = next((t for t, m in reversed(moved) if not np.isnan(m) and m < 0.15), 0.0)
        f["gk_absmot_I"] = abs(f.get("gk_lat_0.3_N", np.nan))
    # keeper offset along the goal line relative to the kicker-goal axis, at the ref frame
    if k0 is not None and g0 is not None:
        f["gk_offset_N"] = ((g0[:2] - k0[:2]) @ l) / max(np.linalg.norm(g0[:2] - k0[:2]), 1e-6)
        f["kk_dist0_I"] = np.linalg.norm(g0[:2] - k0[:2])
    # ---- kicker approach
    if k0 is not None:
        for t in (0.2, 0.5, 1.0):
            kt_ = centre_at(kt, t, W, tol=0.12)
            if kt_ is None:
                f[f"k_lat_{t}_N"] = f[f"k_fwd_{t}_I"] = f[f"k_ang_{t}_N"] = np.nan
                continue
            d = k0[:2] - kt_[:2]
            f[f"k_lat_{t}_N"] = d @ l / kh
            f[f"k_fwd_{t}_I"] = d @ u / kh
            f[f"k_ang_{t}_N"] = np.arctan2(d @ l, d @ u)
        k1 = centre_at(kt, 0.1, W)
        if k1 is not None:
            f["k_laststep_lat_N"] = ((k0[:2] - k1[:2]) @ l) / kh
            f["k_laststep_fwd_I"] = ((k0[:2] - k1[:2]) @ u) / kh
            f["k_hchg_I"] = k0[3] / max(k1[3], 1e-6) - 1
    return f


def build():
    RC = pd.read_csv(CACHE_DIR / "roles_clip.csv").set_index("clip")
    RF = pd.read_parquet(CACHE_DIR / "roles_frames.parquet")
    TR = pd.read_parquet(CACHE_DIR / "tracks.parquet")
    pub, priv = load_public(), load_private()
    rows = []
    rf_g = {c: g for c, g in RF.groupby("clip")}
    tr_g = {c: g for c, g in TR.groupby("clip")}
    for df in (pub, priv):
        for _, m in df.iterrows():
            clip = m["clip"]
            rc = RC.loc[clip]
            f = {"clip": clip}
            if rc.kicker_found or rc.keeper_found:
                f.update(clip_features(rc, rf_g.get(clip, RF.iloc[0:0]), tr_g.get(clip, TR.iloc[0:0]), int(rc.W), int(rc.H)))
            else:
                f.update(kicker_found_I=0, keeper_found_I=0)
            f["foot_N"] = 1.0 if m.preferred_foot == "right" else -1.0
            f["gender_I"] = 1.0 if m.gender == "M" else 0.0
            rows.append(f)
    F = pd.DataFrame(rows).set_index("clip")
    F.to_parquet(CACHE_DIR / "features_v1.parquet")
    print(f"features: {F.shape}, NaN share {F.isna().mean().mean():.3f}")
    return F


def mirror(F: pd.DataFrame) -> pd.DataFrame:
    M = F.copy()
    for c in M.columns:
        if c.endswith("_N"):
            M[c] = -M[c]
        elif not c.endswith("_I"):
            raise ValueError(f"untagged feature {c}")
    return M


def swap_lr(P: np.ndarray, target: str) -> np.ndarray:
    cls = CLASSES[target]
    Q = P.copy()
    if "left" in cls:
        i, j = cls.index("left"), cls.index("right")
        Q[:, [i, j]] = P[:, [j, i]]
    return Q


def train(F: pd.DataFrame, run="gbm_v1", swap_labels=True, seeds=(0, 1, 2)):
    import lightgbm as lgb
    pub, priv = load_public(), load_private()
    folds = get_folds(pub).values
    Xp, Xq = F.loc[pub["clip"]].values.astype(np.float32), F.loc[priv["clip"]].values.astype(np.float32)
    Xp_m, Xq_m = mirror(F.loc[pub["clip"]]).values.astype(np.float32), mirror(F.loc[priv["clip"]]).values.astype(np.float32)
    out_dir = RUNS_DIR / run
    out_dir.mkdir(parents=True, exist_ok=True)
    oof_all, test_all, metrics = {}, {}, {}
    params = dict(n_estimators=400, learning_rate=0.03, num_leaves=15, min_child_samples=25, subsample=0.8,
                  subsample_freq=1, colsample_bytree=0.7, reg_lambda=5.0, verbose=-1)
    for t in TARGETS:
        cls = CLASSES[t]
        y = pub[t].map({c: i for i, c in enumerate(cls)}).values
        y_m = pub[t].map(FLIP_MAP[t]).map({c: i for i, c in enumerate(cls)}).values if swap_labels else y
        oof = np.zeros((len(pub), len(cls)))
        test = np.zeros((len(priv), len(cls)))
        for seed in seeds:
            for k in range(5):
                tr, va = folds != k, folds == k
                X_tr = np.vstack([Xp[tr], Xp_m[tr]])
                y_tr = np.concatenate([y[tr], y_m[tr]])
                m = lgb.LGBMClassifier(random_state=seed + 100 * k, **params)
                m.fit(X_tr, y_tr)
                p = m.predict_proba(Xp[va])
                pm = m.predict_proba(Xp_m[va])
                oof[va] += 0.5 * (p + swap_lr(pm, t)) / len(seeds)
                test += 0.5 * (m.predict_proba(Xq) + swap_lr(m.predict_proba(Xq_m), t)) / (5 * len(seeds))
        oof_all[t], test_all[t] = oof, test
        np.save(out_dir / f"oof_{t.split('_')[0]}.npy", oof)
        np.save(out_dir / f"test_{t.split('_')[0]}.npy", test)
        metrics[t] = float((np.argmax(oof, 1) == y).mean())
    pred = pd.DataFrame({t: np.asarray(CLASSES[t])[np.argmax(oof_all[t], 1)] for t in TARGETS})
    score = score_report(pub, pred, pd.Series(folds), run)
    metrics["score"] = score
    # per-source accuracy histogram (orientation bugs show up as sources at ~0)
    acc_src = pd.Series((pred["kick_direction"].values == pub["kick_direction"].values), index=pub.source_id.values).groupby(level=0).mean()
    print("per-source kick accuracy quantiles:", acc_src.quantile([0, .1, .25, .5, .75, .9, 1]).round(2).to_dict(),
          "| sources <= 0.2:", int((acc_src <= 0.2).sum()))
    if swap_labels:
        tp = pd.DataFrame({t: np.asarray(CLASSES[t])[np.argmax(test_all[t], 1)] for t in TARGETS})
        tp.insert(0, "clip", priv["clip"].values)
        write_submission(tp, out_dir / "submission.csv")
    json.dump(metrics, open(out_dir / "metrics.json", "w"), indent=1)
    return score


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--no-train", action="store_true")
    a = ap.parse_args()
    F = build()
    if a.no_train:
        return
    print("\n=== gbm_v1 (mirror augmentation + flip-TTA) ===")
    train(F, "gbm_v1")
    print("\n=== swap-disable test (labels NOT swapped for mirrored rows; keeper accuracy must collapse) ===")
    train(F, "gbm_v1_noswap", swap_labels=False, seeds=(0,))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/features_v2.py
"""Tabular features v2: keypoint kinematics of the kicker and the keeper from
pose_crops.parquet, expressed in the kicker's frame, merged with the v1 box
features, then LightGBM (mirror augmentation + flip-TTA) via features_v1.train.

Mirror tags. Coordinates are isotropic in units of the frame width. With
u = kicker->keeper unit vector and l = (u_y, -u_x) = kicker-left:
  * displacement-type vectors (same joint over time, or joint minus body centre):
        projection on l -> `_N` (negates under mirroring), on u -> `_I`
  * left-minus-right joint vectors (L_hip - R_hip, L_wrist - R_wrist, ...):
        the joint labels AND the geometry flip, so projection on l -> `_I`, on u -> `_N`
  * "is the right-side joint" indicators -> `_N`; image-vertical L-R differences -> `_N`

Run:  python src/features_v2.py
"""
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import pyarrow  # noqa: F401  (must precede cv2/torch imports elsewhere: pandas 3 + pyarrow on Windows)
import numpy as np
import pandas as pd

from config import CACHE_DIR
from cv import load_private, load_public
from features_v1 import train

NOSE, LEYE, REYE, LEAR, REAR, LSH, RSH, LEL, REL, LWR, RWR, LHIP, RHIP, LKNEE, RKNEE, LANK, RANK = range(17)
TAUS = [0.0, 0.04, 0.08, 0.12, 0.16, 0.20, 0.30, 0.40]
KP_CONF = 0.3


def sample_track(g: pd.DataFrame, W: int):
    """g: pose rows of one (clip, role) -> dict tau -> [17,2] array (NaN where missing), plus quality."""
    g = g.sort_values("t_from_end_s")
    t = g.t_from_end_s.values
    dt = np.median(np.diff(t)) if len(t) > 1 else 0.04
    tol = max(0.03, 0.6 * dt)
    kx = g[[f"kx{k}" for k in range(17)]].values / W
    ky = g[[f"ky{k}" for k in range(17)]].values / W
    kc = g[[f"kc{k}" for k in range(17)]].values
    kx[kc < KP_CONF] = np.nan
    ky[kc < KP_CONF] = np.nan
    out = {}
    for tau in TAUS:
        j = int(np.argmin(np.abs(t - tau)))
        out[tau] = np.stack([kx[j], ky[j]], 1) if abs(t[j] - tau) <= tol else np.full((17, 2), np.nan)
    qual = {"found_frac": float(g.found.mean()), "kp_conf": float(np.nanmean(kc[g.found.values == 1])) if g.found.any() else np.nan}
    # body scale: median torso length (shoulder-mid to hip-mid), fallback 0.35 * box height
    sh = (np.stack([kx[:, LSH], ky[:, LSH]], 1) + np.stack([kx[:, RSH], ky[:, RSH]], 1)) / 2
    hp = (np.stack([kx[:, LHIP], ky[:, LHIP]], 1) + np.stack([kx[:, RHIP], ky[:, RHIP]], 1)) / 2
    torso = np.linalg.norm(sh - hp, axis=1)
    s = np.nanmedian(torso) if np.isfinite(torso).any() else np.nan
    if not np.isfinite(s) or s < 1e-3:
        s = 0.35 * np.nanmedian(g.box_h.values) / W
    # dense ankle speed over the last 0.16 s (for the plant/kick foot assignment)
    m = t <= 0.16 + 1e-6
    speeds = {}
    for name, k in (("L", LANK), ("R", RANK)):
        p = np.stack([kx[m, k], ky[m, k]], 1)
        d = np.linalg.norm(np.diff(p, axis=0), axis=1)
        speeds[name] = np.nanmean(d) / s if np.isfinite(d).any() else np.nan
    return out, s, qual, speeds


def mid(P, a, b):
    return (P[a] + P[b]) / 2


def proj(v, l, u, name, kind, f, scale=1.0):
    """Store lateral/forward projections with the right mirror tags."""
    if v is None or not np.all(np.isfinite(v)):
        f[f"{name}_lat_{'N' if kind == 'disp' else 'I'}"] = np.nan
        f[f"{name}_fwd_{'I' if kind == 'disp' else 'N'}"] = np.nan
        return
    f[f"{name}_lat_{'N' if kind == 'disp' else 'I'}"] = float(v @ l) / scale
    f[f"{name}_fwd_{'I' if kind == 'disp' else 'N'}"] = float(v @ u) / scale


def kicker_features(K, s, qual, speeds, l, u, foot_right: bool):
    f = {"k_kpconf_I": qual["kp_conf"], "k_found_I": qual["found_frac"], "k_scale_I": s}
    P0, P4, P8, P16, P20, P40 = K[0.0], K[0.04], K[0.08], K[0.16], K[0.20], K[0.40]
    hip0 = mid(P0, LHIP, RHIP)
    # hip trajectory
    for tau, P in ((0.08, P8), (0.20, P20), (0.40, P40)):
        proj(hip0 - mid(P, LHIP, RHIP), l, u, f"k_hipdisp{tau}", "disp", f, s)
    # plant vs kick foot by kinematics (lower ankle speed over the last 0.16 s = plant foot)
    sL, sR = speeds["L"], speeds["R"]
    if np.isfinite(sL) and np.isfinite(sR):
        plant_right = sR < sL
        f["k_plant_right_N"] = 1.0 if plant_right else -1.0
        f["k_kickfoot_is_pref_I"] = 1.0 if (not plant_right) == foot_right else 0.0
        f["k_ankle_speed_ratio_I"] = (max(sL, sR) + 1e-6) / (min(sL, sR) + 1e-6)
        pa, ka = (RANK, LANK) if plant_right else (LANK, RANK)
        pk, kk = (RKNEE, LKNEE) if plant_right else (LKNEE, RKNEE)
        proj(P0[pa] - hip0, l, u, "k_plant_hip", "disp", f, s)
        proj(P0[ka] - hip0, l, u, "k_kick_hip", "disp", f, s)
        proj(P0[kk] - hip0, l, u, "k_kickknee_hip", "disp", f, s)
        proj(P0[ka] - P0[pa], l, u, "k_kick_plant", "disp", f, s)
        proj(P0[ka] - P8[ka], l, u, "k_kickvel", "disp", f, s)       # kick-foot swing over 0.08 s
        proj(P0[pa] - P0[pk], l, u, "k_plantshank", "disp", f, s)   # plant shank orientation
        f["k_kickfoot_height_I"] = float(P0[pa][1] - P0[ka][1]) / s if np.all(np.isfinite([P0[pa][1], P0[ka][1]])) else np.nan
        proj(P4[ka] - P8[ka], l, u, "k_kickvel_early", "disp", f, s)
    else:
        for k in ("k_plant_right_N", "k_kickfoot_is_pref_I", "k_ankle_speed_ratio_I", "k_kickfoot_height_I"):
            f[k] = np.nan
        for n in ("k_plant_hip", "k_kick_hip", "k_kickknee_hip", "k_kick_plant", "k_kickvel", "k_plantshank", "k_kickvel_early"):
            proj(None, l, u, n, "disp", f)
    # body orientation (left-minus-right vectors)
    for tau, P in ((0.0, P0), (0.16, P16)):
        proj(P[LHIP] - P[RHIP], l, u, f"k_hipline{tau}", "lr", f, s)
        proj(P[LSH] - P[RSH], l, u, f"k_shline{tau}", "lr", f, s)
        proj(mid(P, LSH, RSH) - mid(P, LHIP, RHIP), l, u, f"k_trunk{tau}", "disp", f, s)
    f["k_twist_fwd_N"] = f.get("k_shline0.0_fwd_N", np.nan) - f.get("k_hipline0.0_fwd_N", np.nan)
    f["k_trunk_dlat_N"] = f.get("k_trunk0.0_lat_N", np.nan) - f.get("k_trunk0.16_lat_N", np.nan)
    f["k_hipline_dfwd_N"] = f.get("k_hipline0.0_fwd_N", np.nan) - f.get("k_hipline0.16_fwd_N", np.nan)
    proj(P0[NOSE] - mid(P0, LEAR, REAR), l, u, "k_gaze", "disp", f, s)
    proj(P0[LWR] - P0[RWR], l, u, "k_arms", "lr", f, s)
    f["k_wrist_dy_N"] = float(P0[LWR][1] - P0[RWR][1]) / s if np.all(np.isfinite([P0[LWR][1], P0[RWR][1]])) else np.nan
    f["k_ear_vis_N"] = (float(np.isfinite(P0[LEAR][0])) - float(np.isfinite(P0[REAR][0])))
    return f


def knee_angle(P, hip, knee, ank):
    a, b = P[hip] - P[knee], P[ank] - P[knee]
    if not (np.all(np.isfinite(a)) and np.all(np.isfinite(b))):
        return np.nan
    c = a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9)
    return float(np.degrees(np.arccos(np.clip(c, -1, 1))))


def keeper_features(G, s, qual, l, u):
    f = {"g_kpconf_I": qual["kp_conf"], "g_found_I": qual["found_frac"], "g_scale_I": s}
    P0, P4, P8, P16, P30, P40 = G[0.0], G[0.04], G[0.08], G[0.16], G[0.30], G[0.40]
    hip0 = mid(P0, LHIP, RHIP)
    for tau, P in ((0.08, P8), (0.16, P16), (0.30, P30), (0.40, P40)):
        proj(hip0 - mid(P, LHIP, RHIP), l, u, f"g_hipdisp{tau}", "disp", f, s)
        proj(mid(P0, LSH, RSH) - mid(P, LSH, RSH), l, u, f"g_shdisp{tau}", "disp", f, s)
    # velocity and acceleration of the hip centre (per second, in body scales)
    h4, h8 = mid(P4, LHIP, RHIP), mid(P8, LHIP, RHIP)
    v1 = (hip0 - h4) / 0.04 if np.all(np.isfinite(h4)) else None
    v2 = (h4 - h8) / 0.04 if np.all(np.isfinite(h4)) and np.all(np.isfinite(h8)) else None
    proj(v1, l, u, "g_vel", "disp", f, s)
    proj((v1 - v2) if (v1 is not None and v2 is not None) else None, l, u, "g_acc", "disp", f, s)
    for tau, P in ((0.0, P0), (0.16, P16)):
        proj(mid(P, LSH, RSH) - mid(P, LHIP, RHIP), l, u, f"g_trunk{tau}", "disp", f, s)
        proj(mid(P, LHIP, RHIP) - mid(P, LANK, RANK), l, u, f"g_com{tau}", "disp", f, s)   # CoM over feet
        proj(P[LANK] - P[RANK], l, u, f"g_feet{tau}", "lr", f, s)
        proj(P[LWR] - P[RWR], l, u, f"g_wrists{tau}", "lr", f, s)
        f[f"g_wrist_dy{tau}_N"] = float(P[LWR][1] - P[RWR][1]) / s if np.all(np.isfinite([P[LWR][1], P[RWR][1]])) else np.nan
        f[f"g_knee_asym{tau}_N"] = knee_angle(P, LHIP, LKNEE, LANK) - knee_angle(P, RHIP, RKNEE, RANK)
        f[f"g_feet_spread{tau}_I"] = float(np.linalg.norm(P[LANK] - P[RANK])) / s if np.all(np.isfinite(P[LANK])) and np.all(np.isfinite(P[RANK])) else np.nan
    f["g_trunk_dlat_N"] = f.get("g_trunk0.0_lat_N", np.nan) - f.get("g_trunk0.16_lat_N", np.nan)
    f["g_com_dlat_N"] = f.get("g_com0.0_lat_N", np.nan) - f.get("g_com0.16_lat_N", np.nan)
    proj(P0[NOSE] - mid(P0, LEAR, REAR), l, u, "g_gaze", "disp", f, s)
    f["g_height_px_I"] = s
    return f


def build():
    P = pd.read_parquet(CACHE_DIR / "pose_crops.parquet")
    RC = pd.read_csv(CACHE_DIR / "roles_clip.csv").set_index("clip")
    V1 = pd.read_parquet(CACHE_DIR / "features_v1.parquet")
    meta = pd.concat([load_public(), load_private()], ignore_index=True).set_index("clip")
    rows = {}
    for (clip, role), g in P.groupby(["clip", "role"]):
        rc = RC.loc[clip]
        if pd.isna(rc.u_x):
            continue
        W = int(rc.W)
        u = np.array([rc.u_x, rc.u_y])
        l = np.array([u[1], -u[0]])
        K, s, qual, speeds = sample_track(g, W)
        f = rows.setdefault(clip, {})
        if role == "kicker":
            f.update(kicker_features(K, s, qual, speeds, l, u, meta.loc[clip, "preferred_foot"] == "right"))
        else:
            f.update(keeper_features(K, s, qual, l, u))
    V2 = pd.DataFrame.from_dict(rows, orient="index")
    V2.index.name = "clip"
    F = V1.join(V2, how="left")
    bad = [c for c in F.columns if not (c.endswith("_N") or c.endswith("_I"))]
    assert not bad, bad
    F.to_parquet(CACHE_DIR / "features_v2.parquet")
    print(f"features v2: {F.shape} (v1 {V1.shape[1]} + pose {V2.shape[1]}), NaN share {F.isna().mean().mean():.3f}")
    return F


if __name__ == "__main__":
    F = build()
    print("\n=== gbm_v2 (v1 box + v2 pose features) ===")
    train(F, "gbm_v2")
    print("\n=== gbm_v2_pose_only ===")
    pose_cols = [c for c in F.columns if c.startswith(("k_", "g_")) or c in ("foot_N", "gender_I", "kicker_found_I", "keeper_found_I")]
    train(F[pose_cols], "gbm_v2_pose_only", seeds=(0,))


In [ ]:
%%writefile src/features_v3.py
"""Tabular features v3 = v2 + ball-anchored kicker geometry.

Ball = confidence-weighted median of the sports-ball detections within 2 kicker
heights of the kicker in the last 0.35 s (the ball is stationary until contact).
Features: plant foot / kicking foot / hip / kicking knee relative to the ball at
tau = 0, 0.04, 0.08 (lateral `_N`, forward `_I`, in body scales), the kick-foot
approach direction relative to the ball, and ball quality flags.

Run:  python src/features_v3.py
"""
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import pyarrow  # noqa: F401
import numpy as np
import pandas as pd

from config import CACHE_DIR
from cv import load_private, load_public
from features_v1 import train
from features_v2 import LANK, LHIP, LKNEE, RANK, RHIP, RKNEE, mid, proj, sample_track

BALL = 32


def ball_positions():
    D = pd.read_parquet(CACHE_DIR / "detections.parquet", columns=["clip", "cls", "conf", "t_from_end_s", "x1", "y1", "x2", "y2"])
    B = D[(D.cls == BALL) & (D.t_from_end_s <= 0.35)]
    RC = pd.read_csv(CACHE_DIR / "roles_clip.csv").set_index("clip")
    out = {}
    for clip, g in B.groupby("clip"):
        rc = RC.loc[clip]
        if not rc.kicker_found:
            continue
        W, H = rc.W, rc.H
        kx, ky, kh = rc.kcx * W, rc.kcy * H, rc.kh * H
        cx, cy = (g.x1 + g.x2) / 2, (g.y1 + g.y2) / 2
        near = g[np.hypot(cx - kx, cy - (ky + 0.4 * kh)) < 2.0 * kh]
        if len(near) == 0:
            continue
        w = near.conf.values
        bx = np.average((near.x1 + near.x2) / 2, weights=w)
        by = np.average((near.y1 + near.y2) / 2, weights=w)
        out[clip] = (bx / W, by / W, float(near.conf.max()), len(near))
    return out


def build():
    V2 = pd.read_parquet(CACHE_DIR / "features_v2.parquet")
    P = pd.read_parquet(CACHE_DIR / "pose_crops.parquet")
    RC = pd.read_csv(CACHE_DIR / "roles_clip.csv").set_index("clip")
    balls = ball_positions()
    print(f"ball anchor found for {len(balls)} clips")
    rows = {}
    for (clip, role), g in P.groupby(["clip", "role"]):
        if role != "kicker":
            continue
        rc = RC.loc[clip]
        f = rows.setdefault(clip, {})
        if pd.isna(rc.u_x) or clip not in balls:
            f["ball_found_I"] = 0.0
            continue
        W = int(rc.W)
        u = np.array([rc.u_x, rc.u_y])
        l = np.array([u[1], -u[0]])
        bx, by, bconf, n = balls[clip]
        ball = np.array([bx, by])
        K, s, qual, speeds = sample_track(g, W)
        f.update(ball_found_I=1.0, ball_conf_I=bconf, ball_n_I=float(n))
        sL, sR = speeds["L"], speeds["R"]
        plant_right = (sR < sL) if (np.isfinite(sL) and np.isfinite(sR)) else None
        for tau in (0.0, 0.04, 0.08):
            P_ = K[tau]
            proj(mid(P_, LHIP, RHIP) - ball, l, u, f"b_hip{tau}", "disp", f, s)
            if plant_right is not None:
                pa, ka = (RANK, LANK) if plant_right else (LANK, RANK)
                kk = LKNEE if plant_right else RKNEE
                proj(P_[pa] - ball, l, u, f"b_plant{tau}", "disp", f, s)
                proj(P_[ka] - ball, l, u, f"b_kick{tau}", "disp", f, s)
                proj(P_[kk] - ball, l, u, f"b_kickknee{tau}", "disp", f, s)
            else:
                for n_ in (f"b_plant{tau}", f"b_kick{tau}", f"b_kickknee{tau}"):
                    proj(None, l, u, n_, "disp", f)
        # kick-foot approach direction toward the ball over the last 0.08 s
        if plant_right is not None:
            ka = LANK if plant_right else RANK
            v = K[0.0][ka] - K[0.08][ka]
            d = ball - K[0.08][ka]
            if np.all(np.isfinite(v)) and np.all(np.isfinite(d)) and np.linalg.norm(v) > 1e-6:
                f["b_kickdir_lat_N"] = float(v @ l) / (np.linalg.norm(v) + 1e-9)
                f["b_kickdir_fwd_I"] = float(v @ u) / (np.linalg.norm(v) + 1e-9)
                f["b_kick_dist08_I"] = float(np.linalg.norm(d)) / s
                f["b_kick_dist0_I"] = float(np.linalg.norm(ball - K[0.0][ka])) / s
            else:
                f["b_kickdir_lat_N"] = f["b_kickdir_fwd_I"] = f["b_kick_dist08_I"] = f["b_kick_dist0_I"] = np.nan
        # ball relative to the kicker's run-up line (hip at 0.4 -> hip at 0)
        h40, h0 = mid(K[0.4], LHIP, RHIP), mid(K[0.0], LHIP, RHIP)
        if np.all(np.isfinite(h40)) and np.all(np.isfinite(h0)) and np.linalg.norm(h0 - h40) > 1e-6:
            dirv = (h0 - h40) / np.linalg.norm(h0 - h40)
            perp = np.array([dirv[1], -dirv[0]])
            f["b_ball_off_runup_N"] = float((ball - h0) @ perp) / s * (1.0 if (perp @ l) >= 0 else -1.0)
            f["b_runup_angle_N"] = float(np.arctan2(dirv @ l, dirv @ u))
        else:
            f["b_ball_off_runup_N"] = f["b_runup_angle_N"] = np.nan
    V3 = pd.DataFrame.from_dict(rows, orient="index")
    V3.index.name = "clip"
    F = V2.join(V3, how="left")
    bad = [c for c in F.columns if not (c.endswith("_N") or c.endswith("_I"))]
    assert not bad, bad
    F.to_parquet(CACHE_DIR / "features_v3.parquet")
    print(f"features v3: {F.shape} (ball cols {V3.shape[1]}), NaN share {F.isna().mean().mean():.3f}")
    return F


if __name__ == "__main__":
    F = build()
    print("\n=== gbm_v3 (v2 + ball-anchored features) ===")
    train(F, "gbm_v3")


In [ ]:
%%writefile src/framecache.py
"""Frame cache A: 30 time-sampled frames per clip, letterboxed to 352x208
(16:9, multiples of 16), JPEG q88, one .npz per clip in CACHE_DIR/framesA/.

Frame k (0..23) is the frame nearest (t_end - k/24 s): the last ~0.96 s at 24 Hz.
Frames 24..29 are context frames at 1.25, 1.5, 2.0, 2.5, 3.5, 5.0 s before the
end. Offsets beyond the clip start clamp to the first frame (valid=False).

Build:  python src/framecache.py            (both partitions, 8 threads)
Load:   from framecache import load_frames  -> uint8 [T, 208, 352, 3] RGB
"""
import os
import sys
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import cv2
import numpy as np

from config import CACHE_DIR, GT_CSV, PRIVATE_CSV, PRIVATE_DIR, PUBLIC_DIR
from video import indices_at_offsets

cv2.setNumThreads(1)

OFFSETS_A = [k / 24.0 for k in range(24)] + [1.25, 1.5, 2.0, 2.5, 3.5, 5.0]
N_FRAMES_A = len(OFFSETS_A)
DENSE16 = list(range(15, -1, -1))              # k = 15..0, chronological, 0.625 s
WIDE16 = [29, 27, 26, 25] + list(range(22, -1, -2))  # 5.0, 2.5, 2.0, 1.25 s + 22..0 step 2
CACHE_A = CACHE_DIR / "framesA"
W_A, H_A = 352, 208
JPEG_Q = 88


def frame_timestamps(path):
    cap = cv2.VideoCapture(str(path))
    ts = []
    try:
        while cap.grab():
            ts.append(cap.get(cv2.CAP_PROP_POS_MSEC))
    finally:
        cap.release()
    ts = np.asarray(ts, dtype=np.float64)
    if len(ts) > 1 and np.all(np.diff(ts) <= 0):
        fps = cv2.VideoCapture(str(path)).get(cv2.CAP_PROP_FPS) or 25.0
        ts = np.arange(len(ts)) * 1000.0 / fps
    return ts


def read_selected(path, wanted):
    wanted = set(wanted)
    out, i = {}, 0
    cap = cv2.VideoCapture(str(path))
    try:
        while i <= max(wanted):
            ok, f = cap.read()
            if not ok:
                break
            if i in wanted:
                out[i] = f
            i += 1
    finally:
        cap.release()
    return out


def letterbox_wh(frame, w=W_A, h=H_A):
    fh, fw = frame.shape[:2]
    s = min(w / fw, h / fh)
    nw, nh = max(1, round(fw * s)), max(1, round(fh * s))
    r = cv2.resize(frame, (nw, nh), interpolation=cv2.INTER_AREA if s < 1 else cv2.INTER_LINEAR)
    out = np.zeros((h, w, 3), np.uint8)
    y0, x0 = (h - nh) // 2, (w - nw) // 2
    out[y0:y0 + nh, x0:x0 + nw] = r
    return out


def build_one(job):
    partition, clip, path = job
    stem = clip.replace(".mp4", "")
    out = CACHE_A / f"{stem}.npz"
    if out.exists():
        return clip, "cached"
    ts = frame_timestamps(path)
    if len(ts) == 0:
        return clip, "DECODE_FAIL"
    idx = indices_at_offsets(ts, OFFSETS_A)
    frames = read_selected(path, set(idx))
    H, W = next(iter(frames.values())).shape[:2]
    t_end = ts[-1]
    bufs, offs = [], [0]
    for i in idx:
        ok, enc = cv2.imencode(".jpg", letterbox_wh(frames[i]), [cv2.IMWRITE_JPEG_QUALITY, JPEG_Q])
        b = enc.tobytes()
        bufs.append(b)
        offs.append(offs[-1] + len(b))
    valid = np.array([(t_end - o * 1000.0) >= ts[0] - 1.0 for o in OFFSETS_A], dtype=bool)
    np.savez(out, jpeg=np.frombuffer(b"".join(bufs), dtype=np.uint8), offsets=np.asarray(offs, np.int64),
             frame_idx=np.asarray(idx, np.int32), ts_ms=ts[idx].astype(np.float32),
             t_from_end=np.asarray(OFFSETS_A, np.float32), valid=valid,
             W=W, H=H, n_frames=len(ts), duration_s=np.float32((t_end + np.median(np.diff(ts)) if len(ts) > 1 else t_end) / 1000.0),
             partition=partition)
    return clip, "ok"


def load_frames(clip, idx=None, rgb=True):
    """Decode cached JPEGs. clip may be '0001_public.mp4' or '0001_public'.
    idx: list of cache indices (0..29); default all 30. Returns uint8 [T, 208, 352, 3]."""
    stem = clip.replace(".mp4", "")
    z = np.load(CACHE_A / f"{stem}.npz")
    buf, offs = z["jpeg"], z["offsets"]
    idx = range(N_FRAMES_A) if idx is None else idx
    out = np.empty((len(idx), H_A, W_A, 3), np.uint8)
    for j, k in enumerate(idx):
        img = cv2.imdecode(buf[offs[k]:offs[k + 1]], cv2.IMREAD_COLOR)
        out[j] = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if rgb else img
    return out


def load_meta(clip):
    stem = clip.replace(".mp4", "")
    z = np.load(CACHE_A / f"{stem}.npz")
    return {k: z[k] for k in ("frame_idx", "ts_ms", "t_from_end", "valid", "W", "H", "n_frames", "duration_s")}


def main():
    import pandas as pd

    CACHE_A.mkdir(parents=True, exist_ok=True)
    gt = pd.read_csv(GT_CSV)
    pm = pd.read_csv(PRIVATE_CSV)
    jobs = [("public", c, str(PUBLIC_DIR / c)) for c in gt["clip"]]
    jobs += [("private", c, str(PRIVATE_DIR / c)) for c in pm["clip"]]
    print(f"building frame cache A for {len(jobs)} clips -> {CACHE_A}", flush=True)
    t0, status = time.time(), {}
    with ThreadPoolExecutor(max_workers=8) as ex:
        futs = [ex.submit(build_one, j) for j in jobs]
        for n, f in enumerate(as_completed(futs), 1):
            clip, st = f.result()
            status[clip] = st
            if n % 200 == 0:
                print(f"  {n}/{len(jobs)}  {time.time() - t0:.0f}s", flush=True)
    bad = [c for c, s in status.items() if s == "DECODE_FAIL"]
    print(f"done in {time.time() - t0:.0f}s; cached={sum(s == 'cached' for s in status.values())} "
          f"new={sum(s == 'ok' for s in status.values())} failed={len(bad)} {bad[:5]}")
    rows = []
    for partition, clip, _ in jobs:
        try:
            m = load_meta(clip)
            rows.append({"clip": clip, "partition": partition, "n_frames": int(m["n_frames"]), "W": int(m["W"]),
                         "H": int(m["H"]), "duration_s": float(m["duration_s"]), "n_valid": int(m["valid"].sum())})
        except Exception as e:  # noqa: BLE001
            rows.append({"clip": clip, "partition": partition, "error": str(e)})
    pd.DataFrame(rows).to_csv(CACHE_DIR / "framesA_index.csv", index=False)
    (CACHE_A / "_DONE").write_text("ok")
    size = sum(f.stat().st_size for f in CACHE_A.glob("*.npz")) / 2**30
    print(f"index written; cache size {size:.2f} GB")
    x = load_frames(jobs[0][1], DENSE16)
    print("sample load:", x.shape, x.dtype)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/gbm_kpraw.py
"""GBM on the RAW keypoint tensor of train_kpnet (per joint lat/fwd/conf at 11 taus in the kicker frame),
flattened, with the kpnet mirror (joint permutation + sign) as augmentation and flip-TTA. Complements the
hand-crafted features: the trees can pick any joint/time combination.

Run:  set KPNET_POSE=pose_rtm_m && python src/gbm_kpraw.py --subset kicker --run gbm_kpraw_k
      (subset: kicker = kicker channels + meta, keeper = keeper channels + meta, all)
"""
import argparse
import json
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import pyarrow  # noqa: F401
import numpy as np
import pandas as pd

from config import CLASSES, FLIP_MAP, RUNS_DIR, TARGETS
from cv import get_folds, load_private, load_public, score_report, write_submission
import train_kpnet as K


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--subset", default="kicker", choices=["kicker", "keeper", "all"])
    ap.add_argument("--run", required=True)
    ap.add_argument("--seeds", type=int, default=3)
    ap.add_argument("--leaves", type=int, default=15)
    ap.add_argument("--trees", type=int, default=400)
    ap.add_argument("--colsample", type=float, default=0.3)
    ap.add_argument("--last", type=int, default=0, help="keep only the last N time steps (taus 0.04*(N-1) .. 0)")
    a = ap.parse_args()
    import lightgbm as lgb

    X, meta = K.build_tensor()                    # [n, T, C] raw channels, meta rows = public then private
    spec = K.ChannelSpec()
    Xm = K.mirror(X, spec)
    if a.last:
        X, Xm = X[:, -a.last:, :], Xm[:, -a.last:, :]
    if a.subset == "kicker":
        ch = list(range(0, K.PER_ROLE)) + list(range(K.META0, K.C))
    elif a.subset == "keeper":
        ch = list(range(K.PER_ROLE, 2 * K.PER_ROLE)) + list(range(K.META0, K.C))
    else:
        ch = list(range(K.C))
    Xf, Xmf = X[:, :, ch].reshape(len(X), -1), Xm[:, :, ch].reshape(len(X), -1)
    pub, priv = load_public(), load_private()
    assert list(meta["clip"][: len(pub)]) == list(pub["clip"]) and list(meta["clip"][len(pub):]) == list(priv["clip"])
    n_pub = len(pub)
    Xp, Xp_m, Xq, Xq_m = Xf[:n_pub], Xmf[:n_pub], Xf[n_pub:], Xmf[n_pub:]
    folds = get_folds(pub).values
    print(f"{a.run}: tensor {X.shape} -> {a.subset} features {Xp.shape[1]} (pose {K.POSE_SRC}, NJ={K.NJ})")
    params = dict(n_estimators=a.trees, learning_rate=0.03, num_leaves=a.leaves, min_child_samples=25, subsample=0.8,
                  subsample_freq=1, colsample_bytree=a.colsample, reg_lambda=5.0, verbose=-1)
    out_dir = RUNS_DIR / a.run
    out_dir.mkdir(parents=True, exist_ok=True)
    oof_all, test_all, metrics = {}, {}, {}
    for t in TARGETS:
        cls = CLASSES[t]
        y = pub[t].map({c: i for i, c in enumerate(cls)}).values
        y_m = pub[t].map(FLIP_MAP[t]).map({c: i for i, c in enumerate(cls)}).values
        oof, test = np.zeros((n_pub, len(cls))), np.zeros((len(priv), len(cls)))
        for seed in range(a.seeds):
            for k in range(5):
                tr, va = folds != k, folds == k
                m = lgb.LGBMClassifier(random_state=seed + 100 * k, **params)
                m.fit(np.vstack([Xp[tr], Xp_m[tr]]), np.concatenate([y[tr], y_m[tr]]))
                oof[va] += 0.5 * (m.predict_proba(Xp[va]) + K.swap_lr_probs(m.predict_proba(Xp_m[va]), t)) / a.seeds
                test += 0.5 * (m.predict_proba(Xq) + K.swap_lr_probs(m.predict_proba(Xq_m), t)) / (5 * a.seeds)
        oof_all[t], test_all[t] = oof, test
        np.save(out_dir / f"oof_{t.split('_')[0]}.npy", oof)
        np.save(out_dir / f"test_{t.split('_')[0]}.npy", test)
        metrics[t] = float((oof.argmax(1) == y).mean())
        print(f"  {t}: {metrics[t]:.4f}", flush=True)
    pred = pd.DataFrame({t: np.asarray(CLASSES[t])[np.argmax(oof_all[t], 1)] for t in TARGETS})
    metrics["score"] = score_report(pub, pred, pd.Series(folds), a.run)
    tp = pd.DataFrame({t: np.asarray(CLASSES[t])[np.argmax(test_all[t], 1)] for t in TARGETS})
    tp.insert(0, "clip", priv["clip"].values)
    write_submission(tp, out_dir / "submission.csv")
    json.dump({**metrics, "subset": a.subset, "pose": K.POSE_SRC, "params": params}, open(out_dir / "metrics.json", "w"), indent=1)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/gbm_subset.py
"""Train the GBM (features_v1.train: mirror aug + flip-TTA, 3 seeds) on a column subset of a feature table.

Used for the kicker-only kick head: keeper features add noise to the kick head (kicker-only 0.602 vs all 0.592
on features_v8), so a kicker-only member is stacked for kick_direction.

Run:  python src/gbm_subset.py --features features_v8.parquet --subset kicker --run gbm_v8k
      python src/gbm_subset.py --features features_v8.parquet --subset keeper --run gbm_v8g
"""
import argparse
import os
import re
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import pyarrow  # noqa: F401
import pandas as pd

from config import CACHE_DIR
from features_v1 import train

KEEPER_RE = re.compile(r"^x?(rg_|g_|fg_|gc|gh|dist_kk|keeper)")      # 'x' prefix = RTMPose-x copies (features_v10)
KICKER_RE = re.compile(r"^x?(rk_|k_|k2_|fk_|kc|kh|kicker|b_)")
META = {"foot_N", "gender_I", "u_x_N", "u_y_I", "kicker_found_I", "keeper_found_I"}


def subset_columns(cols, which):
    if which == "kicker":
        keep = [c for c in cols if not KEEPER_RE.match(c)]
    elif which == "keeper":
        keep = [c for c in cols if not KICKER_RE.match(c) or c in META]
    else:
        keep = list(cols)
    return keep


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--features", default="features_v8.parquet")
    ap.add_argument("--subset", default="kicker", choices=["kicker", "keeper", "all"])
    ap.add_argument("--run", required=True)
    ap.add_argument("--seeds", type=int, default=3)
    a = ap.parse_args()
    F = pd.read_parquet(CACHE_DIR / a.features)
    cols = subset_columns(F.columns, a.subset)
    dropped = [c for c in F.columns if c not in cols]
    print(f"{a.features}: {F.shape[1]} cols -> {a.subset} subset {len(cols)} cols; dropped e.g. {dropped[:8]} ...")
    print(f"=== {a.run} ===")
    train(F[cols], a.run, seeds=tuple(range(a.seeds)))


In [ ]:
%%writefile src/hier_head.py
"""Hierarchical kicker head for kick_direction: two binary LightGBMs instead of one 3-class softmax.

  M: middle vs not-middle, trained on all rows + mirrored rows (the middle label is mirror-invariant: 2x data,
     no label swap) -> p_m.
  S: left vs right, trained on the left/right rows + their mirrored copies with swapped labels, flip-TTA at
     prediction -> p_L = P(left | not middle).
  Combined 3-class member: [ (1-p_m) p_L, p_m, (1-p_m)(1-p_L) ] in the CLASSES order (left, middle, right).

The 3-class heads almost never predict 'middle' (14% of the labels, 16/181 correct in stack_tab40); a dedicated
binary head gives the stack a cleaner P(middle). Kick-only member (no keeper / goal files -> stack.py skips them).

Run:  python src/hier_head.py --features features_v9a2b.parquet --run gbm_hier_ka2b
"""
import argparse
import json
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import pyarrow  # noqa: F401
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

from config import CACHE_DIR, CLASSES, RUNS_DIR
from cv import get_folds, load_private, load_public
from features_v1 import mirror
from gbm_subset import subset_columns

PARAMS = dict(n_estimators=400, learning_rate=0.03, num_leaves=15, min_child_samples=25, subsample=0.8,
              subsample_freq=1, colsample_bytree=0.7, reg_lambda=5.0, verbose=-1)


def main():
    import lightgbm as lgb
    ap = argparse.ArgumentParser()
    ap.add_argument("--features", default="features_v9a2b.parquet")
    ap.add_argument("--run", required=True)
    ap.add_argument("--seeds", type=int, default=3)
    ap.add_argument("--subset", default="kicker")
    a = ap.parse_args()
    pub, priv = load_public(), load_private()
    folds = get_folds(pub).values
    F = pd.read_parquet(CACHE_DIR / a.features)
    F = F[subset_columns(F.columns, a.subset)]
    Xp, Xq = F.loc[pub["clip"]].values.astype(np.float32), F.loc[priv["clip"]].values.astype(np.float32)
    Xp_m, Xq_m = mirror(F.loc[pub["clip"]]).values.astype(np.float32), mirror(F.loc[priv["clip"]]).values.astype(np.float32)
    cls = CLASSES["kick_direction"]
    y = pub["kick_direction"].map({c: i for i, c in enumerate(cls)}).values
    iL, iM, iR = cls.index("left"), cls.index("middle"), cls.index("right")
    y_mid = (y == iM).astype(int)
    y_left = (y == iL).astype(int)
    lr_rows = y != iM
    oof_m, oof_l = np.zeros(len(pub)), np.zeros(len(pub))
    test_m, test_l = np.zeros(len(priv)), np.zeros(len(priv))
    seeds = range(a.seeds)
    for seed in seeds:
        for k in range(5):
            tr, va = folds != k, folds == k
            # M head: all training rows + mirrored, same label
            mM = lgb.LGBMClassifier(random_state=seed + 100 * k, **PARAMS)
            mM.fit(np.vstack([Xp[tr], Xp_m[tr]]), np.concatenate([y_mid[tr], y_mid[tr]]))
            oof_m[va] += 0.5 * (mM.predict_proba(Xp[va])[:, 1] + mM.predict_proba(Xp_m[va])[:, 1]) / len(seeds)
            test_m += 0.5 * (mM.predict_proba(Xq)[:, 1] + mM.predict_proba(Xq_m)[:, 1]) / (5 * len(seeds))
            # S head: left/right rows + mirrored with swapped labels; flip-TTA: P(left|x) = 1 - P(left|mirror x)
            t2 = tr & lr_rows
            mS = lgb.LGBMClassifier(random_state=seed + 100 * k, **PARAMS)
            mS.fit(np.vstack([Xp[t2], Xp_m[t2]]), np.concatenate([y_left[t2], 1 - y_left[t2]]))
            oof_l[va] += 0.5 * (mS.predict_proba(Xp[va])[:, 1] + 1 - mS.predict_proba(Xp_m[va])[:, 1]) / len(seeds)
            test_l += 0.5 * (mS.predict_proba(Xq)[:, 1] + 1 - mS.predict_proba(Xq_m)[:, 1]) / (5 * len(seeds))
    oof = np.zeros((len(pub), 3)); test = np.zeros((len(priv), 3))
    oof[:, iL], oof[:, iM], oof[:, iR] = (1 - oof_m) * oof_l, oof_m, (1 - oof_m) * (1 - oof_l)
    test[:, iL], test[:, iM], test[:, iR] = (1 - test_m) * test_l, test_m, (1 - test_m) * (1 - test_l)
    out = RUNS_DIR / a.run
    out.mkdir(parents=True, exist_ok=True)
    np.save(out / "oof_kick.npy", oof); np.save(out / "test_kick.npy", test)
    acc = float((oof.argmax(1) == y).mean())
    auc = float(roc_auc_score(y_mid, oof_m))
    lr_acc = float(((oof_l[lr_rows] > 0.5) == y_left[lr_rows]).mean())
    # best accuracy reachable by a middle threshold on top of the L/R decision
    lr_pred = np.where(oof_l > 0.5, iL, iR)
    best = max(((np.where(oof_m > th, iM, lr_pred) == y).mean(), th) for th in np.arange(0.15, 0.6, 0.01))
    n_mid = int((oof.argmax(1) == iM).sum())
    print(f"[{a.run}] 3-class acc {acc:.4f} | middle AUC {auc:.3f} (prior {y_mid.mean():.3f}), predicted middle {n_mid} | "
          f"L/R head acc on L/R rows {lr_acc:.4f} | best acc with middle threshold {best[0]:.4f} at p_m > {best[1]:.2f}")
    json.dump({"kick_direction": acc, "middle_auc": auc, "lr_acc": lr_acc, "best_thresh_acc": best[0], "best_thresh": best[1],
               "features": a.features, "subset": a.subset}, open(out / "metrics.json", "w"), indent=1)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/inventory.py
"""Scan every clip once: decode all frames, record true frame count, per-frame
timestamps (to detect variable frame rate), resolution and duration.

Output: CACHE_DIR/clip_inventory.csv  (one row per clip, public + private)

Run:  python src/inventory.py
"""
import os
import sys
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import cv2
import numpy as np

from config import CACHE_DIR, GT_CSV, PRIVATE_CSV, PRIVATE_DIR, PUBLIC_DIR

cv2.setNumThreads(1)

# Threads, not processes: cv2 releases the GIL while decoding, and spawned
# worker processes (each re-importing cv2/numpy) exhausted the Windows commit
# limit on this 16 GB laptop with a nearly-full C: (pagefile cannot grow).
N_WORKERS = 8


def scan_clip(args):
    partition, clip, path = args
    row = {"clip": clip, "partition": partition, "path": path,
           "file_bytes": os.path.getsize(path) if os.path.exists(path) else -1,
           "decode_ok": False}
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        row["error"] = "cannot open"
        return row
    row["width"] = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    row["height"] = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    row["fps_reported"] = float(cap.get(cv2.CAP_PROP_FPS))
    row["n_frames_reported"] = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    ts = []
    last = None
    n = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        ts.append(cap.get(cv2.CAP_PROP_POS_MSEC))
        last = frame
        n += 1
    cap.release()
    row["n_frames_decoded"] = n
    if n == 0:
        row["error"] = "no frames"
        return row
    ts = np.asarray(ts, dtype=np.float64)
    dt = np.diff(ts) if n > 1 else np.array([1000.0 / max(row["fps_reported"], 1e-6)])
    row["first_ts_ms"] = float(ts[0])
    row["last_ts_ms"] = float(ts[-1])
    row["dt_mean_ms"] = float(dt.mean())
    row["dt_std_ms"] = float(dt.std())
    row["dt_min_ms"] = float(dt.min())
    row["dt_max_ms"] = float(dt.max())
    row["fps_measured"] = 1000.0 / row["dt_mean_ms"] if row["dt_mean_ms"] > 0 else np.nan
    # duration = time of last frame + one frame interval
    row["duration_s"] = (row["last_ts_ms"] + row["dt_mean_ms"]) / 1000.0
    row["last_frame_mean_intensity"] = float(last.mean())
    row["last_frame_std_intensity"] = float(last.std())
    row["decode_ok"] = True
    return row


def main():
    import pandas as pd

    gt = pd.read_csv(GT_CSV)
    pm = pd.read_csv(PRIVATE_CSV)
    jobs = [("public", c, str(PUBLIC_DIR / c)) for c in gt["clip"]]
    jobs += [("private", c, str(PRIVATE_DIR / c)) for c in pm["clip"]]
    print(f"scanning {len(jobs)} clips with {N_WORKERS} workers ...", flush=True)
    t0 = time.time()
    rows = []
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        futs = [ex.submit(scan_clip, j) for j in jobs]
        for i, f in enumerate(as_completed(futs), 1):
            rows.append(f.result())
            if i % 200 == 0:
                print(f"  {i}/{len(jobs)}  {time.time() - t0:.0f}s", flush=True)
    inv = pd.DataFrame(rows).sort_values(["partition", "clip"]).reset_index(drop=True)
    meta = pd.concat([gt.assign(partition="public"), pm.assign(partition="private")], ignore_index=True)
    inv = inv.merge(meta[["clip", "source_id", "preferred_foot", "gender", "resolution", "fps"]], on="clip", how="left")
    out = CACHE_DIR / "clip_inventory.csv"
    inv.to_csv(out, index=False)
    print(f"wrote {out}  ({len(inv)} rows) in {time.time() - t0:.0f}s")

    ok = inv[inv.decode_ok]
    print("\ndecode failures:", (~inv.decode_ok).sum())
    print("reported vs decoded frame count mismatch:", (ok.n_frames_reported != ok.n_frames_decoded).sum())
    print("\nduration_s quantiles:\n", ok.groupby("partition").duration_s.describe(percentiles=[.05, .25, .5, .75, .95]).round(2).T)
    print("\nframes decoded quantiles:\n", ok.groupby("partition").n_frames_decoded.describe(percentiles=[.05, .25, .5, .75, .95]).round(0).T)
    print("\nfps_reported counts:\n", ok.fps_reported.round(2).value_counts().head(10))
    print("\nresolution counts:\n", ok.groupby(["width", "height"]).size().sort_values(ascending=False).head(10))
    vfr = ok[ok.dt_std_ms > 0.5 * ok.dt_mean_ms]
    print(f"\npossible variable-frame-rate clips (dt_std > 0.5*dt_mean): {len(vfr)}")
    print("clips shorter than 0.5 s:", (ok.duration_s < 0.5).sum(), " | shorter than 1.0 s:", (ok.duration_s < 1.0).sum())


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/late_table.py
"""Pure late-window kicker table: the kicker-only columns of features_v9<sfx> restricted to the last T seconds
before contact (every column whose time token is later than T is dropped) and WITHOUT the keeper box columns
(gk_*, kk_dist*, p_keeper*) that the kicker regex of gbm_subset.py lets through.

Both restrictions matter together: with every time step the keeper columns help a little (the kicker features
are noisy), but on the last 0.04 s the kicker features are strong and the keeper columns only make the head copy
the goalkeeper, who guesses the side right 56% of the time. On the a2b table the LightGBM kick head goes
0.636 (all columns) -> 0.651 (late window) -> 0.664 (late window, no keeper columns).

Writes CACHE_DIR/features_v9<sfx>_late04pure.parquet (T = 0.04) which gbm_subset.py / model_subset.py train on
with --subset all.
Run:  python src/late_table.py --sfx a2b      (also: a, x, '' for the m table)
"""
import argparse
import os
import re
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import pyarrow  # noqa: F401
import pandas as pd

from config import CACHE_DIR
from gbm_subset import subset_columns

LATER_THAN_004 = r"(?<![\d.])0\.(08|12|16|2|24|3|32|4|40)(?!\d)"   # time tokens later than 0.04 s before contact
KEEPER_BOX = r"(gk_|kk_dist|p_keeper)"

if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--sfx", default="a2b", help="features_v9<sfx> table suffix: a2b, a, x or m (= features_v9, no suffix)")
    a = ap.parse_args()
    a.sfx = "" if a.sfx == "m" else a.sfx
    F = pd.read_parquet(CACHE_DIR / f"features_v9{a.sfx}.parquet")
    kicker = subset_columns(F.columns, "kicker")
    cols = [c for c in kicker if not re.search(LATER_THAN_004, c) and not re.match(KEEPER_BOX, c)]
    out = CACHE_DIR / f"features_v9{a.sfx}_late04pure.parquet"
    F[cols].to_parquet(out)
    print(f"features_v9{a.sfx}: {len(F.columns)} cols -> kicker {len(kicker)} -> late-window, no keeper box {len(cols)} -> {out.name}")


In [ ]:
%%writefile src/model_subset.py
"""Non-LightGBM learners on a feature-table column subset (same protocol as features_v1.train: source-grouped
5-fold CV, mirror augmentation with label swap, flip-TTA, seed averaging) -> stack members with a different
inductive bias (random forest / extra trees / XGBoost / logistic regression).

Run:  python src/model_subset.py --features features_v9.parquet --subset kicker --model et --run et_v9k
"""
import argparse
import json
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import pyarrow  # noqa: F401
import numpy as np
import pandas as pd

from config import CACHE_DIR, CLASSES, FLIP_MAP, RUNS_DIR, TARGETS
from cv import get_folds, load_private, load_public, score_report, write_submission
from features_v1 import mirror, swap_lr
from gbm_subset import subset_columns


def make_model(name, seed, jobs=4):
    if name == "rf":
        from sklearn.ensemble import RandomForestClassifier
        return RandomForestClassifier(n_estimators=600, min_samples_leaf=3, max_features=0.3, n_jobs=jobs, random_state=seed)
    if name == "et":
        from sklearn.ensemble import ExtraTreesClassifier
        return ExtraTreesClassifier(n_estimators=800, min_samples_leaf=3, max_features=0.3, n_jobs=jobs, random_state=seed)
    if name == "xgb":
        from xgboost import XGBClassifier
        return XGBClassifier(n_estimators=400, learning_rate=0.03, max_depth=4, subsample=0.8, colsample_bytree=0.5,
                             min_child_weight=5, reg_lambda=5.0, n_jobs=jobs, random_state=seed, verbosity=0)
    if name == "lr":
        from sklearn.linear_model import LogisticRegression
        from sklearn.pipeline import make_pipeline
        from sklearn.preprocessing import StandardScaler
        return make_pipeline(StandardScaler(), LogisticRegression(C=0.05, max_iter=2000))
    raise ValueError(name)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--features", default="features_v9.parquet")
    ap.add_argument("--subset", default="kicker", choices=["kicker", "keeper", "all"])
    ap.add_argument("--model", default="et", choices=["rf", "et", "xgb", "lr"])
    ap.add_argument("--run", required=True)
    ap.add_argument("--seeds", type=int, default=3)
    ap.add_argument("--jobs", type=int, default=4, help="n_jobs; use 1 if joblib dies with 'concurrent send_bytes'")
    a = ap.parse_args()
    F = pd.read_parquet(CACHE_DIR / a.features)
    cols = subset_columns(F.columns, a.subset)
    F = F[cols]
    pub, priv = load_public(), load_private()
    folds = get_folds(pub).values
    Xp, Xq = F.loc[pub["clip"]].values.astype(np.float32), F.loc[priv["clip"]].values.astype(np.float32)
    Xp_m, Xq_m = mirror(F.loc[pub["clip"]]).values.astype(np.float32), mirror(F.loc[priv["clip"]]).values.astype(np.float32)
    if a.model in ("rf", "et", "lr"):                     # sklearn trees / LR need finite values
        med = np.nanmedian(np.vstack([Xp, Xp_m]), 0)
        med = np.where(np.isfinite(med), med, 0.0)
        for X in (Xp, Xq, Xp_m, Xq_m):
            bad = ~np.isfinite(X)
            X[bad] = np.broadcast_to(med, X.shape)[bad]
    out_dir = RUNS_DIR / a.run
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f"{a.run}: {a.model} on {a.features} {a.subset} subset ({len(cols)} cols), seeds {a.seeds}", flush=True)
    oof_all, test_all, metrics = {}, {}, {}
    for t in TARGETS:
        cls = CLASSES[t]
        y = pub[t].map({c: i for i, c in enumerate(cls)}).values
        y_m = pub[t].map(FLIP_MAP[t]).map({c: i for i, c in enumerate(cls)}).values
        oof, test = np.zeros((len(pub), len(cls))), np.zeros((len(priv), len(cls)))
        for seed in range(a.seeds):
            for k in range(5):
                tr, va = folds != k, folds == k
                m = make_model(a.model, seed + 100 * k, a.jobs)
                m.fit(np.vstack([Xp[tr], Xp_m[tr]]), np.concatenate([y[tr], y_m[tr]]))
                oof[va] += 0.5 * (m.predict_proba(Xp[va]) + swap_lr(m.predict_proba(Xp_m[va]), t)) / a.seeds
                test += 0.5 * (m.predict_proba(Xq) + swap_lr(m.predict_proba(Xq_m), t)) / (5 * a.seeds)
        oof_all[t], test_all[t] = oof, test
        np.save(out_dir / f"oof_{t.split('_')[0]}.npy", oof)
        np.save(out_dir / f"test_{t.split('_')[0]}.npy", test)
        metrics[t] = float((oof.argmax(1) == y).mean())
        print(f"  {t}: {metrics[t]:.4f}", flush=True)
    pred = pd.DataFrame({t: np.asarray(CLASSES[t])[np.argmax(oof_all[t], 1)] for t in TARGETS})
    metrics["score"] = score_report(pub, pred, pd.Series(folds), a.run)
    tp = pd.DataFrame({t: np.asarray(CLASSES[t])[np.argmax(test_all[t], 1)] for t in TARGETS})
    tp.insert(0, "clip", priv["clip"].values)
    write_submission(tp, out_dir / "submission.csv")
    json.dump({**metrics, "model": a.model, "subset": a.subset, "features": a.features}, open(out_dir / "metrics.json", "w"), indent=1)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/pose3d.py
"""3D pose of the KICKER for every frame of the last WINDOW_S seconds: RTMW3D-x (rtmlib RTMPose3d, COCO-WholeBody
133 keypoints with root-relative depth z) on the kicker role box. Why: the clips are side views (the kicker->keeper
axis is horizontal in the image), so the kick's lateral direction is the camera depth axis, which 2D keypoints only
see through foreshortening; the 3D head gives the depth component of the feet / hips / shoulders directly.

Kept keypoints: 0-16 COCO body + 17-22 feet (l big toe, l small toe, l heel, r big toe, r small toe, r heel).
Per keypoint: cx, cy (model crop pixels), z (model depth units), s (score) and ix, iy (image pixels).
Output: CACHE_DIR/pose3d_k[_s<i>of<n>].parquet (resumable; shards for parallel CPU processes).

Run:  python src/pose3d.py --threads 8 --shard 0/2      python src/pose3d.py --threads 8 --shard 1/2
"""
import argparse
import os
import sys
import time
from concurrent.futures import ThreadPoolExecutor

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
os.environ.setdefault("XDG_CACHE_HOME", r"D:\cv3\torch")

import cv2
import numpy as np
import pandas as pd

from config import CACHE_DIR, PRIVATE_DIR, PUBLIC_DIR
from pose_rtm import interp_boxes, load_clip

cv2.setNumThreads(1)

URL = "https://huggingface.co/Soykaf/RTMW3D-x/resolve/main/onnx/rtmw3d-x_8xb64_cocktail14-384x288-b0a0eab7_20240626.onnx"
NK = 23
BOX_SCALE = 1.0


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--limit", type=int, default=0)
    ap.add_argument("--threads", type=int, default=0)
    ap.add_argument("--shard", default="0/1", help="i/n: process clips[i::n] and write a per-shard parquet")
    a = ap.parse_args()
    si, sn = (int(x) for x in a.shard.split("/"))
    OUT = CACHE_DIR / ("pose3d_k.parquet" if sn == 1 else f"pose3d_k_s{si}of{sn}.parquet")

    import onnxruntime as ort
    from rtmlib import RTMPose3d
    model = RTMPose3d(URL, model_input_size=(288, 384), backend="onnxruntime", device="cpu")
    if a.threads:
        so = ort.SessionOptions()
        so.intra_op_num_threads = a.threads
        model.session = ort.InferenceSession(model.onnx_model, so, providers=["CPUExecutionProvider"])

    RF = pd.read_parquet(CACHE_DIR / "roles_frames.parquet")
    RC = pd.read_csv(CACHE_DIR / "roles_clip.csv")
    clips = RC[RC.kicker_found == 1]["clip"].tolist()[si::sn]
    done, existing = set(), None
    if OUT.exists():
        existing = pd.read_parquet(OUT)
        done = set(existing["clip"])
    clips = [c for c in clips if c not in done]
    if a.limit:
        clips = clips[: a.limit]
    print(f"shard {a.shard}: {len(done)} clips cached, {len(clips)} to do", flush=True)

    def path_of(c):
        return str((PUBLIC_DIR if c.endswith("_public.mp4") else PRIVATE_DIR) / c)

    jobs = [(c, path_of(c), RF[(RF["clip"] == c) & (RF.role == "kicker")]) for c in clips]
    rows, t0, n_done = [], time.time(), 0

    def flush():
        nonlocal existing, rows
        if not rows:
            return
        new = pd.DataFrame(rows)
        existing = new if existing is None else pd.concat([existing, new], ignore_index=True)
        tmp = OUT.with_suffix(".tmp.parquet")
        existing.to_parquet(tmp, index=False)
        os.replace(tmp, OUT)
        rows = []

    with ThreadPoolExecutor(max_workers=2) as ex:
        futs = [ex.submit(load_clip, j) for j in jobs[:4]]
        nxt = 4
        while futs:
            clip, data = futs.pop(0).result()
            if nxt < len(jobs):
                futs.append(ex.submit(load_clip, jobs[nxt]))
                nxt += 1
            if data is None:
                print("DECODE FAIL", clip, flush=True)
                continue
            ts, t_from_end, idx, frames, tr = data
            if len(tr) == 0:
                continue
            boxes = interp_boxes(tr, t_from_end[idx])
            for i, fi in enumerate(idx):
                cx, cy, w, h = boxes[i]
                w, h = w * BOX_SCALE, h * BOX_SCALE
                bbox = [cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2]
                kp, sc, _, kp2d = model(frames[fi], bboxes=[bbox])
                kp, sc, kp2d = kp[0][:NK], sc[0][:NK], kp2d[0][:NK]
                row = {"clip": clip, "frame_idx": int(fi), "t_from_end_s": float(t_from_end[fi]),
                       "box_cx": cx, "box_cy": cy, "box_w": w, "box_h": h, "pconf": float(sc[:17].mean())}
                for k in range(NK):
                    row[f"cx{k}"], row[f"cy{k}"], row[f"z{k}"], row[f"s{k}"] = float(kp[k, 0]), float(kp[k, 1]), float(kp[k, 2]), float(sc[k])
                    row[f"ix{k}"], row[f"iy{k}"] = float(kp2d[k, 0]), float(kp2d[k, 1])
                rows.append(row)
            n_done += 1
            if n_done % 25 == 0:
                flush()
                el = time.time() - t0
                print(f"  {n_done}/{len(jobs)}  {el:.0f}s ({el / n_done:.2f}s/clip)", flush=True)
    flush()
    print(f"done {n_done} clips in {time.time() - t0:.0f}s -> {OUT}")
    if existing is not None:
        print("rows:", len(existing), "| mean score body:", round(float(existing[[f"s{k}" for k in range(17)]].values.mean()), 3),
              "feet:", round(float(existing[[f"s{k}" for k in range(17, 23)]].values.mean()), 3))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/pose_crops.py
"""Stage 3: YOLO11x-pose on upscaled crops of the kicker and the keeper for
EVERY frame in the last WINDOW_S seconds (the players are only 30-80 px tall in
the full frame, so full-frame pose misses them).

Role boxes come from roles_frames.parquet (sampled every 0.1 s); the box centre
and size are interpolated in time for the dense frames. Each crop is a square
of CROP_K * box height centred on the box, resized to CROP_PX, and the pose
detection nearest the crop centre is kept. Keypoints are stored in ORIGINAL
frame pixel coordinates.

Output: CACHE_DIR/pose_crops.parquet, one row per (clip, role, frame).
Run:  python src/pose_crops.py [--limit N]
"""
import argparse
import os
import sys
import time
from concurrent.futures import ThreadPoolExecutor

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import cv2
import numpy as np
import pandas as pd

from config import CACHE_DIR, MODELS_DIR, PRIVATE_DIR, PUBLIC_DIR
from framecache import frame_timestamps, read_selected

cv2.setNumThreads(1)

WINDOW_S = 0.4
CROP_K = {"kicker": 2.4, "keeper": 2.8}
POSE_WEIGHTS = "yolo11m-pose.pt"   # x-pose was 5 s/clip; crops are upscaled so m is enough
CROP_PX = 448
OUT = CACHE_DIR / "pose_crops.parquet"
KP = 17


def interp_boxes(tr: pd.DataFrame, t_dense: np.ndarray) -> np.ndarray:
    """tr: role boxes at sampled t_from_end (ascending). Returns [n, 4] boxes (cx, cy, w, h) at t_dense."""
    tr = tr.sort_values("t_from_end_s")
    t = tr.t_from_end_s.values
    cx, cy = ((tr.x1 + tr.x2) / 2).values, ((tr.y1 + tr.y2) / 2).values
    w, h = (tr.x2 - tr.x1).values, (tr.y2 - tr.y1).values
    out = np.stack([np.interp(t_dense, t, v) for v in (cx, cy, w, h)], axis=1)  # np.interp clamps outside
    return out


def crop_square(frame, cx, cy, side):
    H, W = frame.shape[:2]
    side = int(round(side))
    x0, y0 = int(round(cx - side / 2)), int(round(cy - side / 2))
    out = np.full((side, side, 3), 114, np.uint8)
    sx0, sy0 = max(0, x0), max(0, y0)
    sx1, sy1 = min(W, x0 + side), min(H, y0 + side)
    if sx1 > sx0 and sy1 > sy0:
        out[sy0 - y0:sy1 - y0, sx0 - x0:sx1 - x0] = frame[sy0:sy1, sx0:sx1]
    scale = CROP_PX / side
    return cv2.resize(out, (CROP_PX, CROP_PX), interpolation=cv2.INTER_LINEAR if scale > 1 else cv2.INTER_AREA), x0, y0, scale


def load_clip(job):
    clip, path, roles_tr = job
    ts = frame_timestamps(path)
    if len(ts) == 0:
        return clip, None
    t_end = ts[-1]
    t_from_end = (t_end - ts) / 1000.0
    idx = np.where(t_from_end <= WINDOW_S + 1e-6)[0]
    frames = read_selected(path, set(idx.tolist()))
    return clip, (ts, t_from_end, idx, frames, roles_tr)


def pick_pose(res, cx_c, cy_c):
    """Pose result on a crop -> (box xyxy, kps [17,3]) of the person nearest the crop centre, or None."""
    if res.boxes is None or len(res.boxes) == 0:
        return None
    xyxy = res.boxes.xyxy.cpu().numpy()
    conf = res.boxes.conf.cpu().numpy()
    c = (xyxy[:, :2] + xyxy[:, 2:]) / 2
    d = np.hypot(c[:, 0] - cx_c, c[:, 1] - cy_c) / CROP_PX
    score = d - 0.15 * conf  # prefer central, then confident
    j = int(np.argmin(score))
    if d[j] > 0.35:
        return None
    kxy = res.keypoints.xy[j].cpu().numpy()
    kc = res.keypoints.conf[j].cpu().numpy() if res.keypoints.conf is not None else np.zeros(KP)
    return xyxy[j], conf[j], kxy, kc


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--limit", type=int, default=0)
    ap.add_argument("--conf", type=float, default=0.15)
    ap.add_argument("--weights", default=POSE_WEIGHTS)
    a = ap.parse_args()

    import torch
    from ultralytics import YOLO
    model = YOLO(str(MODELS_DIR / a.weights))

    RF = pd.read_parquet(CACHE_DIR / "roles_frames.parquet")
    RC = pd.read_csv(CACHE_DIR / "roles_clip.csv")
    clips = RC[(RC.kicker_found == 1) | (RC.keeper_found == 1)]["clip"].tolist()
    done = set()
    existing = None
    if OUT.exists():
        existing = pd.read_parquet(OUT)
        done = set(existing["clip"])
    clips = [c for c in clips if c not in done]
    if a.limit:
        clips = clips[: a.limit]
    print(f"{len(done)} clips cached, {len(clips)} to do", flush=True)

    def path_of(c):
        return str((PUBLIC_DIR if c.endswith("_public.mp4") else PRIVATE_DIR) / c)

    jobs = [(c, path_of(c), RF[RF["clip"] == c]) for c in clips]
    rows, t0, n_done = [], time.time(), 0

    def flush():
        nonlocal existing, rows
        if not rows:
            return
        new = pd.DataFrame(rows)
        existing = new if existing is None else pd.concat([existing, new], ignore_index=True)
        tmp = OUT.with_suffix(".tmp.parquet")   # atomic replace for concurrent readers
        existing.to_parquet(tmp, index=False)
        os.replace(tmp, OUT)
        rows = []

    with ThreadPoolExecutor(max_workers=2) as ex:
        futs = [ex.submit(load_clip, j) for j in jobs[:4]]
        nxt = 4
        while futs:
            clip, data = futs.pop(0).result()
            if nxt < len(jobs):
                futs.append(ex.submit(load_clip, jobs[nxt]))
                nxt += 1
            if data is None:
                print("DECODE FAIL", clip, flush=True)
                continue
            ts, t_from_end, idx, frames, roles_tr = data
            crops, meta = [], []
            for role in ("kicker", "keeper"):
                tr = roles_tr[roles_tr.role == role]
                if len(tr) == 0:
                    continue
                boxes = interp_boxes(tr, t_from_end[idx])
                for i, fi in enumerate(idx):
                    cx, cy, w, h = boxes[i]
                    side = max(CROP_K[role] * h, 1.4 * w, 64)
                    crop, x0, y0, scale = crop_square(frames[fi], cx, cy, side)
                    crops.append(crop)
                    meta.append((role, int(fi), float(ts[fi]), float(t_from_end[fi]), x0, y0, scale, cx, cy, w, h))
            results = []
            for s in range(0, len(crops), 16):
                try:
                    results.extend(model.predict(crops[s:s + 16], imgsz=CROP_PX, conf=a.conf, device=0,
                                                 half=True, verbose=False))
                except Exception as e:  # noqa: BLE001
                    if "out of memory" not in str(e).lower():
                        raise
                    torch.cuda.empty_cache()
                    time.sleep(20)
                    results.extend(model.predict(crops[s:s + 16], imgsz=CROP_PX, conf=a.conf, device=0,
                                                 half=True, verbose=False))
            for (role, fi, t_ms, tfe, x0, y0, scale, cx, cy, w, h), res in zip(meta, results):
                row = {"clip": clip, "role": role, "frame_idx": fi, "ts_ms": t_ms, "t_from_end_s": tfe,
                       "box_cx": cx, "box_cy": cy, "box_w": w, "box_h": h, "found": 0, "pconf": np.nan,
                       "px1": np.nan, "py1": np.nan, "px2": np.nan, "py2": np.nan}
                p = pick_pose(res, CROP_PX / 2, CROP_PX / 2)
                if p is not None:
                    xyxy, pc, kxy, kc = p
                    row.update(found=1, pconf=float(pc),
                               px1=x0 + xyxy[0] / scale, py1=y0 + xyxy[1] / scale,
                               px2=x0 + xyxy[2] / scale, py2=y0 + xyxy[3] / scale)
                    for k in range(KP):
                        row[f"kx{k}"] = x0 + kxy[k, 0] / scale if kc[k] > 0 else np.nan
                        row[f"ky{k}"] = y0 + kxy[k, 1] / scale if kc[k] > 0 else np.nan
                        row[f"kc{k}"] = float(kc[k])
                else:
                    for k in range(KP):
                        row[f"kx{k}"] = row[f"ky{k}"] = np.nan
                        row[f"kc{k}"] = 0.0
                rows.append(row)
            n_done += 1
            if n_done % 25 == 0:
                flush()
                el = time.time() - t0
                print(f"  {n_done}/{len(jobs)}  {el:.0f}s ({el / n_done:.2f}s/clip)", flush=True)
    flush()
    print(f"done {n_done} clips in {time.time() - t0:.0f}s -> {OUT}")
    if existing is not None:
        print("rows:", len(existing), "| pose found rate by role:", existing.groupby("role").found.mean().round(3).to_dict(),
              "| mean kp conf (found):", round(existing[existing.found == 1][[f"kc{k}" for k in range(KP)]].values.mean(), 3))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/pose_merge.py
"""Average several RTMPose keypoint tables (different models and/or test-time augmentations) into one:
confidence-weighted mean of the coordinates per (clip, role, frame) and the mean confidence. Reduces keypoint
noise where the estimators agree and down-weights the less confident one where they disagree.

Writes CACHE_DIR/pose_rtm_<out>.parquet (same schema as pose_rtm_{m,x}.parquet) plus kp_disagree_mean / _feet
(mean pairwise distance between the inputs, in box heights).
Run:  python src/pose_merge.py                                   (m + x -> a, as before)
      python src/pose_merge.py --inputs m x mf4 xf --out a2       (m, x and their TTA runs -> a2)
"""
import argparse
import os
import sys
from itertools import combinations

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import pyarrow  # noqa: F401
import numpy as np
import pandas as pd

from config import CACHE_DIR
from pose_rtm import KP

KEYS = ["clip", "role", "frame_idx"]

if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--inputs", nargs="+", default=["m", "x"])
    ap.add_argument("--out", default="a")
    a = ap.parse_args()
    kx_c = [f"kx{k}" for k in range(KP)]
    ky_c = [f"ky{k}" for k in range(KP)]
    kc_c = [f"kc{k}" for k in range(KP)]
    tabs = {s: pd.read_parquet(CACHE_DIR / f"pose_rtm_{s}.parquet").set_index(KEYS).sort_index() for s in a.inputs}
    common = None
    for s, T in tabs.items():
        common = T.index if common is None else common.intersection(T.index)
    print("rows " + ", ".join(f"{s} {len(T)}" for s, T in tabs.items()) + f", common {len(common)}")
    tabs = {s: T.loc[common] for s, T in tabs.items()}
    A = tabs[a.inputs[0]].copy()
    W = {s: T[kc_c].values + 1e-6 for s, T in tabs.items()}
    wsum = sum(W.values())
    A[kx_c] = sum(T[kx_c].values * W[s] for s, T in tabs.items()) / wsum
    A[ky_c] = sum(T[ky_c].values * W[s] for s, T in tabs.items()) / wsum
    A[kc_c] = np.mean([T[kc_c].values for T in tabs.values()], 0)
    A["pconf"] = A[kc_c[:17]].mean(1)
    # pairwise disagreement between the inputs (pixels, normalised by the box height) as quality columns
    D = []
    for s1, s2 in combinations(a.inputs, 2):
        T1, T2 = tabs[s1], tabs[s2]
        D.append(np.hypot(T1[kx_c].values - T2[kx_c].values, T1[ky_c].values - T2[ky_c].values) / T1["box_h"].values[:, None])
    d = np.mean(D, 0)
    A["kp_disagree_mean"] = d.mean(1)
    A["kp_disagree_feet"] = d[:, 20:26].mean(1)
    A = A.drop(columns=[c for c in ("kp_tta_spread",) if c in A.columns]).reset_index()
    tmp = CACHE_DIR / f"pose_rtm_{a.out}.tmp.parquet"
    A.to_parquet(tmp, index=False)
    os.replace(tmp, CACHE_DIR / f"pose_rtm_{a.out}.parquet")
    print(f"wrote pose_rtm_{a.out}.parquet: {A.shape}; mean disagreement (box heights) all {A.kp_disagree_mean.mean():.3f}, "
          f"feet {A.kp_disagree_feet.mean():.3f}")


In [ ]:
%%writefile src/pose_rtm.py
"""Stage 3b: RTMPose (Halpe-26: COCO-17 + head/neck/hip + big toe / small toe / heel of each
foot) top-down on the kicker and keeper role boxes for EVERY frame in the last WINDOW_S
seconds. Same sampling and output schema as pose_crops.py (keypoints in ORIGINAL frame pixels),
with 26 keypoints instead of 17, so features_v2.sample_track works unchanged on kx0..kx16 and the
foot keypoints (20..25) add the plant-/kick-foot orientation.

Top-down: the interpolated role box (padding 1.25) is warped to the model input (192x256 for
rtmpose-m, 288x384 for rtmpose-x), i.e. the 30-80 px players are upscaled 3-5x by the affine.

Output: CACHE_DIR/pose_rtm.parquet (+ _m / _x suffix by mode), one row per (clip, role, frame).
Run:  python src/pose_rtm.py [--mode balanced|performance] [--limit N] [--viz]
"""
import argparse
import os
import sys
import time
from concurrent.futures import ThreadPoolExecutor

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import pyarrow  # noqa: F401
import cv2
import numpy as np
import pandas as pd

from config import CACHE_DIR, MODELS_DIR, PRIVATE_DIR, PUBLIC_DIR
from framecache import frame_timestamps, read_selected
from pose_crops import interp_boxes

os.environ.setdefault("XDG_CACHE_HOME", str(MODELS_DIR))   # rtmlib downloads -> D:\cv3\models\rtmlib
cv2.setNumThreads(2)

WINDOW_S = 0.4
KP = 26
NAMES = ["nose", "l_eye", "r_eye", "l_ear", "r_ear", "l_sh", "r_sh", "l_el", "r_el", "l_wr", "r_wr", "l_hip", "r_hip",
         "l_knee", "r_knee", "l_ank", "r_ank", "head", "neck", "hip", "l_toe", "r_toe", "l_stoe", "r_stoe", "l_heel", "r_heel"]
MODES = {
    "balanced": ("https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/onnx_sdk/rtmpose-m_simcc-body7_pt-body7-halpe26_700e-256x192-4d3e73dd_20230605.zip", (192, 256)),
    "performance": ("https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/onnx_sdk/rtmpose-x_simcc-body7_pt-body7-halpe26_700e-384x288-7fb6e239_20230606.zip", (288, 384)),
}
SKELETON = [(15, 13), (13, 11), (16, 14), (14, 12), (11, 12), (5, 11), (6, 12), (5, 6), (5, 7), (6, 8), (7, 9), (8, 10),
            (0, 5), (0, 6), (15, 24), (15, 20), (20, 22), (16, 25), (16, 21), (21, 23)]
# left <-> right joint swap under a horizontal flip (Halpe-26 order of NAMES)
FLIP_PERM = np.arange(KP)
for _l, _r in [(1, 2), (3, 4), (5, 6), (7, 8), (9, 10), (11, 12), (13, 14), (15, 16), (20, 21), (22, 23), (24, 25)]:
    FLIP_PERM[_l], FLIP_PERM[_r] = _r, _l
TTA = {"none": [(False, 1.0)], "flip": [(False, 1.0), (True, 1.0)],
       "flip4": [(False, 1.0), (True, 1.0), (False, 1.15), (True, 1.15)]}


def out_path(mode, tta="none"):
    return CACHE_DIR / f"pose_rtm_{'m' if mode == 'balanced' else 'x'}{'' if tta == 'none' else 'f' if tta == 'flip' else 'f4'}.parquet"


def infer_tta(model, frame, bbox, passes):
    """Run the pose model over the TTA passes (horizontal flip with L/R swap, box scale jitter) and return the
    confidence-weighted mean keypoints [26, 2], the mean scores [26] and the mean spread between passes (px)."""
    H, W = frame.shape[:2]
    x1, y1, x2, y2 = bbox
    cx, cy, w, h = (x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1
    kps, scs = [], []
    flipped = None
    for flip, scale in passes:
        bw, bh = w * scale, h * scale
        b = [cx - bw / 2, cy - bh / 2, cx + bw / 2, cy + bh / 2]
        if flip:
            if flipped is None:
                flipped = np.ascontiguousarray(frame[:, ::-1])
            bf = [W - 1 - b[2], b[1], W - 1 - b[0], b[3]]
            kp, sc = model(flipped, bboxes=[bf])
            kp, sc = kp[0][FLIP_PERM].copy(), sc[0][FLIP_PERM].copy()
            kp[:, 0] = W - 1 - kp[:, 0]
        else:
            kp, sc = model(frame, bboxes=[b])
            kp, sc = kp[0], sc[0]
        kps.append(kp)
        scs.append(sc)
    if len(kps) == 1:
        return kps[0], scs[0], 0.0
    K, S = np.stack(kps), np.stack(scs)                     # [n, 26, 2], [n, 26]
    wgt = np.clip(S, 1e-3, None)[..., None]
    mean = (K * wgt).sum(0) / wgt.sum(0)
    spread = float(np.linalg.norm(K - mean, axis=2).mean())
    return mean, S.mean(0), spread


def load_clip(job):
    clip, path, roles_tr = job
    ts = frame_timestamps(path)
    if len(ts) == 0:
        return clip, None
    t_from_end = (ts[-1] - ts) / 1000.0
    idx = np.where(t_from_end <= WINDOW_S + 1e-6)[0]
    frames = read_selected(path, set(idx.tolist()))
    return clip, (ts, t_from_end, idx, frames, roles_tr)


def draw(frame, box, kp, sc, thr=0.3):
    cx, cy, w, h = box
    side = int(max(2.2 * h, 1.6 * w, 64))
    x0, y0 = int(cx - side / 2), int(cy - side / 2)
    out = np.full((side, side, 3), 114, np.uint8)
    H, W = frame.shape[:2]
    sx0, sy0, sx1, sy1 = max(0, x0), max(0, y0), min(W, x0 + side), min(H, y0 + side)
    if sx1 > sx0 and sy1 > sy0:
        out[sy0 - y0:sy1 - y0, sx0 - x0:sx1 - x0] = frame[sy0:sy1, sx0:sx1]
    s = 256 / side
    out = cv2.resize(out, (256, 256), interpolation=cv2.INTER_CUBIC)
    P = (kp - [x0, y0]) * s
    for a, b in SKELETON:
        if sc[a] > thr and sc[b] > thr:
            col = (0, 255, 255) if a >= 20 or b >= 20 else (0, 200, 0)
            cv2.line(out, tuple(P[a].astype(int)), tuple(P[b].astype(int)), col, 1, cv2.LINE_AA)
    for k in range(KP):
        if sc[k] > thr:
            col = (0, 0, 255) if k in (15, 20, 22, 24) else (255, 0, 0) if k in (16, 21, 23, 25) else (255, 255, 255)
            cv2.circle(out, tuple(P[k].astype(int)), 2, col, -1, cv2.LINE_AA)
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--mode", default="balanced", choices=list(MODES))
    ap.add_argument("--limit", type=int, default=0)
    ap.add_argument("--viz", action="store_true", help="write a contact sheet of the last-frame poses (first 24 clips)")
    ap.add_argument("--threads", type=int, default=0, help="onnxruntime intra-op threads (0 = default)")
    ap.add_argument("--tta", default="none", choices=list(TTA), help="test-time augmentation: flip = + mirrored pass, "
                    "flip4 = + mirrored and 1.15x box passes; keypoints are confidence-averaged over the passes")
    a = ap.parse_args()

    import onnxruntime as ort
    from rtmlib import RTMPose
    url, size = MODES[a.mode]
    if a.threads:
        so = ort.SessionOptions()
        so.intra_op_num_threads = a.threads
    model = RTMPose(url, model_input_size=size, backend="onnxruntime", device="cpu")
    if a.threads:
        model.session = ort.InferenceSession(model.onnx_model, so, providers=["CPUExecutionProvider"])
    OUT = out_path(a.mode, a.tta)
    passes = TTA[a.tta]

    RF = pd.read_parquet(CACHE_DIR / "roles_frames.parquet")
    RC = pd.read_csv(CACHE_DIR / "roles_clip.csv")
    clips = RC[(RC.kicker_found == 1) | (RC.keeper_found == 1)]["clip"].tolist()
    done, existing = set(), None
    if OUT.exists():
        existing = pd.read_parquet(OUT)
        done = set(existing["clip"])
    clips = [c for c in clips if c not in done]
    if a.limit:
        clips = clips[: a.limit]
    print(f"{a.mode}: {len(done)} clips cached, {len(clips)} to do", flush=True)

    def path_of(c):
        return str((PUBLIC_DIR if c.endswith("_public.mp4") else PRIVATE_DIR) / c)

    jobs = [(c, path_of(c), RF[RF["clip"] == c]) for c in clips]
    rows, t0, n_done, tiles = [], time.time(), 0, []

    def flush():
        nonlocal existing, rows
        if not rows:
            return
        new = pd.DataFrame(rows)
        existing = new if existing is None else pd.concat([existing, new], ignore_index=True)
        tmp = OUT.with_suffix(".tmp.parquet")
        existing.to_parquet(tmp, index=False)
        os.replace(tmp, OUT)
        rows = []

    with ThreadPoolExecutor(max_workers=2) as ex:
        futs = [ex.submit(load_clip, j) for j in jobs[:4]]
        nxt = 4
        while futs:
            clip, data = futs.pop(0).result()
            if nxt < len(jobs):
                futs.append(ex.submit(load_clip, jobs[nxt]))
                nxt += 1
            if data is None:
                print("DECODE FAIL", clip, flush=True)
                continue
            ts, t_from_end, idx, frames, roles_tr = data
            for role in ("kicker", "keeper"):
                tr = roles_tr[roles_tr.role == role]
                if len(tr) == 0:
                    continue
                boxes = interp_boxes(tr, t_from_end[idx])
                for i, fi in enumerate(idx):
                    cx, cy, w, h = boxes[i]
                    bbox = [cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2]
                    kp, sc, spread = infer_tta(model, frames[fi], bbox, passes)
                    row = {"clip": clip, "role": role, "frame_idx": int(fi), "ts_ms": float(ts[fi]),
                           "t_from_end_s": float(t_from_end[fi]), "box_cx": cx, "box_cy": cy, "box_w": w, "box_h": h,
                           "found": 1, "pconf": float(sc[:17].mean()), "kp_tta_spread": spread / max(h, 1.0),
                           "px1": bbox[0], "py1": bbox[1], "px2": bbox[2], "py2": bbox[3]}
                    for k in range(KP):
                        row[f"kx{k}"], row[f"ky{k}"], row[f"kc{k}"] = float(kp[k, 0]), float(kp[k, 1]), float(sc[k])
                    rows.append(row)
                    if a.viz and role == "kicker" and fi == idx[-1] and len(tiles) < 24:
                        tiles.append(cv2.putText(draw(frames[fi], boxes[i], kp, sc), clip.split("_")[0], (3, 12),
                                                 cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1))
            n_done += 1
            if n_done % 25 == 0:
                flush()
                el = time.time() - t0
                print(f"  {n_done}/{len(jobs)}  {el:.0f}s ({el / n_done:.2f}s/clip)", flush=True)
    flush()
    print(f"done {n_done} clips in {time.time() - t0:.0f}s -> {OUT}")
    if a.viz and tiles:
        while len(tiles) % 6:
            tiles.append(np.zeros_like(tiles[0]))
        sheet = np.vstack([np.hstack(tiles[r:r + 6]) for r in range(0, len(tiles), 6)])
        p = CACHE_DIR / f"viz_pose_rtm_{a.mode}.jpg"
        cv2.imwrite(str(p), sheet, [cv2.IMWRITE_JPEG_QUALITY, 90])
        print("viz ->", p)
    if existing is not None:
        print("rows:", len(existing), "| mean kp conf by role:",
              existing.groupby("role")[[f"kc{k}" for k in range(17)]].mean().mean(1).round(3).to_dict(),
              "| feet conf:", existing[[f"kc{k}" for k in range(20, 26)]].mean().round(3).to_dict())


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/probe_frozen.py
"""Probes on the frozen-encoder tables written by feat_frozen.py.

Feature assembly per clip and flip (axis 1 of the tables: 0 = original, 1 = mirrored):
  DINOv2 dinov2b_full [N, 2, 30, 1536]: LAST8 (k=0..7) mean | last (k=0) | delta (k=0 - k=7) |
                                        MID (k=8..23) mean | context (k=24..29) mean     -> 5 x 1536
  SSv2   videomae_ssv2_full [N, 2, 942]: [fc_norm pooled token | 174 logits]              -> 942
  metadata: preferred_foot one-hot (swapped on flipped rows) + gender one-hot             -> 4
Training rows = 1300 original + 1300 flipped (labels mapped through config.FLIP_MAP; the flipped
row sits in the same source-grouped fold as its clip). Per fold: StandardScaler -> PCA(256, whiten),
both fit on the training rows only -> per target LogisticRegression (C grid) and a torch MLP
(5 seeds). Inference (OOF and private): p = 0.5 * (f(orig) + unswap(f(flip))), unswap = swap the
left/right probability columns. Private predictions average the 5 fold models.

Outputs: RUNS_DIR/frozen_<subset>/ (subset in dinov2b / ssv2 / both)
  variants/<variant>/{oof,test}_{kick,gk,goal}.npy    every variant
  oof_*.npy, test_*.npy, submission.csv, metrics.json best variant by OOF weighted score

Usage: python src/probe_frozen.py [--subsets both dinov2b ssv2] [--seeds 5] [--device cpu]
"""
import argparse
import json
import os
import sys
import time
from pathlib import Path

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import numpy as np
import pandas as pd
import torch
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from torch import nn

from config import CACHE_DIR, CLASSES, FLIP_MAP, RUNS_DIR, TARGETS, WEIGHTS
from cv import get_folds, load_private, load_public, probs_to_labels, score_report, write_submission


def composite(y_true: pd.DataFrame, pred: pd.DataFrame, mask: np.ndarray | None = None) -> float:
    m = np.ones(len(y_true), bool) if mask is None else mask
    return float(sum(WEIGHTS[t] * (y_true[t].values[m] == pred[t].values[m]).mean() for t in TARGETS))

FEATS_DIR = CACHE_DIR / "feats"
SHORT = {"kick_direction": "kick", "goalkeeper_direction": "gk", "goal_outcome": "goal"}
SUBSETS = {"dinov2b": ["dino"], "ssv2": ["vmae"], "both": ["dino", "vmae"]}
C_GRID = [0.003, 0.01, 0.03, 0.1]
N_PCA = 256


# ----------------------------------------------------------------------------- features
def dino_pooled(path: Path, chunk: int = 100) -> np.ndarray:
    """[N, 2, 30, 1536] fp16 on disk -> [N, 2, 5*1536] float32 (chunked to keep RAM low)."""
    d = np.load(path, mmap_mode="r")
    n = d.shape[0]
    out = np.empty((n, 2, 5 * d.shape[-1]), np.float32)
    for i in range(0, n, chunk):
        x = np.asarray(d[i:i + chunk], dtype=np.float32)
        out[i:i + chunk] = np.concatenate(
            [x[:, :, 0:8].mean(2), x[:, :, 0], x[:, :, 0] - x[:, :, 7], x[:, :, 8:24].mean(2), x[:, :, 24:30].mean(2)],
            axis=-1)
    return out


def metadata_onehot(meta: pd.DataFrame) -> np.ndarray:
    """[N, 2, 4]: foot_left, foot_right, gender_M, gender_F; foot swapped on the flipped copy."""
    foot = meta["preferred_foot"].values
    gender = meta["gender"].values
    orig = np.stack([foot == "left", foot == "right", gender == "M", gender == "F"], 1).astype(np.float32)
    flip = orig.copy()
    flip[:, [0, 1]] = orig[:, [1, 0]]
    return np.stack([orig, flip], 1)


def load_blocks(feats_dir: Path, pub: pd.DataFrame, priv: pd.DataFrame) -> dict[str, np.ndarray]:
    index = pd.read_csv(feats_dir / "index.csv")
    expected = list(pub["clip"]) + list(priv["clip"])
    assert list(index["clip"]) == expected, "feats/index.csv row order differs from load_public()+load_private()"
    blocks = {}
    if (feats_dir / "dinov2b_full.npy").exists():
        blocks["dino"] = dino_pooled(feats_dir / "dinov2b_full.npy")
    if (feats_dir / "videomae_ssv2_full.npy").exists():
        blocks["vmae"] = np.asarray(np.load(feats_dir / "videomae_ssv2_full.npy"), dtype=np.float32)
    blocks["meta"] = metadata_onehot(pd.concat([pub, priv], ignore_index=True))
    for k, v in blocks.items():
        print(f"  block {k}: {v.shape}")
    return blocks


def assemble(blocks: dict[str, np.ndarray], names: list[str], with_meta: bool = True) -> np.ndarray:
    return np.concatenate([blocks[n] for n in names] + ([blocks["meta"]] if with_meta else []), axis=-1)


# ----------------------------------------------------------------------------- labels / flips
def encode_labels(df: pd.DataFrame) -> dict[str, np.ndarray]:
    return {t: np.array([CLASSES[t].index(v) for v in df[t]]) for t in TARGETS}


def flipped_labels(df: pd.DataFrame) -> dict[str, np.ndarray]:
    return {t: np.array([CLASSES[t].index(FLIP_MAP[t][v]) for v in df[t]]) for t in TARGETS}


def unswap(p: np.ndarray, target: str) -> np.ndarray:
    """Probabilities predicted on the mirrored input -> original orientation (swap left/right columns)."""
    cls = CLASSES[target]
    perm = [cls.index(FLIP_MAP[target][c]) for c in cls]
    return p[:, perm]


# ----------------------------------------------------------------------------- MLP
class MLP(nn.Module):
    def __init__(self, d_in: int, n_cls: int, hidden: int = 512, p_in: float = 0.5, p_h: float = 0.3):
        super().__init__()
        self.net = nn.Sequential(nn.Dropout(p_in), nn.Linear(d_in, hidden), nn.GELU(), nn.Dropout(p_h),
                                 nn.Linear(hidden, n_cls))

    def forward(self, x):
        return self.net(x)


def train_mlp(z: np.ndarray, y: np.ndarray, n_cls: int, seed: int, device: str, epochs: int = 40,
              bs: int = 128, lr: float = 1e-3, wd: float = 1e-2, ls: float = 0.1) -> MLP:
    torch.manual_seed(seed)
    model = MLP(z.shape[1], n_cls).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    lossf = nn.CrossEntropyLoss(label_smoothing=ls)
    zt = torch.as_tensor(z, dtype=torch.float32, device=device)
    yt = torch.as_tensor(y, dtype=torch.long, device=device)
    g = torch.Generator(device="cpu").manual_seed(seed)
    model.train()
    for _ in range(epochs):
        perm = torch.randperm(len(zt), generator=g).to(device)
        for i in range(0, len(perm), bs):
            idx = perm[i:i + bs]
            opt.zero_grad(set_to_none=True)
            lossf(model(zt[idx]), yt[idx]).backward()
            opt.step()
    model.eval()
    return model


@torch.no_grad()
def mlp_proba(models: list[MLP], z: np.ndarray, device: str) -> np.ndarray:
    zt = torch.as_tensor(z, dtype=torch.float32, device=device)
    return np.mean([torch.softmax(m(zt), 1).cpu().numpy() for m in models], axis=0)


# ----------------------------------------------------------------------------- probes
def run_subset(name: str, X: np.ndarray, pub: pd.DataFrame, priv: pd.DataFrame, folds: pd.Series,
               n_seeds: int, device: str, out_root: Path, meta: np.ndarray | None = None) -> dict:
    """meta: optional [N, 2, 4] one-hot block appended AFTER scaler+PCA (passthrough) instead of inside them."""
    n_pub = len(pub)
    y_orig, y_flip = encode_labels(pub), flipped_labels(pub)
    X_pub, X_priv = X[:n_pub], X[n_pub:]
    M_pub, M_priv = (meta[:n_pub], meta[n_pub:]) if meta is not None else (None, None)
    variants = [f"lr_C{c:g}" for c in C_GRID] + ["mlp"]
    oof = {v: {t: np.zeros((n_pub, len(CLASSES[t]))) for t in TARGETS} for v in variants}
    oof_notta = {v: {t: np.zeros((n_pub, len(CLASSES[t]))) for t in TARGETS} for v in variants}
    test = {v: {t: np.zeros((len(priv), len(CLASSES[t]))) for t in TARGETS} for v in variants}
    fold_ids = sorted(folds.unique())
    t_sub = time.time()
    for k in fold_ids:
        t0 = time.time()
        tr = (folds != k).values
        va = ~tr
        Z_tr_raw = np.concatenate([X_pub[tr, 0], X_pub[tr, 1]])                 # orig rows then flipped rows
        y_tr = {t: np.concatenate([y_orig[t][tr], y_flip[t][tr]]) for t in TARGETS}
        scaler = StandardScaler().fit(Z_tr_raw)
        pca = PCA(n_components=N_PCA, whiten=True, svd_solver="randomized", random_state=0)
        pca.fit(scaler.transform(Z_tr_raw))
        tf = lambda a, m=None: (pca.transform(scaler.transform(a)) if m is None  # noqa: E731
                                else np.concatenate([pca.transform(scaler.transform(a)), m], axis=1))
        if meta is None:
            Z_tr = tf(Z_tr_raw)
            Z_va = [tf(X_pub[va, 0]), tf(X_pub[va, 1])]
            Z_te = [tf(X_priv[:, 0]), tf(X_priv[:, 1])]
        else:
            Z_tr = tf(Z_tr_raw, np.concatenate([M_pub[tr, 0], M_pub[tr, 1]]))
            Z_va = [tf(X_pub[va, 0], M_pub[va, 0]), tf(X_pub[va, 1], M_pub[va, 1])]
            Z_te = [tf(X_priv[:, 0], M_priv[:, 0]), tf(X_priv[:, 1], M_priv[:, 1])]
        for t in TARGETS:
            n_cls = len(CLASSES[t])
            predictors = {}
            for c in C_GRID:
                lr = LogisticRegression(C=c, max_iter=3000).fit(Z_tr, y_tr[t])
                predictors[f"lr_C{c:g}"] = lr.predict_proba
            mlps = [train_mlp(Z_tr, y_tr[t], n_cls, seed, device) for seed in range(n_seeds)]
            predictors["mlp"] = lambda z, m=mlps: mlp_proba(m, z, device)
            for v, f in predictors.items():
                p_va = [f(Z_va[0]), f(Z_va[1])]
                p_te = [f(Z_te[0]), f(Z_te[1])]
                oof_notta[v][t][va] = p_va[0]
                oof[v][t][va] = 0.5 * (p_va[0] + unswap(p_va[1], t))
                test[v][t] += 0.5 * (p_te[0] + unswap(p_te[1], t)) / len(fold_ids)
        print(f"  [{name}] fold {k}: {tr.sum()} train clips x2, {va.sum()} val, "
              f"PCA var {pca.explained_variance_ratio_.sum():.3f}, {time.time() - t0:.0f}s", flush=True)

    # ---- reports
    shared = set(pub["source_id"]) & set(priv["source_id"])
    unseen = ~pub["source_id"].isin(shared).values
    out_dir = out_root / f"frozen_{name}"
    metrics = {"subset": name, "n_pca": N_PCA, "n_seeds_mlp": n_seeds, "meta_passthrough": meta is not None,
               "variants": {}}
    best, best_score = None, -1.0
    print(f"\n==== subset {name} ({time.time() - t_sub:.0f}s) ====")
    for v in variants:
        pred = probs_to_labels(oof[v])
        score = score_report(pub, pred, folds, f"{name}/{v}")
        s_nt = composite(pub, probs_to_labels(oof_notta[v]))
        s_unseen = composite(pub, pred, unseen)
        print(f"   no-TTA {s_nt:.4f} | unseen-source clips ({unseen.sum()}) {s_unseen:.4f}")
        per_fold = [composite(pub, pred, (folds == k).values) for k in fold_ids]
        metrics["variants"][v] = {
            "score": score, "score_no_tta": s_nt, "score_unseen_sources": s_unseen,
            "acc": {t: float((pub[t].values == pred[t].values).mean()) for t in TARGETS},
            "fold_scores": per_fold,
            "test_pred_dist": {t: pd.Series(probs_to_labels(test[v])[t]).value_counts().to_dict() for t in TARGETS},
        }
        vd = out_dir / "variants" / v
        vd.mkdir(parents=True, exist_ok=True)
        for t in TARGETS:
            np.save(vd / f"oof_{SHORT[t]}.npy", oof[v][t].astype(np.float32))
            np.save(vd / f"test_{SHORT[t]}.npy", test[v][t].astype(np.float32))
        if score > best_score:
            best, best_score = v, score
    metrics["best_variant"] = best
    for t in TARGETS:
        np.save(out_dir / f"oof_{SHORT[t]}.npy", oof[best][t].astype(np.float32))
        np.save(out_dir / f"test_{SHORT[t]}.npy", test[best][t].astype(np.float32))
    sub = probs_to_labels(test[best])
    sub.insert(0, "clip", priv["clip"].values)
    print(f"best variant for {name}: {best} ({best_score:.4f})")
    write_submission(sub, out_dir / "submission.csv")
    (out_dir / "metrics.json").write_text(json.dumps(metrics, indent=2))
    return metrics


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--subsets", nargs="+", default=list(SUBSETS), choices=list(SUBSETS))
    ap.add_argument("--seeds", type=int, default=5, help="MLP seeds averaged")
    ap.add_argument("--device", default="cpu", help="device for the MLP (cpu is enough for a 256->512 net)")
    ap.add_argument("--feats", default=str(FEATS_DIR))
    ap.add_argument("--runs", default=str(RUNS_DIR))
    ap.add_argument("--meta-passthrough", action="store_true",
                    help="append the metadata one-hot after scaler+PCA (keeps the foot prior undiluted); "
                         "outputs go to frozen_<subset>_mp/")
    args = ap.parse_args()
    torch.set_num_threads(4)

    pub, priv = load_public(), load_private()
    folds = get_folds(pub)
    print(f"public {len(pub)} private {len(priv)}; folds {folds.value_counts().sort_index().to_dict()}")
    blocks = load_blocks(Path(args.feats), pub, priv)
    results = {}
    for s in args.subsets:
        if not all(b in blocks for b in SUBSETS[s]):
            print(f"subset {s}: missing table(s) {[b for b in SUBSETS[s] if b not in blocks]}, skipping")
            continue
        X = assemble(blocks, SUBSETS[s], with_meta=not args.meta_passthrough)
        tag = f"{s}_mp" if args.meta_passthrough else s
        print(f"\nsubset {tag}: features {X.shape}" + (" + 4 metadata dims after PCA" if args.meta_passthrough else ""))
        results[tag] = run_subset(tag, X, pub, priv, folds, args.seeds, args.device, Path(args.runs),
                                  meta=blocks["meta"] if args.meta_passthrough else None)

    print("\n================ summary (OOF, flip-TTA) ================")
    print(f"{'subset':8s} {'variant':10s} {'score':>7s} {'kick':>6s} {'gk':>6s} {'goal':>6s} {'unseen':>7s} {'noTTA':>7s}")
    for s, m in results.items():
        for v, r in m["variants"].items():
            a = r["acc"]
            print(f"{s:8s} {v:10s} {r['score']:7.4f} {a['kick_direction']:6.3f} {a['goalkeeper_direction']:6.3f} "
                  f"{a['goal_outcome']:6.3f} {r['score_unseen_sources']:7.4f} {r['score_no_tta']:7.4f}")
    print("chance bars: metadata-only 0.5504, majority 0.5172")
    summary = Path(args.runs) / "frozen_summary.json"
    old = json.loads(summary.read_text()) if summary.exists() else {}
    old.update(results)
    summary.write_text(json.dumps(old, indent=2))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/roles.py
"""Stage 2: kicker / keeper identification from the cached detections.

1. Link person boxes across the sampled frames into tracklets (greedy, backwards
   from the last frame).
2. At a reference frame (~0.23 s before the end = the click frame) build
   per-candidate features; train two LightGBM pickers (kicker, keeper) supervised
   by public_position_clicks.csv; evaluate with source-grouped CV.
3. Apply to every clip (public + private): role boxes on every sampled frame
   (via the tracklets) + per-clip geometry: goal side, kicker->keeper axis u and
   the kicker-right axis r = (-u_y, u_x).

Outputs: CACHE_DIR/roles_frames.parquet, CACHE_DIR/roles_clip.csv
Run:  python src/roles.py [--viz]
"""
import argparse
import os
import pickle
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold

from config import CACHE_DIR, CLICKS_CSV, SEED
from cv import load_private, load_public

DET = CACHE_DIR / "detections.parquet"
OUT_FRAMES = CACHE_DIR / "roles_frames.parquet"
OUT_CLIP = CACHE_DIR / "roles_clip.csv"
OUT_MODEL = CACHE_DIR / "roles_models.pkl"
REF_T = 0.23          # seconds before the end: the annotated click frame
PERSON, BALL = 0, 32

FEATS_K = ["cx", "cy", "w", "h", "aspect", "conf", "h_rank", "conf_rank", "n_cand", "dcx_med", "dcy_med",
           "dist_center", "nn_dist", "n_near", "track_len", "y_bottom",
           "mot_e_dx", "mot_e_dy", "mot_e_speed", "mot_e_gap", "mot_f_dx", "mot_f_dy", "mot_f_speed",
           "h_change", "w_change", "ball_dist", "ball_found"]
FEATS_G = FEATS_K + ["p_kicker", "rel_dx", "rel_dy", "rel_absdx", "rel_dist", "rel_h", "rel_higher", "rel_same_track"]


# ----------------------------------------------------------------------------- tracklets
def build_tracks(fr: pd.DataFrame, W: int) -> pd.DataFrame:
    """Greedy backward linking of boxes across sampled frames. Adds 'track'."""
    fr = fr.copy()
    fr["track"] = -1
    nxt, prev = 0, {}
    for t in sorted(fr.t_from_end_s.unique()):          # 0.0 (last frame) first
        sel = fr.index[fr.t_from_end_s == t]
        cx = ((fr.loc[sel, "x1"] + fr.loc[sel, "x2"]) / 2).values
        cy = ((fr.loc[sel, "y1"] + fr.loc[sel, "y2"]) / 2).values
        h = (fr.loc[sel, "y2"] - fr.loc[sel, "y1"]).values
        pairs = []
        for i in range(len(sel)):
            for tid, (pcx, pcy, ph, pt) in prev.items():
                gap = max(abs(t - pt), 0.1)
                gate = max(1.0 * ph, 0.04 * W) * np.sqrt(gap / 0.1)
                d = np.hypot(cx[i] - pcx, cy[i] - pcy)
                if d < gate:
                    pairs.append((d / gate, i, tid))
        pairs.sort()
        used_i, used_t = set(), set()
        for _, i, tid in pairs:
            if i in used_i or tid in used_t:
                continue
            fr.loc[sel[i], "track"] = tid
            used_i.add(i)
            used_t.add(tid)
        for i in range(len(sel)):
            if fr.loc[sel[i], "track"] < 0:
                fr.loc[sel[i], "track"] = nxt
                nxt += 1
        new = {int(fr.loc[sel[i], "track"]): (cx[i], cy[i], h[i], t) for i in range(len(sel))}
        for tid, v in prev.items():                       # keep unmatched tracks alive 0.5 s
            if tid not in new and (t - v[3]) <= 0.5:
                new[tid] = v
        prev = new
    return fr


# ----------------------------------------------------------------------------- features
def candidate_features(fr: pd.DataFrame, balls: pd.DataFrame, W: int, H: int):
    """Per-candidate feature rows at the reference frame. Returns (DataFrame, t_ref) or (None, None)."""
    ts = np.array(sorted(fr.t_from_end_s.unique()))
    if len(ts) == 0:
        return None, None
    # reference frame: nearest to REF_T that has at least one person, within 0.5 s
    order = ts[np.argsort(np.abs(ts - REF_T))]
    t_ref = next((t for t in order if abs(t - REF_T) <= 0.5), None)
    if t_ref is None:
        return None, None
    ref = fr[fr.t_from_end_s == t_ref]
    n = len(ref)
    cx = ((ref.x1 + ref.x2) / 2).values
    cy = ((ref.y1 + ref.y2) / 2).values
    w = (ref.x2 - ref.x1).values
    h = (ref.y2 - ref.y1).values
    med_cx, med_cy = np.median(cx), np.median(cy)
    last = fr[fr.t_from_end_s == ts.min()]
    bc = None
    if len(balls):
        bb = balls[balls.t_from_end_s <= 0.35]
        if len(bb):
            bc = np.c_[(bb.x1 + bb.x2) / 2, (bb.y1 + bb.y2) / 2]
    rows = []
    for i, (idx, b) in enumerate(ref.iterrows()):
        tr = fr[fr.track == b.track].sort_values("t_from_end_s")
        f = {"det_index": idx, "track": int(b.track), "cx": cx[i] / W, "cy": cy[i] / H, "w": w[i] / W, "h": h[i] / H,
             "aspect": w[i] / max(h[i], 1), "conf": b.conf,
             "h_rank": int((h > h[i]).sum()), "conf_rank": int((ref.conf.values > b.conf).sum()), "n_cand": n,
             "dcx_med": (cx[i] - med_cx) / W, "dcy_med": (cy[i] - med_cy) / H,
             "dist_center": np.hypot(cx[i] / W - 0.5, cy[i] / H - 0.5), "track_len": len(tr),
             "y_bottom": b.y2 / H}
        others = np.hypot(cx - cx[i], cy - cy[i])
        others = others[others > 0]
        f["nn_dist"] = (others.min() / H) if len(others) else 1.0
        f["n_near"] = int((others < 0.1 * W).sum())
        # motion from the earliest frame within 1.0 s before the ref frame
        early = tr[(tr.t_from_end_s > t_ref) & (tr.t_from_end_s <= 1.05)]
        if len(early):
            e = early.iloc[-1]
            gap = e.t_from_end_s - t_ref
            ecx, ecy = (e.x1 + e.x2) / 2, (e.y1 + e.y2) / 2
            f["mot_e_dx"], f["mot_e_dy"] = (cx[i] - ecx) / W, (cy[i] - ecy) / H
            f["mot_e_speed"] = np.hypot(cx[i] - ecx, cy[i] - ecy) / max(h[i], 1) / max(gap, 0.05)
            f["mot_e_gap"] = gap
        else:
            f["mot_e_dx"] = f["mot_e_dy"] = f["mot_e_speed"] = f["mot_e_gap"] = np.nan
        # motion from the ref frame to the last frame
        fut = tr[tr.t_from_end_s < t_ref]
        if len(fut):
            l = fut.iloc[0]
            gap = t_ref - l.t_from_end_s
            lcx, lcy = (l.x1 + l.x2) / 2, (l.y1 + l.y2) / 2
            f["mot_f_dx"], f["mot_f_dy"] = (lcx - cx[i]) / W, (lcy - cy[i]) / H
            f["mot_f_speed"] = np.hypot(lcx - cx[i], lcy - cy[i]) / max(h[i], 1) / max(gap, 0.05)
            f["h_change"] = (l.y2 - l.y1) / max(h[i], 1) - 1
            f["w_change"] = (l.x2 - l.x1) / max(w[i], 1) - 1
        else:
            f["mot_f_dx"] = f["mot_f_dy"] = f["mot_f_speed"] = f["h_change"] = f["w_change"] = np.nan
        if bc is not None:
            d = np.hypot(bc[:, 0] - cx[i], bc[:, 1] - b.y2).min()
            f["ball_dist"], f["ball_found"] = d / max(h[i], 1), 1
        else:
            f["ball_dist"], f["ball_found"] = np.nan, 0
        rows.append(f)
    return pd.DataFrame(rows), t_ref


def keeper_relational(cand: pd.DataFrame, kicker_row) -> pd.DataFrame:
    c = cand.copy()
    if kicker_row is None:
        for k in ["rel_dx", "rel_dy", "rel_absdx", "rel_dist", "rel_h", "rel_higher", "rel_same_track"]:
            c[k] = np.nan
        return c
    c["rel_dx"] = c.cx - kicker_row.cx
    c["rel_dy"] = c.cy - kicker_row.cy
    c["rel_absdx"] = c.rel_dx.abs()
    c["rel_dist"] = np.hypot(c.rel_dx, c.rel_dy)
    c["rel_h"] = c.h / max(kicker_row.h, 1e-3)
    c["rel_higher"] = (c.cy < kicker_row.cy).astype(int)
    c["rel_same_track"] = (c.track == kicker_row.track).astype(int)
    return c


def match_click(cand: pd.DataFrame, det: pd.DataFrame, x: float, y: float):
    """Index (in cand) of the box matching a click: smallest box containing it
    (boxes expanded 20%), else nearest centre within 0.75*h. None if no match."""
    best, best_area = None, np.inf
    for i, r in cand.iterrows():
        b = det.loc[r.det_index]
        w, h = b.x2 - b.x1, b.y2 - b.y1
        if b.x1 - 0.2 * w <= x <= b.x2 + 0.2 * w and b.y1 - 0.2 * h <= y <= b.y2 + 0.2 * h and w * h < best_area:
            best, best_area = i, w * h
    if best is not None:
        return best
    d = np.hypot(cand.cx.values * det.W.iloc[0] - x, cand.cy.values * det.H.iloc[0] - y)
    j = int(np.argmin(d))
    if d[j] < 0.75 * cand.h.values[j] * det.H.iloc[0]:
        return cand.index[j]
    return None


# ----------------------------------------------------------------------------- main
def prepare_all(det: pd.DataFrame):
    """Tracklets + candidate tables for every clip. Returns dict clip -> info."""
    info = {}
    persons = det[det.cls == PERSON]
    balls = det[det.cls == BALL]
    for clip, fr in persons.groupby("clip", sort=False):
        W, H = int(fr.W.iloc[0]), int(fr.H.iloc[0])
        fr = build_tracks(fr, W)
        cand, t_ref = candidate_features(fr, balls[balls["clip"] == clip], W, H)
        info[clip] = {"fr": fr, "cand": cand, "t_ref": t_ref, "W": W, "H": H}
    return info


def fit_lgb(X, y, seed=SEED):
    import lightgbm as lgb
    m = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=15, min_child_samples=10,
                           subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=2.0,
                           random_state=seed, verbose=-1)
    m.fit(X, y)
    return m


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--viz", action="store_true")
    a = ap.parse_args()

    import time
    for attempt in range(6):   # detect.py may be rewriting the file concurrently
        try:
            det = pd.read_parquet(DET)
            break
        except Exception as e:  # noqa: BLE001
            if attempt == 5:
                raise
            print(f"parquet read failed ({str(e)[:60]}...), retrying in 5 s", flush=True)
            time.sleep(5)
    det = det[det.det_id >= 0]
    pub, priv = load_public(), load_private()
    meta = pd.concat([pub, priv], ignore_index=True).set_index("clip")
    clicks = pd.read_csv(CLICKS_CSV).set_index("clip")
    print(f"detections for {det['clip'].nunique()} clips ({det[det.partition == 'public']['clip'].nunique()} public)")

    info = prepare_all(det)

    # ---------------- training tables from the clicks
    rows_k, rows_g, stats = [], [], {"clicked": 0, "no_cand": 0, "kicker_unmatched": 0, "keeper_unmatched": 0}
    for clip in clicks.index:
        if clip not in info or info[clip]["cand"] is None:
            if clip in info:
                stats["no_cand"] += 1
            continue
        stats["clicked"] += 1
        c, fr, W, H = info[clip]["cand"], info[clip]["fr"], info[clip]["W"], info[clip]["H"]
        ck = clicks.loc[clip]
        ik = match_click(c, fr, ck.kicker_x_norm * W, ck.kicker_y_norm * H)
        ig = match_click(c, fr, ck.keeper_x_norm * W, ck.keeper_y_norm * H)
        if ik is None:
            stats["kicker_unmatched"] += 1
        if ig is None:
            stats["keeper_unmatched"] += 1
        if ik == ig:
            ik = ig = None  # ambiguous
        src = meta.loc[clip, "source_id"]
        t = c.copy()
        t["y"] = (t.index == ik).astype(int) if ik is not None else 0
        t["clip"], t["source_id"], t["has_label"] = clip, src, ik is not None
        rows_k.append(t)
        kr = c.loc[ik] if ik is not None else None
        g = keeper_relational(c, kr)
        g["p_kicker"] = (g.index == ik).astype(float) if ik is not None else np.nan
        g["y"] = (g.index == ig).astype(int) if ig is not None else 0
        g["clip"], g["source_id"], g["has_label"] = clip, src, ig is not None
        rows_g.append(g)
    TK, TG = pd.concat(rows_k), pd.concat(rows_g)
    print("click matching:", stats)

    # ---------------- grouped CV of the pickers (hit rate per clip)
    def cv_eval(T, feats, name):
        T = T[T.has_label].copy()
        clips = T["clip"].unique()
        groups = T.groupby("clip").source_id.first()
        hits, oof, fold_models = {}, {}, {}
        gkf = GroupKFold(n_splits=5)
        for tr, va in gkf.split(clips, groups=groups.loc[clips].values):
            trc, vac = set(clips[tr]), set(clips[va])
            m = fit_lgb(T[T["clip"].isin(trc)][feats], T[T["clip"].isin(trc)].y)
            V = T[T["clip"].isin(vac)].copy()
            V["p"] = m.predict_proba(V[feats])[:, 1]
            for clip, g in V.groupby("clip"):
                hits[clip] = int(g.y.values[np.argmax(g.p.values)] == 1)
                oof[clip] = g
            for s in set(groups.loc[list(vac)].values):   # cross-fit: held-out model per source
                fold_models[s] = m
        hr = np.mean(list(hits.values()))
        print(f"{name}: grouped-CV hit rate {hr:.3f} on {len(hits)} labelled clips")
        return hits, oof, fold_models

    hits_k, _, fmk = cv_eval(TK, FEATS_K, "kicker picker")
    hits_g, _, fmg = cv_eval(TG, FEATS_G, "keeper picker (true-kicker relational feats)")

    # ---------------- fit on all labelled clips, apply everywhere
    mk = fit_lgb(TK[TK.has_label][FEATS_K], TK[TK.has_label].y)
    mg = fit_lgb(TG[TG.has_label][FEATS_G], TG[TG.has_label].y)
    with open(OUT_MODEL, "wb") as f:
        pickle.dump({"kicker": mk, "keeper": mg, "feats_k": FEATS_K, "feats_g": FEATS_G}, f)

    clip_rows, frame_rows, track_rows = [], [], []
    for clip in meta.index:
        if clip not in info or info[clip]["cand"] is None:
            clip_rows.append({"clip": clip, "kicker_found": 0, "keeper_found": 0})
            continue
        d = info[clip]
        c, fr, W, H = d["cand"], d["fr"], d["W"], d["H"]
        track_rows.append(fr[["clip", "frame_idx", "t_from_end_s", "track", "conf", "x1", "y1", "x2", "y2"]])
        # cross-fit: clips from a clicked (training) source use the model that held that source out
        src = meta.loc[clip, "source_id"]
        mk_c, mg_c = fmk.get(src, mk), fmg.get(src, mg)
        pk = mk_c.predict_proba(c[FEATS_K])[:, 1]
        ik = int(np.argmax(pk))
        kicker_found = pk[ik] >= 0.05
        kr = c.iloc[ik] if kicker_found else None
        g = keeper_relational(c, kr)
        g["p_kicker"] = pk
        pg = mg_c.predict_proba(g[FEATS_G])[:, 1]
        if kicker_found:
            pg[ik] = -1.0
        ig = int(np.argmax(pg))
        keeper_found = pg[ig] >= 0.05
        row = {"clip": clip, "W": W, "H": H, "t_ref": d["t_ref"], "n_cand": len(c),
               "kicker_found": int(kicker_found), "keeper_found": int(keeper_found),
               "p_kicker": float(pk[ik]), "p_keeper": float(pg[ig]) if keeper_found else np.nan}
        roles = {}
        if kicker_found:
            roles["kicker"] = int(c.iloc[ik].track)
            row.update(kcx=c.iloc[ik].cx, kcy=c.iloc[ik].cy, kh=c.iloc[ik].h)
        if keeper_found:
            roles["keeper"] = int(c.iloc[ig].track)
            row.update(gcx=c.iloc[ig].cx, gcy=c.iloc[ig].cy, gh=c.iloc[ig].h)
        if kicker_found and keeper_found:
            ux, uy = c.iloc[ig].cx - c.iloc[ik].cx, (c.iloc[ig].cy - c.iloc[ik].cy) * H / W  # pixel-isotropic
            nrm = np.hypot(ux, uy) + 1e-9
            ux, uy = ux / nrm, uy / nrm
            row.update(u_x=ux, u_y=uy, r_x=-uy, r_y=ux, goal_side="L" if ux < 0 else "R", dist_kk=nrm)
        clip_rows.append(row)
        for role, tid in roles.items():
            for _, b in fr[fr.track == tid].iterrows():
                frame_rows.append({"clip": clip, "frame_idx": int(b.frame_idx), "t_from_end_s": float(b.t_from_end_s),
                                   "role": role, "x1": b.x1, "y1": b.y1, "x2": b.x2, "y2": b.y2, "conf": b.conf})
    RC = pd.DataFrame(clip_rows)
    RC["partition"] = RC["clip"].map(lambda c: "public" if c.endswith("_public.mp4") else "private")
    RF = pd.DataFrame(frame_rows)
    RC.to_csv(OUT_CLIP, index=False)
    RF.to_parquet(OUT_FRAMES, index=False)
    pd.concat(track_rows, ignore_index=True).to_parquet(CACHE_DIR / "tracks.parquet", index=False)

    # ---------------- report
    for part in ["public", "private"]:
        s = RC[RC.partition == part]
        print(f"{part}: kicker found {s.kicker_found.mean():.3f}, keeper found {s.keeper_found.mean():.3f}, "
              f"both {(s.kicker_found & s.keeper_found).mean():.3f}, n={len(s)}")
    both = RC[(RC.kicker_found == 1) & (RC.keeper_found == 1)]
    j = both.set_index("clip").join(clicks, how="inner")
    if len(j):
        agree = (np.sign(j.gcx - j.kcx) == np.sign(j.keeper_x_norm - j.kicker_x_norm)).mean()
        print(f"goal-side agreement with the clicks: {agree:.3f} on {len(j)} clips")
        dk = np.hypot((j.kcx - j.kicker_x_norm), (j.kcy - j.kicker_y_norm))
        dg = np.hypot((j.gcx - j.keeper_x_norm), (j.gcy - j.keeper_y_norm))
        print(f"picked-centre to click distance (norm): kicker median {dk.median():.3f} (<0.05: {(dk < .05).mean():.3f}), "
              f"keeper median {dg.median():.3f} (<0.05: {(dg < .05).mean():.3f})")
    print("goal side counts:", RC.goal_side.value_counts(dropna=False).to_dict())
    print(f"wrote {OUT_CLIP} ({len(RC)} rows), {OUT_FRAMES} ({len(RF)} rows)")

    if a.viz:
        viz(RC, RF, clicks)


def viz(RC, RF, clicks, n_each=24):
    """Contact sheets of the picked boxes (kicker green, keeper magenta; clicks as circles)."""
    import cv2
    from framecache import OFFSETS_A, W_A, H_A, load_frames
    out_dir = CACHE_DIR / "viz_roles"
    out_dir.mkdir(exist_ok=True)
    rng = np.random.RandomState(SEED)
    unclicked = RC[(RC.partition == "public") & (~RC["clip"].isin(clicks.index))]
    sets = {"public_unclicked": unclicked, "private": RC[RC.partition == "private"],
            "public_clicked": RC[(RC.partition == "public") & (RC["clip"].isin(clicks.index))]}
    for name, s in sets.items():
        pick = s.sample(min(n_each, len(s)), random_state=rng)
        tiles = []
        for _, r in pick.iterrows():
            clip = r["clip"]  # r.clip would be the Series.clip method
            k = int(np.argmin(np.abs(np.array(OFFSETS_A) - (r.t_ref if pd.notna(r.t_ref) else REF_T))))
            img = cv2.cvtColor(load_frames(clip, [k])[0], cv2.COLOR_RGB2BGR)
            sx, sy = W_A / r.W, H_A / r.H
            fb = RF[RF["clip"] == clip]
            for role, col in [("kicker", (0, 255, 0)), ("keeper", (255, 0, 255))]:
                b = fb[(fb.role == role)]
                if len(b):
                    b = b.iloc[(b.t_from_end_s - (r.t_ref if pd.notna(r.t_ref) else REF_T)).abs().argsort().iloc[0]]
                    cv2.rectangle(img, (int(b.x1 * sx), int(b.y1 * sy)), (int(b.x2 * sx), int(b.y2 * sy)), col, 2)
            if clip in clicks.index:
                ck = clicks.loc[clip]
                cv2.circle(img, (int(ck.kicker_x_norm * W_A), int(ck.kicker_y_norm * H_A)), 6, (0, 255, 0), 1)
                cv2.circle(img, (int(ck.keeper_x_norm * W_A), int(ck.keeper_y_norm * H_A)), 6, (255, 0, 255), 1)
            lab = f"{clip[:4]} side={r.goal_side if isinstance(r.goal_side, str) else '?'} pk={r.p_kicker:.2f} pg={r.p_keeper if pd.notna(r.p_keeper) else 0:.2f}"
            cv2.rectangle(img, (0, 0), (W_A, 14), (0, 0, 0), -1)
            cv2.putText(img, lab, (2, 11), cv2.FONT_HERSHEY_SIMPLEX, 0.38, (255, 255, 255), 1, cv2.LINE_AA)
            tiles.append(img)
        while len(tiles) % 4:
            tiles.append(np.zeros((H_A, W_A, 3), np.uint8))
        rows = [np.hstack(tiles[i:i + 4]) for i in range(0, len(tiles), 4)]
        for s_i in range(0, len(rows), 6):
            sheet = np.vstack(rows[s_i:s_i + 6])
            p = out_dir / f"{name}_{s_i // 6 + 1}.jpg"
            cv2.imwrite(str(p), sheet, [cv2.IMWRITE_JPEG_QUALITY, 88])
            print("wrote", p)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/selftrain.py
"""Clean self-training test for the kicker-only kick head: per fold, the fold model (trained without the
validation fold) pseudo-labels the most confident private clips; the model is refit on train + pseudo rows
(mirror-augmented, down-weighted) and evaluated on the validation fold. OOF stays leak-free because the
pseudo-labels for fold k never see fold-k labels. Writes a normal member run dir.

Run:  python src/selftrain.py --features features_v10.parquet --model et --run et_v10k_st --frac 0.5 --w 0.5
"""
import argparse
import json
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import pyarrow  # noqa: F401
import numpy as np
import pandas as pd

from config import CACHE_DIR, CLASSES, FLIP_MAP, RUNS_DIR, TARGETS
from cv import get_folds, load_private, load_public, score_report, write_submission
from features_v1 import mirror, swap_lr
from gbm_subset import subset_columns
from model_subset import make_model


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--features", default="features_v10.parquet")
    ap.add_argument("--subset", default="kicker")
    ap.add_argument("--model", default="et", choices=["et", "rf", "xgb", "lgb"])
    ap.add_argument("--run", required=True)
    ap.add_argument("--seeds", type=int, default=3)
    ap.add_argument("--frac", type=float, default=0.5, help="share of private clips pseudo-labelled (most confident)")
    ap.add_argument("--w", type=float, default=0.5, help="sample weight of pseudo rows")
    ap.add_argument("--rounds", type=int, default=1)
    a = ap.parse_args()
    F = pd.read_parquet(CACHE_DIR / a.features)
    cols = subset_columns(F.columns, a.subset)
    F = F[cols]
    pub, priv = load_public(), load_private()
    folds = get_folds(pub).values
    Xp, Xq = F.loc[pub["clip"]].values.astype(np.float32), F.loc[priv["clip"]].values.astype(np.float32)
    Xp_m, Xq_m = mirror(F.loc[pub["clip"]]).values.astype(np.float32), mirror(F.loc[priv["clip"]]).values.astype(np.float32)
    if a.model in ("et", "rf"):
        med = np.nanmedian(np.vstack([Xp, Xp_m]), 0)
        med = np.where(np.isfinite(med), med, 0.0)
        for X in (Xp, Xq, Xp_m, Xq_m):
            bad = ~np.isfinite(X)
            X[bad] = np.broadcast_to(med, X.shape)[bad]

    def model(seed):
        if a.model == "lgb":
            import lightgbm as lgb
            return lgb.LGBMClassifier(n_estimators=400, learning_rate=0.03, num_leaves=15, min_child_samples=25, subsample=0.8,
                                      subsample_freq=1, colsample_bytree=0.7, reg_lambda=5.0, verbose=-1, random_state=seed)
        return make_model(a.model, seed)

    out_dir = RUNS_DIR / a.run
    out_dir.mkdir(parents=True, exist_ok=True)
    oof_all, test_all, metrics = {}, {}, {}
    base_metrics = {}
    for t in TARGETS:
        cls = CLASSES[t]
        y = pub[t].map({c: i for i, c in enumerate(cls)}).values
        y_m = pub[t].map(FLIP_MAP[t]).map({c: i for i, c in enumerate(cls)}).values
        swap = np.array([cls.index(FLIP_MAP[t][c]) for c in cls])      # class index -> mirrored class index
        oof, test = np.zeros((len(pub), len(cls))), np.zeros((len(priv), len(cls)))
        oof0 = np.zeros((len(pub), len(cls)))
        for seed in range(a.seeds):
            for k in range(5):
                tr, va = folds != k, folds == k
                X_tr = np.vstack([Xp[tr], Xp_m[tr]])
                y_tr = np.concatenate([y[tr], y_m[tr]])
                m = model(seed + 100 * k)
                m.fit(X_tr, y_tr)
                oof0[va] += 0.5 * (m.predict_proba(Xp[va]) + swap_lr(m.predict_proba(Xp_m[va]), t)) / a.seeds
                for _ in range(a.rounds):
                    pq = 0.5 * (m.predict_proba(Xq) + swap_lr(m.predict_proba(Xq_m), t))
                    conf = pq.max(1)
                    sel = conf >= np.quantile(conf, 1 - a.frac)
                    yq = pq.argmax(1)[sel]
                    X_st = np.vstack([X_tr, Xq[sel], Xq_m[sel]])
                    y_st = np.concatenate([y_tr, yq, swap[yq]])
                    w_st = np.concatenate([np.ones(len(y_tr)), np.full(2 * sel.sum(), a.w)])
                    m = model(seed + 100 * k + 7)
                    m.fit(X_st, y_st, sample_weight=w_st)
                oof[va] += 0.5 * (m.predict_proba(Xp[va]) + swap_lr(m.predict_proba(Xp_m[va]), t)) / a.seeds
                test += 0.5 * (m.predict_proba(Xq) + swap_lr(m.predict_proba(Xq_m), t)) / (5 * a.seeds)
        oof_all[t], test_all[t] = oof, test
        np.save(out_dir / f"oof_{t.split('_')[0]}.npy", oof)
        np.save(out_dir / f"test_{t.split('_')[0]}.npy", test)
        metrics[t] = float((oof.argmax(1) == y).mean())
        base_metrics[t] = float((oof0.argmax(1) == y).mean())
        print(f"  {t}: self-trained {metrics[t]:.4f}  (baseline same seeds {base_metrics[t]:.4f})", flush=True)
    pred = pd.DataFrame({t: np.asarray(CLASSES[t])[np.argmax(oof_all[t], 1)] for t in TARGETS})
    metrics["score"] = score_report(pub, pred, pd.Series(folds), a.run)
    tp = pd.DataFrame({t: np.asarray(CLASSES[t])[np.argmax(test_all[t], 1)] for t in TARGETS})
    tp.insert(0, "clip", priv["clip"].values)
    write_submission(tp, out_dir / "submission.csv")
    json.dump({**metrics, "baseline": base_metrics, "model": a.model, "frac": a.frac, "w": a.w, "rounds": a.rounds},
              open(out_dir / "metrics.json", "w"), indent=1)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/stack.py
"""Level-2 stacking of model members.

Each member is a run directory under RUNS_DIR with oof_{kick,goalkeeper|gk,goal}.npy
(rows = load_public order) and test_*.npy (rows = load_private order), probabilities
in the ORIGINAL label frame. Per target: multinomial logistic regression on the
concatenated member log-probabilities + preferred_foot + gender, evaluated with the
shared source-grouped folds; optional additive bias on the `middle` log-prob
(tuned on OOF, applied only if it helps by >= MIN_GAIN); goal = constant `goal`
unless the stacked goal model beats it by >= MIN_GAIN.

Run:  python src/stack.py --name stackA --members gbm_v3 stack7_kicker_s42 stack7_keeper_s42
"""
import argparse
import json
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import pyarrow  # noqa: F401
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

from config import CLASSES, RUNS_DIR, TARGETS, WEIGHTS
from cv import get_folds, load_private, load_public, score_report, write_submission

MIN_GAIN = 0.005
ALIASES = {"kick_direction": ["kick"], "goalkeeper_direction": ["goalkeeper", "gk"], "goal_outcome": ["goal"]}


def load_member(run: str, target: str, split: str, n_rows: int):
    d = RUNS_DIR / run
    for a in ALIASES[target]:
        p = d / f"{split}_{a}.npy"
        if p.exists():
            x = np.load(p).astype(np.float64)
            assert x.shape == (n_rows, len(CLASSES[target])), f"{p}: {x.shape}"
            return x
    raise FileNotFoundError(f"{run}: no {split} file for {target}")


def fill_nan(P: np.ndarray, prior: np.ndarray):
    P = P.copy()
    bad = ~np.isfinite(P).all(1)
    P[bad] = prior
    P = np.clip(P, 1e-6, 1)
    return P / P.sum(1, keepdims=True), int(bad.sum())


def cv_fit_predict(X, y, folds, C):
    oof = np.zeros((len(y), len(np.unique(y))))
    for k in np.unique(folds):
        tr, va = folds != k, folds == k
        m = LogisticRegression(C=C, max_iter=3000)
        m.fit(X[tr], y[tr])
        oof[va] = m.predict_proba(X[va])
    return oof


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--name", required=True)
    ap.add_argument("--members", nargs="+", required=True)
    ap.add_argument("--Cs", nargs="+", type=float, default=[0.03, 0.1, 0.3, 1.0])
    ap.add_argument("--kick-members", nargs="+", default=None, help="members used for kick_direction (default: all)")
    ap.add_argument("--gk-members", nargs="+", default=None, help="members used for goalkeeper_direction (default: all)")
    a = ap.parse_args()
    members_for = {"kick_direction": a.kick_members or a.members,
                   "goalkeeper_direction": a.gk_members or a.members,
                   "goal_outcome": a.members}

    pub, priv = load_public(), load_private()
    folds = get_folds(pub).values
    meta_pub = np.c_[np.where(pub.preferred_foot == "right", 1.0, -1.0), (pub.gender == "M").astype(float)]
    meta_priv = np.c_[np.where(priv.preferred_foot == "right", 1.0, -1.0), (priv.gender == "M").astype(float)]
    out_dir = RUNS_DIR / a.name
    out_dir.mkdir(parents=True, exist_ok=True)

    final_oof, final_test, report = {}, {}, {}
    for t in TARGETS:
        cls = CLASSES[t]
        y = pub[t].map({c: i for i, c in enumerate(cls)}).values
        prior = np.bincount(y, minlength=len(cls)) / len(y)
        oofs, tests, names = [], [], []
        print(f"\n### {t}")
        for run in members_for[t]:
            try:
                o = load_member(run, t, "oof", len(pub))
                te = load_member(run, t, "test", len(priv))
            except FileNotFoundError as e:
                print("  skip:", e)
                continue
            o, nb = fill_nan(o, prior)
            te, nbt = fill_nan(te, prior)
            acc = (o.argmax(1) == y).mean()
            print(f"  {run:28s} acc {acc:.4f}  (nan rows oof {nb}, test {nbt})")
            oofs.append(o)
            tests.append(te)
            names.append(run)
        if not oofs:
            raise SystemExit("no members")
        # equal-weight average as a reference
        avg = np.mean(oofs, 0)
        print(f"  {'mean of members':28s} acc {(avg.argmax(1) == y).mean():.4f}")
        X = np.hstack([np.log(o) for o in oofs] + [meta_pub])
        Xt = np.hstack([np.log(te) for te in tests] + [meta_priv])
        best = None
        for C in a.Cs:
            o = cv_fit_predict(X, y, folds, C)
            acc = (o.argmax(1) == y).mean()
            print(f"  stack LR C={C:<5} acc {acc:.4f}")
            if best is None or acc > best[1]:
                best = (C, acc, o)
        C, acc, oof_s = best
        # additive bias on the rare class (middle / stands in the middle)
        bias_col, bias_val = None, 0.0
        if "middle" in cls:
            bias_col = cls.index("middle")
        elif "stands in the middle" in cls:
            bias_col = cls.index("stands in the middle")
        if bias_col is not None:
            lp = np.log(np.clip(oof_s, 1e-9, 1))
            grid = np.arange(-1.0, 1.51, 0.1)
            accs = [((lp + np.eye(len(cls))[bias_col] * b).argmax(1) == y).mean() for b in grid]
            j = int(np.argmax(accs))
            if accs[j] >= acc + MIN_GAIN:
                bias_val = float(grid[j])
                print(f"  middle-class bias {bias_val:+.1f}: acc {acc:.4f} -> {accs[j]:.4f}")
                acc = accs[j]
            else:
                print(f"  middle-class bias: best {accs[j]:.4f} at {grid[j]:+.1f}, not adopted (gain < {MIN_GAIN})")
        m = LogisticRegression(C=C, max_iter=3000).fit(X, y)
        test_s = m.predict_proba(Xt)
        if bias_val:
            e = np.eye(len(cls))[bias_col] * bias_val
            oof_s = np.exp(np.log(np.clip(oof_s, 1e-9, 1)) + e)
            oof_s /= oof_s.sum(1, keepdims=True)
            test_s = np.exp(np.log(np.clip(test_s, 1e-9, 1)) + e)
            test_s /= test_s.sum(1, keepdims=True)
        if t == "goal_outcome":
            const = prior.argmax()
            const_acc = (y == const).mean()
            if acc < const_acc + MIN_GAIN:
                print(f"  goal: stacked {acc:.4f} < constant {const_acc:.4f} + {MIN_GAIN} -> using constant '{cls[const]}'")
                oof_s = np.tile(np.eye(len(cls))[const], (len(pub), 1))
                test_s = np.tile(np.eye(len(cls))[const], (len(priv), 1))
                acc = const_acc
        final_oof[t], final_test[t] = oof_s, test_s
        report[t] = {"acc": float(acc), "C": C, "bias": bias_val, "members": names}
        np.save(out_dir / f"oof_{ALIASES[t][0]}.npy", oof_s)
        np.save(out_dir / f"test_{ALIASES[t][0]}.npy", test_s)

    pred = pd.DataFrame({t: np.asarray(CLASSES[t])[final_oof[t].argmax(1)] for t in TARGETS})
    score = score_report(pub, pred, pd.Series(folds), a.name)
    report["score"] = score
    # per-source kick histogram sanity
    acc_src = pd.Series(pred.kick_direction.values == pub.kick_direction.values, index=pub.source_id.values).groupby(level=0).mean()
    print("per-source kick acc quantiles:", acc_src.quantile([0, .1, .5, .9, 1]).round(2).to_dict(), "| sources <= 0.2:", int((acc_src <= 0.2).sum()))
    tp = pd.DataFrame({t: np.asarray(CLASSES[t])[final_test[t].argmax(1)] for t in TARGETS})
    tp.insert(0, "clip", priv["clip"].values)
    write_submission(tp, out_dir / "submission.csv")
    json.dump(report, open(out_dir / "metrics.json", "w"), indent=1)
    print(json.dumps({k: v for k, v in report.items() if k != "score"}, indent=1))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/stack2.py
"""Second-stage stacker: re-fits the kick head on a first-stage stack's OOF probabilities for kick AND keeper with
pairwise interaction terms (log p_kick[i] * log p_gk[j]) + foot / gender, source-grouped CV. Tests whether the kick
decision depends on the keeper prediction non-additively (the stage-1 LR is linear in the member log-probs).
Keeper and goal are copied from stage 1.

Run:  python src/stack2.py --base stack_tab40 --name stack2_tab40
"""
import argparse
import json
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import pyarrow  # noqa: F401
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

from config import CLASSES, RUNS_DIR, TARGETS
from cv import get_folds, load_private, load_public, score_report, write_submission


def feats(pk, pg, meta):
    lk, lg = np.log(np.clip(pk, 1e-6, 1)), np.log(np.clip(pg, 1e-6, 1))
    inter = np.einsum("ni,nj->nij", lk, lg).reshape(len(lk), -1)
    return np.hstack([lk, lg, inter, meta])


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--base", required=True)
    ap.add_argument("--name", required=True)
    ap.add_argument("--Cs", nargs="+", type=float, default=[0.01, 0.03, 0.1, 0.3, 1.0])
    a = ap.parse_args()
    pub, priv = load_public(), load_private()
    folds = get_folds(pub).values
    meta_pub = np.c_[np.where(pub.preferred_foot == "right", 1.0, -1.0), (pub.gender == "M").astype(float)]
    meta_priv = np.c_[np.where(priv.preferred_foot == "right", 1.0, -1.0), (priv.gender == "M").astype(float)]
    B = RUNS_DIR / a.base
    ok, og = np.load(B / "oof_kick.npy"), np.load(B / "oof_goalkeeper.npy")
    tk, tg = np.load(B / "test_kick.npy"), np.load(B / "test_goalkeeper.npy")
    cls = CLASSES["kick_direction"]
    y = pub["kick_direction"].map({c: i for i, c in enumerate(cls)}).values
    X, Xt = feats(ok, og, meta_pub), feats(tk, tg, meta_priv)
    base_acc = (ok.argmax(1) == y).mean()
    print(f"stage-1 kick acc {base_acc:.4f}")
    best = None
    for C in a.Cs:
        oof = np.zeros_like(ok)
        for k in np.unique(folds):
            tr, va = folds != k, folds == k
            m = LogisticRegression(C=C, max_iter=5000).fit(X[tr], y[tr])
            oof[va] = m.predict_proba(X[va])
        acc = (oof.argmax(1) == y).mean()
        print(f"  stage-2 LR C={C:<5} kick acc {acc:.4f}")
        if best is None or acc > best[1]:
            best = (C, acc, oof)
    C, acc, oof2 = best
    test2 = LogisticRegression(C=C, max_iter=5000).fit(X, y).predict_proba(Xt)
    out = RUNS_DIR / a.name
    out.mkdir(parents=True, exist_ok=True)
    final_oof = {"kick_direction": oof2, "goalkeeper_direction": og, "goal_outcome": np.load(B / "oof_goal.npy")}
    final_test = {"kick_direction": test2, "goalkeeper_direction": tg, "goal_outcome": np.load(B / "test_goal.npy")}
    for t, al in (("kick_direction", "kick"), ("goalkeeper_direction", "goalkeeper"), ("goal_outcome", "goal")):
        np.save(out / f"oof_{al}.npy", final_oof[t])
        np.save(out / f"test_{al}.npy", final_test[t])
    pred = pd.DataFrame({t: np.asarray(CLASSES[t])[final_oof[t].argmax(1)] for t in TARGETS})
    score = score_report(pub, pred, pd.Series(folds), a.name)
    tp = pd.DataFrame({t: np.asarray(CLASSES[t])[final_test[t].argmax(1)] for t in TARGETS})
    tp.insert(0, "clip", priv["clip"].values)
    write_submission(tp, out / "submission.csv")
    json.dump({"score": score, "base": a.base, "C": C, "kick_acc": float(acc), "stage1_kick_acc": float(base_acc),
               "test_kick_changed": int((test2.argmax(1) != tk.argmax(1)).sum())}, open(out / "metrics.json", "w"), indent=1)
    print(f"test kick labels changed vs stage 1: {(test2.argmax(1) != tk.argmax(1)).sum()} / {len(tk)}")


In [ ]:
%%writefile src/train_kpnet.py
"""kpnet: a small temporal network on keypoint trajectories of the kicker and
the keeper (pose_crops.parquet), in the kicker frame of features_v2.

Per clip a tensor [T=11, C=114]: time grid tau = 0.40, 0.36, ..., 0.00 s before
the last frame (index 0 = earliest). Per role (kicker, keeper):
  17 joints x (lateral, forward, confidence)   lateral = (p - origin) . l,
                                               forward = (p - origin) . u, / body scale
  hip-centre velocity (lat, fwd), log box-height change, pose-present flag
then 4 metadata channels broadcast over time: preferred_foot (+1 right / -1 left),
gender, kicker_found, keeper_found.
Origin: kicker hip centre at tau=0 (kicker), keeper hip centre at tau=0.4 or the
earliest available (keeper). Body scale: median torso length (fallback 0.35*box_h/W).

Mirror = negate every lateral channel + preferred_foot, swap the L/R joint blocks,
swap left<->right labels of kick / keeper. Train aug: mirror p=0.5, coordinate
jitter 0.02, time shift +-1 step (p=0.3), channel dropout 0.1. Mirror TTA at test.
Standardisation uses train-fold statistics on identity+mirror (so mirroring and
standardising commute exactly).

Outputs: RUNS_DIR/<run>/{oof,test}_{kick,goalkeeper,goal}.npy, metrics.json,
         submission.csv (argmax kick/keeper, constant "goal").
Run:  python src/train_kpnet.py --arch cnn            -> runs/kpnet
      python src/train_kpnet.py --arch gru --run kpnet_gru
      python src/train_kpnet.py --noswap --seeds 1 --run kpnet_noswap   (sanity test b)
"""
import argparse
import json
import os
import sys
import time

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import pyarrow  # noqa: F401  (must precede torch: pandas 3 + pyarrow on Windows)
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from config import CACHE_DIR, CLASSES, RUNS_DIR, TARGETS, WEIGHTS
from cv import get_folds, load_private, load_public, score_report, write_submission

LSH, RSH, LHIP, RHIP = 5, 6, 11, 12
POSE_SRC = os.environ.get("KPNET_POSE", "pose_crops")   # pose_crops (YOLO COCO-17) | pose_rtm_m / pose_rtm_x (RTMPose Halpe-26)
NJ = 26 if POSE_SRC.startswith("pose_rtm") else 17
PAIRS = [(1, 2), (3, 4), (5, 6), (7, 8), (9, 10), (11, 12), (13, 14), (15, 16)]
if NJ == 26:
    PAIRS += [(20, 21), (22, 23), (24, 25)]            # big toes, small toes, heels (17 head / 18 neck / 19 hip are midline)
TAUS = np.round(np.arange(0.40, -0.001, -0.04), 2)   # 0.40 ... 0.00  (forward in time)
T = len(TAUS)                                          # 11
DT = 0.04
GRID_TOL = 0.5 * DT + 1e-6
KP_CONF = 0.3
VEL_CLIP = 20.0                # body scales per second
Z_CLIP = 6.0                   # clamp of standardised channels
ROLES = ("kicker", "keeper")
PER_ROLE = NJ * 3 + 4          # joints (lat, fwd, conf) + hipvel (lat, fwd) + boxchg + present
META = ["foot", "gender", "kicker_found", "keeper_found"]
META0 = 2 * PER_ROLE
C = META0 + len(META)          # 114
TENSOR_CACHE = CACHE_DIR / ("kpnet_tensor.npz" if POSE_SRC == "pose_crops" else f"kpnet_tensor_{POSE_SRC}.npz")


# ----------------------------------------------------------------------------- channels
def ch(r, j, k):
    """role r (0 kicker / 1 keeper), joint j, k: 0 lat, 1 fwd, 2 conf."""
    return r * PER_ROLE + 3 * j + k


def ch_extra(r, k):
    """k: 0 hipvel lat, 1 hipvel fwd, 2 box-height change, 3 present."""
    return r * PER_ROLE + 3 * NJ + k


class ChannelSpec:
    """Mirror signs / permutation and standardisation masks for the C channels."""

    def __init__(self):
        sign = np.ones(C, np.float32)
        perm = np.arange(C)
        is_coord = np.zeros(C, bool)
        conf_of = np.full(C, -1)
        std_mask = np.zeros(C, bool)
        names = [""] * C
        for r, role in enumerate(ROLES):
            for j in range(NJ):
                sign[ch(r, j, 0)] = -1.0
                is_coord[ch(r, j, 0)] = is_coord[ch(r, j, 1)] = True
                conf_of[ch(r, j, 0)] = conf_of[ch(r, j, 1)] = ch(r, j, 2)
                std_mask[ch(r, j, 0)] = std_mask[ch(r, j, 1)] = True
                names[ch(r, j, 0)], names[ch(r, j, 1)], names[ch(r, j, 2)] = f"{role}_j{j}_lat", f"{role}_j{j}_fwd", f"{role}_j{j}_conf"
            sign[ch_extra(r, 0)] = -1.0
            std_mask[ch_extra(r, 0)] = std_mask[ch_extra(r, 1)] = std_mask[ch_extra(r, 2)] = True
            for k, n in enumerate(("hipvel_lat", "hipvel_fwd", "boxchg", "present")):
                names[ch_extra(r, k)] = f"{role}_{n}"
            for a, b in PAIRS:
                for k in range(3):
                    perm[ch(r, a, k)] = ch(r, b, k)
                    perm[ch(r, b, k)] = ch(r, a, k)
        sign[META0] = -1.0   # preferred foot
        for k, n in enumerate(META):
            names[META0 + k] = n
        assert sorted(perm.tolist()) == list(range(C))
        self.sign, self.perm, self.is_coord, self.conf_of, self.std_mask, self.names = sign, perm, is_coord, conf_of, std_mask, names
        self.coord_idx = np.where(is_coord)[0]
        self.coord_conf_idx = conf_of[is_coord]

    def to(self, dev):
        self.t_sign = torch.tensor(self.sign, device=dev)
        self.t_perm = torch.tensor(self.perm, device=dev, dtype=torch.long)
        self.t_coord = torch.tensor(self.coord_idx, device=dev, dtype=torch.long)
        self.t_coord_conf = torch.tensor(self.coord_conf_idx, device=dev, dtype=torch.long)
        return self


def mirror(X, spec):
    """X [n, T, C] (numpy or torch) in RAW space -> mirrored tensor."""
    if isinstance(X, np.ndarray):
        return X[:, :, spec.perm] * spec.sign
    return X[:, :, spec.t_perm] * spec.t_sign


def swap_lr_probs(P, target):
    cls = CLASSES[target]
    if "left" not in cls:
        return P
    i, j = cls.index("left"), cls.index("right")
    Q = P.clone() if isinstance(P, torch.Tensor) else P.copy()
    Q[:, [i, j]] = P[:, [j, i]]
    return Q


# ----------------------------------------------------------------------------- feature tensor
def role_block(g: pd.DataFrame, W: float, u: np.ndarray, l: np.ndarray, role: str) -> np.ndarray:
    """Pose rows of one (clip, role) -> [T, PER_ROLE] raw channels."""
    g = g.sort_values("t_from_end_s")
    t = g.t_from_end_s.values
    found = g.found.values == 1
    kx = g[[f"kx{k}" for k in range(NJ)]].values / W
    ky = g[[f"ky{k}" for k in range(NJ)]].values / W
    kc = g[[f"kc{k}" for k in range(NJ)]].values
    valid = np.isfinite(kx) & np.isfinite(ky) & (kc >= KP_CONF) & found[:, None]
    kx = np.where(valid, kx, np.nan)
    ky = np.where(valid, ky, np.nan)
    # body scale: median torso length, fallback 0.35 * box height
    sh = np.stack([(kx[:, LSH] + kx[:, RSH]) / 2, (ky[:, LSH] + ky[:, RSH]) / 2], 1)
    hp = np.stack([(kx[:, LHIP] + kx[:, RHIP]) / 2, (ky[:, LHIP] + ky[:, RHIP]) / 2], 1)
    torso = np.linalg.norm(sh - hp, axis=1)
    s = np.nanmedian(torso) if np.isfinite(torso).any() else np.nan
    if not np.isfinite(s) or s < 1e-3:
        s = 0.35 * np.nanmedian(g.box_h.values) / W
    # grid sampling: nearest frame within half a grid interval
    pos = np.full((T, NJ, 2), np.nan)
    conf = np.zeros((T, NJ), np.float32)
    present = np.zeros(T, np.float32)
    box_h = np.full(T, np.nan)
    box_c = np.full((T, 2), np.nan)
    for ti, tau in enumerate(TAUS):
        j = int(np.argmin(np.abs(t - tau)))
        if abs(t[j] - tau) > GRID_TOL:
            continue
        box_h[ti] = g.box_h.values[j]
        box_c[ti] = (g.box_cx.values[j] / W, g.box_cy.values[j] / W)
        if found[j]:
            present[ti] = 1.0
            pos[ti] = np.stack([kx[j], ky[j]], 1)
            conf[ti] = np.where(valid[j], kc[j], 0.0)
    hip = (pos[:, LHIP] + pos[:, RHIP]) / 2                       # NaN if either hip missing
    # origin: kicker -> hip centre at tau=0 (last index), keeper -> at tau=0.4 (first index) or earliest available
    order = range(T - 1, -1, -1) if role == "kicker" else range(T)
    origin = None
    for ti in order:
        if np.all(np.isfinite(hip[ti])):
            origin = hip[ti]
            break
    if origin is None:
        for ti in order:                                           # fallback: interpolated box centre
            if np.all(np.isfinite(box_c[ti])):
                origin = box_c[ti]
                break
    if origin is None:
        origin = np.zeros(2)
    d = pos - origin                                               # [T, NJ, 2]
    lat = np.nan_to_num(d @ l / s, nan=0.0)
    fwd = np.nan_to_num(d @ u / s, nan=0.0)
    block = np.zeros((T, PER_ROLE), np.float32)
    block[:, 0:3 * NJ:3] = lat
    block[:, 1:3 * NJ:3] = fwd
    block[:, 2:3 * NJ:3] = conf
    for ti in range(1, T):
        if np.all(np.isfinite(hip[ti])) and np.all(np.isfinite(hip[ti - 1])):
            v = (hip[ti] - hip[ti - 1]) / DT / s
            block[ti, ch_extra(0, 0)] = np.clip(v @ l, -VEL_CLIP, VEL_CLIP)   # detection jumps -> outliers
            block[ti, ch_extra(0, 1)] = np.clip(v @ u, -VEL_CLIP, VEL_CLIP)
        if np.isfinite(box_h[ti]) and np.isfinite(box_h[ti - 1]) and box_h[ti - 1] > 0 and box_h[ti] > 0:
            block[ti, ch_extra(0, 2)] = np.log(box_h[ti] / box_h[ti - 1])
    block[:, ch_extra(0, 3)] = present
    return block


def build_tensor(force=False):
    pub, priv = load_public(), load_private()
    meta = pd.concat([pub, priv], ignore_index=True)
    clips = meta["clip"].tolist()
    if TENSOR_CACHE.exists() and not force:
        z = np.load(TENSOR_CACHE, allow_pickle=True)
        if list(z["clips"]) == clips and z["X"].shape[1:] == (T, C):
            print(f"loaded cached tensor {z['X'].shape} from {TENSOR_CACHE}")
            return z["X"], meta
    t0 = time.time()
    P = pd.read_parquet(CACHE_DIR / f"{POSE_SRC}.parquet")
    RC = pd.read_csv(CACHE_DIR / "roles_clip.csv").set_index("clip")
    groups = {k: g for k, g in P.groupby(["clip", "role"])}
    X = np.zeros((len(clips), T, C), np.float32)
    n_geom = 0
    for i, clip in enumerate(clips):
        m = meta.iloc[i]
        X[i, :, META0 + 0] = 1.0 if m.preferred_foot == "right" else -1.0
        X[i, :, META0 + 1] = 1.0 if m.gender == "M" else 0.0
        if clip not in RC.index:
            continue
        rc = RC.loc[clip]
        X[i, :, META0 + 2] = float(rc.kicker_found)
        X[i, :, META0 + 3] = float(rc.keeper_found)
        if pd.isna(rc.u_x):
            continue
        W = float(rc.W)
        u = np.array([rc.u_x, rc.u_y], dtype=float)
        l = np.array([u[1], -u[0]])
        for r, role in enumerate(ROLES):
            g = groups.get((clip, role))
            if g is None or len(g) == 0:
                continue
            X[i, :, r * PER_ROLE:(r + 1) * PER_ROLE] = role_block(g, W, u, l, role)
        n_geom += 1
    assert np.isfinite(X).all()
    np.savez_compressed(TENSOR_CACHE, X=X, clips=np.array(clips))
    print(f"built tensor {X.shape} ({n_geom} clips with geometry) in {time.time() - t0:.0f}s -> {TENSOR_CACHE}")
    return X, meta


# ----------------------------------------------------------------------------- standardisation
class Stats:
    def __init__(self, X_tr: torch.Tensor, spec: ChannelSpec):
        """Train-fold channel mean/std on identity + mirror, over present entries for coordinate channels."""
        Xu = torch.cat([X_tr, mirror(X_tr, spec)], 0)
        mask = torch.ones_like(Xu)
        mask[:, :, spec.t_coord] = (Xu[:, :, spec.t_coord_conf] > 0).float()
        n = mask.sum((0, 1)).clamp(min=1.0)
        mean = (Xu * mask).sum((0, 1)) / n
        var = (((Xu - mean) ** 2) * mask).sum((0, 1)) / n
        std = var.sqrt() + 1e-6
        sm = torch.tensor(spec.std_mask, device=X_tr.device)
        self.mean = torch.where(sm, mean, torch.zeros_like(mean))
        self.std = torch.where(sm, std, torch.ones_like(std))
        self.spec = spec

    def __call__(self, X: torch.Tensor) -> torch.Tensor:
        Z = ((X - self.mean) / self.std).clamp(-Z_CLIP, Z_CLIP)
        present = (X[:, :, self.spec.t_coord_conf] > 0).float()
        Z[:, :, self.spec.t_coord] = Z[:, :, self.spec.t_coord] * present
        return Z


# ----------------------------------------------------------------------------- models
class ConvNet(nn.Module):
    def __init__(self, c_in=C, hid=128, p=0.3, n_blocks=3):
        super().__init__()
        layers = []
        for i in range(n_blocks):
            layers += [nn.Conv1d(c_in if i == 0 else hid, hid, 3, padding=1), nn.BatchNorm1d(hid), nn.GELU(), nn.Dropout(p)]
        self.body = nn.Sequential(*layers)
        self.heads = nn.ModuleList([nn.Linear(2 * hid, len(CLASSES[t])) for t in TARGETS])

    def forward(self, x):                      # x [n, T, C]
        h = self.body(x.transpose(1, 2))       # [n, hid, T]
        z = torch.cat([h.mean(2), h.amax(2)], 1)
        return [head(z) for head in self.heads]


class GRUNet(nn.Module):
    def __init__(self, c_in=C, hid=128, p=0.3):
        super().__init__()
        self.inp = nn.Sequential(nn.Linear(c_in, hid), nn.GELU(), nn.Dropout(p))
        self.gru = nn.GRU(hid, hid, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(p)
        self.heads = nn.ModuleList([nn.Linear(4 * hid, len(CLASSES[t])) for t in TARGETS])

    def forward(self, x):
        h, _ = self.gru(self.inp(x))           # [n, T, 2*hid]
        h = self.drop(h)
        z = torch.cat([h.mean(1), h.amax(1)], 1)
        return [head(z) for head in self.heads]


def make_model(arch):
    return {"cnn": ConvNet, "gru": GRUNet}[arch]()


# ----------------------------------------------------------------------------- augmentation
def augment(xb, yb, ysw, spec, stats, swap_labels, p_mirror=0.5, jitter=0.02, p_shift=0.3, p_drop=0.1):
    """xb raw [n,T,C]; yb / ysw [n,3] original / label-swapped targets. Returns standardised xb, targets."""
    n = xb.shape[0]
    dev = xb.device
    flip = torch.rand(n, device=dev) < p_mirror
    xm = mirror(xb, spec)
    xb = torch.where(flip[:, None, None], xm, xb)
    if swap_labels:
        yb = torch.where(flip[:, None], ysw, yb)
    if jitter > 0:
        present = (xb[:, :, spec.t_coord_conf] > 0).float()
        xb[:, :, spec.t_coord] = xb[:, :, spec.t_coord] + jitter * torch.randn(n, T, len(spec.coord_idx), device=dev) * present
    if p_shift > 0:
        shift = torch.randint(-1, 2, (n,), device=dev) * (torch.rand(n, device=dev) < p_shift).long()
        idx = (torch.arange(T, device=dev)[None, :] - shift[:, None]).clamp(0, T - 1)       # [n, T]
        xb = torch.gather(xb, 1, idx[:, :, None].expand(-1, -1, C))
    zb = stats(xb)
    if p_drop > 0:
        keep = (torch.rand(n, 1, C, device=dev) >= p_drop).float() / (1 - p_drop)
        zb = zb * keep
    return zb, yb


# ----------------------------------------------------------------------------- train / predict
@torch.no_grad()
def predict(model, X_raw, spec, stats, swap_labels=True, bs=512):
    """-> (p_id, p_mir): dict target -> [n, k] probabilities, mirror predictions mapped back to the original frame."""
    model.eval()
    out = {"id": {t: [] for t in TARGETS}, "mir": {t: [] for t in TARGETS}}
    for tag in ("id", "mir"):
        for s in range(0, len(X_raw), bs):
            xb = X_raw[s:s + bs]
            if tag == "mir":
                xb = mirror(xb, spec)
            logits = model(stats(xb))
            for t, lg in zip(TARGETS, logits):
                p = F.softmax(lg.float(), 1)
                if tag == "mir" and swap_labels:
                    p = swap_lr_probs(p, t)
                out[tag][t].append(p.cpu().numpy())
    return ({t: np.concatenate(v) for t, v in out["id"].items()}, {t: np.concatenate(v) for t, v in out["mir"].items()})


def train_one(arch, X_tr, Y_tr, Ysw_tr, spec, stats, seed, swap_labels, epochs=60, bs=64, lr=1e-3, wd=1e-2, dev="cuda"):
    torch.manual_seed(seed)
    np.random.seed(seed)
    model = make_model(arch).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    n = len(X_tr)
    steps_per_epoch = max(1, n // bs)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs * steps_per_epoch)
    w = torch.tensor([WEIGHTS[t] for t in TARGETS], device=dev)
    g = torch.Generator(device=dev)
    g.manual_seed(seed)
    hist = []
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(n, device=dev, generator=g)
        tot = 0.0
        for b in range(steps_per_epoch):                      # drop_last (BatchNorm)
            idx = perm[b * bs:(b + 1) * bs]
            xb, yb = augment(X_tr[idx].clone(), Y_tr[idx], Ysw_tr[idx], spec, stats, swap_labels)
            logits = model(xb)
            loss = sum(w[i] * F.cross_entropy(logits[i], yb[:, i], label_smoothing=0.1) for i in range(3))
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            sched.step()
            tot += loss.item()
        hist.append(tot / steps_per_epoch)
    return model, hist


def per_source_hist(pred_kick: np.ndarray, pub: pd.DataFrame):
    acc = pd.Series(pred_kick == pub["kick_direction"].values, index=pub["source_id"].values).groupby(level=0).mean()
    bins = pd.cut(acc, [-0.001, 0.2, 0.4, 0.6, 0.8, 1.0]).value_counts().sort_index()
    print("per-source kick accuracy histogram:", {str(k): int(v) for k, v in bins.items()},
          "| quantiles:", acc.quantile([0, .1, .25, .5, .75, .9, 1]).round(2).to_dict(),
          "| sources <= 0.2:", int((acc <= 0.2).sum()), "of", len(acc))
    return {"hist": {str(k): int(v) for k, v in bins.items()}, "n_sources_le_0.2": int((acc <= 0.2).sum()),
            "n_sources": int(len(acc)), "min": float(acc.min()), "median": float(acc.median())}


def run(arch="cnn", run_name="kpnet", seeds=(0, 1, 2), epochs=60, bs=64, swap_labels=True, dev=None, force_tensor=False):
    dev = dev or ("cuda" if torch.cuda.is_available() else "cpu")
    torch.set_num_threads(2)
    X, meta = build_tensor(force=force_tensor)
    pub, priv = load_public(), load_private()
    n_pub = len(pub)
    folds = get_folds(pub).values
    spec = ChannelSpec().to(dev)
    Xp = torch.tensor(X[:n_pub], device=dev)
    Xq = torch.tensor(X[n_pub:], device=dev)
    enc = {t: {c: i for i, c in enumerate(CLASSES[t])} for t in TARGETS}
    Y = np.stack([pub[t].map(enc[t]).values for t in TARGETS], 1)
    from config import FLIP_MAP
    Ysw = np.stack([pub[t].map(FLIP_MAP[t]).map(enc[t]).values for t in TARGETS], 1)
    Yt, Yswt = torch.tensor(Y, device=dev, dtype=torch.long), torch.tensor(Ysw, device=dev, dtype=torch.long)
    out_dir = RUNS_DIR / run_name
    out_dir.mkdir(parents=True, exist_ok=True)
    oof_id = {t: np.zeros((n_pub, len(CLASSES[t]))) for t in TARGETS}
    oof_mir = {t: np.zeros((n_pub, len(CLASSES[t]))) for t in TARGETS}
    test = {t: np.zeros((len(priv), len(CLASSES[t]))) for t in TARGETS}
    n_models = len(seeds) * 5
    fold_times, fold_scores = [], []
    print(f"=== {run_name}: arch={arch} seeds={list(seeds)} epochs={epochs} bs={bs} swap_labels={swap_labels} dev={dev} C={C} T={T}")
    for k in range(5):
        t0 = time.time()
        tr, va = np.where(folds != k)[0], np.where(folds == k)[0]
        tr_t = torch.tensor(tr, device=dev)
        stats = Stats(Xp[tr_t], spec)
        for seed in seeds:
            model, hist = train_one(arch, Xp[tr_t], Yt[tr_t], Yswt[tr_t], spec, stats, seed + 100 * k, swap_labels, epochs, bs, dev=dev)
            p_id, p_mir = predict(model, Xp[torch.tensor(va, device=dev)], spec, stats, swap_labels)
            q_id, q_mir = predict(model, Xq, spec, stats, swap_labels)
            for t in TARGETS:
                oof_id[t][va] += p_id[t] / len(seeds)
                oof_mir[t][va] += p_mir[t] / len(seeds)
                test[t] += 0.5 * (q_id[t] + q_mir[t]) / n_models
            print(f"  fold {k} seed {seed}: train loss {hist[0]:.3f} -> {hist[-1]:.3f}", flush=True)
        oof_k = {t: 0.5 * (oof_id[t][va] + oof_mir[t][va]) for t in TARGETS}
        pred_k = pd.DataFrame({t: np.asarray(CLASSES[t])[np.argmax(oof_k[t], 1)] for t in TARGETS})
        acc_k = {t: float((pred_k[t].values == pub[t].values[va]).mean()) for t in TARGETS}
        sc_k = sum(WEIGHTS[t] * acc_k[t] for t in TARGETS)
        fold_scores.append(sc_k)
        fold_times.append(time.time() - t0)
        print(f"  fold {k}: score {sc_k:.4f} ({' | '.join(f'{t.split('_')[0]} {acc_k[t]:.3f}' for t in TARGETS)})  [{fold_times[-1]:.0f}s]", flush=True)
    oof = {t: 0.5 * (oof_id[t] + oof_mir[t]) for t in TARGETS}
    for t in TARGETS:
        np.save(out_dir / f"oof_{t.split('_')[0]}.npy", oof[t])
        np.save(out_dir / f"test_{t.split('_')[0]}.npy", test[t])
    pred = pd.DataFrame({t: np.asarray(CLASSES[t])[np.argmax(oof[t], 1)] for t in TARGETS})
    score = score_report(pub, pred, pd.Series(folds), run_name)
    metrics = {t: float((pred[t].values == pub[t].values).mean()) for t in TARGETS}
    metrics.update(score=score, arch=arch, seeds=list(seeds), epochs=epochs, swap_labels=swap_labels,
                   fold_scores=fold_scores, fold_time_s=fold_times)
    # sanity (a): mirror consistency (identity-only vs mirror-only OOF accuracy)
    acc_id = {t: float((np.argmax(oof_id[t], 1) == Y[:, i]).mean()) for i, t in enumerate(TARGETS)}
    acc_mir = {t: float((np.argmax(oof_mir[t], 1) == Y[:, i]).mean()) for i, t in enumerate(TARGETS)}
    agree = {t: float((np.argmax(oof_id[t], 1) == np.argmax(oof_mir[t], 1)).mean()) for t in TARGETS}
    print("sanity (a) mirror consistency: identity-only acc", {t.split('_')[0]: round(v, 4) for t, v in acc_id.items()},
          "| mirror-only acc", {t.split('_')[0]: round(v, 4) for t, v in acc_mir.items()},
          "| max |diff|", round(max(abs(acc_id[t] - acc_mir[t]) for t in TARGETS), 4),
          "| argmax agreement", {t.split('_')[0]: round(v, 3) for t, v in agree.items()})
    metrics["mirror_test"] = {"acc_identity": acc_id, "acc_mirror": acc_mir, "argmax_agreement": agree}
    # sanity (c): per-source kick accuracy
    metrics["per_source"] = per_source_hist(pred["kick_direction"].values, pub)
    json.dump(metrics, open(out_dir / "metrics.json", "w"), indent=1)
    if swap_labels:
        tp = pd.DataFrame({"clip": priv["clip"].values,
                           "kick_direction": np.asarray(CLASSES["kick_direction"])[np.argmax(test["kick_direction"], 1)],
                           "goalkeeper_direction": np.asarray(CLASSES["goalkeeper_direction"])[np.argmax(test["goalkeeper_direction"], 1)],
                           "goal_outcome": "goal"})
        write_submission(tp, out_dir / "submission.csv")
    print(f"mean wall time per fold: {np.mean(fold_times):.0f}s  | outputs -> {out_dir}")
    return metrics


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--arch", default="cnn", choices=["cnn", "gru"])
    ap.add_argument("--run", default=None)
    ap.add_argument("--seeds", type=int, default=3)
    ap.add_argument("--epochs", type=int, default=60)
    ap.add_argument("--bs", type=int, default=64)
    ap.add_argument("--noswap", action="store_true", help="sanity test (b): mirror augmentation WITHOUT the label swap")
    ap.add_argument("--device", default=None)
    ap.add_argument("--rebuild", action="store_true")
    a = ap.parse_args()
    run_name = a.run or (("kpnet" if a.arch == "cnn" else f"kpnet_{a.arch}")
                         + ("" if POSE_SRC == "pose_crops" else f"_{POSE_SRC.replace('pose_', '')}"))
    if a.noswap and a.run is None:
        run_name += "_noswap"
    run(a.arch, run_name, seeds=tuple(range(a.seeds)), epochs=a.epochs, bs=a.bs, swap_labels=not a.noswap,
        dev=a.device, force_tensor=a.rebuild)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/train_stack7.py
"""Stage 2b: stack-7 ConvNeXt-tiny multi-task model on the canonical crop caches (crops.py).

Input: 7 crop frames at tau = 0.40, 0.32, 0.24, 0.16, 0.08, 0.04, 0.00 s before the end
(cache indices 5, 7, 9, 11, 13, 14, 15 of the 16-frame grid; asserted against the stored
t_from_end), ImageNet-normalised per frame and stacked as 21 channels at 224x224.
Model: timm convnext_tiny.fb_in22k_ft_in1k with in_chans=21 (timm adapts the stem),
global pool (768) ++ emb(goal_side of the presented image L/R/U -> 8) ++ emb(preferred_foot
canonical -> 4) ++ emb(gender -> 4) -> dropout 0.3 -> three heads (3/3/2).
Loss 0.60 CE(kick) + 0.25 CE(keeper) + 0.15 CE(goal), label smoothing 0.1. AdamW (backbone
1e-4, heads/embeddings 1e-3, wd 0.05), 1 warm-up epoch + cosine, 15 epochs, batch 32, bf16
autocast, EMA 0.998 for evaluation.

Canonical label frame: the crops of goal_side == 'R' clips are mirrored, so their training
labels are swapped through config.FLIP_MAP (kick / keeper left<->right, preferred_foot
swapped); predictions of mirrored clips are un-swapped back to the original frame.
Augmentation: hflip p=0.5 (labels + foot + goal_side swapped), RandomResizedCrop scale
(0.8, 1.0) with the same window for all 7 frames, colour jitter 0.2, end-jitter p=0.25
(all 7 cache indices one step = 0.04 s earlier).
Inference: TTA over {canonical, flipped-canonical with swapped embeddings}, un-swapped.

Run:  python src/train_stack7.py --run stack7_kicker_s42 --view kicker            (full 5-fold run)
      python src/train_stack7.py --smoke --view kicker                            (48 clips, 2 folds, 1 epoch)
Outputs: D:\\cv3\\models\\<run>\\fold{k}.pt, D:\\cv3\\runs\\<run>\\{log.jsonl, fold{k}_preds.npz,
         oof_{kick,gk,goal}.npy (load_public order, original label frame, NaN rows = no crops),
         test_{kick,gk,goal}.npy (load_private order), metrics.json, submission.csv}
"""
import argparse
import faulthandler
import json
import math
import os
import sys
import time
from copy import deepcopy

faulthandler.enable()
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import config  # noqa: F401  (HF_HOME / TORCH_HOME on D: before timm / torch hub are imported)

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

try:  # pandas 3 lazily loads pyarrow; importing it after cv2 + torch crashes read_csv (DLL clash)
    import pyarrow  # noqa: F401
except ImportError:
    pass

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

from config import CLASSES, FLIP_MAP, MODELS_DIR, RUNS_DIR, SEED, TARGETS, WEIGHTS
from crops import CROP_PX, N_FRAMES, STACK7_TAUS, crops_dir, load_crop_meta, load_crops, stack_indices
from cv import accuracy_by_target, get_folds, load_private, load_public, probs_to_labels, score_report, write_submission

cv2.setNumThreads(1)

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], np.float32)
SIDE_IDX = {"L": 0, "R": 1, "U": 2}
SIDE_FLIP = np.array([1, 0, 2])          # hflip of the presented image: L<->R, U stays
FOOT_IDX = {"left": 0, "right": 1}
GENDER_IDX = {"M": 0, "F": 1}
N_CLASSES = [len(CLASSES[t]) for t in TARGETS]
SHORT = {"kick_direction": "kick", "goalkeeper_direction": "gk", "goal_outcome": "goal"}
# column permutation that un-swaps a prediction made on the mirrored clip (involution)
FLIP_PERM = {t: np.array([CLASSES[t].index(FLIP_MAP[t][c]) for c in CLASSES[t]]) for t in TARGETS}
FLIP_PERM_T = {t: torch.as_tensor(FLIP_PERM[t]) for t in TARGETS}
STACK_IDX = stack_indices(STACK7_TAUS)              # [5, 7, 9, 11, 13, 14, 15]
STACK_IDX_JIT = stack_indices(STACK7_TAUS, shift=1)  # one step earlier: [4, 6, 8, 10, 12, 13, 14]
assert STACK_IDX == [5, 7, 9, 11, 13, 14, 15], STACK_IDX
BACKBONE = "convnext_tiny.fb_in22k_ft_in1k"


# ----------------------------------------------------------------------------- data
def crop_meta_table(clips, view):
    """DataFrame indexed by clip: mirrored, side_unknown, fallback (from the npz; None rows = no crops)."""
    rows = {}
    for c in clips:
        f = crops_dir(view) / c.replace(".mp4", ".npz")
        if not f.exists():
            continue
        m = load_crop_meta(c, view)
        assert np.allclose(m["t_from_end"][STACK_IDX], STACK7_TAUS, atol=1e-5), (c, m["t_from_end"][STACK_IDX])
        rows[c] = {"mirrored": m["mirrored"], "side_unknown": m["side_unknown"], "fallback": m["fallback"]}
    return pd.DataFrame.from_dict(rows, orient="index")


def encode_meta(df: pd.DataFrame, cm: pd.DataFrame) -> np.ndarray:
    """int64 [n, 3] in the CANONICAL frame: presented goal side (L if known, else U), preferred_foot
    (swapped for mirrored clips), gender. Never fps / resolution / duration."""
    foot = df["preferred_foot"].map(FOOT_IDX).values
    gender = df["gender"].map(GENDER_IDX).values
    if np.isnan(foot.astype(float)).any() or np.isnan(gender.astype(float)).any():
        raise ValueError("unknown preferred_foot / gender value")
    mirrored = cm.loc[df["clip"].values, "mirrored"].values.astype(bool)
    unknown = cm.loc[df["clip"].values, "side_unknown"].values.astype(bool)
    foot = np.where(mirrored, 1 - foot, foot)
    side = np.where(unknown, SIDE_IDX["U"], SIDE_IDX["L"])
    return np.stack([side, foot, gender], 1).astype(np.int64)


def encode_labels(df: pd.DataFrame, cm: pd.DataFrame) -> np.ndarray:
    """int64 [n, 3] class indices in the CANONICAL frame (mirrored clips: FLIP_MAP applied)."""
    out = np.stack([df[t].map({c: i for i, c in enumerate(CLASSES[t])}).values for t in TARGETS], 1)
    if np.isnan(out.astype(float)).any():
        raise ValueError("unknown label value")
    out = out.astype(np.int64)
    mirrored = cm.loc[df["clip"].values, "mirrored"].values.astype(bool)
    for j, t in enumerate(TARGETS):
        out[mirrored, j] = FLIP_PERM[t][out[mirrored, j]]
    return out


def unswap(probs: dict, mirrored: np.ndarray) -> dict:
    """Canonical-frame probabilities -> original label frame (mirrored rows get their columns swapped)."""
    out = {}
    for t in TARGETS:
        p = probs[t].copy()
        p[mirrored] = p[mirrored][:, FLIP_PERM[t]]
        out[t] = p
    return out


class CropDataset(Dataset):
    """Crop cache -> x float16 [21, 224, 224] (7 frames x RGB, ImageNet-normalised), meta int64 [3]."""

    def __init__(self, clips, view, meta, labels, train, end_jitter_p=0.25, aug=True, flip=True):
        self.clips = list(clips)
        self.view = view
        self.meta = np.asarray(meta, np.int64)
        self.labels = None if labels is None else np.asarray(labels, np.int64)
        self.train = train
        self.end_jitter_p = end_jitter_p
        self.aug = aug      # random-resized-crop + colour jitter (ablation switch)
        self.flip = flip    # hflip with label/foot/side swap (ablation switch)
        self._rng, self._rng_seed = None, None

    def __len__(self):
        return len(self.clips)

    def rng(self):
        seed = torch.initial_seed() % 2**32  # differs per worker and per epoch
        if self._rng_seed != seed:
            self._rng, self._rng_seed = np.random.default_rng(seed), seed
        return self._rng

    @staticmethod
    def crop_box(rng, h, w):
        s = rng.uniform(0.8, 1.0)
        r = rng.uniform(0.9, 1.1)
        cw = int(round(min(w, w * math.sqrt(s * r))))
        ch = int(round(min(h, h * math.sqrt(s / r))))
        cw = max(cw, int(math.ceil(0.8 * w)))
        ch = max(ch, int(math.ceil(0.8 * h)))
        return int(rng.integers(0, w - cw + 1)), int(rng.integers(0, h - ch + 1)), cw, ch

    def __getitem__(self, i):
        rng = self.rng() if self.train else None
        idx = STACK_IDX_JIT if (self.train and rng.random() < self.end_jitter_p) else STACK_IDX
        frames = load_crops(self.clips[i], self.view, idx)  # uint8 [7, 224, 224, 3] RGB
        meta = self.meta[i].copy()
        labels = None if self.labels is None else self.labels[i].copy()
        flipped = False
        if self.train and self.aug:
            x0, y0, cw, ch = self.crop_box(rng, CROP_PX, CROP_PX)   # same window for all 7 frames
            if (cw, ch) != (CROP_PX, CROP_PX):
                frames = np.stack([cv2.resize(np.ascontiguousarray(f[y0:y0 + ch, x0:x0 + cw]), (CROP_PX, CROP_PX),
                                              interpolation=cv2.INTER_LINEAR) for f in frames])
        if self.train and self.flip and rng.random() < 0.5:
            frames = frames[:, :, ::-1]
            flipped = True
        x = torch.from_numpy(np.ascontiguousarray(frames)).permute(0, 3, 1, 2).to(
            torch.float32, memory_format=torch.contiguous_format).div_(255.0)   # [7, 3, H, W]
        if self.train and self.aug and rng.random() < 0.8:  # colour jitter 0.2 (brightness, contrast, saturation)
            b, c, s = rng.uniform(0.8, 1.2, size=3)
            x.mul_(b)
            gray = (0.299 * x[:, 0] + 0.587 * x[:, 1] + 0.114 * x[:, 2]).unsqueeze(1)
            x = (x - gray.mean()) * c + gray.mean()
            gray = (0.299 * x[:, 0] + 0.587 * x[:, 1] + 0.114 * x[:, 2]).unsqueeze(1)
            x = gray + (x - gray) * s
            x.clamp_(0.0, 1.0)
        x.sub_(torch.from_numpy(IMAGENET_MEAN).view(1, 3, 1, 1)).div_(torch.from_numpy(IMAGENET_STD).view(1, 3, 1, 1))
        x = x.reshape(3 * len(idx), CROP_PX, CROP_PX)
        if flipped:
            meta[0] = SIDE_FLIP[meta[0]]
            meta[1] = 1 - meta[1]
            if labels is not None:
                labels = np.array([FLIP_PERM[t][labels[j]] for j, t in enumerate(TARGETS)], np.int64)
        out = {"x": x.half(), "meta": torch.from_numpy(meta), "idx": i}
        if labels is not None:
            out["labels"] = torch.from_numpy(labels)
        return out


def worker_init(_):
    cv2.setNumThreads(1)
    torch.set_num_threads(1)


def make_loader(ds, bs, shuffle, workers, drop_last=False):
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, num_workers=workers, pin_memory=True,
                      drop_last=drop_last, persistent_workers=workers > 0, worker_init_fn=worker_init,
                      prefetch_factor=4 if workers > 0 else None)


# ----------------------------------------------------------------------------- model
class StackNet(nn.Module):
    """ConvNeXt-tiny with a 21-channel stem + metadata embeddings -> three linear heads."""

    def __init__(self, backbone=BACKBONE, n_frames=7, drop=0.3, drop_path=0.1, pretrained=True):
        super().__init__()
        import timm
        self.backbone = timm.create_model(backbone, pretrained=pretrained, in_chans=3 * n_frames, num_classes=0,
                                          drop_path_rate=drop_path)
        d = self.backbone.num_features
        self.emb_side = nn.Embedding(3, 8)
        self.emb_foot = nn.Embedding(2, 4)
        self.emb_gender = nn.Embedding(2, 4)
        self.drop = nn.Dropout(drop)
        self.heads = nn.ModuleList([nn.Linear(d + 16, n) for n in N_CLASSES])

    def forward(self, x, meta):
        f = self.backbone(x)
        z = torch.cat([f, self.emb_side(meta[:, 0]), self.emb_foot(meta[:, 1]), self.emb_gender(meta[:, 2])], 1)
        z = self.drop(z)
        return [h(z) for h in self.heads]


def param_groups(model: StackNet, lr_bb, lr_head, wd):
    groups = {}
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        tag, lr = ("backbone", lr_bb) if name.startswith("backbone.") else ("head", lr_head)
        no_decay = p.ndim <= 1 or name.endswith(".bias")
        key = (tag, no_decay)
        if key not in groups:
            groups[key] = {"params": [], "lr": lr, "weight_decay": 0.0 if no_decay else wd,
                           "name": f"{tag}{'_nd' if no_decay else ''}"}
        groups[key]["params"].append(p)
    return list(groups.values())


def cosine_with_warmup(optimizer, warmup_steps, total_steps, min_factor=0.01):
    def f(step):
        if step < warmup_steps:
            return (step + 1) / max(1, warmup_steps)
        p = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return min_factor + (1 - min_factor) * 0.5 * (1 + math.cos(math.pi * min(1.0, p)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, f)


class EMA:
    """EMA of all floating-point state (decay 0.998 with the usual (1+n)/(10+n) warm-up ramp)."""

    def __init__(self, model: nn.Module, decay=0.998):
        self.module = deepcopy(model).eval()
        for p in self.module.parameters():
            p.requires_grad_(False)
        self.decay, self.n = decay, 0
        src, dst = model.state_dict(), self.module.state_dict()
        self.src = [src[k] for k in src if src[k].is_floating_point()]
        self.dst = [dst[k] for k in src if src[k].is_floating_point()]

    @torch.no_grad()
    def update(self):
        self.n += 1
        d = min(self.decay, (1 + self.n) / (10 + self.n))
        torch._foreach_lerp_(self.dst, self.src, 1.0 - d)


# ----------------------------------------------------------------------------- train / eval
def multitask_loss(logits, labels, smoothing=0.1):
    return sum(WEIGHTS[t] * F.cross_entropy(logits[j].float(), labels[:, j], label_smoothing=smoothing)
               for j, t in enumerate(TARGETS))


AMP = {"dtype": torch.bfloat16}   # set by pick_amp_dtype(): bf16 on Ampere+ (sm80), fp16 + GradScaler on T4/V100/P100


def pick_amp_dtype(device):
    """bf16 autocast needs sm80+; on older GPUs (Kaggle T4/P100) cuDNN has no bf16 engines for the depthwise
    convolutions ("FIND was unable to find an engine"), so use fp16 with loss scaling there."""
    if device.type == "cuda" and torch.cuda.get_device_capability(device)[0] < 8:
        AMP["dtype"] = torch.float16
    else:
        AMP["dtype"] = torch.bfloat16
    return AMP["dtype"]


def cuda_sync(device):
    if device.type == "cuda":
        torch.cuda.synchronize()


def max_mem_gb(device):
    return torch.cuda.max_memory_allocated() / 2**30 if device.type == "cuda" else 0.0


def with_oom_retry(fn, what, retries=3, wait=20):
    """The GPU may be shared with another job: on a CUDA OOM free the cache, wait, and retry."""
    for attempt in range(retries + 1):
        try:
            return fn()
        except torch.OutOfMemoryError:
            if attempt == retries:
                raise
            print(f"   CUDA OOM in {what}; retrying in {wait}s ({attempt + 1}/{retries})", flush=True)
            torch.cuda.empty_cache()
            time.sleep(wait)


@torch.no_grad()
def predict(model, loader, device):
    """Flip-TTA probabilities in the CANONICAL frame, {target: [n, n_classes]} in dataset order.
    TTA = {image as cached, hflip with goal_side L<->R and preferred_foot swapped}; the flipped
    prediction's left/right columns are swapped back before averaging."""
    model.eval()
    outs = {t: [] for t in TARGETS}
    order = []
    perm = {t: FLIP_PERM_T[t].to(device) for t in TARGETS}
    side_flip = torch.as_tensor(SIDE_FLIP).to(device)
    for batch in loader:
        x = batch["x"].to(device, non_blocking=True).float().contiguous(memory_format=torch.channels_last)
        meta = batch["meta"].to(device)
        meta_f = meta.clone()
        meta_f[:, 0] = side_flip[meta_f[:, 0]]
        meta_f[:, 1] = 1 - meta_f[:, 1]

        def fwd():
            with torch.autocast(device.type, dtype=AMP["dtype"], enabled=device.type == "cuda"):
                return model(x, meta), model(x.flip(-1), meta_f)
        lo, lf = with_oom_retry(fwd, "predict")
        for j, t in enumerate(TARGETS):
            p = lo[j].float().softmax(-1)
            pf = lf[j].float().softmax(-1)[:, perm[t]]
            outs[t].append(((p + pf) * 0.5).cpu().numpy())
        order.append(batch["idx"].numpy())
    order = np.concatenate(order)
    assert np.array_equal(order, np.arange(len(order))), "loader must be sequential"
    return {t: np.concatenate(outs[t]).astype(np.float32) for t in TARGETS}


def eval_metrics(probs, true_df):
    """probs in the ORIGINAL label frame vs the original labels."""
    pred = probs_to_labels(probs)
    acc = accuracy_by_target(true_df.reset_index(drop=True), pred)
    acc["score"] = float(sum(WEIGHTS[t] * acc[t] for t in TARGETS))
    return acc


def fmt_acc(acc):
    return " ".join(f"{t.split('_')[0]} {acc[t]:.3f}" for t in TARGETS) + f" | score {acc['score']:.4f}"


def train_fold(k, args, pub, folds, priv, cm_pub, cm_priv, run_dir, model_dir, device, log):
    torch.manual_seed(args.seed + k)
    np.random.seed(args.seed + k)
    tr = np.where(folds.values != k)[0]
    va = np.where(folds.values == k)[0]
    meta_pub, lab_pub = encode_meta(pub, cm_pub), encode_labels(pub, cm_pub)
    mir_va = cm_pub.loc[pub["clip"].values[va], "mirrored"].values.astype(bool)
    mir_te = cm_priv.loc[priv["clip"].values, "mirrored"].values.astype(bool)
    ds_tr = CropDataset(pub["clip"].values[tr], args.view, meta_pub[tr], lab_pub[tr], train=True, end_jitter_p=args.end_jitter,
                        aug=not args.no_aug, flip=not args.no_flip)
    ds_va = CropDataset(pub["clip"].values[va], args.view, meta_pub[va], lab_pub[va], train=False)
    ds_te = CropDataset(priv["clip"].values, args.view, encode_meta(priv, cm_priv), None, train=False)
    dl_tr = make_loader(ds_tr, args.bs, True, args.workers, drop_last=len(ds_tr) >= 2 * args.bs)
    dl_va = make_loader(ds_va, args.eval_bs, False, args.workers)
    dl_te = make_loader(ds_te, args.eval_bs, False, args.workers) if len(ds_te) else None

    model = StackNet(args.backbone, len(STACK_IDX), args.drop, args.drop_path).to(device).to(memory_format=torch.channels_last)
    ema = EMA(model, args.ema)
    opt = torch.optim.AdamW(param_groups(model, args.lr, args.head_lr, args.wd), betas=(0.9, 0.999))
    total_steps = len(dl_tr) * args.epochs
    sched = cosine_with_warmup(opt, warmup_steps=int(len(dl_tr) * args.warmup), total_steps=total_steps)
    bb_group = [g for g in opt.param_groups if g["name"] == "backbone"][0]
    print(f"[fold {k}] train {len(tr)} val {len(va)} test {len(ds_te)} | {len(dl_tr)} it/epoch, {total_steps} steps | "
          f"params {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M", flush=True)

    scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda" and AMP["dtype"] == torch.float16))

    def train_step(x, meta, labels, update=True):
        with torch.autocast(device.type, dtype=AMP["dtype"], enabled=device.type == "cuda"):
            logits = model(x, meta)
        loss = multitask_loss(logits, labels, args.smoothing)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()          # identity when the scaler is disabled (bf16 / CPU)
        if args.clip > 0:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), args.clip)
        if update:
            scaler.step(opt)
            scaler.update()
            sched.step()
            ema.update()
        return loss

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    iter_times = []
    val_probs = None
    last_batch = None
    for epoch in range(args.epochs):
        model.train()
        t_ep = time.time()
        loss_sum, n_loss = 0.0, 0
        t_it = time.time()
        for i, batch in enumerate(dl_tr):
            x = batch["x"].to(device, non_blocking=True).float().contiguous(memory_format=torch.channels_last)
            meta = batch["meta"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)
            loss = with_oom_retry(lambda: train_step(x, meta, labels), "train step")
            loss_sum += loss.item()
            n_loss += 1
            cuda_sync(device)
            iter_times.append(time.time() - t_it)
            t_it = time.time()
            if args.smoke or (i + 1) % 20 == 0:
                print(f"   ep {epoch} it {i + 1}/{len(dl_tr)} loss {loss_sum / n_loss:.4f} lr_bb {bb_group['lr']:.2e} "
                      f"{np.mean(iter_times[-20:]):.3f}s/it", flush=True)
            if args.smoke:
                last_batch = (x, meta, labels)
        if args.smoke and last_batch is not None:   # steady-state compute timing (no data wait, no autotune)
            bench = []
            for _ in range(6):
                cuda_sync(device)
                t_b = time.time()
                with_oom_retry(lambda: train_step(*last_batch, update=False), "bench step")
                cuda_sync(device)
                bench.append(time.time() - t_b)
            iter_times = iter_times + bench[2:]
            print(f"   smoke bench: {len(last_batch[0])}-clip batch fwd+bwd median {np.median(bench[2:]):.3f}s/it "
                  f"(first data-loader iteration incl. worker start-up + cudnn autotune: {iter_times[0]:.1f}s)", flush=True)
        train_time = time.time() - t_ep
        t_ev = time.time()
        val_probs = unswap(predict(ema.module, dl_va, device), mir_va)
        acc = eval_metrics(val_probs, pub.iloc[va][TARGETS])
        rec = {"fold": k, "epoch": epoch, "train_loss": loss_sum / max(1, n_loss), "lr_backbone": bb_group["lr"],
               "val_ema": acc, "train_s": round(train_time, 1), "eval_s": round(time.time() - t_ev, 1),
               "sec_per_epoch": round(train_time + time.time() - t_ev, 1),
               "sec_per_iter": float(np.median(iter_times[-len(dl_tr):])),
               "max_mem_gb": max_mem_gb(device)}
        log.write(json.dumps(rec) + "\n")
        log.flush()
        print(f"[fold {k}] epoch {epoch + 1}/{args.epochs} loss {rec['train_loss']:.4f} | EMA val {fmt_acc(acc)} | "
              f"{train_time:.0f}s train {rec['eval_s']:.0f}s eval {rec['sec_per_iter']:.3f}s/it {rec['max_mem_gb']:.2f}GB",
              flush=True)

    test_probs = None
    if dl_te is not None:
        t0 = time.time()
        test_probs = unswap(predict(ema.module, dl_te, device), mir_te)
        print(f"[fold {k}] test inference {len(ds_te)} clips in {time.time() - t0:.0f}s", flush=True)
    sd = {n: (v.to(torch.bfloat16) if v.is_floating_point() else v) for n, v in ema.module.state_dict().items()}
    torch.save({"state_dict": sd, "backbone": args.backbone, "view": args.view, "fold": k, "epochs": args.epochs,
                "ema_steps": ema.n, "stack_idx": STACK_IDX, "taus": STACK7_TAUS}, model_dir / f"fold{k}.pt")
    np.savez(run_dir / f"fold{k}_preds.npz", val_idx=pub["row"].values[va], test_idx=priv["row"].values,
             **{f"val_{t}": val_probs[t] for t in TARGETS},
             **({f"test_{t}": test_probs[t] for t in TARGETS} if test_probs is not None else {}))
    stats = {"sec_per_iter_median": float(np.median(iter_times[1:] if len(iter_times) > 1 else iter_times)),
             "max_mem_gb": max_mem_gb(device)}
    del model, ema, opt, dl_tr, dl_va, dl_te
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return stats


def aggregate(args, pub_all, folds_all, priv_all, run_dir):
    """fold{k}_preds.npz -> oof_*.npy (load_public order), test_*.npy (load_private order), metrics, submission."""
    n_pub, n_priv = len(pub_all), len(priv_all)
    oof = {t: np.full((n_pub, len(CLASSES[t])), np.nan, np.float32) for t in TARGETS}
    test_sum = {t: np.zeros((n_priv, len(CLASSES[t])), np.float64) for t in TARGETS}
    test_cnt = np.zeros(n_priv)
    done = []
    for k in range(int(folds_all.max()) + 1):
        f = run_dir / f"fold{k}_preds.npz"
        if not f.exists():
            continue
        z = np.load(f)
        for t in TARGETS:
            oof[t][z["val_idx"]] = z[f"val_{t}"]
            if f"test_{t}" in z:
                test_sum[t][z["test_idx"]] += z[f"test_{t}"]
        if f"test_{TARGETS[0]}" in z:
            test_cnt[z["test_idx"]] += 1
        done.append(k)
    if not done:
        print("no fold predictions found")
        return
    covered_te = test_cnt > 0
    test = {}
    for t in TARGETS:
        p = np.full((n_priv, len(CLASSES[t])), np.nan, np.float32)
        p[covered_te] = (test_sum[t][covered_te] / test_cnt[covered_te, None]).astype(np.float32)
        test[t] = p
    for t in TARGETS:
        np.save(run_dir / f"oof_{SHORT[t]}.npy", oof[t])
        np.save(run_dir / f"test_{SHORT[t]}.npy", test[t])
    covered = ~np.isnan(oof[TARGETS[0]][:, 0])
    oof_c = {t: oof[t][covered] for t in TARGETS}
    pred = probs_to_labels(oof_c)
    true = pub_all.loc[covered, TARGETS].reset_index(drop=True)
    score = score_report(true, pred, folds_all[covered].reset_index(drop=True), f"{args.run} OOF (EMA, flip-TTA)")
    acc = accuracy_by_target(true, pred)
    per_fold = {}
    for k in done:
        m = (folds_all[covered].values == k)
        per_fold[k] = {**accuracy_by_target(true[m].reset_index(drop=True), pred[m].reset_index(drop=True))}
        per_fold[k]["score"] = float(sum(WEIGHTS[t] * per_fold[k][t] for t in TARGETS))
    # private rows without crops (roles not available yet) -> class prior of the public set, loudly
    n_missing = int((~covered_te).sum())
    if n_missing:
        print(f"WARNING: {n_missing}/{n_priv} private clips have no crops -> filled with the public class prior",
              flush=True)
    test_filled = {}
    for t in TARGETS:
        prior = pub_all[t].value_counts(normalize=True).reindex(CLASSES[t]).fillna(0).values.astype(np.float32)
        p = test[t].copy()
        p[~covered_te] = prior
        test_filled[t] = p
    test_pred = probs_to_labels(test_filled)
    dist = {t: test_pred[t].value_counts(normalize=True).round(3).to_dict() for t in TARGETS}
    metrics = {"run": args.run, "backbone": args.backbone, "view": args.view, "epochs": args.epochs, "seed": args.seed,
               "folds_done": done, "n_oof": int(covered.sum()), "n_test_covered": int(covered_te.sum()),
               "oof_acc": acc, "oof_score": score, "per_fold": per_fold, "test_pred_distribution": dist,
               "smoke": bool(args.smoke), "stack_idx": STACK_IDX, "taus": STACK7_TAUS}
    (run_dir / "metrics.json").write_text(json.dumps(metrics, indent=2))
    print("private predicted distribution:", dist)
    sub = pd.concat([priv_all[["clip"]].reset_index(drop=True), test_pred], axis=1)
    if args.smoke:
        sub.to_csv(run_dir / "submission_smoke.csv", index=False)
        print(f"smoke: wrote {run_dir / 'submission_smoke.csv'} ({len(sub)} rows; template validation skipped)")
    else:
        write_submission(sub, run_dir / "submission.csv")
    return metrics


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--run", default=None)
    ap.add_argument("--view", default="kicker", choices=["kicker", "keeper"])
    ap.add_argument("--backbone", default=BACKBONE)
    ap.add_argument("--epochs", type=int, default=15)
    ap.add_argument("--folds", default="0,1,2,3,4")
    ap.add_argument("--seed", type=int, default=SEED)
    ap.add_argument("--bs", type=int, default=32)
    ap.add_argument("--eval-bs", type=int, default=32)
    ap.add_argument("--workers", type=int, default=3)
    ap.add_argument("--lr", type=float, default=1e-4, help="backbone lr")
    ap.add_argument("--head-lr", type=float, default=1e-3, help="heads + embeddings lr")
    ap.add_argument("--wd", type=float, default=0.05)
    ap.add_argument("--warmup", type=float, default=1.0, help="linear warm-up epochs before the cosine decay")
    ap.add_argument("--clip", type=float, default=0.0, help="grad-norm clip (0 = off)")
    ap.add_argument("--ema", type=float, default=0.998)
    ap.add_argument("--smoothing", type=float, default=0.1)
    ap.add_argument("--drop", type=float, default=0.3)
    ap.add_argument("--drop-path", type=float, default=0.1)
    ap.add_argument("--end-jitter", type=float, default=0.25)
    ap.add_argument("--cpu", action="store_true", help="force CPU training (the laptop GPU is unreliable)")
    ap.add_argument("--threads", type=int, default=8, help="torch CPU threads when --cpu")
    ap.add_argument("--no-aug", action="store_true", help="ablation: no random-resized-crop / colour jitter")
    ap.add_argument("--no-flip", action="store_true", help="ablation: no hflip label-swap augmentation")
    ap.add_argument("--skip-done", action="store_true", help="skip folds whose fold{k}_preds.npz exists")
    ap.add_argument("--smoke", action="store_true", help="48 public clips, 2 folds, 1 epoch")
    args = ap.parse_args()
    if args.run is None:
        args.run = f"smoke_stack7_{args.view}" if args.smoke else f"stack7_{args.view}_s{args.seed}"
    fold_list = [int(v) for v in args.folds.split(",")]

    pub_all, priv_all = load_public(), load_private()
    folds_all = get_folds(pub_all)
    pub_all["row"], priv_all["row"] = np.arange(len(pub_all)), np.arange(len(priv_all))
    cm_pub = crop_meta_table(pub_all["clip"], args.view)
    cm_priv = crop_meta_table(priv_all["clip"], args.view)
    has_pub = pub_all["clip"].isin(cm_pub.index).values
    has_priv = priv_all["clip"].isin(cm_priv.index).values
    print(f"crops_{args.view}: {has_pub.sum()}/{len(pub_all)} public and {has_priv.sum()}/{len(priv_all)} private clips "
          f"have crops -> training on that subset", flush=True)
    if len(cm_pub):
        print(f"   public crops: mirrored {cm_pub.mirrored.mean():.3f} side_unknown {cm_pub.side_unknown.mean():.3f} "
              f"fallback {cm_pub.fallback.mean():.3f}", flush=True)
    pub, folds = pub_all[has_pub].reset_index(drop=True), folds_all[has_pub].reset_index(drop=True)
    priv = priv_all[has_priv].reset_index(drop=True)
    if args.smoke:
        sel = np.concatenate([np.where(folds.values == 0)[0][:24], np.where(folds.values == 1)[0][:24]])
        pub, folds = pub.iloc[sel].reset_index(drop=True), folds.iloc[sel].reset_index(drop=True)
        priv = priv.iloc[:16].reset_index(drop=True)
        fold_list, args.epochs = [0, 1], 1
        print(f"smoke: {len(pub)} public clips (folds 0/1), {len(priv)} private clips, 1 epoch", flush=True)
    for k in fold_list:
        assert (folds.values == k).sum() > 0, f"fold {k} has no clips with crops"

    run_dir, model_dir = RUNS_DIR / args.run, MODELS_DIR / args.run
    run_dir.mkdir(parents=True, exist_ok=True)
    model_dir.mkdir(parents=True, exist_ok=True)
    (run_dir / "args.json").write_text(json.dumps(vars(args), indent=2))
    device = torch.device("cuda" if torch.cuda.is_available() and not args.cpu else "cpu")
    if device.type == "cpu":
        torch.set_num_threads(args.threads)
        print(f"running on CPU with {args.threads} threads (bf16 autocast disabled)", flush=True)
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    if device.type == "cuda":
        print(f"GPU {torch.cuda.get_device_name(device)} capability {torch.cuda.get_device_capability(device)} "
              f"-> autocast {pick_amp_dtype(device)}", flush=True)
    print(f"run {args.run}: view={args.view} backbone={args.backbone} epochs={args.epochs} folds={fold_list} bs={args.bs} "
          f"workers={args.workers} seed={args.seed} | stack idx {STACK_IDX} (taus {STACK7_TAUS}) | "
          f"public {len(pub)} private {len(priv)}", flush=True)

    t_run = time.time()
    stats_all = []
    with open(run_dir / "log.jsonl", "a") as log:
        for k in fold_list:
            if args.skip_done and (run_dir / f"fold{k}_preds.npz").exists():
                print(f"[fold {k}] already done, skipping")
                continue
            t0 = time.time()
            stats = train_fold(k, args, pub, folds, priv, cm_pub, cm_priv, run_dir, model_dir, device, log)
            stats_all.append(stats)
            print(f"[fold {k}] done in {(time.time() - t0) / 60:.1f} min | median {stats['sec_per_iter_median']:.3f} s/iter "
                  f"| max VRAM {stats['max_mem_gb']:.2f} GB", flush=True)
    aggregate(args, pub_all, folds_all, priv_all, run_dir)
    if stats_all:
        print(f"SUMMARY sec/iter median {np.median([s['sec_per_iter_median'] for s in stats_all]):.3f} | "
              f"max VRAM {max(s['max_mem_gb'] for s in stats_all):.2f} GB", flush=True)
    print(f"total {(time.time() - t_run) / 60:.1f} min")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/train_videomae.py
"""Multi-task VideoMAE fine-tune (plan "end-to-end-finetune", Stages B-D) on
  * frame cache A   (--view dense16 | wide16): 16 full frames 352x208 squashed to 224x224
                    (or kept at 208x352 with --size 208x352), or
  * the canonical crop caches of crops.py (--view kicker_crop | keeper_crop): 16 crops
    224x224 at tau 0.60..0.00 s, already mirrored so the goal is on the image-left
    (mirrored=1 for goal_side 'R'); training labels and preferred_foot are swapped into
    that canonical frame through config.FLIP_MAP, predictions of mirrored clips are
    swapped back (left/right columns) for the OOF and test outputs.

Flip + label-swap augmentation, metadata embeddings (preferred_foot, gender, presented
goal side L/R/U), three heads (kick 3 / keeper 3 / goal 2), AdamW + layer-wise lr decay +
cosine + EMA, 5 source-grouped folds, flip-TTA, fold-ensembled private predictions and a
validated submission.

Run:  python src/train_videomae.py --run vmae_k400_dense16            (full 5-fold run, K400 init)
      python src/train_videomae.py --run vmae_ssv2_kicker --view kicker_crop   (SSv2 init by default)
      python src/train_videomae.py --smoke [--view keeper_crop]        (24 clips, 2 folds, 1 epoch)
      python src/train_videomae.py --run x --view wide16 --size 208x352 --grad-ckpt

Outputs: D:\\cv3\\models\\<run>\\fold{k}.pt (bf16 EMA state dict)
         D:\\cv3\\runs\\<run>\\{log.jsonl, fold{k}_preds.npz, oof_*.npy, test_*.npy,
                                metrics.json, submission.csv}   (probabilities in the ORIGINAL label frame)
"""
import argparse
import faulthandler
import json
import math
import os
import sys
import time
from copy import deepcopy

faulthandler.enable()
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import config  # noqa: F401  (sets HF_HOME / TORCH_HOME before torch & transformers are imported)

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")  # less fragmentation on a shared 8 GB GPU
for _v in ("OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):  # numpy BLAS thread pools cost RAM in every DataLoader worker
    os.environ.setdefault(_v, "1")

try:  # pandas 3 lazily loads pyarrow for string columns; if cv2 AND torch are imported first,
    import pyarrow  # noqa: F401  # read_csv dies with an access violation (DLL clash). Load it first.
except ImportError:
    pass

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

from config import CLASSES, FLIP_MAP, MODELS_DIR, RUNS_DIR, SEED, TARGETS, WEIGHTS
from crops import N_FRAMES as N_CROP_FRAMES, TAUS16, crops_dir, load_crop_meta, load_crops
from cv import accuracy_by_target, get_folds, load_private, load_public, probs_to_labels, score_report, write_submission
from framecache import CACHE_A, DENSE16, WIDE16, load_frames

cv2.setNumThreads(1)

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], np.float32)
FOOT_IDX = {"left": 0, "right": 1}
GENDER_IDX = {"M": 0, "F": 1}
SIDE_IDX = {"L": 0, "R": 1, "U": 2}          # goal side of the PRESENTED image (canonical crops: L unless unknown)
SIDE_FLIP = np.array([1, 0, 2])              # hflip of the presented image: L<->R, U stays
CROP_VIEWS = {"kicker_crop": "kicker", "keeper_crop": "keeper"}   # --view -> crops.py view name
VIEWS = {"dense16": DENSE16, "wide16": WIDE16,
         "kicker_crop": list(range(N_CROP_FRAMES)), "keeper_crop": list(range(N_CROP_FRAMES))}
N_CLASSES = [len(CLASSES[t]) for t in TARGETS]
# column permutation that un-swaps a prediction made on the mirrored clip (involution)
FLIP_PERM = {t: np.array([CLASSES[t].index(FLIP_MAP[t][c]) for c in CLASSES[t]]) for t in TARGETS}
FLIP_PERM_T = {t: torch.as_tensor(FLIP_PERM[t]) for t in TARGETS}
K400_INIT = "MCG-NJU/videomae-base-finetuned-kinetics"
SSV2_INIT = "MCG-NJU/videomae-base-finetuned-ssv2"       # direction-aware init: default for the crop views
DEFAULT_INIT = K400_INIT
SHORT = {"kick_direction": "kick", "goalkeeper_direction": "gk", "goal_outcome": "goal"}


# ----------------------------------------------------------------------------- canonical frame
def crop_meta_table(clips, crop_view):
    """{clip: (mirrored, side_unknown, fallback)} read from the crop npz files (authoritative, like
    train_stack7.py); raises if any clip has no crops."""
    out, missing = {}, []
    for c in clips:
        if not (crops_dir(crop_view) / c.replace(".mp4", ".npz")).exists():
            missing.append(c)
            continue
        m = load_crop_meta(c, crop_view)
        assert np.allclose(m["t_from_end"], TAUS16, atol=1e-5), (c, m["t_from_end"])
        out[c] = (int(m["mirrored"]), int(m["side_unknown"]), int(m["fallback"]))
    if missing:
        raise FileNotFoundError(f"{len(missing)} clips without crops_{crop_view}, e.g. {missing[:5]}")
    return out


def canon_arrays(df: pd.DataFrame, view: str):
    """(mirrored bool[n], side_unknown bool[n]) for the clips of df. Frame cache A views are never
    mirrored and carry no goal-side information (side 'U')."""
    if view in CROP_VIEWS:
        tab = crop_meta_table(df["clip"].values, CROP_VIEWS[view])
        mirrored = np.array([tab[c][0] for c in df["clip"].values], bool)
        unknown = np.array([tab[c][1] for c in df["clip"].values], bool)
        return mirrored, unknown
    return np.zeros(len(df), bool), np.ones(len(df), bool)


def encode_meta(df: pd.DataFrame, mirrored: np.ndarray, side_unknown: np.ndarray) -> np.ndarray:
    """int64 [n, 3] in the CANONICAL (presented-image) frame: preferred_foot (left=0,right=1; swapped
    for mirrored clips), gender (M=0,F=1), presented goal side (L unless unknown -> U).
    Never fps/resolution/duration."""
    foot = df["preferred_foot"].map(FOOT_IDX)
    gender = df["gender"].map(GENDER_IDX)
    if foot.isna().any() or gender.isna().any():
        raise ValueError("unknown preferred_foot / gender value")
    foot = np.where(mirrored, 1 - foot.values, foot.values)
    side = np.where(side_unknown, SIDE_IDX["U"], SIDE_IDX["L"])
    return np.stack([foot, gender.values, side], 1).astype(np.int64)


def encode_labels(df: pd.DataFrame, mirrored: np.ndarray) -> np.ndarray:
    """int64 [n, 3]: class index in config.CLASSES order, in the CANONICAL frame (mirrored clips:
    kick/keeper left<->right through FLIP_MAP, goal unchanged)."""
    cols = [df[t].map({c: i for i, c in enumerate(CLASSES[t])}) for t in TARGETS]
    out = np.stack([c.values for c in cols], 1)
    if np.isnan(out.astype(float)).any():
        raise ValueError("unknown label value")
    out = out.astype(np.int64)
    for j, t in enumerate(TARGETS):
        out[mirrored, j] = FLIP_PERM[t][out[mirrored, j]]
    return out


def unswap(probs: dict, mirrored: np.ndarray) -> dict:
    """Canonical-frame probabilities -> original label frame (mirrored rows get left/right swapped)."""
    out = {}
    for t in TARGETS:
        p = probs[t].copy()
        p[mirrored] = p[mirrored][:, FLIP_PERM[t]]
        out[t] = p
    return out


# ----------------------------------------------------------------------------- data
class ClipDataset(Dataset):
    """Cache clip -> pixel_values float16 [16, 3, H, W] (ImageNet-normalised), meta int64 [3].

    Train-time augmentation (identical for all 16 frames of a clip):
      * end-jitter p=0.25: indices shifted one cache step earlier (cache A: k=16..1; crops: first frame repeated)
      * random resized crop: area scale (0.8, 1.0), aspect x(0.9, 1.1), width never < 80 %
      * brightness / contrast / saturation +-0.2 with p=0.8
      * horizontal flip p=0.5 with label swap (kick/keeper left<->right, preferred_foot and goal side swapped)
    """

    def __init__(self, clips, meta, labels, view, size, train, flip=True):
        self.clips = list(clips)
        self.meta = np.asarray(meta, np.int64)
        self.labels = None if labels is None else np.asarray(labels, np.int64)
        self.view = view
        self.crop_view = CROP_VIEWS.get(view)
        self.idx = list(VIEWS[view])
        self.size = tuple(size)  # (H, W)
        self.train = train
        self.flip = flip        # hflip + label swap augmentation (ablation switch --no-flip)
        self._rng, self._rng_seed = None, None

    def __len__(self):
        return len(self.clips)

    def rng(self):
        seed = torch.initial_seed() % 2**32  # differs per worker and per epoch
        if self._rng_seed != seed:
            self._rng, self._rng_seed = np.random.default_rng(seed), seed
        return self._rng

    def frame_indices(self, rng):
        idx = self.idx
        if self.train and rng.random() < 0.25:  # end-jitter: one cache step earlier
            if self.crop_view:                   # chronological 0..15 -> 0,0,1..14
                idx = [max(0, j - 1) for j in idx]
            else:                                # cache A: k (steps before the end) + 1, grid frames only
                idx = [min(k + 1, 23) if k < 24 else k for k in idx]
        return idx

    def load(self, clip, idx):
        if self.crop_view:
            return load_crops(clip, self.crop_view, idx)  # uint8 [T, 224, 224, 3] RGB, canonical orientation
        return load_frames(clip, idx)                      # uint8 [T, 208, 352, 3] RGB

    @staticmethod
    def crop_box(rng, h, w):
        s = rng.uniform(0.8, 1.0)
        r = rng.uniform(0.9, 1.1)
        cw = int(round(min(w, w * math.sqrt(s * r))))
        ch = int(round(min(h, h * math.sqrt(s / r))))
        cw = max(cw, int(math.ceil(0.8 * w)))
        ch = max(ch, int(math.ceil(0.8 * h)))
        x0 = int(rng.integers(0, w - cw + 1))
        y0 = int(rng.integers(0, h - ch + 1))
        return x0, y0, cw, ch

    def __getitem__(self, i):
        rng = self.rng() if self.train else None
        idx = self.frame_indices(rng) if self.train else self.idx
        frames = self.load(self.clips[i], idx)
        meta = self.meta[i].copy()
        labels = None if self.labels is None else self.labels[i].copy()
        flipped = False
        if self.train:
            x0, y0, cw, ch = self.crop_box(rng, frames.shape[1], frames.shape[2])
            frames = frames[:, y0:y0 + ch, x0:x0 + cw]
            if self.flip and rng.random() < 0.5:
                frames = frames[:, :, ::-1]
                flipped = True
        th, tw = self.size
        if frames.shape[1] != th or frames.shape[2] != tw:
            interp = cv2.INTER_AREA if frames.shape[2] >= tw else cv2.INTER_LINEAR
            frames = np.stack([cv2.resize(np.ascontiguousarray(f), (tw, th), interpolation=interp) for f in frames])
        # [T,3,H,W]; force NCHW-contiguous: broadcast ops on a permuted (channels-last) view are ~20x slower
        x = torch.from_numpy(np.ascontiguousarray(frames)).permute(0, 3, 1, 2).to(
            torch.float32, memory_format=torch.contiguous_format).div_(255.0)
        if self.train and rng.random() < 0.8:
            b, c, s = rng.uniform(0.8, 1.2, size=3)
            x.mul_(b)
            gray = (0.299 * x[:, 0] + 0.587 * x[:, 1] + 0.114 * x[:, 2]).unsqueeze(1)
            x = (x - gray.mean()) * c + gray.mean()
            gray = (0.299 * x[:, 0] + 0.587 * x[:, 1] + 0.114 * x[:, 2]).unsqueeze(1)
            x = gray + (x - gray) * s
            x.clamp_(0.0, 1.0)
        x.sub_(torch.from_numpy(IMAGENET_MEAN).view(1, 3, 1, 1)).div_(torch.from_numpy(IMAGENET_STD).view(1, 3, 1, 1))
        if flipped:
            meta[0] = 1 - meta[0]            # preferred_foot left<->right
            meta[2] = SIDE_FLIP[meta[2]]     # presented goal side L<->R (U stays)
            if labels is not None:
                labels = np.array([FLIP_PERM[t][labels[j]] for j, t in enumerate(TARGETS)], np.int64)
        out = {"pixel_values": x.half(), "meta": torch.from_numpy(meta), "idx": i}
        if labels is not None:
            out["labels"] = torch.from_numpy(labels)
        return out


def worker_init(_):
    cv2.setNumThreads(1)
    torch.set_num_threads(1)


def make_loader(ds, bs, shuffle, workers, persistent=True, drop_last=False):
    """Windows spawns workers (each re-imports torch, ~0.4 GB RSS): keep the train loader persistent,
    eval loaders non-persistent so their workers only live while evaluating (16 GB RAM is tight)."""
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, num_workers=workers, pin_memory=True,
                      drop_last=drop_last, persistent_workers=persistent and workers > 0, worker_init_fn=worker_init,
                      prefetch_factor=4 if workers > 0 else None)


# ----------------------------------------------------------------------------- model
class VideoMAEMultiTask(nn.Module):
    """VideoMAE encoder (mean-pooled + fc_norm) + metadata embeddings -> three linear heads."""

    def __init__(self, init: str, num_frames: int, image_size, grad_ckpt: bool = False):
        super().__init__()
        from transformers import VideoMAEConfig, VideoMAEForVideoClassification

        cfg = VideoMAEConfig.from_pretrained(init)
        cfg.num_frames = num_frames
        cfg.image_size = list(image_size) if image_size[0] != image_size[1] else int(image_size[0])
        self.backbone = VideoMAEForVideoClassification.from_pretrained(
            init, config=cfg, ignore_mismatched_sizes=True, attn_implementation="sdpa")
        self.backbone.classifier = nn.Identity()  # -> 768-d pooled feature (after fc_norm)
        if grad_ckpt:
            self.backbone.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        d = cfg.hidden_size
        self.emb_foot = nn.Embedding(2, 8)
        self.emb_gender = nn.Embedding(2, 8)
        self.emb_side = nn.Embedding(3, 8)   # presented goal side L/R/U ('U' for the full-frame views)
        self.drop = nn.Dropout(0.3)
        self.heads = nn.ModuleList([nn.Linear(d + 24, n) for n in N_CLASSES])
        self.num_layers = cfg.num_hidden_layers

    def forward(self, pixel_values, meta):
        feat = self.backbone(pixel_values=pixel_values).logits
        z = torch.cat([feat, self.emb_foot(meta[:, 0]), self.emb_gender(meta[:, 1]), self.emb_side(meta[:, 2])], 1)
        z = self.drop(z)
        return [h(z) for h in self.heads]

    def pin_pos_embed(self, device):
        """The sinusoidal table is a plain tensor attribute (not a buffer): move it once."""
        emb = self.backbone.videomae.embeddings
        emb.position_embeddings = emb.position_embeddings.to(device)


def param_groups(model: VideoMAEMultiTask, lr_bb, lr_head, wd, layer_decay):
    n_layers = model.num_layers
    groups = {}
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if name.startswith("backbone."):
            sub = name[len("backbone."):]
            if sub.startswith("videomae.embeddings"):
                lid = 0
            elif sub.startswith("videomae.encoder.layer."):
                lid = int(sub.split(".")[3]) + 1
            else:  # fc_norm
                lid = n_layers + 1
            lr = lr_bb * layer_decay ** (n_layers + 1 - lid)
            tag = f"L{lid:02d}"
        else:
            lr, tag = lr_head, "head"
        no_decay = p.ndim <= 1 or name.endswith(".bias")
        key = (tag, no_decay)
        if key not in groups:
            groups[key] = {"params": [], "lr": lr, "weight_decay": 0.0 if no_decay else wd, "name": f"{tag}{'_nd' if no_decay else ''}"}
        groups[key]["params"].append(p)
    return list(groups.values())


def cosine_with_warmup(optimizer, warmup_steps, total_steps, min_factor=0.01):
    def f(step):
        if step < warmup_steps:
            return (step + 1) / max(1, warmup_steps)
        p = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return min_factor + (1 - min_factor) * 0.5 * (1 + math.cos(math.pi * min(1.0, p)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, f)


class EMA:
    """Exponential moving average of all floating-point state (decay 0.998, with the usual
    warm-up ramp min(decay, (1+n)/(10+n)) so the first steps do not keep the random heads)."""

    def __init__(self, model: nn.Module, decay=0.998):
        self.module = deepcopy(model).eval()
        for p in self.module.parameters():
            p.requires_grad_(False)
        self.decay, self.n = decay, 0
        src, dst = model.state_dict(), self.module.state_dict()
        self.src = [src[k] for k in src if src[k].is_floating_point()]
        self.dst = [dst[k] for k in src if src[k].is_floating_point()]

    @torch.no_grad()
    def update(self):
        self.n += 1
        d = min(self.decay, (1 + self.n) / (10 + self.n))
        torch._foreach_lerp_(self.dst, self.src, 1.0 - d)


# ----------------------------------------------------------------------------- train / eval
def multitask_loss(logits, labels, smoothing=0.1):
    return sum(WEIGHTS[t] * F.cross_entropy(logits[j].float(), labels[:, j], label_smoothing=smoothing)
               for j, t in enumerate(TARGETS))


@torch.no_grad()
def predict(model, loader, device):
    """Flip-TTA probabilities in the CANONICAL (presented-image) frame, {target: [n, n_classes]} in
    dataset order. TTA = {image as cached, hflip with preferred_foot and goal side swapped}; the flipped
    prediction's left/right columns are swapped back before averaging."""
    model.eval()
    outs = {t: [] for t in TARGETS}
    order = []
    perm = {t: FLIP_PERM_T[t].to(device) for t in TARGETS}
    side_flip = torch.as_tensor(SIDE_FLIP).to(device)
    for batch in loader:
        x = batch["pixel_values"].to(device, non_blocking=True).float()
        meta = batch["meta"].to(device)
        meta_f = meta.clone()
        meta_f[:, 0] = 1 - meta_f[:, 0]
        meta_f[:, 2] = side_flip[meta_f[:, 2]]
        with torch.autocast("cuda", dtype=torch.bfloat16):
            lo = model(x, meta)
            lf = model(x.flip(-1), meta_f)
        for j, t in enumerate(TARGETS):
            p = lo[j].float().softmax(-1)
            pf = lf[j].float().softmax(-1)[:, perm[t]]
            outs[t].append(((p + pf) * 0.5).cpu().numpy())
        order.append(batch["idx"].numpy())
    order = np.concatenate(order)
    assert np.array_equal(order, np.arange(len(order))), "loader must be sequential"
    return {t: np.concatenate(outs[t]).astype(np.float32) for t in TARGETS}


def eval_metrics(probs, true: pd.DataFrame):
    """probs in the ORIGINAL label frame vs the original label strings."""
    pred = probs_to_labels(probs)
    acc = accuracy_by_target(true.reset_index(drop=True), pred)
    acc["score"] = float(sum(WEIGHTS[t] * acc[t] for t in TARGETS))
    return acc


def fmt_acc(acc):
    return " ".join(f"{t.split('_')[0]} {acc[t]:.3f}" for t in TARGETS) + f" | score {acc['score']:.4f}"


def train_fold(k, args, pub, folds, priv, run_dir, model_dir, device, log):
    torch.manual_seed(args.seed + k)
    np.random.seed(args.seed + k)
    tr = np.where(folds.values != k)[0]
    va = np.where(folds.values == k)[0]
    mir_pub, unk_pub = canon_arrays(pub, args.view)
    mir_te, unk_te = canon_arrays(priv, args.view)
    meta_pub, lab_pub = encode_meta(pub, mir_pub, unk_pub), encode_labels(pub, mir_pub)
    mir_va = mir_pub[va]
    ds_tr = ClipDataset(pub["clip"].values[tr], meta_pub[tr], lab_pub[tr], args.view, args.size, train=True,
                        flip=not args.no_flip)
    ds_va = ClipDataset(pub["clip"].values[va], meta_pub[va], lab_pub[va], args.view, args.size, train=False)
    ds_te = ClipDataset(priv["clip"].values, encode_meta(priv, mir_te, unk_te), None, args.view, args.size, train=False)
    dl_tr = make_loader(ds_tr, args.bs, True, args.workers, persistent=True)
    dl_va = make_loader(ds_va, args.eval_bs, False, args.eval_workers, persistent=False)
    dl_te = make_loader(ds_te, args.eval_bs, False, args.eval_workers, persistent=False)

    model = VideoMAEMultiTask(args.init, 16, args.size, args.grad_ckpt).to(device)
    model.pin_pos_embed(device)
    ema = EMA(model, args.ema)
    # fused AdamW: one in-place kernel, no per-step temporaries (the multi-tensor path OOMed on the shared GPU)
    opt = torch.optim.AdamW(param_groups(model, args.lr, args.head_lr, args.wd, args.llrd), betas=(0.9, 0.999), fused=True)
    steps_per_epoch = math.ceil(len(dl_tr) / args.accum)
    total_steps = steps_per_epoch * args.epochs
    sched = cosine_with_warmup(opt, warmup_steps=steps_per_epoch * args.warmup, total_steps=total_steps)
    last_group = [g for g in opt.param_groups if g["name"] == f"L{model.num_layers:02d}"][0]
    print(f"[fold {k}] train {len(tr)} val {len(va)} test {len(ds_te)} | mirrored train {mir_pub[tr].mean():.3f} "
          f"val {mir_va.mean():.3f} test {mir_te.mean():.3f}, side unknown {unk_pub.mean():.3f} | {len(dl_tr)} it/epoch, "
          f"{steps_per_epoch} opt steps/epoch, {total_steps} total | params "
          f"{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M", flush=True)

    torch.cuda.reset_peak_memory_stats()
    iter_times = []
    val_probs = None
    for epoch in range(args.epochs):
        model.train()
        t_ep = time.time()
        loss_sum, n_loss = 0.0, 0
        opt.zero_grad(set_to_none=True)
        t_it = time.time()
        for i, batch in enumerate(dl_tr):
            x = batch["pixel_values"].to(device, non_blocking=True).float()
            meta = batch["meta"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)
            with torch.autocast("cuda", dtype=torch.bfloat16):
                logits = model(x, meta)
            loss = multitask_loss(logits, labels, args.smoothing)
            (loss / args.accum).backward()
            loss_sum += loss.item()
            n_loss += 1
            if (i + 1) % args.accum == 0 or (i + 1) == len(dl_tr):
                torch.nn.utils.clip_grad_norm_(model.parameters(), args.clip)
                opt.step()
                opt.zero_grad(set_to_none=True)
                sched.step()
                ema.update()
            torch.cuda.synchronize()
            iter_times.append(time.time() - t_it)
            t_it = time.time()
            if args.smoke or (i + 1) % 50 == 0:
                print(f"   ep {epoch} it {i + 1}/{len(dl_tr)} loss {loss_sum / n_loss:.4f} "
                      f"lr {last_group['lr']:.2e} {np.mean(iter_times[-50:]):.3f}s/it", flush=True)
        train_time = time.time() - t_ep
        t_ev = time.time()
        val_probs = unswap(predict(ema.module, dl_va, device), mir_va)   # -> original label frame
        acc = eval_metrics(val_probs, pub.iloc[va][TARGETS])
        rec = {"fold": k, "epoch": epoch, "train_loss": loss_sum / max(1, n_loss), "lr_last_block": last_group["lr"],
               "val_ema": acc, "train_s": round(train_time, 1), "eval_s": round(time.time() - t_ev, 1),
               "sec_per_iter": float(np.median(iter_times[-len(dl_tr):])),
               "max_mem_gb": torch.cuda.max_memory_allocated() / 2**30}
        if args.eval_raw:
            rec["val_raw"] = eval_metrics(unswap(predict(model, dl_va, device), mir_va), pub.iloc[va][TARGETS])
        log.write(json.dumps(rec) + "\n")
        log.flush()
        print(f"[fold {k}] epoch {epoch + 1}/{args.epochs} loss {rec['train_loss']:.4f} | EMA val {fmt_acc(acc)}"
              + (f" | raw {fmt_acc(rec['val_raw'])}" if args.eval_raw else "")
              + f" | {train_time:.0f}s train {rec['eval_s']:.0f}s eval {rec['sec_per_iter']:.3f}s/it "
              f"{rec['max_mem_gb']:.2f}GB", flush=True)

    t0 = time.time()
    test_probs = unswap(predict(ema.module, dl_te, device), mir_te)       # -> original label frame
    print(f"[fold {k}] test inference {len(ds_te)} clips in {time.time() - t0:.0f}s", flush=True)
    sd = {n: (v.to(torch.bfloat16) if v.is_floating_point() else v) for n, v in ema.module.state_dict().items()}
    torch.save({"state_dict": sd, "init": args.init, "view": args.view, "size": list(args.size), "fold": k,
                "epochs": args.epochs, "ema_steps": ema.n}, model_dir / f"fold{k}.pt")
    np.savez(run_dir / f"fold{k}_preds.npz", val_idx=va,
             **{f"val_{t}": val_probs[t] for t in TARGETS}, **{f"test_{t}": test_probs[t] for t in TARGETS})
    stats = {"sec_per_iter_median": float(np.median(iter_times[1:] if len(iter_times) > 1 else iter_times)),
             "max_mem_gb": torch.cuda.max_memory_allocated() / 2**30}
    del model, ema, opt, dl_tr, dl_va, dl_te
    torch.cuda.empty_cache()
    return stats


def aggregate(args, pub, folds, priv, run_dir):
    """Collect fold{k}_preds.npz (original label frame) -> oof_*.npy, test_*.npy, metrics.json, submission.csv."""
    n_pub, n_priv = len(pub), len(priv)
    oof = {t: np.full((n_pub, len(CLASSES[t])), np.nan, np.float32) for t in TARGETS}
    test_sum = {t: np.zeros((n_priv, len(CLASSES[t])), np.float64) for t in TARGETS}
    done = []
    for k in range(folds.max() + 1):
        f = run_dir / f"fold{k}_preds.npz"
        if not f.exists():
            continue
        z = np.load(f)
        for t in TARGETS:
            oof[t][z["val_idx"]] = z[f"val_{t}"]
            test_sum[t] += z[f"test_{t}"]
        done.append(k)
    if not done:
        print("no fold predictions found")
        return
    test = {t: (test_sum[t] / len(done)).astype(np.float32) for t in TARGETS}
    for t in TARGETS:
        np.save(run_dir / f"oof_{SHORT[t]}.npy", oof[t])
        np.save(run_dir / f"test_{SHORT[t]}.npy", test[t])
    covered = ~np.isnan(oof[TARGETS[0]][:, 0])
    oof_c = {t: oof[t][covered] for t in TARGETS}
    pred = probs_to_labels(oof_c)
    true = pub.loc[covered, TARGETS].reset_index(drop=True)
    score = score_report(true, pred, folds[covered].reset_index(drop=True), f"{args.run} OOF (EMA, flip-TTA)")
    acc = accuracy_by_target(true, pred)
    per_fold = {}
    for k in done:
        m = (folds[covered].values == k)
        per_fold[k] = {**accuracy_by_target(true[m].reset_index(drop=True), pred[m].reset_index(drop=True))}
        per_fold[k]["score"] = float(sum(WEIGHTS[t] * per_fold[k][t] for t in TARGETS))
    test_pred = probs_to_labels(test)
    dist = {t: test_pred[t].value_counts(normalize=True).round(3).to_dict() for t in TARGETS}
    metrics = {"run": args.run, "init": args.init, "view": args.view, "size": list(args.size), "epochs": args.epochs,
               "folds_done": done, "n_oof": int(covered.sum()), "oof_acc": acc, "oof_score": score,
               "per_fold": per_fold, "test_pred_distribution": dist, "smoke": bool(args.smoke)}
    (run_dir / "metrics.json").write_text(json.dumps(metrics, indent=2))
    print("private predicted distribution:", dist)
    sub = pd.concat([priv[["clip"]].reset_index(drop=True), test_pred], axis=1)
    if args.smoke:
        sub.to_csv(run_dir / "submission_smoke.csv", index=False)
        print(f"smoke: wrote {run_dir / 'submission_smoke.csv'} ({len(sub)} rows; template validation skipped)")
    else:
        write_submission(sub, run_dir / "submission.csv")
    return metrics


def parse_size(s):
    if "x" in s:
        h, w = (int(v) for v in s.lower().split("x"))
    else:
        h = w = int(s)
    if h % 16 or w % 16:
        raise ValueError("size must be multiples of 16")
    return (h, w)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--run", default=None)
    ap.add_argument("--init", default=None, help=f"HF checkpoint; default {K400_INIT} (cache A views) / {SSV2_INIT} (crop views)")
    ap.add_argument("--view", default="dense16", choices=list(VIEWS))
    ap.add_argument("--size", default="224", help="224 or HxW, e.g. 208x352 (cache A views only)")
    ap.add_argument("--epochs", type=int, default=12)
    ap.add_argument("--folds", default="0,1,2,3,4")
    ap.add_argument("--bs", type=int, default=4)
    ap.add_argument("--accum", type=int, default=4)
    ap.add_argument("--eval-bs", type=int, default=8)
    ap.add_argument("--workers", type=int, default=4, help="train DataLoader workers (persistent)")
    ap.add_argument("--eval-workers", type=int, default=2, help="val/test DataLoader workers (spawned per eval; 0 = in-process)")
    ap.add_argument("--lr", type=float, default=5e-5, help="backbone peak lr (last block)")
    ap.add_argument("--head-lr", type=float, default=5e-4)
    ap.add_argument("--llrd", type=float, default=0.75)
    ap.add_argument("--wd", type=float, default=0.05)
    ap.add_argument("--warmup", type=int, default=1, help="warm-up epochs")
    ap.add_argument("--clip", type=float, default=1.0)
    ap.add_argument("--ema", type=float, default=0.998)
    ap.add_argument("--smoothing", type=float, default=0.1)
    ap.add_argument("--grad-ckpt", action="store_true")
    ap.add_argument("--eval-raw", action="store_true", help="also evaluate the non-EMA weights each epoch")
    ap.add_argument("--skip-done", action="store_true", help="skip folds whose fold{k}_preds.npz exists")
    ap.add_argument("--smoke", action="store_true", help="24 public clips, 2 folds, 1 epoch, 8 private clips")
    ap.add_argument("--seed", type=int, default=SEED, help="per-fold seed base (seed + fold)")
    ap.add_argument("--no-flip", action="store_true", help="ablation: no hflip label-swap augmentation")
    args = ap.parse_args()
    args.size = parse_size(args.size)
    is_crop = args.view in CROP_VIEWS
    if args.init is None:
        args.init = SSV2_INIT if is_crop else K400_INIT
    if is_crop and args.size != (224, 224):
        print(f"WARNING: crop caches are 224x224; --size {args.size} will resize them", flush=True)
    if args.run is None:
        args.run = f"smoke_{args.view}" if args.smoke else f"vmae_{args.init.split('-')[-1]}_{args.view}_{args.size[0]}x{args.size[1]}"
    fold_list = [int(v) for v in args.folds.split(",")]

    if not is_crop and not (CACHE_A / "_DONE").exists():
        print(f"WARNING: {CACHE_A / '_DONE'} not found - frame cache A may be incomplete", flush=True)
    pub, priv = load_public(), load_private()
    folds = get_folds(pub)
    if args.smoke:
        sel = np.concatenate([np.where(folds.values == 0)[0][:12], np.where(folds.values == 1)[0][:12]])
        pub, folds = pub.iloc[sel].reset_index(drop=True), folds.iloc[sel].reset_index(drop=True)
        priv = priv.iloc[:8].reset_index(drop=True)
        fold_list, args.epochs = [0, 1], 1
        cdir = crops_dir(CROP_VIEWS[args.view]) if is_crop else CACHE_A
        missing = [c for c in list(pub["clip"]) + list(priv["clip"]) if not (cdir / c.replace(".mp4", ".npz")).exists()]
        if missing:
            raise FileNotFoundError(f"{len(missing)} smoke clips missing from {cdir}, e.g. {missing[:3]}")
    if is_crop:  # fail early (and report the canonical-frame statistics) before any GPU work
        mir, unk = canon_arrays(pub, args.view)
        mir_te, unk_te = canon_arrays(priv, args.view)
        print(f"crops_{CROP_VIEWS[args.view]}: public mirrored {mir.mean():.3f} side_unknown {unk.mean():.3f} | "
              f"private mirrored {mir_te.mean():.3f} side_unknown {unk_te.mean():.3f}", flush=True)

    run_dir, model_dir = RUNS_DIR / args.run, MODELS_DIR / args.run
    run_dir.mkdir(parents=True, exist_ok=True)
    model_dir.mkdir(parents=True, exist_ok=True)
    (run_dir / "args.json").write_text(json.dumps({**vars(args), "size": list(args.size)}, indent=2))
    device = torch.device("cuda")
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print(f"run {args.run}: init={args.init} view={args.view} size={args.size} epochs={args.epochs} folds={fold_list} "
          f"bs={args.bs}x{args.accum} workers={args.workers}/{args.eval_workers} grad_ckpt={args.grad_ckpt} "
          f"| public {len(pub)} private {len(priv)}", flush=True)

    t_run = time.time()
    with open(run_dir / "log.jsonl", "a") as log:
        for k in fold_list:
            if args.skip_done and (run_dir / f"fold{k}_preds.npz").exists():
                print(f"[fold {k}] already done, skipping")
                continue
            t0 = time.time()
            stats = train_fold(k, args, pub, folds, priv, run_dir, model_dir, device, log)
            print(f"[fold {k}] done in {(time.time() - t0) / 60:.1f} min | median {stats['sec_per_iter_median']:.3f} s/iter "
                  f"| max VRAM {stats['max_mem_gb']:.2f} GB", flush=True)
    aggregate(args, pub, folds, priv, run_dir)
    print(f"total {(time.time() - t_run) / 60:.1f} min | torch.cuda.max_memory_allocated "
          f"{torch.cuda.max_memory_allocated() / 2**30:.2f} GB")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/video.py
"""Video decoding helpers.

Clips are short (median 1.9 s) and H.264 seeking in OpenCV is unreliable, so we
always decode sequentially and keep every frame (optionally downscaled) in
memory, then pick frames by timestamp. All sampling is relative to the END of
the clip, which is the last pre-contact moment.
"""
import cv2
import numpy as np

cv2.setNumThreads(1)


def decode_clip(path, short_side: int | None = None):
    """Decode every frame. Returns (frames: list[HxWx3 uint8 BGR], ts_ms: float64 array).

    short_side: downscale so min(H, W) == short_side (never upscales)."""
    cap = cv2.VideoCapture(str(path))
    frames, ts = [], []
    try:
        while True:
            ok, f = cap.read()
            if not ok:
                break
            t = cap.get(cv2.CAP_PROP_POS_MSEC)
            if short_side is not None:
                h, w = f.shape[:2]
                s = short_side / min(h, w)
                if s < 1:
                    f = cv2.resize(f, (round(w * s), round(h * s)), interpolation=cv2.INTER_AREA)
            frames.append(f)
            ts.append(t)
    finally:
        cap.release()
    if not frames:
        raise RuntimeError(f"no frames decoded from {path}")
    ts = np.asarray(ts, dtype=np.float64)
    # Some containers report 0 for every timestamp; fall back to the nominal fps.
    if len(ts) > 1 and np.all(np.diff(ts) <= 0):
        fps = cv2.VideoCapture(str(path)).get(cv2.CAP_PROP_FPS) or 25.0
        ts = np.arange(len(frames)) * 1000.0 / fps
    return frames, ts


def indices_at_offsets(ts_ms: np.ndarray, offsets_s) -> list[int]:
    """Index of the frame closest to (t_end - offset) for each offset in seconds.
    Offsets beyond the clip start clamp to the first frame."""
    end = ts_ms[-1]
    return [int(np.argmin(np.abs(ts_ms - (end - o * 1000.0)))) for o in offsets_s]


def indices_uniform_window(ts_ms: np.ndarray, window_s: float, n: int) -> list[int]:
    """n indices uniformly spaced in time over the last window_s seconds
    (the window is clamped to the clip start, so short clips are stretched)."""
    end = ts_ms[-1]
    start = max(ts_ms[0], end - window_s * 1000.0)
    targets = np.linspace(start, end, n)
    return [int(np.argmin(np.abs(ts_ms - t))) for t in targets]


def last_n_indices(n_frames: int, n: int) -> list[int]:
    """The last n frame indices, left-padded by repeating index 0 for short clips."""
    idx = list(range(max(0, n_frames - n), n_frames))
    return [0] * (n - len(idx)) + idx


def to_rgb(frame_bgr: np.ndarray) -> np.ndarray:
    return cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)


def letterbox(frame: np.ndarray, size: int, pad_value: int = 114) -> np.ndarray:
    """Resize keeping aspect ratio, pad to size x size (used for fixed-size encoders)."""
    h, w = frame.shape[:2]
    s = size / max(h, w)
    nh, nw = max(1, round(h * s)), max(1, round(w * s))
    r = cv2.resize(frame, (nw, nh), interpolation=cv2.INTER_AREA if s < 1 else cv2.INTER_LINEAR)
    out = np.full((size, size, 3), pad_value, dtype=np.uint8)
    y0, x0 = (size - nh) // 2, (size - nw) // 2
    out[y0:y0 + nh, x0:x0 + nw] = r
    return out


if __name__ == "__main__":
    import os
    import sys
    import time

    sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
    from config import PUBLIC_DIR

    for clip in ["0001_public.mp4", "0002_public.mp4", "0003_public.mp4"]:
        t0 = time.time()
        fr, ts = decode_clip(PUBLIC_DIR / clip, short_side=360)
        print(f"{clip}: {len(fr)} frames {fr[0].shape} first_ts={ts[0]:.1f} last_ts={ts[-1]:.1f} "
              f"dt={np.median(np.diff(ts)):.2f}ms decode={time.time() - t0:.2f}s")
        print("   offsets [0, .1, .25, .5, 1.0]s ->", indices_at_offsets(ts, [0, .1, .25, .5, 1.0]),
              "| uniform 8 over last 0.5s ->", indices_uniform_window(ts, 0.5, 8),
              "| last 8 ->", last_n_indices(len(fr), 8))


In [ ]:
%%writefile src/viz_contact.py
"""Contact sheets for eyeballing the data: last frame of sampled clips with
labels, the hand-clicked kicker (green) / keeper (magenta) positions, and
YOLO11-pose detections drawn on top.

Run:  python src/viz_contact.py --partition public --n 36 --out <dir>
"""
import argparse
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import cv2
import numpy as np
import pandas as pd

from config import CLICKS_CSV, GT_CSV, MODELS_DIR, PRIVATE_CSV, PRIVATE_DIR, PUBLIC_DIR, SEED
from video import decode_clip, letterbox

TILE_W, TILE_H, COLS, ROWS = 480, 270, 4, 3


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--partition", default="public")
    ap.add_argument("--n", type=int, default=36)
    ap.add_argument("--out", required=True)
    ap.add_argument("--imgsz", type=int, default=1280)
    ap.add_argument("--conf", type=float, default=0.15)
    ap.add_argument("--seed", type=int, default=SEED)
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)

    from ultralytics import YOLO
    pose = YOLO(str(MODELS_DIR / "yolo11x-pose.pt"))

    if a.partition == "public":
        df = pd.read_csv(GT_CSV)
        clicks = pd.read_csv(CLICKS_CSV).set_index("clip")
        root = PUBLIC_DIR
    else:
        df = pd.read_csv(PRIVATE_CSV)
        clicks = None
        root = PRIVATE_DIR
    # one clip per source, sources in random order -> maximal variety
    rng = np.random.RandomState(a.seed)
    srcs = np.array(df["source_id"].unique(), dtype=object)
    rng.shuffle(srcs)
    picks = [df[df.source_id == s].sample(1, random_state=a.seed).iloc[0] for s in srcs[: a.n]]

    tiles = []
    for row in picks:
        clip = row["clip"]  # row.clip would be the Series.clip method
        frames, ts = decode_clip(root / clip)
        last = frames[-1]
        r = pose.predict(last, imgsz=a.imgsz, conf=a.conf, device=0, verbose=False)[0]
        img = r.plot(line_width=2, font_size=6, labels=False)
        H, W = img.shape[:2]
        if clicks is not None and clip in clicks.index:
            c = clicks.loc[clip]
            cv2.circle(img, (int(c.kicker_x_norm * W), int(c.kicker_y_norm * H)), 14, (0, 255, 0), 3)
            cv2.circle(img, (int(c.keeper_x_norm * W), int(c.keeper_y_norm * H)), 14, (255, 0, 255), 3)
        tile = letterbox(img, max(TILE_W, TILE_H))
        tile = tile[(tile.shape[0] - TILE_H) // 2:(tile.shape[0] - TILE_H) // 2 + TILE_H, :TILE_W] if tile.shape[1] >= TILE_W else tile
        tile = cv2.resize(tile, (TILE_W, TILE_H))
        lab = f"{clip[:4]} {row.source_id} {row.preferred_foot[0]}{row.gender} {W}x{H} {len(frames)}f {ts[-1] / 1000:.1f}s"
        cv2.rectangle(tile, (0, 0), (TILE_W, 34), (0, 0, 0), -1)
        cv2.putText(tile, lab, (4, 13), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255, 255, 255), 1, cv2.LINE_AA)
        if "kick_direction" in row:
            lab2 = f"kick={row.kick_direction} gk={row.goalkeeper_direction} {row.goal_outcome}  persons={len(r.boxes)}"
        else:
            lab2 = f"persons={len(r.boxes)}"
        cv2.putText(tile, lab2, (4, 29), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (0, 255, 255), 1, cv2.LINE_AA)
        tiles.append(tile)
        print(lab, "|", lab2, flush=True)

    per = COLS * ROWS
    for s in range(0, len(tiles), per):
        chunk = tiles[s:s + per]
        while len(chunk) < per:
            chunk.append(np.zeros((TILE_H, TILE_W, 3), np.uint8))
        rows_img = [np.hstack(chunk[i * COLS:(i + 1) * COLS]) for i in range(ROWS)]
        sheet = np.vstack(rows_img)
        out = os.path.join(a.out, f"{a.partition}_sheet{s // per + 1}.jpg")
        cv2.imwrite(out, sheet, [cv2.IMWRITE_JPEG_QUALITY, 88])
        print("wrote", out)


if __name__ == "__main__":
    main()


In [ ]:
import os, sys, pathlib

os.environ.setdefault("CV3_ROOT", r"D:\cv3")       # <- change here (or set CV3_ROOT before launching Jupyter) if the data root differs

# Run from the project root (the folder that holds src/), whether the notebook was opened from notebook/ or from the root.
here = pathlib.Path.cwd().resolve()
PROJECT = here if (here / "src").exists() else here.parent
assert (PROJECT / "src" / "config.py").exists(), f"cannot find src/config.py next to or above {here}"
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT / "src"))

import pyarrow  # noqa: F401  (Windows: import pyarrow before cv2/torch, see notes)
import pandas as pd
import config

print("project root  :", PROJECT)
print("data root     :", config.ROOT)
print("kernel python :", sys.executable)

In [ ]:
# The `!python` cells must use the kernel's interpreter and see the same CV3_ROOT.
!python -c "import sys, os; print('shell python  :', sys.executable); print('CV3_ROOT      :', os.environ.get('CV3_ROOT'))"
!python -c "import torch, ultralytics, timm, lightgbm, xgboost, sklearn, onnxruntime, pyarrow, cv2; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| ultralytics', ultralytics.__version__, '| timm', timm.__version__, '| lightgbm', lightgbm.__version__, '| xgboost', xgboost.__version__, '| sklearn', sklearn.__version__, '| onnxruntime', onnxruntime.__version__, '| pyarrow', pyarrow.__version__, '| cv2', cv2.__version__)"

In [ ]:
# Verify the data layout.
for name, d, n in [("public clips  (data/train)", config.PUBLIC_DIR, 1300), ("private clips (data/val)", config.PRIVATE_DIR, 900)]:
    k = len(list(d.glob("*.mp4"))) if d.exists() else -1
    print(f"{name:28s} {str(d):40s} {k:5d} mp4  (expected {n})")
for p in [config.GT_CSV, config.PRIVATE_CSV, config.CLICKS_CSV, config.TEMPLATE_CSV]:
    print(f"{p.name:28s} {'ok' if p.exists() else 'MISSING':8s}", f"{len(pd.read_csv(p))} rows" if p.exists() else "")
for d in [config.CACHE_DIR, config.MODELS_DIR, config.RUNS_DIR]:
    print(f"{str(d):40s} exists={d.exists()}  files={len(list(d.iterdir())) if d.exists() else 0}")

In [ ]:
# Pretrained detector / pose weights (Ultralytics release assets). Downloaded once into D:\cv3\models if missing.
from ultralytics.utils.downloads import attempt_download_asset

for w in ["yolo11x.pt", "yolo11m-pose.pt"]:
    p = config.MODELS_DIR / w
    if p.exists():
        print(f"{w:16s} ok  ({p.stat().st_size / 1e6:.0f} MB)")
    else:
        try:
            attempt_download_asset(str(p))
            print(f"{w:16s} downloaded -> {p}")
        except Exception as e:  # noqa: BLE001
            print(f"{w:16s} download failed ({e}); download it manually from https://github.com/ultralytics/assets/releases into {config.MODELS_DIR}")
# RTMPose ONNX models are fetched automatically by rtmlib on first use (stage 5b).

## Folds, metric and chance baselines (`src/cv.py`, `src/baseline_meta.py`)

Every model in this project is evaluated with the **same** 5-fold split: `StratifiedGroupKFold` on `kick_direction`, grouped by `source_id` (clips from the same shootout or compilation never appear in both train and validation folds), seed 42, cached once in `cache/folds_5_42.csv`. `cv.py` also defines the competition metric (`0.60 * acc(kick) + 0.25 * acc(keeper) + 0.15 * acc(goal)`), the left/right label-swap helper used by the mirror augmentation, and the submission writer that validates a file against `submission_template.csv`.

`baseline_meta.py` reproduces the chance bars from metadata alone: majority class = 0.5172, preferred-foot majority = 0.5504 (the README's "educated guess" of 0.550). Nothing below uses `fps`, `resolution` or clip duration as a model input (the private partition contains frame rates absent from the public one).

In [ ]:
!python src/cv.py
!python src/baseline_meta.py

## Stage 1: clip inventory (`src/inventory.py`)

Decodes every clip once with OpenCV and records the true frame count, per-frame timestamps (to detect variable frame rate), resolution and duration. The project samples frames by *time before the last frame*, never by frame index, because frame rates range from 15 to 60 fps. No pretrained model. **~3-5 min** with 8 decoder threads. Writes `cache/clip_inventory.csv` (one row per clip, 2,200 rows). Finding: no variable-frame-rate clips; durations 0.3-9.9 s; 15 fps occurs only in the private partition (31 clips).

In [ ]:
!python src/inventory.py

## Stage 2: frame cache A (`src/framecache.py`)

A small, RAM-friendly frame cache used by the near-duplicate check, contact sheets and the (dropped) full-frame video models: 30 time-sampled frames per clip (the last ~0.96 s at 24 Hz plus six context frames back to 5 s), letterboxed to 352x208, JPEG q88, one `.npz` per clip in `cache/framesA/` (1.4 GB). No pretrained model. **~10 min**; existing files are skipped on re-runs.

In [ ]:
!python src/framecache.py

## Stage 3: person and ball detection (`src/detect.py`)

**Pretrained model: YOLO11x** (Ultralytics, COCO detection weights `yolo11x.pt`). For every clip we decode the frames nearest 16 fixed offsets before the last frame (every 0.1 s over the final second, then 1.25, 1.5, 2, 2.5 and 3 s) and run the detector at `imgsz=1280`, `conf=0.10`, fp16, keeping the `person` and `sports ball` classes. The high input resolution matters: the players are only 30-80 px tall in 1280x720 broadcast frames. The ball is rarely detected at this scale and is used only as a weak picker feature.

One row per detected box is appended to `cache/detections.parquet` (atomic writes every 25 clips), so the job **resumes** if interrupted and skips cached clips. **~1 h on the RTX 4060** for 2,200 clips x 16 frames. `--limit 20` runs a smoke test.

In [ ]:
!python src/detect.py

## Stage 4: kicker / keeper identification and the kicker frame (`src/roles.py`)

1. **Tracklets.** Person boxes are linked across the sampled frames by greedy backward matching from the last frame (distance gate scaled by box height and time gap).
2. **Click-supervised pickers.** At the reference frame 0.23 s before the end (the frame the organisers clicked on), every candidate box gets geometric and motion features (position, size, rank, motion over the last 0.5 s, neighbours, ball distance). Two LightGBM pickers (kicker first, then keeper with features relative to the chosen kicker) are supervised by `public_position_clicks.csv` and evaluated with source-grouped CV. The pickers are **cross-fit**: public clips get the prediction of the fold model that held out their source, private clips get the all-data model, so public and private features are equally "clean". Measured hit rates: kicker 0.942, keeper 0.985; goal-side agreement with the clicks 0.993.
3. **Geometry.** Per clip: `u` = unit vector kicker -> keeper (pixel-isotropic), kicker-right `r = (-u_y, u_x)`, kicker-left `l = -r`, and `goal_side` (L/R). Every later feature is a projection on `l` (lateral, sign flips under mirroring, tag `_N`) or on `u` (forward, invariant, tag `_I`), which makes "left" mean the kicker's left in every camera view and makes feature-level mirror augmentation exact.

No pretrained model (LightGBM pickers trained here). **~2 min.** Writes `cache/roles_clip.csv` (per-clip roles, geometry, `kicker_found`/`keeper_found` flags), `cache/roles_frames.parquet` (role boxes per sampled frame), `cache/tracks.parquet` (all tracklets, used for camera-pan compensation) and, with `--viz`, contact sheets in `cache/viz_roles/`.

In [ ]:
!python src/roles.py --viz

## Stage 5a: YOLO pose on up-scaled crops (`src/pose_crops.py`)

**Pretrained model: YOLO11m-pose** (Ultralytics, COCO-17 keypoints, `yolo11m-pose.pt`). Full-frame pose misses the small players, so for **every frame in the last 0.4 s** we cut a square crop around the (time-interpolated) kicker and keeper boxes (2.4x / 2.8x the box height), resize it to 448 px and keep the pose nearest the crop centre; keypoints are mapped back to original frame pixels. Keypoints were found in 97% (kicker) / 94% (keeper) of frames. GPU, **~0.5-1 h**; cached clips are skipped. Writes `cache/pose_crops.parquet`, one row per (clip, role, frame) with `kx0..kx16, ky0..ky16, kc0..kc16`.

In [ ]:
!python src/pose_crops.py

## Stage 5b: RTMPose Halpe-26 on the same crops, mirrored-crop TTA and merged keypoint tables (`src/pose_rtm.py`, `src/pose_merge.py`)

**Pretrained models: RTMPose-m and RTMPose-x, Halpe-26 "body7" ONNX checkpoints** (OpenMMLab, run through `rtmlib` on CPU with `onnxruntime`). Halpe-26 = COCO-17 + head, neck, pelvis + **big toe, small toe and heel of each foot**; the foot keypoints give the orientation of the plant foot and the kicking foot, the single most useful kick-direction cue. Top-down: the interpolated role box (padding 1.25) is warped to the model input (192x256 for m, 288x384 for x), i.e. the players are up-scaled 3-5x. Same frames and output schema as stage 5a with 26 keypoints.

**Test-time augmentation and merging.** Keypoints are noisy on 30-80 px players, so each model is also run on mirrored crops (`--tta flip4`: original, mirrored, and two shifted variants for RTMPose-m; `--tta flip` for RTMPose-x) and the passes are averaged with their confidences. `pose_merge.py` then averages whole tables, weighting each keypoint by its confidence. The merge must be **balanced**: `mm` = m + its TTA pass, `xx` = x + its TTA pass, `a2b` = mm + x, `a3` = mm + xx, and `a` = m + x. A flat three-way mean (m + x + mf4) gives the smaller model two thirds of the weight and loses 1.5 points on every head; the balanced `a2b` table is the main kicker table of the final model, and the heads on `a`, `x` and `m` are averaged with it.

CPU only, resumable. **`balanced` (RTMPose-m): 0.56 s/clip, ~20 min per pass. `performance` (RTMPose-x): 2.9 s/clip, 2-4 h per pass**; the TTA runs multiply this by the number of passes (they ran overnight). The ONNX models are downloaded by rtmlib on first use into the torch hub cache. Writes `cache/pose_rtm_{m,x,mf4,xf}.parquet` and the merged `cache/pose_rtm_{a,mm,xx,a2b,a3}.parquet`; `--viz` writes a contact sheet of last-frame skeletons.

In [ ]:
!python src/pose_rtm.py --mode balanced --viz                 # RTMPose-m            -> pose_rtm_m
!python src/pose_rtm.py --mode performance --viz              # RTMPose-x            -> pose_rtm_x
!python src/pose_rtm.py --mode balanced --tta flip4           # RTMPose-m, 4-way TTA -> pose_rtm_mf4
!python src/pose_rtm.py --mode performance --tta flip         # RTMPose-x, flip TTA  -> pose_rtm_xf

!python src/pose_merge.py --inputs m x --out a                # a   = m + x
!python src/pose_merge.py --inputs m mf4 --out mm             # mm  = m + its TTA pass
!python src/pose_merge.py --inputs x xf --out xx              # xx  = x + its TTA pass
!python src/pose_merge.py --inputs mm x --out a2b             # a2b = balanced (m, mf4) + x   <- main kicker table
!python src/pose_merge.py --inputs mm xx --out a3             # a3  = (m, mf4) + (x, xf)      <- raw-keypoint head only

## Stage 6 (only for the video models): canonical crop caches (`src/crops.py`)

For the crop-based CNN / VideoMAE experiments: 16 frames at 0.60, 0.56, ..., 0.00 s before the end, cropped with one square fixed in image coordinates (kicker view 3.0x box height, keeper view 4.0x), mirrored horizontally when the goal is on the right so that **the goal is always on the image-left** (labels and preferred foot are swapped accordingly through `config.FLIP_MAP`), 224x224 JPEG, one `.npz` per clip and view. No pretrained model. CPU, **~10 min**, existing files skipped. Writes `cache/crops_kicker/`, `cache/crops_keeper/` (~0.5 GB each) and index CSVs. These caches were zipped and uploaded as the Kaggle dataset `ay3140/cv3-crops` for stage 10. This stage can be skipped if the VideoMAE member is not reproduced.

In [ ]:
!python src/crops.py --view both

## Stage 7: feature tables and LightGBM heads (`src/features_*.py`)

Each script builds one feature table in `cache/features_v*.parquet` (rows = all 2,200 clips, columns tagged `_N` / `_I` for the mirror rule) and trains a LightGBM head with `features_v1.train`: 400 trees, learning rate 0.03, 15 leaves, `min_child_samples` 25, subsample 0.8, `colsample_bytree` 0.7, L2 = 5, three seeds, five source-grouped folds. Training rows are doubled by **feature-level mirror augmentation** (negate `_N` columns, swap left/right labels and the preferred foot); predictions use **flip-TTA** (average of the original and the un-swapped mirrored prediction). Each run writes `runs/<name>/{oof,test}_{kick,goalkeeper,goal}.npy`, `metrics.json` and a `submission.csv`.

| Script | Table | Features added | Head(s) | OOF kick / keeper |
|---|---|---|---|---|
| `features_v1.py` | v1 | boxes only: keeper lateral displacement, velocity and acceleration on `l` (camera pan compensated by the median motion of the other tracks), kicker approach speed/angle, box size changes, role flags, preferred foot, gender | `gbm_v1` (+ `gbm_v1_noswap`: the swap-disable sanity test, keeper accuracy must collapse) | 0.458 / 0.767 |
| `features_v2.py` | v2 = v1 + YOLO pose | joint positions and velocities in the kicker frame at 8 time points over the last 0.4 s, hip/shoulder lines, trunk lean, gaze, arm swing, plant-vs-kick foot (assigned kinematically: the slower ankle is the plant foot) | `gbm_v2` | 0.533 / 0.847 |
| `features_flow.py` | v4 = v2 + flow | Farneback optical flow inside the kicker and keeper boxes over the last frames, border-median pan subtraction, lateral/forward components in body heights per second, legs vs upper body | `gbm_v4` | 0.568 / 0.863 |
| `features_rtm.py --mode balanced` | v7 = v1 + RTM-m + flow; v8 = v2 + RTM-m + flow | the v2 body features recomputed on RTMPose keypoints plus **foot orientation** (toe - heel unit vectors of plant and kick foot, signed angles vs the forward axis and the run-up) | `gbm_v7`, `gbm_v8` | 0.587 / 0.877; 0.592 / 0.872 |
| `features_rtm.py --mode performance --suffix x` | v7x, v8x (RTMPose-x) | same, from the larger pose model | `gbm_v7x`, `gbm_v8x` | 0.591 / 0.882 (v8x) |
| `features_rtm.py --mode avg --suffix a` | v7a, v8a (merged m + x) | same, from the merged keypoints | `gbm_v7a`, `gbm_v8a` | 0.611 / 0.880 (v8a) |
| `features_rtm2.py --mode balanced` | v9 = v8 + round-2 kicker | smoothed tracks (3-frame median), limb orientations (plant/kick foot, shank, thigh) at 9 time points, angular velocities, kick-foot swing, knee angles, trunk/gaze/hip velocities, arm positions | `gbm_v9k` (kicker-only columns), `gbm_v9` | 0.608 / -; 0.605 / 0.874 |
| `features_rtm2.py --mode performance --suffix x` | v9x | same from RTMPose-x | `gbm_v9kx`, `gbm_v9x` | 0.618 / -; 0.616 / 0.883 |
| `features_rtm2.py --mode avg --suffix a` | v9a | same from the merged m + x keypoints | `gbm_v9ka`, `gbm_v9a` | 0.632 / -; 0.625 / 0.877 |
| `features_rtm.py` / `features_rtm2.py --mode a2b` | v7a2b, v8a2b, v9a2b | same from the balanced TTA merge `a2b` (the main kicker table of stage 8) | `gbm_v9ka2b` | 0.636 / - |
| `features_union.py` | v10 = v9 + RTMPose-x copies (prefix `x`) | both pose estimators in one table | `gbm_v10k` | 0.630 / - |
| `features_rtm2g.py` | v11 = v10 + round-2 keeper (m and x) | keeper hip/shoulder/feet displacements over 9 time points, trunk and head lean, foot directions, knee angles, centre of mass over the feet, onset of the lateral hip motion | `gbm_v11g` (keeper-only columns) | - / 0.887 |

`features_v3.py` (ball-anchored kicker geometry from the YOLO ball detections) gave no gain (kick 0.530) and is not part of the final stack; it is left commented out. CPU only; each script takes **a few minutes** (the flow extraction ~15 min); the feature tables are rebuilt each time (cheap), the pose/detection caches are not.

In [ ]:
!python src/features_v1.py
!python src/features_v2.py
# !python src/features_v3.py            # ball-anchored features: no gain, not used
!python src/features_flow.py
!python src/features_rtm.py --mode balanced                      # v7, v8   (RTMPose-m)
!python src/features_rtm.py --mode performance --suffix x        # v7x, v8x (RTMPose-x)
!python src/features_rtm.py --mode avg --suffix a                # v7a, v8a (merged m + x)   -> gbm_v8a
!python src/features_rtm.py --mode a2b --suffix a2b --no-train   # v7a2b, v8a2b (balanced TTA merge)
!python src/features_rtm2.py --mode balanced                     # v9       -> gbm_v9k, gbm_v9
!python src/features_rtm2.py --mode performance --suffix x       # v9x      -> gbm_v9kx, gbm_v9x
!python src/features_rtm2.py --mode avg --suffix a               # v9a      -> gbm_v9ka, gbm_v9a
!python src/features_rtm2.py --mode a2b --suffix a2b --kick-only # v9a2b    -> gbm_v9ka2b (reference head, kick 0.636)
!python src/features_union.py
!python src/features_rtm2g.py                                    # v11 (keeper round 2) -> gbm_v11g

## Stage 8: kicker-only / keeper-only heads, and the pure late-window kicker heads (`src/gbm_subset.py`, `src/model_subset.py`, `src/late_table.py`, `src/avg_members.py`)

**a) Column subsets and other learners.** A kick-direction head trained on **all** features mostly learns to "follow the keeper" (sources where the keeper dived the wrong way have kick accuracy ~0.1 with keeper accuracy ~0.9). Restricting the kick head to kicker columns and the keeper head to keeper columns gives better and more diverse stack members. `gbm_subset.py` applies the column filter to the LightGBM recipe; `model_subset.py` trains other learners with the same protocol (mirror augmentation, flip-TTA, 3 seeds): extra-trees (800 trees, `min_samples_leaf` 3, `max_features` 0.3) and XGBoost (400 trees, depth 4, learning rate 0.03). Members of the final stack produced here: `xgb_v9ka` (kick 0.622), `et_v9k` (0.609), `et_v9kx` (0.636), `xgb_v9kx` (0.618), and the keeper heads `gbm_v8g` (0.876), `et_v9g` (0.884), `et_v9gx` (0.889), `et_v11g` (0.894, the best keeper head).

**b) The pure late-window kicker heads (the lever that mattered most).** Two things were still wrong with the "kicker-only" subset. First, 18 columns describing the **goalkeeper's box motion** (`gk_*`: lateral displacement, velocity, commit time) slipped through the kicker filter, so every kick head still saw the keeper, who guesses the side right only 56% of the time. Second, features from **earlier than 0.04 s before contact** are noise for the kick: the kicker's hips, plant foot and swinging leg only commit in the last two frames. `late_table.py` keeps the kicker columns of a `features_v9*` table whose time token is 0.0 or 0.04 s and drops the keeper-box columns (454 -> 222 columns on `a2b`). The two restrictions only work **together** (keeper columns dropped alone: 0.632; late window alone: 0.651; both: **0.664**, up from 0.636, in all five folds). The same table is given to LightGBM, XGBoost and ExtraTrees for the `a2b`, `a`, `x` and `m` keypoint tables and the ten heads are averaged into one member, `gbm_kick_ens_pure10` (kick 0.650; the `a2b` LightGBM head alone is the strongest single head). Family ablations on the pure table (dropping the RTMPose, YOLO-pose, round-2 or flow families) all lose 1-3 points, and a hyper-parameter sweep found the default LightGBM settings best.

No pretrained model. CPU, **1-5 min per head** (ExtraTrees runs single-process, `--jobs 1`, because the joblib worker pool crashed under load on this machine).

In [ ]:
# a) column-subset heads with other learners (members of the final stack)
!python src/model_subset.py --features features_v9a.parquet --subset kicker --model xgb --run xgb_v9ka
!python src/model_subset.py --features features_v9.parquet  --subset kicker --model et  --run et_v9k
!python src/model_subset.py --features features_v9x.parquet --subset kicker --model et  --run et_v9kx
!python src/model_subset.py --features features_v9x.parquet --subset kicker --model xgb --run xgb_v9kx
!python src/gbm_subset.py   --features features_v8.parquet  --subset keeper --run gbm_v8g
!python src/model_subset.py --features features_v9.parquet  --subset keeper --model et  --run et_v9g
!python src/model_subset.py --features features_v9x.parquet --subset keeper --model et  --run et_v9gx
!python src/model_subset.py --features features_v11.parquet --subset keeper --model et  --run et_v11g

# b) pure late-window kicker tables (last 0.04 s, no keeper-box columns) for the a2b, a, x and m keypoint tables
!python src/late_table.py --sfx a2b
!python src/late_table.py --sfx a
!python src/late_table.py --sfx x
!python src/late_table.py --sfx m
# LightGBM, XGBoost and ExtraTrees on every column of each pure table
!python src/gbm_subset.py   --features features_v9a2b_late04pure.parquet --subset all --run gbm_v9ka2b_late04pure
!python src/gbm_subset.py   --features features_v9a_late04pure.parquet   --subset all --run gbm_v9ka_late04pure
!python src/gbm_subset.py   --features features_v9x_late04pure.parquet   --subset all --run gbm_v9kx_late04pure
!python src/gbm_subset.py   --features features_v9_late04pure.parquet    --subset all --run gbm_v9k_late04pure
!python src/model_subset.py --features features_v9a2b_late04pure.parquet --subset all --model xgb --run xgb_v9ka2b_late04pure
!python src/model_subset.py --features features_v9a_late04pure.parquet   --subset all --model xgb --run xgb_v9ka_late04pure
!python src/model_subset.py --features features_v9x_late04pure.parquet   --subset all --model xgb --run xgb_v9kx_late04pure
!python src/model_subset.py --features features_v9a2b_late04pure.parquet --subset all --model et  --run et_v9ka2b_late04pure --jobs 1
!python src/model_subset.py --features features_v9a_late04pure.parquet   --subset all --model et  --run et_v9ka_late04pure   --jobs 1
!python src/model_subset.py --features features_v9x_late04pure.parquet   --subset all --model et  --run et_v9kx_late04pure   --jobs 1
# one averaged member for the stack (kick 0.650)
!python src/avg_members.py --run gbm_kick_ens_pure10 --members gbm_v9ka2b_late04pure gbm_v9ka_late04pure gbm_v9kx_late04pure gbm_v9k_late04pure xgb_v9ka2b_late04pure xgb_v9ka_late04pure xgb_v9kx_late04pure et_v9ka2b_late04pure et_v9ka_late04pure et_v9kx_late04pure

## Stage 9: raw keypoint tensor members (`src/gbm_kpraw.py`, `src/train_kpnet.py`)

Both scripts read the same per-clip tensor `[T=11, C]`: for each role and each of the 26 Halpe keypoints the lateral / forward coordinate in the kicker frame (body-scale normalised) and the confidence, at 0.40, 0.36, ..., 0.00 s before the end, plus hip velocity, box-height change, presence flags and four metadata channels. The environment variable `KPNET_POSE` selects the pose table (`pose_rtm_m`, `pose_rtm_x`, or a merged table such as `pose_rtm_a2b`); the tensor is cached in `cache/kpnet_tensor_<pose>.npz`. Mirroring = negate lateral channels and the preferred foot, permute the left/right joint blocks, swap the labels.

* `gbm_kpraw.py`: LightGBM on the flattened tensor (`colsample_bytree` 0.3) so the trees can pick any joint/time combination, kicker channels only. The late-window finding of stage 8 carries over: with `--last 2` (only the 0.04 s and 0.00 s steps) the head improves on every table (a2b 0.607 -> 0.625, a3 0.608 -> 0.623, m 0.596 -> 0.610); the three late heads are averaged into `gbm_kpraw_late_ens` (kick 0.634). **~5 min each.**
* `train_kpnet.py`: a small temporal 1-D CNN trained on CPU, 60 epochs, 3 seeds, mirror augmentation p=0.5, coordinate jitter, time shift, channel dropout, mirror TTA, plus a mirror-consistency test. `kpnet_rtm` (CNN on RTMPose-m: kick 0.528, keeper 0.874) and `kpnet_rtmx` (RTMPose-x: 0.535 / 0.875) are weak alone but add diversity to the stack. **~5-10 min each** (45-100 s per fold).

No pretrained model (the pose estimators of stage 5b are the only pretrained components upstream).

In [ ]:
%env KPNET_POSE=pose_rtm_m
!python src/gbm_kpraw.py --subset kicker --run gbm_kpraw_k_late04 --last 2
!python src/train_kpnet.py --arch cnn --run kpnet_rtm

%env KPNET_POSE=pose_rtm_x
!python src/train_kpnet.py --arch cnn --run kpnet_rtmx

%env KPNET_POSE=pose_rtm_a2b
!python src/gbm_kpraw.py --subset kicker --run gbm_kpraw_ka2b_late04 --last 2

%env KPNET_POSE=pose_rtm_a3
!python src/gbm_kpraw.py --subset kicker --run gbm_kpraw_ka3_late04 --last 2

!python src/avg_members.py --run gbm_kpraw_late_ens --members gbm_kpraw_ka2b_late04 gbm_kpraw_ka3_late04 gbm_kpraw_k_late04

## Stage 9b: two small extra kick members (`src/camera.py`, `src/features_cam.py`, `src/pose3d.py`, `src/features_3d.py`)

Both are weak on their own and worth only a few clips in the stack; they are kept because the final stack was fitted with them.

* **Camera-corrected run-up path.** `camera.py` estimates the broadcast camera's pan/zoom per frame from background feature tracks (LK flow outside the player boxes, RANSAC 4-DoF similarity per step, chained to the last frame); `features_cam.py` re-expresses the kicker's whole run-up (last 1 s) in that stabilised frame: path angle, curvature, deceleration, straightness, start offset, corrected hip/foot displacements. CPU, **~1 h**. Head `gbm_camka`: kick 0.517.
* **3D pose.** `pose3d.py` runs **RTMW3D-x** (OpenMMLab, 133 keypoints with a root-relative depth coordinate, via `rtmlib`) on the kicker crops; `features_3d.py` builds the same foot / leg / hip / shoulder / trunk descriptors with the depth component included. The depth estimate does not resolve the body's yaw (both hips land at the same depth), so the head is weak: `gbm_3d_only` kick 0.569, +2-3 clips in the stack. CPU only, **~9 h** for 2,200 clips; optional (the stack skips the member if the run folder is missing).

In [ ]:
# Camera-corrected run-up path (kick 0.517 alone; a diverse member)
!python src/camera.py --workers 8
!python src/features_cam.py --mode avg           # -> gbm_camka (also trains gbm_v13ka / gbm_v13ga, not used)

# 3D kicker pose, RTMW3D-x on CPU (~8 s per clip; the two shards can run in parallel). Kick 0.569 alone.
!python src/pose3d.py --threads 8 --shard 0/2
!python src/pose3d.py --threads 8 --shard 1/2
!python src/features_3d.py --base a2b             # -> gbm_3d_only (also trains gbm_v9ka2b_3d, not used)

## Stage 10 (optional, GPU): VideoMAE on the player crops (`src/train_videomae.py`, `src/assemble_folds.py`)

**Pretrained models: VideoMAE-base fine-tuned on Something-Something-v2** (`MCG-NJU/videomae-base-finetuned-ssv2`; SSv2 pre-training is direction-aware, e.g. "moving something from left to right") and, for one of the keeper runs, **VideoMAE-large fine-tuned on Kinetics-400** (`MCG-NJU/videomae-large-finetuned-kinetics`). The 16-frame canonical crops of stage 6 are fed to the encoder (mean-pooled tokens + `fc_norm`) with preferred-foot / gender / goal-side embeddings and three linear heads; AdamW with layer-wise lr decay 0.75 (backbone 5e-5, heads 5e-4), cosine schedule, 12 epochs, batch 4 x accumulation 4, EMA 0.998, label smoothing 0.1, same 5 source-grouped folds.

These runs need a GPU for hours (2.15 s/iteration on a T4, ~2.5 h per fold) and were trained on Kaggle and Colab with the crop caches uploaded as a dataset; the command is the same everywhere:

```
python src/train_videomae.py --run vmae_keeper      --view keeper_crop --epochs 12 --folds 0,1,2,3,4 --workers 2 --eval-workers 0 --skip-done
python src/train_videomae.py --run vmae_keeper_s1   --view keeper_crop --seed 1   ...   (seeds 1-4: vmae_keeper_s1 ... s4)
python src/train_videomae.py --run vmae_keeper_Lk400 --view keeper_crop --init MCG-NJU/videomae-large-finetuned-kinetics ...
python src/train_videomae.py --run vmae_kicker_noflip --view kicker_crop --no-flip ...   (+ --seed 1/2/3 -> nf_s1..s3, --epochs 20 -> nf_ep20)
```

* **Keeper member `vmae_keeper_ens6`** = average of six keeper runs (SSv2 base with five seeds + the large K400 init): keeper 0.909 (single runs 0.894-0.916 per fold). The video model reads the dive directly and adds diversity to the keypoint keeper heads.
* **Kicker member `vmae_kicker_nf_ens5`** = average of five kicker runs trained *without* the horizontal-flip label-swap augmentation (with it the kicker runs predict the class prior): kick 0.542 (single runs 0.512-0.526; seed averaging helps the weak image signal). Worth about +0.002 in the stack.

Per-fold outputs (`fold{k}_preds.npz`) are downloaded into `<root>/kaggle_out` or `<root>/colab_out/runs` and turned into ordinary stack members by `assemble_folds.py` (missing folds become NaN rows that `stack.py` fills with the class prior); `avg_members.py` averages the runs. The cell below assembles whatever run folders are present; if none are, the stack runs without these members (about -0.004 OOF).

Negative results, documented but not re-run: the same recipe on the kicker crops **with** flip augmentation (validation kick accuracy 0.54 / 0.47 / 0.48 / 0.46 = the class prior) and a **ConvNeXt-tiny stack-7** crop model (`src/train_stack7.py`, 7 frames stacked as 21 channels) that never left the prior either (train loss stuck at ~0.92, OOF kick 0.502). With about 1,000 training clips per fold the crop CNNs/transformers learn little beyond the label balance; the keypoint features are the signal.

In [ ]:
import subprocess

ENSEMBLES = {
    "vmae_keeper_ens6": ["vmae_keeper", "vmae_keeper_s1", "vmae_keeper_s2", "vmae_keeper_s3", "vmae_keeper_s4", "vmae_keeper_Lk400"],
    "vmae_kicker_nf_ens5": ["vmae_kicker_noflip", "vmae_kicker_nf_s1", "vmae_kicker_nf_s2", "vmae_kicker_nf_s3", "vmae_kicker_nf_ep20"],
}
roots = [config.ROOT / "kaggle_out", config.ROOT / "colab_out"]      # downloaded GPU outputs: .../<run>/fold{k}_preds.npz

for ens, members in ENSEMBLES.items():
    present = []
    for m in members:
        if (config.RUNS_DIR / m / "oof_kick.npy").exists():            # already assembled
            present.append(m)
            continue
        src = next((d for r in roots if r.exists() for d in r.rglob(m) if d.is_dir() and list(d.glob("fold*_preds.npz"))), None)
        if src is not None:
            subprocess.run([sys.executable, "src/assemble_folds.py", "--src", str(src), "--run", m], check=True)
            present.append(m)
    if present:
        subprocess.run([sys.executable, "src/avg_members.py", "--run", ens, "--members", *present], check=True)
        print(f"{ens}: averaged {len(present)} run(s): {present}")
    else:
        print(f"{ens}: no fold predictions found under {[str(r) for r in roots]} -> stack.py will skip this member")

## Stage 11: level-2 stack and the submission (`src/stack.py`)

For each target, a multinomial logistic regression on the concatenated **log-probabilities of the members** (OOF probabilities for the labelled clips, fold-averaged test probabilities for the unlabelled ones) plus preferred foot and gender, evaluated on the same source-grouped folds; `C` is chosen from {0.03, 0.1, 0.3, 1.0}. An additive bias on the rare class (`middle` / `stands in the middle`) is applied only if it improves OOF accuracy by at least 0.005. `goal_outcome` falls back to the constant `goal` unless the stacked head beats the base rate (0.722) by 0.005; it never did. Members whose run directory is missing are skipped with a message.

The final stack, `stack_tab88`:

* **kick_direction** (20 members): the pure late-window ensemble `gbm_kick_ens_pure10`, the late raw-keypoint ensemble `gbm_kpraw_late_ens`, the kicker heads `xgb_v9ka gbm_v9a gbm_v8a et_v9kx xgb_v9kx gbm_v9x et_v9k gbm_camka gbm_3d_only vmae_kicker_nf_ens5`, the keeper heads `vmae_keeper_ens6 et_v11g et_v9gx et_v9g gbm_v8g gbm_v8x` and the temporal nets `kpnet_rtmx kpnet_rtm`;
* **goalkeeper_direction** (`--gk-members`): keeper heads only, `vmae_keeper_ens6 et_v11g et_v9gx et_v9g gbm_v8g gbm_v8x kpnet_rtmx kpnet_rtm` (the kicker heads only added noise to the keeper regression: 0.911 -> 0.918).

**OOF: 0.7388 (kick 0.668 | keeper 0.918 | goal 0.722).** Runs in **seconds**; writes `runs/stack_tab88/{oof,test}_*.npy`, `metrics.json` and `submission.csv`. For comparison, the same member set with the old kick ensemble (every time step, keeper columns included) scores 0.7275; the single pure late-window head in its place 0.7340.

In [ ]:
!python src/stack.py --name stack_tab88 --members gbm_kick_ens_pure10 xgb_v9ka gbm_v9a gbm_v8a gbm_3d_only et_v9kx xgb_v9kx gbm_v9x et_v9k gbm_kpraw_late_ens gbm_camka vmae_kicker_nf_ens5 vmae_keeper_ens6 et_v11g et_v9gx et_v9g gbm_v8g gbm_v8x kpnet_rtmx kpnet_rtm --gk-members vmae_keeper_ens6 et_v11g et_v9gx et_v9g gbm_v8g gbm_v8x kpnet_rtmx kpnet_rtm

In [ ]:
# Copy the stack's submission into submissions/ and validate it against the template.
import shutil

src = config.RUNS_DIR / "stack_tab88" / "submission.csv"
dst = PROJECT / "submissions" / "submission_final.csv"
dst.parent.mkdir(exist_ok=True)
shutil.copy2(src, dst)

sub, tmpl = pd.read_csv(dst), pd.read_csv(config.TEMPLATE_CSV)
assert list(sub.columns) == ["clip"] + config.TARGETS, sub.columns.tolist()
assert len(sub) == 900 and set(sub["clip"]) == set(tmpl["clip"]) and not sub["clip"].duplicated().any()
for t in config.TARGETS:
    assert sub[t].isin(config.CLASSES[t]).all(), t
print("wrote", dst, "| rows:", len(sub))
for t in config.TARGETS:
    print(f"  {t:22s} {sub[t].value_counts().to_dict()}")
display(sub.head())

ref = PROJECT / "submissions" / "submission_v25.csv"       # the file that was uploaded, if present
if ref.exists():
    r = pd.read_csv(ref).set_index("clip").loc[sub["clip"]]
    for t in config.TARGETS:
        print(f"agreement with submission_v25.csv on {t}: {(r[t].values == sub[t].values).mean():.3f}")

## Results

Out-of-fold (OOF) scores are on the 1,300 labelled clips with the source-grouped folds. The public leaderboard is a fixed 40% of the 900 unlabelled clips (360 clips, standard error about 0.023); the final ranking uses the other 60%. Model selection was done on the OOF score only.

| Submission | Model | OOF | OOF kick | OOF keeper | Public LB |
|---|---|---|---|---|---|
| v1 | foot-prior kick + `gbm_v1` keeper + constant goal | 0.6010 | 0.502 | 0.767 | 0.5775 |
| v2 | `gbm_v2` (boxes + YOLO pose) | 0.6399 | 0.533 | 0.847 | 0.6631 |
| v4 | stack of `gbm_v4` (flow), `gbm_v2`, keypoint nets | 0.6704 | 0.575 | 0.869 | 0.6911 |
| v5 | + RTMPose foot-orientation heads | 0.6930 | 0.608 | 0.880 | 0.6958 |
| v6 | + kicker-only / keeper-only heads | 0.7043 | 0.627 | 0.879 | 0.7172 |
| v9 | + RTMPose-x heads, raw-keypoint GBMs, VideoMAE keeper | 0.7160 | 0.636 | 0.904 | 0.7486 |
| v13 | + merged keypoint tables, VideoMAE ensembles, camera head (bag of 4 stacks) | 0.7288 | 0.653 | 0.915 | 0.7581 |
| v14 | single stack of the same members | 0.7275 | 0.652 | 0.913 | 0.7600 |
| v20 | pure late-window kicker head (one LightGBM) in place of the old kick ensemble | 0.7340 | 0.662 | 0.913 | 0.7650 |
| v22 | pure late-window ensemble (gbm / xgb / et on four keypoint tables) | 0.7347 | 0.664 | 0.912 | 0.7536 |
| v23 | + late-window raw-keypoint ensemble | 0.7362 | 0.667 | 0.911 | 0.7583 |
| v25 | + keeper-only member set for the keeper target (`stack_tab88`, this notebook) | **0.7388** | 0.668 | 0.918 | (pending when this was written) |

Chance bars: majority class 0.5172, preferred-foot "educated guess" 0.5504. The keeper is easy to read once the dive has started (0.92); the kick (weight 0.60) is the hard part. Between v9 and v19 some sixty stacks with different members (pose TTA, 3D pose, ball geometry, contact alignment, hierarchical heads) all landed at 0.7227-0.7296 OOF: the kick head was stuck at ~0.65 until the keeper-column leak and the late-window finding of stage 8 lifted it to 0.668. Stacks that differ by less than ~0.005 are within fold noise (the standard error of the kick accuracy on 1,300 clips is about 0.013), and the public-leaderboard scores of v13-v25 (0.754-0.765, except v21) are all within one standard error of each other.

## Pretrained models used

Used in the final submission:

1. **YOLO11x** (Ultralytics 8.4.190, COCO object detection, `yolo11x.pt`): person and sports-ball detection at 1280 px (stage 3).
2. **YOLO11m-pose** (Ultralytics, COCO-17 keypoints, `yolo11m-pose.pt`): pose on up-scaled player crops (stage 5a) -> `features_v2`-based tables.
3. **RTMPose-m, Halpe-26, body7** (OpenMMLab RTMPose, ONNX `rtmpose-m_simcc-body7_pt-body7-halpe26_700e-256x192-4d3e73dd_20230605`, run with `rtmlib` 0.0.16 + `onnxruntime` 1.26 on CPU): 26-keypoint pose incl. toes and heels (stage 5b).
4. **RTMPose-x, Halpe-26, body7** (ONNX `rtmpose-x_simcc-body7_pt-body7-halpe26_700e-384x288-7fb6e239_20230606`): same, larger model (stage 5b).
5. **RTMW3D-x** (OpenMMLab, 133 keypoints with depth, ONNX via `rtmlib`): the 3D kicker pose member (stage 9b).
6. **VideoMAE-base fine-tuned on SSv2** (`MCG-NJU/videomae-base-finetuned-ssv2`) and **VideoMAE-large fine-tuned on Kinetics-400** (`MCG-NJU/videomae-large-finetuned-kinetics`), Hugging Face `transformers`: fine-tuned on the canonical player crops (stage 10).

Tried and dropped (documented as negative results, not part of the submission): **ConvNeXt-tiny** (`timm` `convnext_tiny.fb_in22k_ft_in1k`, 21-channel stack-7 crop model); frozen **DINOv2-base** (`facebook/dinov2-base`) and frozen VideoMAE-SSv2 feature probes on full frames (~0.51-0.52, i.e. chance). No external datasets were used; the only supervision is the labelled clips and their position clicks.

## Notes on reproducibility

* **Determinism.** LightGBM / extra-trees / XGBoost / logistic regression runs are seeded (3 seeds averaged) and reproduce exactly on the same machine; YOLO detection in fp16 and the CPU ONNX pose models are deterministic up to floating-point noise, so a from-scratch rerun can differ from the uploaded file on a handful of borderline clips (the last cell prints the agreement with the uploaded file when it is present). The GPU-trained VideoMAE members depend on GPU non-determinism; their per-fold prediction files are kept under `<root>/kaggle_out` and `<root>/colab_out`.
* **Caches.** `cache/detections.parquet`, `pose_crops.parquet`, `pose_rtm_{m,x,mf4,xf}.parquet` and their merges, `pose3d_k_*.parquet`, `camera/`, `crops_*` and `framesA` are the expensive artefacts (about one GPU-hour plus roughly a day of CPU with the pose TTA and 3D passes); with them present the full notebook re-runs in about two hours. Delete a cache file to force its stage to recompute.
* **Hard rules kept.** No `fps`, `resolution` or duration as model inputs; no pooling across unlabelled clips and no per-source normalisation at inference (a per-source feature-centering test lost 4 points); the role picker is cross-fit so labelled-clip features are not cleaner than unlabelled-clip features; a clean per-fold self-training test (`src/selftrain.py`) hurt the kick head and was dropped, so no pseudo-labels are used.
* **Things that did not help** (all measured with the same grouped CV): crop CNNs and VideoMAE on the kicker with flip augmentation, 3D pose beyond a few clips, ball detection and ball geometry (`src/ball_detect.py`, `src/features_ball.py`), aligning clips on the kicking foot's speed peak instead of the clip end, per-source centering, a probability threshold for the rare `middle` class, a hierarchical middle / left-right head (`src/hier_head.py`), a linear kicker head, forcing the metadata's preferred foot as the kicking foot (`FOOT_PREF=1` in `features_rtm.py`), hyper-parameter sweeps and feature-family ablations of the pure late-window table.